<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## بازبینی مجدد پس از Final Health — بررسی متن اصلی Case 4 از 22

در بازبینی مجدد Human Review مشخص شد سؤال پذیرفته‌شده قبلی درباره «تمرین 8» مستقل و روشن نیست و Target Evidence فعلی نیز فقط شامل عنوان تمرین و یک دستور کوتاه است.

پیش از هر تصمیم REVISE یا RETARGET، باید متن واقعی همان کتاب در اطراف پاراگراف‌های 329 و 330 بررسی شود تا مشخص شود:

- تمرین 8 ادامه چه مبحثی بوده است؛
- هدف آموزشی آن از متن قبل یا بعد قابل استنباط هست یا نه؛
- آیا می‌توان از همان منبع سؤال طبیعی و مفهومی ساخت؛
- یا باید Target Evidence به بخش مناسب‌تری از همان موضوع تغییر کند.

این مرحله فقط Diagnostic است و هیچ Human Decision، Registry، Source، Citation یا Queue را تغییر نمی‌دهد.

</div>

In [1]:
# Code Cell 105

from pathlib import Path
import json
import re


CELL105_PILOT_ID = (
    "question-pilot:"
    "909b695797935246c92c7a66ebe0f773"
)

CELL105_WORK_ID = (
    "book:trainer-training-financial-literacy"
)

CELL105_EDITION_ID = (
    "book:trainer-training-financial-literacy:fa-source"
)

CELL105_TARGET_PARAGRAPHS = {
    329,
    330,
}

CELL105_CONTEXT_START = 315
CELL105_CONTEXT_END = 345


cell105_failures = []


def cell105_check(name, condition):
    condition = bool(condition)

    if not condition:
        cell105_failures.append(
            name
        )

    return condition


def cell105_find_repo_root():
    candidates = []

    if "cell21_registry_path" in globals():
        registry_path = Path(
            cell21_registry_path
        ).resolve()

        candidates.extend(
            [
                registry_path.parent,
                *registry_path.parents,
            ]
        )

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        if (
            (
                candidate
                /
                "data"
            ).exists()
            and
            (
                candidate
                /
                "notebooks"
            ).exists()
        ):
            return candidate

    return None


def cell105_extract_paragraph_number(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    direct = record.get(
        "paragraph_number"
    )

    if isinstance(
        direct,
        int,
    ):
        return direct

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "paragraph_number"
        )

        if isinstance(
            value,
            int,
        ):
            return value

    paragraph_id = (
        record.get(
            "paragraph_id"
        )
        or
        record.get(
            "document_id"
        )
    )

    if isinstance(
        paragraph_id,
        str,
    ):
        match = re.search(
            r":p0*(\d+)$",
            paragraph_id,
        )

        if match:
            return int(
                match.group(
                    1
                )
            )

    return None


def cell105_record_work_id(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = record.get(
        "work_id"
    )

    if isinstance(
        value,
        str,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "work_id"
        )

        if isinstance(
            value,
            str,
        ):
            return value

    return None


def cell105_record_edition_id(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = (
        record.get(
            "edition_id"
        )
        or
        record.get(
            "source_id"
        )
    )

    if isinstance(
        value,
        str,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "edition_id"
        )

        if isinstance(
            value,
            str,
        ):
            return value

    return None


def cell105_record_text(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    for key in [
        "text",
        "normalized_text",
        "content",
        "paragraph_text",
    ]:
        value = record.get(
            key
        )

        if (
            isinstance(
                value,
                str,
            )
            and
            value.strip()
        ):
            return value.strip()

    return None


def cell105_heading_path(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = record.get(
        "heading_path"
    )

    if isinstance(
        value,
        list,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "heading_path"
        )

        if isinstance(
            value,
            list,
        ):
            return value

    return None


def cell105_walk(obj):
    if isinstance(
        obj,
        dict,
    ):
        yield obj

        for value in obj.values():
            yield from cell105_walk(
                value
            )

    elif isinstance(
        obj,
        list,
    ):
        for item in obj:
            yield from cell105_walk(
                item
            )


cell105_repo_root = (
    cell105_find_repo_root()
)

cell105_check(
    "REPO_ROOT_NOT_FOUND",
    cell105_repo_root
    is not None,
)


cell105_registry_path = (
    (
        Path(
            cell21_registry_path
        ).resolve()
    )
    if "cell21_registry_path" in globals()
    else
    (
        cell105_repo_root
        /
        "data"
        /
        "phase2"
        /
        "human_review"
        /
        "human_review_registry.json"
    )
)


cell105_book_files = [
    cell105_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "book_ingested_v1.json",

    cell105_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "book_ingested_v2.json",
]


cell105_existing_book_files = [
    path
    for path in cell105_book_files
    if path.exists()
]


cell105_check(
    "BOOK_INGESTED_FILES_NOT_FOUND",
    bool(
        cell105_existing_book_files
    ),
)


cell105_registry = json.loads(
    cell105_registry_path.read_text(
        encoding="utf-8"
    )
)


cell105_case_matches = [
    case
    for case
    in cell105_registry.get(
        "completed_human_review_cases",
        [],
    )
    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL105_PILOT_ID
    )
]


cell105_check(
    "PILOT_CASE_NOT_UNIQUE",
    len(
        cell105_case_matches
    )
    ==
    1,
)


cell105_case = (
    cell105_case_matches[
        0
    ]
    if len(
        cell105_case_matches
    )
    ==
    1
    else
    {}
)


cell105_current_question = None

for draft in (
    cell105_case.get(
        "model_drafts",
        []
    )
    or
    []
):
    if (
        isinstance(
            draft,
            dict,
        )
        and
        draft.get(
            "human_decision"
        )
        ==
        "ACCEPT"
    ):
        cell105_current_question = (
            draft.get(
                "question_text"
            )
        )


cell105_current_answer = (
    (
        cell105_case.get(
            "expected_answer_review"
        )
        or
        {}
    ).get(
        "expected_answer_text"
    )
)


cell105_current_evidence = (
    cell105_case.get(
        "source_evidence",
        []
    )
    or
    []
)


cell105_book_records = []


for book_file in cell105_existing_book_files:
    payload = json.loads(
        book_file.read_text(
            encoding="utf-8"
        )
    )

    for record in cell105_walk(
        payload
    ):
        paragraph_number = (
            cell105_extract_paragraph_number(
                record
            )
        )

        text = cell105_record_text(
            record
        )

        if (
            paragraph_number
            is None
            or
            text
            is None
        ):
            continue

        work_id = cell105_record_work_id(
            record
        )

        edition_id = (
            cell105_record_edition_id(
                record
            )
        )

        paragraph_id = (
            record.get(
                "paragraph_id"
            )
            or
            record.get(
                "document_id"
            )
        )

        identity_text = " ".join(
            str(value)
            for value in [
                work_id,
                edition_id,
                paragraph_id,
            ]
            if value
        )

        exact_source_match = any([
            work_id
            ==
            CELL105_WORK_ID,

            edition_id
            ==
            CELL105_EDITION_ID,

            CELL105_EDITION_ID
            in
            identity_text,
        ])

        if not exact_source_match:
            continue

        if not (
            CELL105_CONTEXT_START
            <=
            paragraph_number
            <=
            CELL105_CONTEXT_END
        ):
            continue

        cell105_book_records.append({
            "paragraph_number":
                paragraph_number,

            "text":
                text,

            "heading_path":
                cell105_heading_path(
                    record
                ),

            "paragraph_id":
                paragraph_id,

            "source_file":
                str(
                    book_file
                ),
        })


cell105_unique_records = {}

for record in cell105_book_records:
    key = (
        record[
            "paragraph_number"
        ],
        record[
            "text"
        ],
    )

    if key not in cell105_unique_records:
        cell105_unique_records[
            key
        ] = record


cell105_context_records = sorted(
    cell105_unique_records.values(),
    key=lambda item: (
        item[
            "paragraph_number"
        ],
        item[
            "text"
        ],
    ),
)


cell105_found_numbers = {
    record[
        "paragraph_number"
    ]
    for record in cell105_context_records
}


cell105_check(
    "TARGET_PARAGRAPHS_NOT_FOUND_IN_BOOK",
    CELL105_TARGET_PARAGRAPHS.issubset(
        cell105_found_numbers
    ),
)


cell105_check(
    "NO_CONTEXT_BEFORE_TARGET",
    any(
        number
        <
        min(
            CELL105_TARGET_PARAGRAPHS
        )
        for number
        in cell105_found_numbers
    ),
)


cell105_check(
    "NO_CONTEXT_AFTER_TARGET",
    any(
        number
        >
        max(
            CELL105_TARGET_PARAGRAPHS
        )
        for number
        in cell105_found_numbers
    ),
)


print(
    "=== CASE 4 POST-FINAL SOURCE CONTEXT DIAGNOSTIC ==="
)


print(
    "\n=== CURRENT ACCEPTED CASE ==="
)

print(
    "Pilot:",
    CELL105_PILOT_ID,
)

print(
    "Current Question:",
    cell105_current_question,
)

print(
    "Current Answer:",
    cell105_current_answer,
)

print(
    "Current Target Evidence:",
    cell105_case.get(
        "target_evidence_id"
    ),
)


print(
    "\n=== CURRENT PROTECTED SOURCE EVIDENCE ==="
)

for item in cell105_current_evidence:
    print(
        "\nDocument:",
        item.get(
            "document_id"
        ),
    )

    print(
        "Text:",
        item.get(
            "text"
        ),
    )


print(
    "\n=== EXACT BOOK CONTEXT ==="
)

print(
    "Work ID:",
    CELL105_WORK_ID,
)

print(
    "Edition ID:",
    CELL105_EDITION_ID,
)

print(
    "Requested paragraph window:",
    (
        CELL105_CONTEXT_START,
        CELL105_CONTEXT_END,
    ),
)

print(
    "Found unique records:",
    len(
        cell105_context_records
    ),
)


for record in cell105_context_records:
    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Paragraph:",
        record[
            "paragraph_number"
        ],
    )

    print(
        "Paragraph ID:",
        record[
            "paragraph_id"
        ],
    )

    print(
        "Heading Path:",
        record[
            "heading_path"
        ],
    )

    print(
        "Text:"
    )

    print(
        record[
            "text"
        ]
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    False,
)

print(
    "Human Decision added:",
    False,
)

print(
    "Question changed:",
    False,
)

print(
    "Answer changed:",
    False,
)

print(
    "Target changed:",
    False,
)

print(
    "Citation changed:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)

print(
    "Commit / Push:",
    False,
)


cell105_gate = (
    len(
        cell105_failures
    )
    ==
    0
)


print(
    "\n=== FAILURES ==="
)

print(
    cell105_failures
)


print(
    "\nCODE CELL 105 GATE:",
    (
        "PASS"
        if cell105_gate
        else
        "FAIL"
    ),
)


assert cell105_gate, (
    "Case 4 source-context diagnostic failed. "
    "Repair this SAME Code Cell 105 before any re-review decision."
)

=== CASE 4 POST-FINAL SOURCE CONTEXT DIAGNOSTIC ===

=== CURRENT ACCEPTED CASE ===
Pilot: question-pilot:909b695797935246c92c7a66ebe0f773
Current Question: مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟
Current Answer: در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی ۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، ۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه به دیگران مطرح شده است.
Current Target Evidence: target-evidence:63c2f0f041fa8caa0f21a1aed165d7ec

=== CURRENT PROTECTED SOURCE EVIDENCE ===

Document: book:trainer-training-financial-literacy:fa-source:p000323
Text: کودکان بین پن

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## بازبینی دوم — مورد 4 از 22 — اصلاح کامل Target، سؤال و پاسخ

### 1. وضعیت ثبت‌شده در بازبینی اول

در Human Review اولیه، این Case با Target زیر پذیرفته شده بود:

**Target Evidence قبلی:**

`target-evidence:8d87341f7847558d90ee76e350c64fa3`

**متن منبع قبلی:**

**پاراگراف 329**

تمرین 8

**پاراگراف 330**

هر مقطع سنی دو نفر را انتخاب کنید و درباره میزان هزینه ماهانه آنها مصاحبه کنید. به آن‌ها اطمینان دهید که این اطلاعات محرمانه خواهد بود. نتیجه را تحلیل کنید.

**سؤال پذیرفته‌شده قبلی:**

در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟

**پاسخ پذیرفته‌شده قبلی:**

باید از هر مقطع سنی دو نفر انتخاب شوند، درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند و در پایان نتیجه مصاحبه‌ها تحلیل شود.

این سؤال و پاسخ در بازبینی اول ACCEPT شده بودند، اما در بازبینی دوم مشخص شد که:

- سؤال بدون مشاهده کتاب مستقل و طبیعی نیست؛
- عبارت «تمرین ۸» برای کاربر معنای مستقلی ندارد؛
- Target قبلی فقط عنوان و دستور یک تمرین را شامل می‌شود؛
- بخش اصلی و آموزشی موضوع بودجه‌بندی درست پیش از این تمرین قرار دارد؛
- بنابراین نگه‌داشتن این Case به شکل قبلی برای Golden Dataset مناسب نیست.

تصمیم بازبینی دوم:

**RETARGET**

نسخه قبلی پاک نمی‌شود و به‌عنوان تاریخچه بازبینی اول حفظ خواهد شد.

---

### 2. منبع بازبینی‌شده و جایگزین

در بررسی متن کامل کتاب، پاراگراف‌های 323 تا 327 به‌عنوان Target جایگزین انتخاب شدند.

#### پاراگراف 323

کودکان بین پنج تا هفت سال : اولین پولی که کودک دریافت می‌کند، تنها از آن نگهداری می‌کند و در واقع فقط پس‌انداز می‌کند و کم‌کم با موقعیت‌هایی که به او پول می‌دهیم (مانند فروشگاه یا مسافرت) خرج کردن را یاد می‌گیرد.

#### پاراگراف 324

کودکان بین هشت تا دوازده سال: دومین ردیف بودجه پول‌توجیبی هم یاد داده می‌شود. خرج پول با نظارت والدین است. هر چند به کودکان اجازهٔ اشتباه می‌دهند، اما همچنان کنارشان هستند و خرج پول با نظارت آن‌هاست. حتی اگر با خرید فرزند مخالف‌اند، اجازه می‌دهند بخرد. مهم این است که جلوی اشتباه گرفته نشود، بلکه بعد از اشتباه، خیلی واقعی درباره آن و چرایی اشتباه بودن آن گفتگو شود. در این سن، پول‌توجیبی فقط در دو ردیف خرج و پس‌انداز صرف می‌شود و می‌توان به هر کدام 50 درصد از دریافتی اختصاص یابد.

#### پاراگراف 325

اگر با این روش پیش رفته باشیم، دیگر «پول خودم است» مطرح نیست؛ چون پدر و مادر همواره با او هستند. هیچ وقت احساس نمی‌کند پول خودش است. اگر کودکان بگویند پول خودشان است،‌ می‌گوییم باید پاسخگوی تصمیم‌شان باشند. ممکن است تصمیم‌شان پیامدهایی داشته یا نداشته باشد. اگر پیامدها، فرصت آموزش را فراهم می‌کرد، تصمیم اشتباه و پیامدش را یادآوری می‌کنیم. یادمان باشد هنگام یادگیری راه رفتن، فرزند را برای زمین خوردن مواخذه نمی‌کنیم.

#### پاراگراف 326

نوجوان دوازده تا پانزده سال:در این دوره ردیف سرمایه‌گذاری هم اضافه می‌شود و پول‌توجیبی به سه بخش تقسیم می‌شود: خرج، پس‌انداز و سرمایه‌گذاری. در این حالت به ترتیب الگوی 50 درصد، 30 درصد و 20 درصد می‌تواند به کارگرفته شود.(تفاوت پس‌انداز و سرمایه‌گذاری: مشخص و نامشخص بودن هدف، بازه زمانی کوتاه مدت و بلندمدت). می‌توان در نشست خانوادگی دربارهٔ سرفصل‌های بودجه‌بندی و خرج با آن‌ها به توافق رسید. در این دوره می‌توان به شکل موقعیتی به او، هدیه به دیگران، بخشش یا حتی قرض به دیگران را پیشنهاد کرد.

#### پاراگراف 327

نوجوان پانزده تا هجده سال: در این دوره ردیف دیگری به پول‌توجیبی اضافه می‌شود: هدیه یا قرض به دیگران. تا پیش از این، به شکل موقعیتی با این ردیف آشنا بوده است. اما از اینجا به بعد با تعیین بخش و درصدی به این موضوع به آن ساختار می‌دهیم؛ الگوی 40 درصد خرج، 30 درصد پس‌انداز، 20 درصد سرمایه‌گذاری و 10 درصد هدیه به دیگران در سن مناسب است. نقش نظارتی والدین همچنان هست و آخر ماه از فرزندشان می‌خواهند توضیح دهد که این درصدها را چطور اعمال کرده است. این درصدها به نوعی خط‌کش است تا جلوی رفتارهای هیجانی نوجوان‌ها را بگیرد تا در هیچ کدام افراط و تفریط نداشته باشند؛ حتی در بخشش!

---

### 3. سؤال جایگزین

**مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟**

Human Reviewer این سؤال را در بازبینی دوم تأیید کرد.

---

### 4. پاسخ جایگزین

**در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی ۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، ۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه به دیگران مطرح شده است.**

این پاسخ فقط از پاراگراف‌های 323 تا 327 همان منبع ساخته شده و دانش بیرونی به آن افزوده نشده است.

---

### 5. تصمیم نهایی بازبینی دوم

- تصمیم درباره Target قبلی: **RETARGET**
- Target قبلی: **حفظ در History**
- سؤال قبلی: **حفظ در History**
- پاسخ قبلی: **حفظ در History**
- Source خام: **بدون تغییر**
- Parser Output: **بدون تغییر**
- Target جدید: **پاراگراف‌های 323 تا 327**
- سؤال جدید: **ACCEPT**
- پاسخ جدید: **ACCEPT**
- Citation جدید: **مبتنی بر پاراگراف‌های 323 تا 327**
- Author Final Review: **NOT_STARTED**
- Case پس از این اصلاح همچنان آماده بازبینی دکتر کمیل رودی است.

</div>

In [2]:
# Code Cell 106
# ============================================================
# SECOND HUMAN REVIEW — CASE 4 / 22
#
# COMPLETE RETARGET + QUESTION + ANSWER + CITATION AMENDMENT
#
# Prerequisite:
#   Cell 105 diagnostic must have PASSed.
#
# This cell:
# - preserves the previously accepted target/question/answer/citation;
# - records the second-review RETARGET;
# - activates book paragraphs 323-327 as replacement evidence;
# - records the newly approved Question;
# - records the replacement Expected Answer;
# - rebuilds citation mapping from the replacement evidence;
# - updates BOTH Registry copies of the case;
# - keeps Author Final Review = NOT_STARTED;
# - writes a backup before changing Registry;
# - reloads and verifies the persisted Registry;
# - is idempotent for this exact amendment.
#
# It DOES NOT:
# - modify raw book source;
# - modify parser output;
# - delete first-review history;
# - record any Author decision;
# - commit or push anything.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Fixed identity
# ============================================================

CELL106_PILOT_ID = (
    "question-pilot:"
    "909b695797935246c92c7a66ebe0f773"
)

CELL106_OLD_TARGET_ID = (
    "target-evidence:"
    "8d87341f7847558d90ee76e350c64fa3"
)

CELL106_REPLACEMENT_PARAGRAPHS = [
    323,
    324,
    325,
    326,
    327,
]

CELL106_NEW_QUESTION = (
    "مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند "
    "و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟"
)

CELL106_NEW_ANSWER = (
    "در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و "
    "پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی "
    "یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز "
    "تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. "
    "در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی "
    "۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری "
    "پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران "
    "نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، "
    "۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه "
    "به دیگران مطرح شده است."
)

CELL106_HUMAN_REASON = (
    "در بازبینی دوم مشخص شد Target قبلی فقط عنوان و دستور "
    "تمرین 8 را شامل می‌شد و سؤال پذیرفته‌شده قبلی بدون مشاهده "
    "کتاب، مستقل و طبیعی نبود. پس از بررسی متن کامل بخش "
    "بودجه‌بندی، پاراگراف‌های 323 تا 327 محتوای آموزشی کامل‌تر "
    "و مناسب‌تری برای همین Pilot فراهم کردند. Human Reviewer "
    "سؤال و پاسخ جایگزین را تأیید کرد."
)


# ============================================================
# 2) Utilities
# ============================================================

cell106_failures = []


def cell106_check(name, condition, details=None):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell106_failures.append(
            failure
        )

    return bool(condition)


def cell106_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell106_stable_id(prefix, payload):

    if not isinstance(
        payload,
        str,
    ):
        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell106_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


# ============================================================
# 3) Cell 105 prerequisite
# ============================================================

cell106_check(
    "CELL105_GATE_NOT_PASS",
    globals().get(
        "cell105_gate"
    )
    is True,
)

cell106_check(
    "CELL105_REGISTRY_PATH_MISSING",
    "cell105_registry_path"
    in globals(),
)

cell106_check(
    "CELL105_CONTEXT_RECORDS_MISSING",
    "cell105_context_records"
    in globals(),
)

cell106_check(
    "CELL105_WORK_ID_MISSING",
    "CELL105_WORK_ID"
    in globals(),
)

cell106_check(
    "CELL105_EDITION_ID_MISSING",
    "CELL105_EDITION_ID"
    in globals(),
)


if cell106_failures:

    print(
        "Prerequisite failures:",
        cell106_failures,
    )

    raise AssertionError(
        "Cell 106 blocked because Cell 105 prerequisites are not valid."
    )


cell106_registry_path = Path(
    cell105_registry_path
).resolve()


cell106_check(
    "REGISTRY_FILE_NOT_FOUND",
    cell106_registry_path.exists(),
    str(
        cell106_registry_path
    ),
)


if cell106_failures:

    raise AssertionError(
        cell106_failures
    )


# ============================================================
# 4) Resolve EXACT replacement paragraphs from Cell 105 output
# ============================================================

cell106_replacement_records = []


for paragraph_number in CELL106_REPLACEMENT_PARAGRAPHS:

    matches = [
        record

        for record
        in cell105_context_records

        if (
            isinstance(
                record,
                dict,
            )
            and
            record.get(
                "paragraph_number"
            )
            ==
            paragraph_number
        )
    ]

    cell106_check(
        (
            "REPLACEMENT_PARAGRAPH_"
            +
            str(
                paragraph_number
            )
            +
            "_NOT_UNIQUE"
        ),
        len(
            matches
        )
        ==
        1,
        {
            "match_count":
                len(
                    matches
                ),
        },
    )

    if len(
        matches
    ) == 1:

        cell106_replacement_records.append(
            deepcopy(
                matches[0]
            )
        )


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Replacement source resolution failed."
    )


cell106_check(
    "REPLACEMENT_ORDER_MISMATCH",
    [
        record[
            "paragraph_number"
        ]

        for record
        in cell106_replacement_records
    ]
    ==
    CELL106_REPLACEMENT_PARAGRAPHS,
)


cell106_replacement_text = "\n\n".join(
    record[
        "text"
    ]

    for record
    in cell106_replacement_records
)


# ============================================================
# 5) Source-grounding sanity checks
# ============================================================

CELL106_REQUIRED_SOURCE_CUES = [
    "پنج تا هفت",
    "هشت تا دوازده",
    "خرج و پس‌انداز",
    "50 درصد",
    "دوازده تا پانزده",
    "سرمایه‌گذاری",
    "30 درصد",
    "20 درصد",
    "پانزده تا هجده",
    "40 درصد",
    "10 درصد",
    "هدیه",
]


for cue in CELL106_REQUIRED_SOURCE_CUES:

    cell106_check(
        (
            "REQUIRED_SOURCE_CUE_MISSING:"
            +
            cue
        ),
        cue
        in
        cell106_replacement_text,
    )


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Replacement evidence does not support the approved Question/Answer."
    )


# ============================================================
# 6) Build normalized replacement source evidence
# ============================================================

cell106_source_evidence = []


for part_index, record in enumerate(
    cell106_replacement_records
):

    paragraph_number = (
        record[
            "paragraph_number"
        ]
    )

    document_id = (
        record[
            "paragraph_id"
        ]
    )

    text = (
        record[
            "text"
        ]
    )

    heading_path = (
        record.get(
            "heading_path"
        )
        or
        [
            "بودجه‌بندی"
        ]
    )

    cell106_source_evidence.append({
        "part_index":
            part_index,

        "role":
            "PROTECTED_REGION_MEMBER",

        "document_id":
            document_id,

        "text":
            text,

        "text_sha256":
            cell106_sha256_text(
                text
            ),

        "char_start":
            0,

        "char_end":
            len(
                text
            ),

        "locator": {
            "work_id":
                CELL105_WORK_ID,

            "edition_id":
                CELL105_EDITION_ID,

            "paragraph_id":
                document_id,

            "paragraph_number":
                paragraph_number,

            "heading_path":
                deepcopy(
                    heading_path
                ),
        },
    })


# ============================================================
# 7) Deterministic replacement IDs
# ============================================================

cell106_target_identity_payload = {
    "pilot_case_id":
        CELL106_PILOT_ID,

    "replacement_paragraphs":
        CELL106_REPLACEMENT_PARAGRAPHS,

    "replacement_documents": [
        {
            "document_id":
                item[
                    "document_id"
                ],

            "text_sha256":
                item[
                    "text_sha256"
                ],
        }

        for item
        in cell106_source_evidence
    ],
}


cell106_new_target_id = (
    cell106_stable_id(
        "target-evidence",
        cell106_target_identity_payload,
    )
)


cell106_new_question_id = (
    cell106_stable_id(
        "question-draft",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "target_evidence_id":
                cell106_new_target_id,

            "question":
                CELL106_NEW_QUESTION,
        },
    )
)


cell106_new_answer_id = (
    cell106_stable_id(
        "expected-answer-draft",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "target_evidence_id":
                cell106_new_target_id,

            "answer":
                CELL106_NEW_ANSWER,
        },
    )
)


cell106_new_citation_id = (
    cell106_stable_id(
        "citation-mapping",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "documents": [
                item[
                    "document_id"
                ]

                for item
                in cell106_source_evidence
            ],
        },
    )
)


cell106_amendment_id = (
    cell106_stable_id(
        "second-human-review",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "old_target_evidence_id":
                CELL106_OLD_TARGET_ID,

            "new_target_evidence_id":
                cell106_new_target_id,

            "new_question_draft_id":
                cell106_new_question_id,

            "new_expected_answer_draft_id":
                cell106_new_answer_id,
        },
    )
)


# ============================================================
# 8) Load Registry fresh from disk
# ============================================================

cell106_registry_before_text = (
    cell106_registry_path.read_text(
        encoding="utf-8"
    )
)


cell106_registry = json.loads(
    cell106_registry_before_text
)


cell106_registry_before_sha256 = (
    hashlib.sha256(
        cell106_registry_before_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 9) Locate Case in BOTH durable Registry copies
# ============================================================

cell106_completed_cases = (
    cell106_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell106_author_review_root = (
    cell106_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell106_author_cases = (
    cell106_author_review_root.get(
        "cases_ready_for_review",
        [],
    )
)


def cell106_find_case(
    case_list,
    location_name,
):

    matches = [
        case

        for case
        in case_list

        if (
            isinstance(
                case,
                dict,
            )
            and
            case.get(
                "pilot_case_id"
            )
            ==
            CELL106_PILOT_ID
        )
    ]

    cell106_check(
        (
            location_name
            +
            "_CASE_NOT_UNIQUE"
        ),
        len(
            matches
        )
        ==
        1,
        {
            "match_count":
                len(
                    matches
                ),
        },
    )

    if len(
        matches
    ) == 1:

        return matches[0]

    return None


cell106_completed_case = (
    cell106_find_case(
        cell106_completed_cases,
        "COMPLETED",
    )
)


cell106_author_case = (
    cell106_find_case(
        cell106_author_cases,
        "AUTHOR_READY",
    )
)


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Case 4 could not be resolved uniquely in Registry."
    )


# ============================================================
# 10) Helper: resolve currently active Question
# ============================================================

def cell106_active_question(case):

    final_question_id = (
        case.get(
            "final_question_draft_id"
        )
    )

    for draft in (
        case.get(
            "model_drafts",
            []
        )
        or
        []
    ):

        if (
            isinstance(
                draft,
                dict,
            )
            and
            draft.get(
                "question_draft_id"
            )
            ==
            final_question_id
        ):

            return draft.get(
                "question_text"
            )

    question_review = (
        case.get(
            "question_review"
        )
        or
        {}
    )

    return question_review.get(
        "question_text"
    )


# ============================================================
# 11) Validate current / historical state before mutation
# ============================================================

CELL106_OLD_QUESTION = (
    "در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد "
    "چه مراحلی باید انجام شود؟"
)


for location_name, case in [
    (
        "COMPLETED",
        cell106_completed_case,
    ),
    (
        "AUTHOR_READY",
        cell106_author_case,
    ),
]:

    existing_second_review = (
        case.get(
            "second_human_review"
        )
        or
        {}
    )

    existing_amendment_id = (
        existing_second_review.get(
            "amendment_id"
        )
    )

    exact_amendment_already_active = (
        existing_amendment_id
        ==
        cell106_amendment_id
        and
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id
        and
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION
        and
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER
    )

    if exact_amendment_already_active:

        continue

    cell106_check(
        (
            location_name
            +
            "_UNEXPECTED_EXISTING_SECOND_REVIEW"
        ),
        existing_amendment_id
        in
        [
            None,
            cell106_amendment_id,
        ],
        existing_amendment_id,
    )

    cell106_check(
        (
            location_name
            +
            "_OLD_TARGET_MISMATCH"
        ),
        case.get(
            "target_evidence_id"
        )
        ==
        CELL106_OLD_TARGET_ID,
        case.get(
            "target_evidence_id"
        ),
    )

    cell106_check(
        (
            location_name
            +
            "_OLD_QUESTION_MISMATCH"
        ),
        cell106_active_question(
            case
        )
        ==
        CELL106_OLD_QUESTION,
        cell106_active_question(
            case
        ),
    )

    author_final_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )

    cell106_check(
        (
            location_name
            +
            "_AUTHOR_REVIEW_ALREADY_STARTED"
        ),
        author_final_review.get(
            "status"
        )
        ==
        "NOT_STARTED",
        author_final_review,
    )


if cell106_failures:

    print(
        "\n=== PRE-WRITE FAILURES ==="
    )

    print(
        cell106_failures
    )

    raise AssertionError(
        "Registry current state does not match the expected Case 4 preconditions."
    )


# ============================================================
# 12) Reuse existing attribution — DO NOT invent attribution
# ============================================================

cell106_existing_citation_entries = (
    (
        cell106_completed_case.get(
            "citation_review"
        )
        or
        {}
    ).get(
        "citation_entries"
    )
    or
    []
)


cell106_attribution_template = None


for entry in cell106_existing_citation_entries:

    if (
        isinstance(
            entry,
            dict,
        )
        and
        isinstance(
            entry.get(
                "attribution"
            ),
            dict,
        )
    ):

        cell106_attribution_template = (
            deepcopy(
                entry[
                    "attribution"
                ]
            )
        )

        break


cell106_check(
    "EXISTING_ATTRIBUTION_TEMPLATE_NOT_FOUND",
    isinstance(
        cell106_attribution_template,
        dict,
    ),
)


if cell106_failures:

    raise AssertionError(
        cell106_failures
    )


# ============================================================
# 13) Build new Citation entries from exact replacement evidence
# ============================================================

cell106_new_citation_entries = []


for citation_index, item in enumerate(
    cell106_source_evidence
):

    cell106_new_citation_entries.append({
        "citation_index":
            citation_index,

        "document_id":
            item[
                "document_id"
            ],

        "source_id":
            CELL105_EDITION_ID,

        "target_evidence_id":
            cell106_new_target_id,

        "role":
            item[
                "role"
            ],

        "char_start":
            item[
                "char_start"
            ],

        "char_end":
            item[
                "char_end"
            ],

        "char_count":
            (
                item[
                    "char_end"
                ]
                -
                item[
                    "char_start"
                ]
            ),

        "locator":
            deepcopy(
                item[
                    "locator"
                ]
            ),

        "locator_origin":
            "COPIED_FROM_SECOND_HUMAN_REVIEW_REPLACEMENT_EVIDENCE",

        "attribution":
            deepcopy(
                cell106_attribution_template
            ),
    })


# ============================================================
# 14) Apply exact amendment to one Registry Case
# ============================================================

cell106_review_timestamp = (
    cell106_now()
)


def cell106_apply_amendment(
    case,
    registry_location,
):

    existing_second_review = (
        case.get(
            "second_human_review"
        )
        or
        {}
    )

    # --------------------------------------------------------
    # Idempotent no-op if this exact amendment is already active
    # --------------------------------------------------------

    if (
        existing_second_review.get(
            "amendment_id"
        )
        ==
        cell106_amendment_id
        and
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id
        and
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION
        and
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER
    ):

        return (
            "ALREADY_APPLIED"
        )


    # --------------------------------------------------------
    # Preserve exact first-review accepted state
    # --------------------------------------------------------

    previous_final_question_id = (
        case.get(
            "final_question_draft_id"
        )
    )

    previous_question_text = (
        cell106_active_question(
            case
        )
    )

    previous_expected_answer_review = (
        deepcopy(
            case.get(
                "expected_answer_review"
            )
        )
    )

    previous_citation_review = (
        deepcopy(
            case.get(
                "citation_review"
            )
        )
    )

    previous_source_evidence = (
        deepcopy(
            case.get(
                "source_evidence"
            )
        )
    )

    previous_target_id = (
        case.get(
            "target_evidence_id"
        )
    )


    history = case.setdefault(
        "post_final_human_review_history",
        [],
    )


    existing_history_ids = {
        item.get(
            "amendment_id"
        )

        for item
        in history

        if isinstance(
            item,
            dict,
        )
    }


    if (
        cell106_amendment_id
        not in
        existing_history_ids
    ):

        history.append({
            "amendment_id":
                cell106_amendment_id,

            "review_record_version":
                "second_human_review_amendment_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW",

            "review_timestamp_utc":
                cell106_review_timestamp,

            "registry_location":
                registry_location,

            "human_decision":
                "RETARGET_AND_ACCEPT_REPLACEMENT",

            "human_review_reason":
                CELL106_HUMAN_REASON,

            "previous_state_preserved":
                True,

            "superseded_accepted_qa": {
                "target_evidence_id":
                    previous_target_id,

                "source_evidence":
                    previous_source_evidence,

                "question_draft_id":
                    previous_final_question_id,

                "question_text":
                    previous_question_text,

                "expected_answer_review":
                    previous_expected_answer_review,

                "citation_review":
                    previous_citation_review,
            },

            "replacement_state": {
                "target_evidence_id":
                    cell106_new_target_id,

                "replacement_paragraphs":
                    deepcopy(
                        CELL106_REPLACEMENT_PARAGRAPHS
                    ),

                "question_draft_id":
                    cell106_new_question_id,

                "question_text":
                    CELL106_NEW_QUESTION,

                "expected_answer_draft_id":
                    cell106_new_answer_id,

                "expected_answer_text":
                    CELL106_NEW_ANSWER,

                "citation_mapping_id":
                    cell106_new_citation_id,
            },

            "raw_source_mutated":
                False,

            "parser_output_mutated":
                False,

            "author_review_recorded":
                False,
        })


    # --------------------------------------------------------
    # Mark old active draft as historically accepted but superseded
    # WITHOUT changing the old human_decision.
    # --------------------------------------------------------

    model_drafts = (
        case.get(
            "model_drafts",
            []
        )
        or
        []
    )


    for draft in model_drafts:

        if (
            isinstance(
                draft,
                dict,
            )
            and
            draft.get(
                "question_draft_id"
            )
            ==
            previous_final_question_id
        ):

            draft[
                "second_review_status"
            ] = (
                "SUPERSEDED_BY_RETARGET"
            )

            draft[
                "superseded_by_question_draft_id"
            ] = (
                cell106_new_question_id
            )


    existing_draft_ids = {
        draft.get(
            "question_draft_id"
        )

        for draft
        in model_drafts

        if isinstance(
            draft,
            dict,
        )
    }


    if (
        cell106_new_question_id
        not in
        existing_draft_ids
    ):

        previous_revisions = [
            int(
                draft.get(
                    "revision",
                    0,
                )
                or
                0
            )

            for draft
            in model_drafts

            if isinstance(
                draft,
                dict,
            )
        ]

        new_revision = (
            max(
                previous_revisions
                or
                [0]
            )
            +
            1
        )

        model_drafts.append({
            "question_draft_id":
                cell106_new_question_id,

            "question_text":
                CELL106_NEW_QUESTION,

            "revision":
                new_revision,

            "machine_validation":
                "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

            "human_decision":
                "ACCEPT",

            "review_origin":
                "SECOND_HUMAN_REVIEW_RETARGET",

            "target_evidence_id":
                cell106_new_target_id,
        })


    case[
        "model_drafts"
    ] = (
        model_drafts
    )


    # --------------------------------------------------------
    # Append second-review Human Review records
    # --------------------------------------------------------

    human_reviews = (
        case.get(
            "human_reviews",
            []
        )
        or
        []
    )


    cell106_retarget_review_event_id = (
        cell106_stable_id(
            "human-review",
            {
                "amendment_id":
                    cell106_amendment_id,

                "event":
                    "RETARGET",
            },
        )
    )


    cell106_question_accept_event_id = (
        cell106_stable_id(
            "human-review",
            {
                "amendment_id":
                    cell106_amendment_id,

                "event":
                    "QUESTION_ACCEPT",
            },
        )
    )


    existing_review_event_ids = {
        review.get(
            "review_event_id"
        )

        for review
        in human_reviews

        if isinstance(
            review,
            dict,
        )
    }


    if (
        cell106_retarget_review_event_id
        not in
        existing_review_event_ids
    ):

        human_reviews.append({
            "review_event_id":
                cell106_retarget_review_event_id,

            "review_record_version":
                "second_human_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_RETARGET",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "human_decision":
                "RETARGET",

            "previous_target_evidence_id":
                previous_target_id,

            "previous_question_draft_id":
                previous_final_question_id,

            "replacement_target_evidence_id":
                cell106_new_target_id,

            "replacement_question_draft_id":
                cell106_new_question_id,

            "previous_review_preserved":
                True,

            "human_review_reason":
                CELL106_HUMAN_REASON,

            "raw_source_mutated":
                False,

            "parser_output_mutated":
                False,
        })


    if (
        cell106_question_accept_event_id
        not in
        existing_review_event_ids
    ):

        human_reviews.append({
            "review_event_id":
                cell106_question_accept_event_id,

            "review_record_version":
                "second_human_question_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_QUESTION",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "target_evidence_id":
                cell106_new_target_id,

            "question_draft_id":
                cell106_new_question_id,

            "question_text":
                CELL106_NEW_QUESTION,

            "human_decision":
                "ACCEPT",

            "human_note":
                (
                    "Human Reviewer explicitly approved the "
                    "replacement Question after reviewing the "
                    "full source context."
                ),

            "external_knowledge_used":
                False,
        })


    case[
        "human_reviews"
    ] = (
        human_reviews
    )


    # --------------------------------------------------------
    # Activate replacement Target + Question
    # --------------------------------------------------------

    case[
        "target_evidence_id"
    ] = (
        cell106_new_target_id
    )

    case[
        "source_evidence"
    ] = deepcopy(
        cell106_source_evidence
    )

    case[
        "final_question_draft_id"
    ] = (
        cell106_new_question_id
    )


    # --------------------------------------------------------
    # Activate replacement Expected Answer
    # --------------------------------------------------------

    case[
        "expected_answer_review"
    ] = {
        "expected_answer_draft_id":
            cell106_new_answer_id,

        "expected_answer_text":
            CELL106_NEW_ANSWER,

        "human_decision":
            "ACCEPT",

        "machine_validation_status":
            "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

        "source_evidence":
            deepcopy(
                cell106_source_evidence
            ),

        "human_review": {
            "review_record_version":
                "second_human_expected_answer_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_EXPECTED_ANSWER",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "human_decision":
                "ACCEPT",

            "expected_answer_text":
                CELL106_NEW_ANSWER,

            "external_knowledge_used":
                False,

            "human_context_used_as_evidence":
                False,

            "human_note":
                (
                    "Human Reviewer accepted the replacement "
                    "Expected Answer after reviewing the exact "
                    "book paragraphs 323-327."
                ),
        },
    }


    # --------------------------------------------------------
    # Activate replacement Citation Mapping
    # --------------------------------------------------------

    case[
        "citation_review"
    ] = {
        "citation_mapping_id":
            cell106_new_citation_id,

        "citation_entries":
            deepcopy(
                cell106_new_citation_entries
            ),

        "human_decision":
            "ACCEPT",

        "machine_validation_status":
            "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

        "human_review": {
            "review_record_version":
                "second_human_citation_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_CITATION",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "citation_mapping_id":
                cell106_new_citation_id,

            "human_decision":
                "ACCEPT",

            "external_locator_lookup_used":
                False,

            "human_note":
                (
                    "Citation Mapping uses only the exact "
                    "replacement book paragraphs 323-327 "
                    "resolved in Cell 105."
                ),
        },
    }


    # --------------------------------------------------------
    # Explicit second-review current status
    # --------------------------------------------------------

    case[
        "second_human_review"
    ] = {
        "amendment_id":
            cell106_amendment_id,

        "status":
            "COMPLETE",

        "decision":
            "RETARGET_AND_ACCEPT_REPLACEMENT",

        "review_timestamp_utc":
            cell106_review_timestamp,

        "previous_target_evidence_id":
            previous_target_id,

        "replacement_target_evidence_id":
            cell106_new_target_id,

        "replacement_question_draft_id":
            cell106_new_question_id,

        "replacement_expected_answer_draft_id":
            cell106_new_answer_id,

        "replacement_citation_mapping_id":
            cell106_new_citation_id,

        "previous_state_preserved":
            True,

        "raw_source_mutated":
            False,

        "parser_output_mutated":
            False,

        "author_review_status":
            "NOT_STARTED",
    }


    # --------------------------------------------------------
    # Keep author review untouched / pending
    # --------------------------------------------------------

    author_final_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )

    author_final_review.setdefault(
        "reviewer",
        "AUTHOR",
    )

    author_final_review[
        "status"
    ] = (
        "NOT_STARTED"
    )

    author_final_review[
        "decision"
    ] = None

    author_final_review[
        "notes"
    ] = None

    case[
        "author_final_review"
    ] = (
        author_final_review
    )


    case[
        "final_status"
    ] = (
        "HUMAN_REVIEW_COMPLETE_"
        "AUTHOR_FINAL_REVIEW_PENDING"
    )


    return (
        "APPLIED"
    )


# ============================================================
# 15) Apply to BOTH durable copies
# ============================================================

cell106_completed_apply_status = (
    cell106_apply_amendment(
        cell106_completed_case,
        "completed_human_review_cases",
    )
)


cell106_author_apply_status = (
    cell106_apply_amendment(
        cell106_author_case,
        "author_review.cases_ready_for_review",
    )
)


# ============================================================
# 16) In-memory validation BEFORE write
# ============================================================

def cell106_validate_final_case(
    case,
    location_name,
):

    failures_before = len(
        cell106_failures
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_TARGET_MISMATCH",
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_QUESTION_ID_MISMATCH",
        case.get(
            "final_question_draft_id"
        )
        ==
        cell106_new_question_id,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_QUESTION_TEXT_MISMATCH",
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_ANSWER_MISMATCH",
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER,
    )

    cell106_check(
        location_name
        +
        "_ANSWER_NOT_ACCEPTED",
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "human_decision"
        )
        ==
        "ACCEPT",
    )

    cell106_check(
        location_name
        +
        "_CITATION_NOT_ACCEPTED",
        (
            case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "human_decision"
        )
        ==
        "ACCEPT",
    )

    cell106_check(
        location_name
        +
        "_CITATION_TARGET_MISMATCH",
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            cell106_new_target_id

            for entry
            in (
                (
                    case.get(
                        "citation_review"
                    )
                    or
                    {}
                ).get(
                    "citation_entries"
                )
                or
                []
            )
        ),
    )

    cell106_check(
        location_name
        +
        "_REPLACEMENT_EVIDENCE_COUNT_MISMATCH",
        len(
            case.get(
                "source_evidence",
                []
            )
            or
            []
        )
        ==
        5,
    )

    cell106_check(
        location_name
        +
        "_SECOND_REVIEW_NOT_COMPLETE",
        (
            case.get(
                "second_human_review"
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "COMPLETE",
    )

    cell106_check(
        location_name
        +
        "_AMENDMENT_ID_MISMATCH",
        (
            case.get(
                "second_human_review"
            )
            or
            {}
        ).get(
            "amendment_id"
        )
        ==
        cell106_amendment_id,
    )

    cell106_check(
        location_name
        +
        "_HISTORY_MISSING",
        any(
            (
                isinstance(
                    record,
                    dict,
                )
                and
                record.get(
                    "amendment_id"
                )
                ==
                cell106_amendment_id
            )

            for record
            in (
                case.get(
                    "post_final_human_review_history",
                    []
                )
                or
                []
            )
        ),
    )

    cell106_check(
        location_name
        +
        "_AUTHOR_REVIEW_STATUS_CHANGED",
        (
            case.get(
                "author_final_review"
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    return (
        len(
            cell106_failures
        )
        ==
        failures_before
    )


cell106_validate_final_case(
    cell106_completed_case,
    "COMPLETED",
)


cell106_validate_final_case(
    cell106_author_case,
    "AUTHOR_READY",
)


cell106_check(
    "AUTHOR_REVIEW_CASE_COUNT_CHANGED",
    cell106_author_review_root.get(
        "case_count"
    )
    ==
    22,
    cell106_author_review_root.get(
        "case_count"
    ),
)


if cell106_failures:

    print(
        "\n=== IN-MEMORY VALIDATION FAILURES ==="
    )

    print(
        cell106_failures
    )

    raise AssertionError(
        "Second Review amendment failed before Registry write."
    )


# ============================================================
# 17) Write only if bytes actually need to change
# ============================================================

cell106_registry_after_text = (
    json.dumps(
        cell106_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell106_change_required = (
    cell106_registry_after_text
    !=
    cell106_registry_before_text
)


cell106_backup_path = None


if cell106_change_required:

    timestamp_for_filename = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )

    cell106_backup_path = (
        cell106_registry_path.parent
        /
        (
            cell106_registry_path.stem
            +
            ".before_case4_second_review_"
            +
            timestamp_for_filename
            +
            cell106_registry_path.suffix
        )
    )

    cell106_backup_path.write_text(
        cell106_registry_before_text,
        encoding="utf-8",
    )


    cell106_temp_path = (
        cell106_registry_path.parent
        /
        (
            cell106_registry_path.name
            +
            ".cell106.tmp"
        )
    )


    cell106_temp_path.write_text(
        cell106_registry_after_text,
        encoding="utf-8",
    )


    # Validate temporary JSON before replacing Registry.
    json.loads(
        cell106_temp_path.read_text(
            encoding="utf-8"
        )
    )


    cell106_temp_path.replace(
        cell106_registry_path
    )


# ============================================================
# 18) Reload exact persisted Registry
# ============================================================

cell106_persisted_text = (
    cell106_registry_path.read_text(
        encoding="utf-8"
    )
)


cell106_persisted_registry = (
    json.loads(
        cell106_persisted_text
    )
)


cell106_registry_after_sha256 = (
    hashlib.sha256(
        cell106_persisted_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 19) Resolve persisted copies independently
# ============================================================

cell106_persisted_completed_matches = [
    case

    for case
    in cell106_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )

    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL106_PILOT_ID
    )
]


cell106_persisted_author_matches = [
    case

    for case
    in (
        (
            cell106_persisted_registry.get(
                "author_review",
                {}
            )
            or
            {}
        ).get(
            "cases_ready_for_review",
            [],
        )
    )

    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL106_PILOT_ID
    )
]


cell106_check(
    "PERSISTED_COMPLETED_CASE_NOT_UNIQUE",
    len(
        cell106_persisted_completed_matches
    )
    ==
    1,
)


cell106_check(
    "PERSISTED_AUTHOR_CASE_NOT_UNIQUE",
    len(
        cell106_persisted_author_matches
    )
    ==
    1,
)


if not cell106_failures:

    cell106_persisted_completed_case = (
        cell106_persisted_completed_matches[
            0
        ]
    )

    cell106_persisted_author_case = (
        cell106_persisted_author_matches[
            0
        ]
    )

    cell106_validate_final_case(
        cell106_persisted_completed_case,
        "PERSISTED_COMPLETED",
    )

    cell106_validate_final_case(
        cell106_persisted_author_case,
        "PERSISTED_AUTHOR_READY",
    )


# ============================================================
# 20) Cross-copy canonical equality
# ============================================================

if not cell106_failures:

    cell106_check(
        "CROSS_COPY_TARGET_MISMATCH",
        cell106_persisted_completed_case.get(
            "target_evidence_id"
        )
        ==
        cell106_persisted_author_case.get(
            "target_evidence_id"
        ),
    )

    cell106_check(
        "CROSS_COPY_QUESTION_MISMATCH",
        cell106_active_question(
            cell106_persisted_completed_case
        )
        ==
        cell106_active_question(
            cell106_persisted_author_case
        ),
    )

    cell106_check(
        "CROSS_COPY_ANSWER_MISMATCH",
        (
            cell106_persisted_completed_case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        (
            cell106_persisted_author_case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        ),
    )

    cell106_check(
        "CROSS_COPY_CITATION_ID_MISMATCH",
        (
            cell106_persisted_completed_case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "citation_mapping_id"
        )
        ==
        (
            cell106_persisted_author_case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "citation_mapping_id"
        ),
    )


# ============================================================
# 21) Final gate
# ============================================================

cell106_gate = (
    len(
        cell106_failures
    )
    ==
    0
)


print(
    "=== CASE 4 SECOND HUMAN REVIEW — COMPLETE RETARGET ==="
)


print(
    "\nPilot:",
    CELL106_PILOT_ID,
)


print(
    "\nPrevious Target:",
    CELL106_OLD_TARGET_ID,
)


print(
    "Replacement Target:",
    cell106_new_target_id,
)


print(
    "Replacement paragraphs:",
    CELL106_REPLACEMENT_PARAGRAPHS,
)


print(
    "\nReplacement Question:"
)

print(
    CELL106_NEW_QUESTION
)


print(
    "\nReplacement Answer:"
)

print(
    CELL106_NEW_ANSWER
)


print(
    "\nQuestion Draft ID:",
    cell106_new_question_id,
)


print(
    "Expected Answer Draft ID:",
    cell106_new_answer_id,
)


print(
    "Citation Mapping ID:",
    cell106_new_citation_id,
)


print(
    "Second Review Amendment ID:",
    cell106_amendment_id,
)


print(
    "\nCompleted copy apply status:",
    cell106_completed_apply_status,
)


print(
    "Author-ready copy apply status:",
    cell106_author_apply_status,
)


print(
    "\nRegistry write required:",
    cell106_change_required,
)


print(
    "Backup path:",
    (
        str(
            cell106_backup_path
        )
        if cell106_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_APPLIED"
    ),
)


print(
    "\nRegistry SHA256 before:",
    cell106_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell106_registry_after_sha256,
)


print(
    "\n=== HISTORY / SAFETY ==="
)


print(
    "Previous Target preserved:",
    True,
)


print(
    "Previous Question preserved:",
    True,
)


print(
    "Previous Answer preserved:",
    True,
)


print(
    "Previous Citation preserved:",
    True,
)


print(
    "Raw Source mutated:",
    False,
)


print(
    "Parser Output mutated:",
    False,
)


print(
    "Author Review recorded:",
    False,
)


print(
    "Author Review status:",
    "NOT_STARTED",
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell106_failures
)


print(
    "\nCODE CELL 106 GATE:",
    (
        "PASS"
        if cell106_gate
        else
        "FAIL"
    ),
)


assert cell106_gate, (
    "Case 4 Second Human Review amendment failed. "
    "Do not continue to another Case until this SAME Cell 106 is repaired."
)


=== IN-MEMORY VALIDATION FAILURES ===
[{'reason': 'AUTHOR_REVIEW_CASE_COUNT_CHANGED', 'details': 28}]


AssertionError: Second Review amendment failed before Registry write.

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## پایلوت‌های مشتق‌شده از بازخورد واقعی دکتر در دموی فاز یک

در این مرحله، ۶ مورد بازخورد واقعی دکتر کمیل رودی از دموی فاز یک به‌عنوان Candidate Pilot جدید وارد بازبینی دوم می‌شوند.

این ۶ مورد با ۲۲ پایلوت قبلی تفاوت منشأ دارند:

- ۲۲ پایلوت قبلی از Corpus فاز دو و Human Review ساخته شده‌اند.
- این ۶ مورد از سؤال‌های واقعی دموی فاز یک و بازخورد مستقیم دکتر کمیل رودی استخراج شده‌اند.
- متن بازخورد دکتر به‌عنوان شواهد مستقیم مؤلف حفظ می‌شود.
- جواب مدل فاز یک به‌عنوان Ground Truth در نظر گرفته نمی‌شود.
- هیچ متن تاریخی حذف یا بازنویسی نمی‌شود.
- سؤال و پاسخ پیشنهادی جدید به‌صورت Artifact مستقل ثبت می‌شود.
- این ۶ مورد هنوز به صف ۲۲تایی Author Review اضافه نمی‌شوند.
- پس از بررسی این Batch، هر ۶ مورد در یک مرحله مستقل به همراه ۲۲ مورد قبلی برای بازبینی نهایی دکتر آماده خواهند شد.

---

### مورد 23 — تعریف سواد مالی

**سؤال اصلی دموی فاز یک:**

سواد مالی درباره پول دار شدن دیگه؟

**بازخورد مستقیم دکتر:**

نه ، ببین سواد مالی درباره مدیریت پول است. نه درباره میزان پول

**سؤال پیشنهادی پایلوت:**

سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟

**پاسخ پیشنهادی:**

سواد مالی درباره مدیریت پول است، نه درباره میزان پول.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 24 — ارزیابی دانش، نگرش و مهارت مالی

**سؤال اصلی دموی فاز یک:**

ایا من از نظر مالی وضعیت قابل قبولی دارم؟ از نظر دانشی و مهارتی یا حتی نگرشی؟

**بازخورد مستقیم دکتر:**

مدیریت مالی شخصی نیازمند دانش و نگرش و مهارت های خاصی است، برا یاینکه بدانید شما به اندازه کافی وضعیت مالی خوبی دارید ، لازم است این پرسشنامه را تکمیل کنید.

**سؤال پیشنهادی پایلوت:**

برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید کرد؟

**پاسخ پیشنهادی:**

مدیریت مالی شخصی به دانش، نگرش و مهارت‌های خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی ما از این جنبه‌ها به اندازه کافی مناسب است، باید پرسشنامه مربوط را تکمیل کنیم.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 25 — تداوم گفت‌وگو در سؤال پیگیری

**سؤال اولیه کاربر:**

نمی دانم پولم کجا می رود ؟ انگار پولم را گم می کنم.

**پاسخ اول مدل:**

پول شما ممکن است به دلیل عدم توجه به جزئیات هزینه‌ها و انجام خریدهایی بدون نظارت، به طور غیر منظمی خرج شود. بهترین راه برای کشف این موضوع، ثبت دقیق هزینه‌هاست تا بدانیم چه چیزی را کجاها خرج کرده‌اید. این کار به شما کمک می‌کند تا بهترین تصمیم‌ها را بگیرید و از هدررفت پول جلوگیری کنید.

**سؤال پیگیری کاربر:**

این جواب خوبی بود. ولی کوتاه است بیشتر توضیح می دهی.

**پاسخ دوم مدل:**

مدل به‌اشتباه موضوع را عوض کرد و درباره قرض گرفتن پاسخ داد.

**بازخورد مستقیم دکتر:**

این خیلی حواب بی ربطی بود.

**وظیفه پایلوت:**

اگر کاربر بعد از پاسخ درباره پیدا کردن مسیر خرج پول بگوید «بیشتر توضیح می‌دهی»، پاسخ باید همان موضوع قبلی را ادامه و بسط دهد و نباید به موضوع نامرتبطی مانند قرض گرفتن تغییر مسیر دهد.

**نوع پایلوت:**

MULTI_TURN_CONTEXT_CONTINUITY

این مورد یک Golden Content Question معمولی نیست؛ یک Regression Pilot برای حفظ Context مکالمه است.

---

### مورد 26 — الگوهای خریدی که باعث هدررفت پول می‌شوند

**زمینه گفت‌وگوی فاز یک:**

کاربر درباره هزینه‌های غیرضروری پرسید و سپس خواست چند نمونه ببیند.

**بازخورد مستقیم دکتر:**

آنچه باعث می شود پول هایم هدر روند ، الگوی خرید نامناسب است مانند

1. خرید با تنوع و حجم زیاد
2. خرید به خاطر تنش عاطفی
3. خرید به خاطر هیجان زود گذر
4. خرید به خاطر پذیرفته شدن در گروه های مرجع
5. خرید به خاطر پذیرش تبلیغات و بازاریابی
6. خرید به خاطر یک دوره سختی و احتیاج به پرداختن به خود
7. خرید به خاطر فداکاری

**سؤال پیشنهادی پایلوت:**

چه الگوهای خریدی می‌توانند باعث هدررفت پول شوند؟

**پاسخ پیشنهادی:**

الگوهای خریدی که می‌توانند باعث هدررفت پول شوند شامل این موارد هستند:

1. خرید با تنوع و حجم زیاد
2. خرید به خاطر تنش عاطفی
3. خرید به خاطر هیجان زودگذر
4. خرید به خاطر پذیرفته شدن در گروه‌های مرجع
5. خرید به خاطر پذیرش تبلیغات و بازاریابی
6. خرید به خاطر یک دوره سختی و نیاز به پرداختن به خود
7. خرید به خاطر فداکاری

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 27 — روش‌های افزایش درآمد

**سؤال اصلی دموی فاز یک:**

روش های افزایش درآمد را توضیح بده

**بازخورد مستقیم دکتر:**

ترتیب آن مناسب نیست.

1. فروش وسایلی که واقعا نیازی نداریم.
2. پیشنهاد پروژه و کار بیشتر به کارفرمایی که الان برایش کار می کنیم.
3. خرید دارایی درآمد زا و سرمایه گذاری دوباره سود آن
4. ایجاد درآمد منفعل یعنی درآمدی که وابسته به فعالیت شغلی من نباشد

**سؤال پیشنهادی پایلوت:**

چه روش‌هایی برای افزایش درآمد پیشنهاد می‌شود؟

**پاسخ پیشنهادی:**

روش‌های پیشنهادی برای افزایش درآمد به این ترتیب‌اند:

1. فروش وسایلی که واقعاً به آن‌ها نیاز نداریم.
2. پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.
3. خرید دارایی درآمدزا و سرمایه‌گذاری دوباره سود آن.
4. ایجاد درآمد منفعل؛ یعنی درآمدی که وابسته به فعالیت شغلی فرد نباشد.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### مورد 28 — احساس فقر با وجود داشتن درآمد

**سؤال اصلی دموی فاز یک:**

من پول در می آورم اما هنوز احساس می کنم فقیر هستم!

**بازخورد مستقیم دکتر:**

جواب خوبی است که مساله فقر مساله پول درآوردن و خرج کردن نیست. مساله رابطه با پول است . پول به معنای چیزی است که نداریم و تلاش می کنیم آن چیز را با پول جبران کنیم. در نتیجه مثلا اعتماد به نفس را به پول به دست بیارویم.
ضمن اینکه همه احساس فقر می کنند چون فقر به معنی نرسیدن به چیزی است که می خواهیم و همه چیزهایی می خواهند که ندارند و در نتیجه فقیر هستند.

**سؤال پیشنهادی پایلوت:**

چرا ممکن است فردی با وجود درآمد داشتن همچنان احساس فقر کند؟

**پاسخ پیشنهادی:**

احساس فقر فقط به میزان پول درآوردن یا خرج کردن مربوط نیست و می‌تواند به رابطه فرد با پول مربوط باشد. ممکن است فرد تلاش کند با پول چیزی را که احساس می‌کند ندارد جبران کند؛ برای مثال، اعتمادبه‌نفس را از طریق پول به دست آورد. همچنین در این دیدگاه، فقر به معنای نرسیدن به چیزی است که فرد می‌خواهد و چون انسان‌ها همواره چیزهایی می‌خواهند که ندارند، احساس فقر می‌تواند حتی با وجود داشتن درآمد باقی بماند.

**نوع پایلوت:**

AUTHOR_FEEDBACK_CONTENT

---

### وضعیت Batch

در پایان Cell بعدی باید دقیقاً ۶ Candidate Pilot جدید با شماره‌های 23 تا 28 ثبت شده باشند.

این مرحله نباید:

- ۲۲ پایلوت قبلی را تغییر دهد؛
- Author Review قبلی را تغییر دهد؛
- Case Count فعلی ۲۲تایی Author Review را به ۲۸ تغییر دهد؛
- Feedback خام دکتر را بازنویسی کند؛
- Commit یا Push انجام دهد.

پس از PASS این Batch، مرحله بعدی فقط یک Promotion Batch خواهد بود که مجموعه بازبینی مؤلف را از ۲۲ به ۲۸ مورد ارتقا می‌دهد.

</div>

In [ ]:
# Code Cell 107
# ============================================================
# PHASE-1 AUTHOR FEEDBACK -> DERIVED PILOTS 23-28
#
# Batch import + pilot construction for ALL SIX cases.
#
# This cell is intentionally self-contained.
#
# It:
# - loads the durable Human Review Registry;
# - preserves the exact historical Phase-1 demo material;
# - creates six deterministic derived Pilot records;
# - classifies five as AUTHOR_FEEDBACK_CONTENT;
# - classifies one as MULTI_TURN_CONTEXT_CONTINUITY;
# - records the proposed final Question/Answer or expected behavior;
# - stores all six in a separate second-review Registry collection;
# - prepares them for later Author Review promotion;
# - verifies that the original 22 Author Review cases remain untouched;
# - creates a Registry backup before writing;
# - reloads and validates the exact persisted state;
# - is idempotent.
#
# It DOES NOT:
# - append these six to the existing 22-case Author Review queue;
# - modify any of the 22 existing completed Human Review cases;
# - treat Phase-1 model answers as authoritative evidence;
# - alter Dr. Roudi's raw historical feedback;
# - commit or push.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Constants
# ============================================================

CELL107_BATCH_VERSION = (
    "phase1_author_feedback_derived_pilots_v1"
)

CELL107_ORIGIN = (
    "PHASE1_DEMO_AUTHOR_FEEDBACK"
)

CELL107_AUTHORITY = (
    "PRIMARY_DIRECT_AUTHOR_FEEDBACK"
)

CELL107_EXPECTED_EXISTING_AUTHOR_CASE_COUNT = 22

CELL107_EXPECTED_NEW_CASE_COUNT = 6

CELL107_FUTURE_TOTAL_REFERENCE_PILOTS = 28


# ============================================================
# 2) Utilities
# ============================================================

cell107_failures = []


def cell107_check(
    name,
    condition,
    details=None,
):

    if not bool(
        condition
    ):

        failure = {
            "reason":
                name,
        }

        if details is not None:

            failure[
                "details"
            ] = details

        cell107_failures.append(
            failure
        )

    return bool(
        condition
    )


def cell107_stable_id(
    prefix,
    payload,
):

    if not isinstance(
        payload,
        str,
    ):

        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        )

    digest = hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell107_sha256_text(
    text,
):

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def cell107_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


def cell107_find_repo_root():

    candidates = []

    if (
        "cell105_registry_path"
        in globals()
    ):

        registry_path = Path(
            cell105_registry_path
        ).resolve()

        candidates.extend(
            [
                registry_path.parent,
                *registry_path.parents,
            ]
        )

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = (
            candidate.resolve()
        )

        if str(
            candidate
        ) in seen:

            continue

        seen.add(
            str(
                candidate
            )
        )

        expected_registry = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        if expected_registry.exists():

            return candidate

    return None


# ============================================================
# 3) Resolve Registry independently
# ============================================================

cell107_repo_root = (
    cell107_find_repo_root()
)


cell107_check(
    "REPO_ROOT_NOT_FOUND",
    cell107_repo_root
    is not None,
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


cell107_registry_path = (
    cell107_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell107_check(
    "REGISTRY_NOT_FOUND",
    cell107_registry_path.exists(),
    str(
        cell107_registry_path
    ),
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


# ============================================================
# 4) Exact historical Phase-1 records
#
# IMPORTANT:
# Raw questions, model answers and author feedback are preserved
# as historical evidence. Derived Q/A are separate fields.
# ============================================================

CELL107_CASES = [


    # --------------------------------------------------------
    # Pilot 23
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            23,

        "feedback_case_number":
            1,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    "سواد مالی درباره پول دار شدن دیگه؟",
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "سواد مالی به رابطۀ انسان با پول می‌پردازد، "
                        "نه فقط به خود پول. پول در سواد مالی، یک "
                        "پدیده‌ای بیرونی نیست، بلکه احساسات و رفتارهای "
                        "انسان نسبت به پول است. سواد مالی به دانش و "
                        "مهارت‌های لازم برای مدیریت بهتر پول و "
                        "تصمیم‌گیری‌های مالی اشاره دارد."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "نه ، ببین سواد مالی درباره مدیریت پول است. "
                "نه درباره میزان پول"
            ),

        "derived_question":
            (
                "سواد مالی درباره میزان پولی است که داریم "
                "یا درباره مدیریت پول؟"
            ),

        "derived_expected_answer":
            (
                "سواد مالی درباره مدیریت پول است، "
                "نه درباره میزان پول."
            ),
    },


    # --------------------------------------------------------
    # Pilot 24
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            24,

        "feedback_case_number":
            2,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    (
                        "ایا من از نظر مالی وضعیت قابل قبولی دارم؟ "
                        "از نظر دانشی و مهارتی یا حتی نگرشی؟"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "پاسخ شما در منابع موجود پاسخ کافی پیدا نشد."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "مدیریت مالی شخصی نیازمند دانش و نگرش و "
                "مهارت های خاصی است، برا یاینکه بدانید شما "
                "به اندازه کافی وضعیت مالی خوبی دارید ، لازم "
                "است این پرسشنامه را تکمیل کنید."
            ),

        "derived_question":
            (
                "برای اینکه بفهمیم از نظر دانش، نگرش و "
                "مهارت‌های مالی وضعیت قابل قبولی داریم، "
                "چه باید کرد؟"
            ),

        "derived_expected_answer":
            (
                "مدیریت مالی شخصی به دانش، نگرش و مهارت‌های "
                "خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی "
                "ما از این جنبه‌ها به اندازه کافی مناسب است، "
                "باید پرسشنامه مربوط را تکمیل کنیم."
            ),
    },


    # --------------------------------------------------------
    # Pilot 25
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            25,

        "feedback_case_number":
            3,

        "pilot_type":
            "MULTI_TURN_CONTEXT_CONTINUITY",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "turn":
                    1,

                "text":
                    (
                        "نمی دانم پولم کجا می رود ؟ "
                        "انگار پولم را گم می کنم."
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    1,

                "text":
                    (
                        "پول شما ممکن است به دلیل عدم توجه به جزئیات "
                        "هزینه‌ها و انجام خریدهایی بدون نظارت، به طور "
                        "غیر منظمی خرج شود. بهترین راه برای کشف این "
                        "موضوع، ثبت دقیق هزینه‌هاست تا بدانیم چه چیزی "
                        "را کجاها خرج کرده‌اید. این کار به شما کمک "
                        "می‌کند تا بهترین تصمیم‌ها را بگیرید و از "
                        "هدررفت پول جلوگیری کنید."
                    ),
            },

            {
                "speaker":
                    "HUMAN",

                "turn":
                    2,

                "text":
                    (
                        "این جواب خوبی بود. ولی کوتاه است "
                        "بیشتر توضیح می دهی."
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    2,

                "text":
                    (
                        "قرض گرفتن یک راه حلی است که ممکن است در "
                        "موقعیت‌هایی مانند نیاز به پول سریع یا سرمایه "
                        "گذاری، مفید باشد. اما اگر به طور مداوم و بدون "
                        "نیت مشخص انجام شود، می‌تواند به یک معضل تبدیل "
                        "شود. قرض گرفتن بهتر است زمانی باشد که بتوانیم "
                        "آن را به خوبی بازپرداخت کنیم و از آن برای "
                        "هزینه‌های ضروری یا سرمایه‌گذاری استفاده کنیم. "
                        "اما اگر به صورت مداوم و بدون برنامه‌ریزی، "
                        "انجام شود، می‌تواند به دلیل افزایش بدهی‌ها و "
                        "مشکلات مالی، به یک معضل تبدیل شود. بهتر است "
                        "از قرض به‌عنوان یک ابزار کمکی استفاده کنیم، "
                        "نه یک راه حل دائمی."
                    ),
            },
        ],

        "raw_author_feedback":
            "این خیلی حواب بی ربطی بود.",

        "derived_question":
            (
                "این جواب خوبی بود. ولی کوتاه است "
                "بیشتر توضیح می‌دهی؟"
            ),

        "derived_expected_answer":
            None,

        "expected_behavior":
            (
                "پاسخ باید همان موضوع Turn قبلی، یعنی ثبت و بررسی "
                "هزینه‌ها برای فهمیدن اینکه پول کجا می‌رود، را "
                "ادامه و بسط دهد. پاسخ نباید بدون ارتباط با Context "
                "به موضوع دیگری مانند قرض گرفتن تغییر مسیر دهد."
            ),

        "golden_content_case":
            False,
    },


    # --------------------------------------------------------
    # Pilot 26
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            26,

        "feedback_case_number":
            4,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "turn":
                    1,

                "text":
                    (
                        "چند تا راهکار بده که بتونم پس انداز کنم"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    1,

                "text":
                    (
                        "سه راهکار برای پس‌انداز عبارتند از:\n"
                        "1. تعیین اهداف مالی مشخص و پس‌اندازی برای "
                        "رسیدن به آن‌ها.\n"
                        "2. کاهش هزینه‌های غیر ضروری و جایگزینی آن‌ها "
                        "با اموری که باعث افزایش ثروت می‌شوند.\n"
                        "3. سرمایه‌گذاری پس‌اندازها در گزینه‌هایی که "
                        "حداقل به اندازۀ تورم بازدهی داشته باشند."
                    ),
            },

            {
                "speaker":
                    "HUMAN",

                "turn":
                    2,

                "text":
                    (
                        "هزینه غیرضروری ندارم. همه اش ضروریه . "
                        "یعنی چی هزینه غیرضروری؟"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    2,

                "text":
                    (
                        "هزینه غیرضروری به معنای خرج کردن پول برای "
                        "چیزهایی است که لازم نیستند یا بهبود کیفیت "
                        "زندگی ما را فراهم نمی‌کنند. به عبارت دیگر، "
                        "چیزهایی که به طور مداوم یا بدون نیاز به "
                        "خریداری می‌شوند و ممکن است باعث هدررفت پول "
                        "شوند. به طور کلی، هزینه‌های غیرضروری، "
                        "هزینه‌هایی هستند که به طور غیرمتعارف یا "
                        "بدون نیتی در حساب کشیده می‌شوند."
                    ),
            },

            {
                "speaker":
                    "HUMAN",

                "turn":
                    3,

                "text":
                    "چند نمونه برام بگو",
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "turn":
                    3,

                "text":
                    (
                        "چند نمونه از گزینه‌های سرمایه‌گذاری عبارتند از:\n"
                        "- سرمایه‌گذاری در اوراق بهادار\n"
                        "- سرمایه‌گذاری در املاک و مستغلات\n"
                        "- سرمایه‌گذاری در صندوق‌های سرمایه\n"
                        "- سرمایه‌گذاری در بانک‌ها با سود بالا\n"
                        "- سرمایه‌گذاری در طلا یا سایر فلزات گرانبها."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "آنچه باعث می شود پول هایم هدر روند ، "
                "الگوی خرید نامناسب است مانند \n"
                "1. خرید با تنوع و حجم زیاد\n"
                "2. خرید به خاطر تنش عاطفی \n"
                "3. خرید به خاطر هیجان زود گذر \n"
                "4. خرید به خاطر پذیرفته شدن در گروه های مرجع \n"
                "5. خرید به خاطر پذیرش تبلیغات و بازاریابی \n"
                "6. خرید به خاطر یک دوره سختی و احتیاج به پرداختن به خود \n"
                "7. خرید به خاطر فداکاری"
            ),

        "derived_question":
            (
                "چه الگوهای خریدی می‌توانند باعث هدررفت پول شوند؟"
            ),

        "derived_expected_answer":
            (
                "الگوهای خریدی که می‌توانند باعث هدررفت پول شوند "
                "شامل این موارد هستند:\n"
                "1. خرید با تنوع و حجم زیاد\n"
                "2. خرید به خاطر تنش عاطفی\n"
                "3. خرید به خاطر هیجان زودگذر\n"
                "4. خرید به خاطر پذیرفته شدن در گروه‌های مرجع\n"
                "5. خرید به خاطر پذیرش تبلیغات و بازاریابی\n"
                "6. خرید به خاطر یک دوره سختی و نیاز به پرداختن به خود\n"
                "7. خرید به خاطر فداکاری"
            ),
    },


    # --------------------------------------------------------
    # Pilot 27
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            27,

        "feedback_case_number":
            5,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    "روش های افزایش درآمد را توضیح بده",
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "روش‌های افزایش درآمد عبارتند از:\n"
                        "1. قیمت‌گذاری مناسب محصولات\n"
                        "2. سرمایه‌گذاری درست\n"
                        "3. خرید دارایی‌های درآمدزا\n"
                        "4. تغییر نقشه کسب‌وکار"
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "ترتیب آن مناسب نیست. \n\n"
                "1. فروش وسایلی که واقعا نیازی نداریم.\n"
                "2. پیشنهاد پروژه و کار بیشتر به کارفرمایی که "
                "الان برایش کار می کنیم. \n"
                "3. خرید دارایی درآمد زا و سرمایه گذاری دوباره سود آن \n"
                "4. ایجاد درآمد منفعل یعنی درآمدی که وابسته به "
                "فعالیت شغلی من نباشد"
            ),

        "derived_question":
            (
                "چه روش‌هایی برای افزایش درآمد پیشنهاد می‌شود؟"
            ),

        "derived_expected_answer":
            (
                "روش‌های پیشنهادی برای افزایش درآمد به این ترتیب‌اند:\n"
                "1. فروش وسایلی که واقعاً به آن‌ها نیاز نداریم.\n"
                "2. پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.\n"
                "3. خرید دارایی درآمدزا و سرمایه‌گذاری دوباره سود آن.\n"
                "4. ایجاد درآمد منفعل؛ یعنی درآمدی که وابسته به "
                "فعالیت شغلی فرد نباشد."
            ),
    },


    # --------------------------------------------------------
    # Pilot 28
    # --------------------------------------------------------

    {
        "reference_pilot_number":
            28,

        "feedback_case_number":
            6,

        "pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "raw_conversation": [
            {
                "speaker":
                    "HUMAN",

                "text":
                    (
                        "من پول در می آورم اما هنوز احساس می کنم "
                        "فقیر هستم!"
                    ),
            },

            {
                "speaker":
                    "PHASE1_MODEL",

                "text":
                    (
                        "شما ممکن است به دلیل ارتباط نادرست با پول "
                        "و ترس از نداشتن پول، احساس فقر کنید. پول تنها "
                        "ابزاری برای ثابت کردن خود و امنیت نیست، بلکه "
                        "رابطه‌ای است که با آن سروکار داریم. اگر "
                        "تصمیم‌های نادرستی را در گذشته گرفته باشید، "
                        "ممکن است این احساس فقر را ایجاد کرده باشد. "
                        "بهتر است به جای توجه به مقدار پول، به رابطه‌ای "
                        "که با پول دارید نیز توجه کنید. همچنین، پول را "
                        "باید به کار برد تا ارزشش حفظ شود. بهترین راه "
                        "برای جلوگیری از احساس فقر، توجه به سواد مالی "
                        "و تصمیم‌گیری‌های صحیح است."
                    ),
            },
        ],

        "raw_author_feedback":
            (
                "جواب خوبی است که مساله فقر مساله پول درآوردن و "
                "خرج کردن نیست. مساله رابطه با پول است . پول به "
                "معنای چیزی است که نداریم و تلاش می کنیم آن چیز را "
                "با پول جبران کنیم. در نتیجه مثلا اعتماد به نفس را "
                "به پول به دست بیارویم.\n"
                "ضمن اینکه همه احساس فقر می کنند چون فقر به معنی "
                "نرسیدن به چیزی است که می خواهیم و همه چیزهایی "
                "می خواهند که ندارند و در نتیجه فقیر هستند."
            ),

        "derived_question":
            (
                "چرا ممکن است فردی با وجود درآمد داشتن "
                "همچنان احساس فقر کند؟"
            ),

        "derived_expected_answer":
            (
                "احساس فقر فقط به میزان پول درآوردن یا خرج کردن "
                "مربوط نیست و می‌تواند به رابطه فرد با پول مربوط "
                "باشد. ممکن است فرد تلاش کند با پول چیزی را که "
                "احساس می‌کند ندارد جبران کند؛ برای مثال، "
                "اعتمادبه‌نفس را از طریق پول به دست آورد. همچنین "
                "در این دیدگاه، فقر به معنای نرسیدن به چیزی است "
                "که فرد می‌خواهد و چون انسان‌ها همواره چیزهایی "
                "می‌خواهند که ندارند، احساس فقر می‌تواند حتی با "
                "وجود داشتن درآمد باقی بماند."
            ),
    },
]


# ============================================================
# 5) Batch structural validation BEFORE touching Registry
# ============================================================

cell107_check(
    "INPUT_CASE_COUNT_MISMATCH",
    len(
        CELL107_CASES
    )
    ==
    CELL107_EXPECTED_NEW_CASE_COUNT,
)


cell107_numbers = [
    item[
        "reference_pilot_number"
    ]

    for item
    in CELL107_CASES
]


cell107_check(
    "REFERENCE_NUMBERS_MISMATCH",
    cell107_numbers
    ==
    [
        23,
        24,
        25,
        26,
        27,
        28,
    ],
)


cell107_check(
    "REFERENCE_NUMBERS_NOT_UNIQUE",
    len(
        set(
            cell107_numbers
        )
    )
    ==
    6,
)


cell107_feedback_numbers = [
    item[
        "feedback_case_number"
    ]

    for item
    in CELL107_CASES
]


cell107_check(
    "FEEDBACK_NUMBERS_MISMATCH",
    cell107_feedback_numbers
    ==
    [
        1,
        2,
        3,
        4,
        5,
        6,
    ],
)


cell107_check(
    "CONTENT_CASE_COUNT_MISMATCH",
    sum(
        1

        for item
        in CELL107_CASES

        if (
            item[
                "pilot_type"
            ]
            ==
            "AUTHOR_FEEDBACK_CONTENT"
        )
    )
    ==
    5,
)


cell107_check(
    "CONTINUITY_CASE_COUNT_MISMATCH",
    sum(
        1

        for item
        in CELL107_CASES

        if (
            item[
                "pilot_type"
            ]
            ==
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )
    )
    ==
    1,
)


for item in CELL107_CASES:

    cell107_check(
        (
            "EMPTY_AUTHOR_FEEDBACK_CASE_"
            +
            str(
                item[
                    "feedback_case_number"
                ]
            )
        ),
        bool(
            item.get(
                "raw_author_feedback"
            )
        ),
    )

    cell107_check(
        (
            "EMPTY_DERIVED_QUESTION_CASE_"
            +
            str(
                item[
                    "feedback_case_number"
                ]
            )
        ),
        bool(
            item.get(
                "derived_question"
            )
        ),
    )

    if (
        item[
            "pilot_type"
        ]
        ==
        "AUTHOR_FEEDBACK_CONTENT"
    ):

        cell107_check(
            (
                "EMPTY_EXPECTED_ANSWER_CASE_"
                +
                str(
                    item[
                        "feedback_case_number"
                    ]
                )
            ),
            bool(
                item.get(
                    "derived_expected_answer"
                )
            ),
        )

    else:

        cell107_check(
            "CONTINUITY_EXPECTED_BEHAVIOR_MISSING",
            bool(
                item.get(
                    "expected_behavior"
                )
            ),
        )


if cell107_failures:

    print(
        cell107_failures
    )

    raise AssertionError(
        "Cell 107 input batch failed structural validation."
    )


# ============================================================
# 6) Build immutable derived Pilot records
# ============================================================

cell107_timestamp = (
    cell107_now()
)


cell107_derived_pilots = []


for source_case in CELL107_CASES:

    feedback_case_number = (
        source_case[
            "feedback_case_number"
        ]
    )

    reference_number = (
        source_case[
            "reference_pilot_number"
        ]
    )

    raw_feedback = (
        source_case[
            "raw_author_feedback"
        ]
    )

    feedback_source_id = (
        cell107_stable_id(
            "author-feedback-source",
            {
                "origin":
                    CELL107_ORIGIN,

                "feedback_case_number":
                    feedback_case_number,

                "raw_author_feedback":
                    raw_feedback,
            },
        )
    )


    pilot_case_id = (
        cell107_stable_id(
            "question-pilot",
            {
                "origin":
                    CELL107_ORIGIN,

                "feedback_case_number":
                    feedback_case_number,

                "reference_pilot_number":
                    reference_number,

                "pilot_type":
                    source_case[
                        "pilot_type"
                    ],

                "feedback_source_id":
                    feedback_source_id,
            },
        )
    )


    question_draft_id = (
        cell107_stable_id(
            "question-draft",
            {
                "pilot_case_id":
                    pilot_case_id,

                "question":
                    source_case[
                        "derived_question"
                    ],
            },
        )
    )


    expected_answer_draft_id = None

    citation_mapping_id = None


    if (
        source_case[
            "pilot_type"
        ]
        ==
        "AUTHOR_FEEDBACK_CONTENT"
    ):

        expected_answer_draft_id = (
            cell107_stable_id(
                "expected-answer-draft",
                {
                    "pilot_case_id":
                        pilot_case_id,

                    "question_draft_id":
                        question_draft_id,

                    "answer":
                        source_case[
                            "derived_expected_answer"
                        ],
                },
            )
        )


        citation_mapping_id = (
            cell107_stable_id(
                "citation-mapping",
                {
                    "pilot_case_id":
                        pilot_case_id,

                    "expected_answer_draft_id":
                        expected_answer_draft_id,

                    "feedback_source_id":
                        feedback_source_id,
                },
            )
        )


    else:

        citation_mapping_id = (
            cell107_stable_id(
                "behavior-mapping",
                {
                    "pilot_case_id":
                        pilot_case_id,

                    "question_draft_id":
                        question_draft_id,

                    "expected_behavior":
                        source_case[
                            "expected_behavior"
                        ],

                    "feedback_source_id":
                        feedback_source_id,
                },
            )
        )


    record = {
        "record_schema":
            "AUTHOR_FEEDBACK_DERIVED_PILOT_V1",

        "batch_version":
            CELL107_BATCH_VERSION,

        "reference_pilot_number":
            reference_number,

        "pilot_case_id":
            pilot_case_id,

        "feedback_case_number":
            feedback_case_number,

        "origin":
            CELL107_ORIGIN,

        "pilot_type":
            source_case[
                "pilot_type"
            ],

        "authority": {
            "authority_class":
                CELL107_AUTHORITY,

            "primary_author":
                "Dr. Komeil Roudi",

            "phase1_model_answer_authoritative":
                False,

            "raw_author_feedback_authoritative":
                True,
        },

        "historical_phase1_demo": {
            "raw_conversation":
                deepcopy(
                    source_case[
                        "raw_conversation"
                    ]
                ),

            "raw_author_feedback":
                raw_feedback,

            "raw_author_feedback_sha256":
                cell107_sha256_text(
                    raw_feedback
                ),

            "raw_feedback_preserved_verbatim":
                True,
        },

        "feedback_source": {
            "feedback_source_id":
                feedback_source_id,

            "source_type":
                "DIRECT_AUTHOR_FEEDBACK",

            "source_scope":
                "PHASE1_DEMO",

            "source_mapping_to_phase2_corpus":
                "NOT_REQUIRED_FOR_AUTHOR_FEEDBACK_AUTHORITY",

            "optional_corpus_cross_mapping_status":
                "NOT_STARTED",
        },

        "derived_question": {
            "question_draft_id":
                question_draft_id,

            "question_text":
                source_case[
                    "derived_question"
                ],

            "review_origin":
                "SECOND_HUMAN_REVIEW_PHASE1_AUTHOR_FEEDBACK",

            "human_decision":
                "ACCEPT",

            "external_knowledge_used":
                False,
        },

        "second_human_review": {
            "status":
                "COMPLETE",

            "decision":
                "ACCEPT_DERIVED_PILOT",

            "review_origin":
                "PHASE1_AUTHOR_FEEDBACK",

            "review_timestamp_utc":
                cell107_timestamp,

            "raw_feedback_preserved":
                True,

            "phase1_model_answer_used_as_ground_truth":
                False,

            "author_review_status":
                "NOT_STARTED",
        },

        "author_final_review": {
            "status":
                "NOT_STARTED",

            "reviewer":
                "AUTHOR",

            "decision":
                None,

            "notes":
                None,
        },

        "promotion": {
            "status":
                "READY_FOR_AUTHOR_REVIEW",

            "currently_in_22_case_author_queue":
                False,

            "future_reference_pilot_number":
                reference_number,

            "future_target_total":
                CELL107_FUTURE_TOTAL_REFERENCE_PILOTS,
        },
    }


    if (
        source_case[
            "pilot_type"
        ]
        ==
        "AUTHOR_FEEDBACK_CONTENT"
    ):

        record[
            "expected_answer_review"
        ] = {
            "expected_answer_draft_id":
                expected_answer_draft_id,

            "expected_answer_text":
                source_case[
                    "derived_expected_answer"
                ],

            "human_decision":
                "ACCEPT",

            "grounding_source":
                "DIRECT_AUTHOR_FEEDBACK",

            "external_knowledge_used":
                False,
        }


        record[
            "citation_review"
        ] = {
            "citation_mapping_id":
                citation_mapping_id,

            "human_decision":
                "ACCEPT",

            "citation_entries": [
                {
                    "citation_index":
                        0,

                    "source_type":
                        "DIRECT_AUTHOR_FEEDBACK",

                    "feedback_source_id":
                        feedback_source_id,

                    "raw_author_feedback_sha256":
                        cell107_sha256_text(
                            raw_feedback
                        ),

                    "authority_class":
                        CELL107_AUTHORITY,
                },
            ],
        }


        record[
            "golden_content_case"
        ] = True


    else:

        record[
            "expected_behavior_review"
        ] = {
            "behavior_mapping_id":
                citation_mapping_id,

            "expected_behavior":
                source_case[
                    "expected_behavior"
                ],

            "human_decision":
                "ACCEPT",

            "grounding_source":
                "DIRECT_AUTHOR_FEEDBACK_PLUS_CONVERSATION_CONTEXT",

            "external_knowledge_used":
                False,
        }


        record[
            "golden_content_case"
        ] = False


        record[
            "regression_role"
        ] = (
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )


    cell107_derived_pilots.append(
        record
    )


# ============================================================
# 7) Deterministic-ID validation
# ============================================================

cell107_pilot_ids = [
    item[
        "pilot_case_id"
    ]

    for item
    in cell107_derived_pilots
]


cell107_check(
    "DERIVED_PILOT_ID_COUNT_MISMATCH",
    len(
        cell107_pilot_ids
    )
    ==
    6,
)


cell107_check(
    "DERIVED_PILOT_IDS_NOT_UNIQUE",
    len(
        set(
            cell107_pilot_ids
        )
    )
    ==
    6,
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


# ============================================================
# 8) Load Registry fresh
# ============================================================

cell107_registry_before_text = (
    cell107_registry_path.read_text(
        encoding="utf-8"
    )
)


cell107_registry = json.loads(
    cell107_registry_before_text
)


cell107_registry_before_sha256 = (
    hashlib.sha256(
        cell107_registry_before_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 9) Protect the existing 22-case collections
# ============================================================

cell107_completed_before = deepcopy(
    cell107_registry.get(
        "completed_human_review_cases",
        []
    )
)


cell107_author_review_before = deepcopy(
    cell107_registry.get(
        "author_review",
        {}
    )
)


cell107_existing_author_cases_before = (
    (
        cell107_author_review_before
        or
        {}
    ).get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell107_check(
    "PREEXISTING_AUTHOR_QUEUE_NOT_22",
    len(
        cell107_existing_author_cases_before
    )
    ==
    CELL107_EXPECTED_EXISTING_AUTHOR_CASE_COUNT,
    len(
        cell107_existing_author_cases_before
    ),
)


if cell107_failures:

    raise AssertionError(
        cell107_failures
    )


# ============================================================
# 10) Create / update separate Second Review Batch
# ============================================================

cell107_second_review_root = (
    cell107_registry.setdefault(
        "second_human_review",
        {},
    )
)


cell107_feedback_batch = (
    cell107_second_review_root.setdefault(
        "phase1_author_feedback_derived_pilots",
        {},
    )
)


existing_records = (
    cell107_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


existing_by_id = {
    item.get(
        "pilot_case_id"
    ):
        item

    for item
    in existing_records

    if isinstance(
        item,
        dict,
    )
}


for new_record in cell107_derived_pilots:

    pilot_id = (
        new_record[
            "pilot_case_id"
        ]
    )

    if pilot_id in existing_by_id:

        cell107_check(
            (
                "EXISTING_RECORD_CONTENT_MISMATCH:"
                +
                pilot_id
            ),
            existing_by_id[
                pilot_id
            ]
            ==
            new_record,
        )

    else:

        existing_records.append(
            deepcopy(
                new_record
            )
        )


if cell107_failures:

    print(
        cell107_failures
    )

    raise AssertionError(
        "Existing Cell 107 batch conflicts with the deterministic rebuild."
    )


# Keep only this Batch's six deterministic records in sorted order.
cell107_batch_ids = set(
    cell107_pilot_ids
)


cell107_exact_batch_records = [
    record

    for record
    in existing_records

    if (
        isinstance(
            record,
            dict,
        )
        and
        record.get(
            "pilot_case_id"
        )
        in
        cell107_batch_ids
    )
]


cell107_exact_batch_records = sorted(
    cell107_exact_batch_records,
    key=lambda item: (
        item[
            "reference_pilot_number"
        ]
    ),
)


cell107_check(
    "PERSISTED_BATCH_CASE_COUNT_NOT_6",
    len(
        cell107_exact_batch_records
    )
    ==
    6,
)


cell107_feedback_batch.update({
    "batch_version":
        CELL107_BATCH_VERSION,

    "origin":
        CELL107_ORIGIN,

    "status":
        "COMPLETE_READY_FOR_AUTHOR_REVIEW_PROMOTION",

    "case_count":
        6,

    "reference_pilot_range":
        [
            23,
            28,
        ],

    "content_pilot_count":
        5,

    "continuity_regression_pilot_count":
        1,

    "future_total_reference_pilots":
        28,

    "currently_promoted_to_author_review":
        False,

    "author_review_queue_count_before_promotion":
        22,

    "cases":
        cell107_exact_batch_records,
})


# ============================================================
# 11) Confirm original 22 collections were not mutated
# ============================================================

cell107_check(
    "COMPLETED_22_CASES_MUTATED",
    cell107_registry.get(
        "completed_human_review_cases",
        []
    )
    ==
    cell107_completed_before,
)


cell107_check(
    "AUTHOR_REVIEW_ROOT_MUTATED",
    cell107_registry.get(
        "author_review",
        {}
    )
    ==
    cell107_author_review_before,
)


if cell107_failures:

    print(
        cell107_failures
    )

    raise AssertionError(
        "Cell 107 attempted to alter the protected existing 22-case state."
    )


# ============================================================
# 12) Serialize / decide whether write is necessary
# ============================================================

cell107_registry_after_text = (
    json.dumps(
        cell107_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell107_change_required = (
    cell107_registry_after_text
    !=
    cell107_registry_before_text
)


cell107_backup_path = None


# ============================================================
# 13) Backup + atomic-ish write
# ============================================================

if cell107_change_required:

    timestamp_for_filename = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )


    cell107_backup_path = (
        cell107_registry_path.parent
        /
        (
            cell107_registry_path.stem
            +
            ".before_phase1_author_feedback_batch_"
            +
            timestamp_for_filename
            +
            cell107_registry_path.suffix
        )
    )


    cell107_backup_path.write_text(
        cell107_registry_before_text,
        encoding="utf-8",
    )


    cell107_temp_path = (
        cell107_registry_path.parent
        /
        (
            cell107_registry_path.name
            +
            ".cell107.tmp"
        )
    )


    cell107_temp_path.write_text(
        cell107_registry_after_text,
        encoding="utf-8",
    )


    # Parse temporary bytes before replacement.
    json.loads(
        cell107_temp_path.read_text(
            encoding="utf-8"
        )
    )


    cell107_temp_path.replace(
        cell107_registry_path
    )


# ============================================================
# 14) Reload exact persisted Registry
# ============================================================

cell107_persisted_text = (
    cell107_registry_path.read_text(
        encoding="utf-8"
    )
)


cell107_persisted_registry = (
    json.loads(
        cell107_persisted_text
    )
)


cell107_registry_after_sha256 = (
    hashlib.sha256(
        cell107_persisted_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 15) Persisted validation
# ============================================================

cell107_persisted_completed = (
    cell107_persisted_registry.get(
        "completed_human_review_cases",
        []
    )
)


cell107_persisted_author_review = (
    cell107_persisted_registry.get(
        "author_review",
        {}
    )
)


cell107_persisted_author_cases = (
    (
        cell107_persisted_author_review
        or
        {}
    ).get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell107_persisted_batch = (
    (
        cell107_persisted_registry.get(
            "second_human_review",
            {}
        )
        or
        {}
    ).get(
        "phase1_author_feedback_derived_pilots",
        {}
    )
    or
    {}
)


cell107_persisted_batch_cases = (
    cell107_persisted_batch.get(
        "cases",
        []
    )
    or
    []
)


cell107_check(
    "PERSISTED_COMPLETED_CASES_CHANGED",
    cell107_persisted_completed
    ==
    cell107_completed_before,
)


cell107_check(
    "PERSISTED_AUTHOR_QUEUE_COUNT_CHANGED",
    len(
        cell107_persisted_author_cases
    )
    ==
    22,
    len(
        cell107_persisted_author_cases
    ),
)


cell107_check(
    "PERSISTED_AUTHOR_REVIEW_ROOT_CHANGED",
    cell107_persisted_author_review
    ==
    cell107_author_review_before,
)


cell107_check(
    "PERSISTED_FEEDBACK_BATCH_COUNT_MISMATCH",
    len(
        cell107_persisted_batch_cases
    )
    ==
    6,
)


cell107_check(
    "PERSISTED_FEEDBACK_BATCH_STATUS_MISMATCH",
    cell107_persisted_batch.get(
        "status"
    )
    ==
    "COMPLETE_READY_FOR_AUTHOR_REVIEW_PROMOTION",
)


cell107_check(
    "PERSISTED_FEEDBACK_BATCH_PROMOTED_TOO_EARLY",
    cell107_persisted_batch.get(
        "currently_promoted_to_author_review"
    )
    is False,
)


cell107_persisted_ids = [
    item.get(
        "pilot_case_id"
    )

    for item
    in cell107_persisted_batch_cases
]


cell107_check(
    "PERSISTED_PILOT_IDS_MISMATCH",
    cell107_persisted_ids
    ==
    cell107_pilot_ids,
)


for record in cell107_persisted_batch_cases:

    cell107_check(
        (
            "AUTHOR_REVIEW_STARTED_TOO_EARLY:"
            +
            record.get(
                "pilot_case_id",
                "UNKNOWN",
            )
        ),
        (
            record.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    cell107_check(
        (
            "PROMOTION_STATUS_MISMATCH:"
            +
            record.get(
                "pilot_case_id",
                "UNKNOWN",
            )
        ),
        (
            record.get(
                "promotion",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "READY_FOR_AUTHOR_REVIEW",
    )


# ============================================================
# 16) Final gate
# ============================================================

cell107_gate = (
    len(
        cell107_failures
    )
    ==
    0
)


print(
    "=== PHASE-1 AUTHOR FEEDBACK -> PILOTS 23-28 ==="
)


print(
    "\nBatch version:",
    CELL107_BATCH_VERSION,
)


print(
    "Origin:",
    CELL107_ORIGIN,
)


print(
    "\nNew Pilot count:",
    len(
        cell107_persisted_batch_cases
    ),
)


print(
    "Content Pilots:",
    sum(
        1

        for item
        in cell107_persisted_batch_cases

        if (
            item.get(
                "pilot_type"
            )
            ==
            "AUTHOR_FEEDBACK_CONTENT"
        )
    ),
)


print(
    "Continuity Regression Pilots:",
    sum(
        1

        for item
        in cell107_persisted_batch_cases

        if (
            item.get(
                "pilot_type"
            )
            ==
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )
    ),
)


print(
    "\n=== DERIVED PILOTS ==="
)


for record in cell107_persisted_batch_cases:

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Reference Pilot:",
        record[
            "reference_pilot_number"
        ],
    )

    print(
        "Pilot ID:",
        record[
            "pilot_case_id"
        ],
    )

    print(
        "Type:",
        record[
            "pilot_type"
        ],
    )

    print(
        "Question:"
    )

    print(
        record[
            "derived_question"
        ][
            "question_text"
        ]
    )

    if record.get(
        "golden_content_case"
    ):

        print(
            "Expected Answer:"
        )

        print(
            (
                record.get(
                    "expected_answer_review"
                )
                or
                {}
            ).get(
                "expected_answer_text"
            )
        )

    else:

        print(
            "Expected Behavior:"
        )

        print(
            (
                record.get(
                    "expected_behavior_review"
                )
                or
                {}
            ).get(
                "expected_behavior"
            )
        )

    print(
        "Promotion:",
        (
            record.get(
                "promotion"
            )
            or
            {}
        ).get(
            "status"
        ),
    )


print(
    "\n=== PROTECTED EXISTING STATE ==="
)


print(
    "Existing completed Human Review cases changed:",
    False,
)


print(
    "Existing Author Review queue count:",
    len(
        cell107_persisted_author_cases
    ),
)


print(
    "New six already promoted:",
    False,
)


print(
    "Future intended total after promotion:",
    28,
)


print(
    "\n=== WRITE / SAFETY ==="
)


print(
    "Registry write required:",
    cell107_change_required,
)


print(
    "Backup path:",
    (
        str(
            cell107_backup_path
        )
        if cell107_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_APPLIED"
    ),
)


print(
    "Registry SHA256 before:",
    cell107_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell107_registry_after_sha256,
)


print(
    "Raw Phase-1 Author Feedback mutated:",
    False,
)


print(
    "Phase-1 model answers treated as authority:",
    False,
)


print(
    "Author Review recorded:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell107_failures
)


print(
    "\nCODE CELL 107 GATE:",
    (
        "PASS"
        if cell107_gate
        else
        "FAIL"
    ),
)


assert cell107_gate, (
    "Phase-1 Author Feedback Batch failed. "
    "Repair this SAME Code Cell 107 before promotion to the 28-case Author Review queue."
)

=== PHASE-1 AUTHOR FEEDBACK -> PILOTS 23-28 ===

Batch version: phase1_author_feedback_derived_pilots_v1
Origin: PHASE1_DEMO_AUTHOR_FEEDBACK

New Pilot count: 6
Content Pilots: 5
Continuity Regression Pilots: 1

=== DERIVED PILOTS ===

Reference Pilot: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Type: AUTHOR_FEEDBACK_CONTENT
Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟
Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.
Promotion: READY_FOR_AUTHOR_REVIEW

Reference Pilot: 24
Pilot ID: question-pilot:70628caeefebbfba48300176fafddc96
Type: AUTHOR_FEEDBACK_CONTENT
Question:
برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید کرد؟
Expected Answer:
مدیریت مالی شخصی به دانش، نگرش و مهارت‌های خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی ما از این جنبه‌ها به اندازه کافی مناسب است، باید پرسشنامه مربوط را تکمیل کنیم.
Promotion: READY_FOR_AUTHOR_REVIEW

Reference Pilot: 25
Pilot ID: question-pilot:77d8

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## بازبینی منبع پایلوت‌های 23 تا 28 — بازخوردهای فاز یک

شش پایلوت 23 تا 28 از بازخوردهای واقعی دکتر کمیل رودی در دموی فاز یک ساخته شده‌اند.

پیش از اینکه این شش مورد هم‌سطح ۲۲ پایلوت قبلی تلقی شوند، باید بررسی شود که سؤال و پاسخ پیشنهادی آن‌ها با Sourceهای واقعی پذیرفته‌شده پروژه سازگار است یا خیر.

### محدوده این مرحله

از آنجا که دموی فاز یک مورد بحث فقط از مقالات دکتر استفاده می‌کرد، Source Mapping در این مرحله ابتدا فقط روی:

**Website + Direct Author / Primary Author**

انجام می‌شود.

Source of Truth:

`data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl`

برای هر یک از پایلوت‌های 23 تا 28 باید:

1. سؤال فعلی خوانده شود.
2. پاسخ/رفتار مورد انتظار خوانده شود.
3. بازخورد تاریخی دکتر به‌عنوان Query Signal استفاده شود.
4. مقالات Direct Author دکتر در Accepted Corpus جستجو شوند.
5. چند Source Candidate واقعی همراه با متن Evidence نمایش داده شوند.
6. میزان پوشش واژگان کلیدی گزارش شود.
7. هنوز هیچ `ACCEPT / REVISE / RETARGET / REJECT` در Registry ثبت نشود.

### مورد خاص Pilot 25

Pilot 25 یک سؤال محتوایی معمولی نیست و نوع آن:

`MULTI_TURN_CONTEXT_CONTINUITY`

است.

برای این مورد، Source Search فقط بررسی می‌کند که موضوع اصلی گفت‌وگو یعنی:

`پول کجا می‌رود / ثبت و بررسی هزینه‌ها`

در Corpus چه پشتوانه‌ای دارد.

خود الزام «حفظ Context مکالمه» از Source مقاله استخراج نمی‌شود و باید به‌عنوان Regression Behavior مستقل باقی بماند.

### ممنوعیت‌ها

این Cell:

- Registry را تغییر نمی‌دهد.
- سؤال یا جواب جدید تولید نمی‌کند.
- Source را خودکار ACCEPT نمی‌کند.
- Author Review Decision ثبت نمی‌کند.
- Target Evidence را تغییر نمی‌دهد.
- ۲۲ پایلوت قبلی را تغییر نمی‌دهد.
- فایل Markdown بازبینی مؤلف را تغییر نمی‌دهد.
- Commit یا Push انجام نمی‌دهد.

خروجی این Cell مبنای تصمیم انسانی مرحله بعد خواهد بود.

</div>

In [ ]:
# Code Cell 108
# ============================================================
# SOURCE MAPPING REVIEW — PILOTS 23-28
#
# READ-ONLY diagnostic.
#
# Purpose:
# - Load the frozen Phase-2 Accepted Corpus.
# - Resolve the six Phase-1 Author-Feedback-derived Pilots.
# - Search ONLY Website Direct-Author / Primary-Author sources.
# - Show real candidate documents + evidence windows.
# - Produce a review report in memory.
#
# NO Registry mutation.
# NO ACCEPT / REJECT decision.
# NO target/citation mutation.
# NO Author decision.
# NO Commit / Push.
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import re
import unicodedata


# ============================================================
# 1) Constants
# ============================================================

CELL108_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
)

CELL108_EXPECTED_CORPUS_RECORDS = 19456
CELL108_EXPECTED_DIRECT_AUTHOR_WEBSITE = 76

CELL108_NEW_PILOT_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]

CELL108_TOP_K = 5


# Search terms are derived ONLY from:
# - actual Phase-1 questions,
# - actual Dr. Roudi feedback,
# - the already-created derived pilot.
#
# No external knowledge is introduced.
CELL108_SEARCH_QUERIES = {

    23: (
        "سواد مالی مدیریت پول میزان پول "
        "پولدار شدن مدیریت بهتر پول"
    ),

    24: (
        "مدیریت مالی شخصی دانش نگرش مهارت "
        "پرسشنامه وضعیت مالی ارزیابی"
    ),

    25: (
        "پولم کجا می رود ثبت هزینه ثبت مخارج "
        "مدیریت هزینه خرج هزینه ها"
    ),

    26: (
        "الگوی خرید نامناسب هدررفت پول "
        "تنش عاطفی هیجان زودگذر گروه مرجع "
        "تبلیغات بازاریابی فداکاری خرید زیاد"
    ),

    27: (
        "افزایش درآمد فروش وسایل کار بیشتر "
        "کارفرما پروژه دارایی درآمدزا "
        "سرمایه گذاری سود درآمد منفعل"
    ),

    28: (
        "احساس فقر رابطه با پول "
        "اعتماد به نفس پول نرسیدن خواسته "
        "احساس فقیر بودن"
    ),
}


# ============================================================
# 2) Utilities
# ============================================================

cell108_failures = []


def cell108_check(name, condition, details=None):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell108_failures.append(
            failure
        )

    return bool(condition)


def cell108_sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def cell108_find_repo_root():

    candidates = []

    # Previous notebook variables are optional.
    for variable_name in [
        "cell107_registry_path",
        "cell108_registry_path",
        "cell105_registry_path",
    ]:

        if variable_name in globals():

            try:
                p = Path(
                    globals()[variable_name]
                ).resolve()

                candidates.extend(
                    [
                        p.parent,
                        *p.parents,
                    ]
                )

            except Exception:
                pass

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_path = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        corpus_path = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "processed"
            /
            "documents_dedup_quality_accepted_v1.jsonl"
        )

        if (
            registry_path.exists()
            and
            corpus_path.exists()
        ):
            return candidate

    return None


# ============================================================
# 3) Persian normalization
# ============================================================

CELL108_STOPWORDS = {
    "از", "به", "در", "با", "برای", "را",
    "که", "و", "یا", "این", "آن", "یک",
    "است", "هست", "هستم", "هستند", "بود",
    "شد", "شود", "می", "شود", "کرد", "کنیم",
    "چه", "چرا", "چگونه", "اگر", "اما",
    "هم", "روی", "خود", "ما", "من", "شما",
    "فرد", "افراد", "مورد", "درباره",
}


def cell108_normalize(text):

    text = "" if text is None else str(text)

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    replacements = {
        "ي": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
        "\u200c": " ",
        "\u200f": " ",
        "\u200e": " ",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(
        r"[^\w\u0600-\u06FF]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def cell108_tokens(text):

    tokens = [
        token

        for token
        in cell108_normalize(
            text
        ).split()

        if (
            len(token) >= 2
            and
            token not in CELL108_STOPWORDS
        )
    ]

    return tokens


# ============================================================
# 4) Flexible metadata helpers
# ============================================================

def cell108_json_blob(value):

    try:
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
        ).lower()

    except Exception:
        return str(
            value
        ).lower()


def cell108_is_direct_author_website(record):

    if (
        str(
            record.get(
                "source_type",
                ""
            )
        ).lower()
        !=
        "website"
    ):
        return False

    # Inspect only authority-related metadata,
    # NOT the article text itself.
    authority_blob = cell108_json_blob({
        "attribution":
            record.get(
                "attribution"
            ),

        "content_origin":
            record.get(
                "content_origin"
            ),

        "source_metadata":
            record.get(
                "source_metadata"
            ),
    })

    positive_markers = [
        '"primary_author"',
        '"direct_author"',
        "primary_author",
        "direct_author",
    ]

    return any(
        marker in authority_blob

        for marker
        in positive_markers
    )


def cell108_get_url(record):

    candidates = []

    locator = (
        record.get(
            "locator"
        )
        or
        {}
    )

    source_metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(locator, dict):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if locator.get(key):
                candidates.append(
                    locator.get(key)
                )

    if isinstance(source_metadata, dict):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if source_metadata.get(key):
                candidates.append(
                    source_metadata.get(key)
                )

    return (
        candidates[0]
        if candidates
        else None
    )


def cell108_get_title(record):

    title = record.get(
        "title"
    )

    if title:
        return str(title)

    metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(metadata, dict):

        for key in [
            "title",
            "article_title",
            "page_title",
        ]:

            if metadata.get(key):
                return str(
                    metadata.get(key)
                )

    return "NO_TITLE"


# ============================================================
# 5) Resolve repository and frozen artifacts
# ============================================================

cell108_repo_root = (
    cell108_find_repo_root()
)


cell108_check(
    "REPO_ROOT_NOT_FOUND",
    cell108_repo_root is not None,
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


cell108_registry_path = (
    cell108_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell108_corpus_path = (
    cell108_repo_root
    /
    "data"
    /
    "phase2"
    /
    "processed"
    /
    "documents_dedup_quality_accepted_v1.jsonl"
)


cell108_check(
    "REGISTRY_NOT_FOUND",
    cell108_registry_path.exists(),
    str(cell108_registry_path),
)


cell108_check(
    "ACCEPTED_CORPUS_NOT_FOUND",
    cell108_corpus_path.exists(),
    str(cell108_corpus_path),
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 6) Verify Accepted Corpus identity
# ============================================================

cell108_corpus_sha256 = (
    cell108_sha256_file(
        cell108_corpus_path
    )
)


cell108_check(
    "ACCEPTED_CORPUS_SHA256_MISMATCH",
    cell108_corpus_sha256
    ==
    CELL108_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL108_EXPECTED_CORPUS_SHA256,

        "actual":
            cell108_corpus_sha256,
    },
)


if cell108_failures:

    print(
        cell108_failures
    )

    raise AssertionError(
        "Cell 108 stopped Fail-Closed: frozen corpus identity mismatch."
    )


# ============================================================
# 7) Load Accepted Corpus
# ============================================================

cell108_corpus = []


with cell108_corpus_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue

        try:

            record = json.loads(
                line
            )

        except Exception as exc:

            cell108_check(
                "CORPUS_JSON_PARSE_FAILURE",
                False,
                {
                    "line":
                        line_number,

                    "error":
                        str(exc),
                },
            )

            break

        cell108_corpus.append(
            record
        )


cell108_check(
    "CORPUS_RECORD_COUNT_MISMATCH",
    len(cell108_corpus)
    ==
    CELL108_EXPECTED_CORPUS_RECORDS,
    len(cell108_corpus),
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 8) Select Direct-Author Website corpus
# ============================================================

cell108_primary_website_records = [
    record

    for record
    in cell108_corpus

    if cell108_is_direct_author_website(
        record
    )
]


cell108_check(
    "DIRECT_AUTHOR_WEBSITE_COUNT_MISMATCH",
    len(
        cell108_primary_website_records
    )
    ==
    CELL108_EXPECTED_DIRECT_AUTHOR_WEBSITE,
    {
        "expected":
            CELL108_EXPECTED_DIRECT_AUTHOR_WEBSITE,

        "actual":
            len(
                cell108_primary_website_records
            ),
    },
)


if cell108_failures:

    print(
        cell108_failures
    )

    raise AssertionError(
        "Cell 108 stopped Fail-Closed: Direct-Author Website scope mismatch."
    )


# ============================================================
# 9) Load six derived Pilots from Registry
# ============================================================

cell108_registry = json.loads(
    cell108_registry_path.read_text(
        encoding="utf-8"
    )
)


cell108_second_review = (
    cell108_registry.get(
        "second_human_review",
        {}
    )
    or
    {}
)


cell108_feedback_batch = (
    cell108_second_review.get(
        "phase1_author_feedback_derived_pilots",
        {}
    )
    or
    {}
)


cell108_feedback_cases = (
    cell108_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


cell108_cases_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell108_feedback_cases

    if isinstance(
        case,
        dict,
    )
}


cell108_check(
    "DERIVED_PILOT_BATCH_NOT_6",
    len(
        cell108_feedback_cases
    )
    ==
    6,
    len(
        cell108_feedback_cases
    ),
)


cell108_check(
    "DERIVED_PILOT_NUMBERS_MISSING",
    sorted(
        cell108_cases_by_number.keys()
    )
    ==
    CELL108_NEW_PILOT_NUMBERS,
    sorted(
        cell108_cases_by_number.keys()
    ),
)


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 10) Build BM25 index over the 76 real Direct-Author articles
# ============================================================

cell108_doc_tokens = []
cell108_doc_term_counts = []
cell108_document_frequency = Counter()


for record in (
    cell108_primary_website_records
):

    text = (
        record.get(
            "text"
        )
        or
        ""
    )

    tokens = cell108_tokens(
        text
    )

    counts = Counter(
        tokens
    )

    cell108_doc_tokens.append(
        tokens
    )

    cell108_doc_term_counts.append(
        counts
    )

    for token in set(tokens):

        cell108_document_frequency[
            token
        ] += 1


cell108_doc_lengths = [
    len(tokens)

    for tokens
    in cell108_doc_tokens
]


cell108_average_doc_length = (
    sum(
        cell108_doc_lengths
    )
    /
    max(
        len(
            cell108_doc_lengths
        ),
        1,
    )
)


def cell108_bm25_score(
    query_tokens,
    doc_index,
    k1=1.5,
    b=0.75,
):

    counts = (
        cell108_doc_term_counts[
            doc_index
        ]
    )

    doc_length = (
        cell108_doc_lengths[
            doc_index
        ]
    )

    N = len(
        cell108_primary_website_records
    )

    score = 0.0

    for term in set(
        query_tokens
    ):

        tf = counts.get(
            term,
            0,
        )

        if tf <= 0:
            continue

        df = (
            cell108_document_frequency.get(
                term,
                0,
            )
        )

        idf = math.log(
            1
            +
            (
                N
                -
                df
                +
                0.5
            )
            /
            (
                df
                +
                0.5
            )
        )

        denominator = (
            tf
            +
            k1
            *
            (
                1
                -
                b
                +
                b
                *
                doc_length
                /
                max(
                    cell108_average_doc_length,
                    1,
                )
            )
        )

        score += (
            idf
            *
            (
                tf
                *
                (
                    k1
                    +
                    1
                )
            )
            /
            denominator
        )

    return score


# ============================================================
# 11) Evidence-window selection
# ============================================================

def cell108_best_window(
    text,
    query_tokens,
    max_chars=700,
):

    text = str(
        text
        or
        ""
    ).strip()

    if not text:
        return ""

    # Preserve actual source text.
    blocks = [
        block.strip()

        for block
        in re.split(
            r"\n\s*\n",
            text,
        )

        if block.strip()
    ]

    if not blocks:
        blocks = [text]

    query_set = set(
        query_tokens
    )

    best_block = None
    best_score = -1

    for block in blocks:

        block_tokens = set(
            cell108_tokens(
                block
            )
        )

        overlap = (
            query_set
            &
            block_tokens
        )

        score = len(
            overlap
        )

        if score > best_score:

            best_score = score
            best_block = block

    if best_block is None:
        best_block = text

    # If the best block is short enough, show it whole.
    if len(
        best_block
    ) <= max_chars:

        return best_block

    normalized_block = (
        cell108_normalize(
            best_block
        )
    )

    first_position = None

    for token in query_tokens:

        position = normalized_block.find(
            token
        )

        if (
            position >= 0
            and
            (
                first_position is None
                or
                position < first_position
            )
        ):
            first_position = position

    # Approximate a useful source window.
    if first_position is None:

        return (
            best_block[:max_chars]
            +
            " ..."
        )

    raw_lower = best_block

    # Find a raw occurrence of one matching term.
    raw_position = None

    for token in query_tokens:

        p = raw_lower.find(
            token
        )

        if p >= 0:

            raw_position = p

            break

    if raw_position is None:
        raw_position = 0

    start = max(
        0,
        raw_position
        -
        200,
    )

    end = min(
        len(
            best_block
        ),
        start
        +
        max_chars,
    )

    excerpt = (
        best_block[
            start:end
        ].strip()
    )

    if start > 0:
        excerpt = "... " + excerpt

    if end < len(
        best_block
    ):
        excerpt = excerpt + " ..."

    return excerpt


# ============================================================
# 12) Search all six Pilots
# ============================================================

cell108_source_mapping_report = []


for pilot_number in (
    CELL108_NEW_PILOT_NUMBERS
):

    case = (
        cell108_cases_by_number[
            pilot_number
        ]
    )

    query_text = (
        CELL108_SEARCH_QUERIES[
            pilot_number
        ]
    )

    query_tokens = (
        cell108_tokens(
            query_text
        )
    )

    query_unique = set(
        query_tokens
    )

    ranked = []


    for doc_index, record in enumerate(
        cell108_primary_website_records
    ):

        score = (
            cell108_bm25_score(
                query_tokens,
                doc_index,
            )
        )

        doc_token_set = set(
            cell108_doc_tokens[
                doc_index
            ]
        )

        matched_terms = sorted(
            query_unique
            &
            doc_token_set
        )

        coverage = (
            len(
                matched_terms
            )
            /
            max(
                len(
                    query_unique
                ),
                1,
            )
        )

        # Small transparent bonus for broader query coverage.
        combined_score = (
            score
            *
            (
                1.0
                +
                coverage
            )
        )

        ranked.append({
            "record":
                record,

            "score":
                combined_score,

            "bm25":
                score,

            "coverage":
                coverage,

            "matched_terms":
                matched_terms,

            "doc_index":
                doc_index,
        })


    ranked.sort(
        key=lambda item: (
            -item[
                "score"
            ],
            str(
                item[
                    "record"
                ].get(
                    "document_id",
                    "",
                )
            ),
        )
    )


    top_candidates = []


    for rank, result in enumerate(
        ranked[
            :CELL108_TOP_K
        ],
        start=1,
    ):

        record = result[
            "record"
        ]

        candidate = {
            "rank":
                rank,

            "document_id":
                record.get(
                    "document_id"
                ),

            "source_id":
                record.get(
                    "source_id"
                ),

            "title":
                cell108_get_title(
                    record
                ),

            "url":
                cell108_get_url(
                    record
                ),

            "score":
                round(
                    result[
                        "score"
                    ],
                    4,
                ),

            "bm25":
                round(
                    result[
                        "bm25"
                    ],
                    4,
                ),

            "query_coverage":
                round(
                    result[
                        "coverage"
                    ],
                    4,
                ),

            "matched_terms":
                result[
                    "matched_terms"
                ],

            "evidence_window":
                cell108_best_window(
                    record.get(
                        "text",
                        "",
                    ),
                    query_tokens,
                ),
        }

        top_candidates.append(
            candidate
        )


    top = (
        top_candidates[0]
        if top_candidates
        else None
    )


    # This is ONLY a retrieval signal.
    # It is explicitly NOT a Human Review decision.
    if top is None:

        retrieval_signal = (
            "NO_CANDIDATE"
        )

    elif (
        top[
            "query_coverage"
        ]
        >=
        0.50
    ):

        retrieval_signal = (
            "STRONG_SOURCE_CANDIDATE"
        )

    elif (
        top[
            "query_coverage"
        ]
        >=
        0.30
    ):

        retrieval_signal = (
            "POSSIBLE_SOURCE_CANDIDATE"
        )

    else:

        retrieval_signal = (
            "WEAK_SOURCE_MATCH_REVIEW_REQUIRED"
        )


    report_item = {
        "reference_pilot_number":
            pilot_number,

        "pilot_case_id":
            case.get(
                "pilot_case_id"
            ),

        "pilot_type":
            case.get(
                "pilot_type"
            ),

        "derived_question":
            (
                (
                    case.get(
                        "derived_question",
                        {}
                    )
                    or
                    {}
                ).get(
                    "question_text"
                )
            ),

        "expected_answer":
            (
                (
                    case.get(
                        "expected_answer_review",
                        {}
                    )
                    or
                    {}
                ).get(
                    "expected_answer_text"
                )
            ),

        "expected_behavior":
            (
                (
                    case.get(
                        "expected_behavior_review",
                        {}
                    )
                    or
                    {}
                ).get(
                    "expected_behavior"
                )
            ),

        "search_query":
            query_text,

        "mapping_purpose":
            (
                "CONTENT_GROUNDING"
                if (
                    case.get(
                        "pilot_type"
                    )
                    ==
                    "AUTHOR_FEEDBACK_CONTENT"
                )
                else
                "CONVERSATION_TOPIC_GROUNDING_ONLY"
            ),

        "retrieval_signal":
            retrieval_signal,

        "top_candidates":
            top_candidates,

        "human_source_decision":
            "NOT_STARTED",
    }


    cell108_source_mapping_report.append(
        report_item
    )


# ============================================================
# 13) Coverage / safety validation
# ============================================================

cell108_check(
    "SOURCE_MAPPING_REPORT_COUNT_NOT_6",
    len(
        cell108_source_mapping_report
    )
    ==
    6,
)


for item in (
    cell108_source_mapping_report
):

    cell108_check(
        (
            "NO_TOP_CANDIDATES_FOR_PILOT_"
            +
            str(
                item[
                    "reference_pilot_number"
                ]
            )
        ),
        len(
            item[
                "top_candidates"
            ]
        )
        ==
        CELL108_TOP_K,
    )


if cell108_failures:
    raise AssertionError(
        cell108_failures
    )


# ============================================================
# 14) Human-readable output
# ============================================================

print(
    "=== CELL 108 — SOURCE MAPPING REVIEW: PILOTS 23-28 ==="
)


print(
    "\n=== FROZEN SOURCE ==="
)


print(
    "Corpus:",
    str(
        cell108_corpus_path
    ),
)


print(
    "Corpus SHA256:",
    cell108_corpus_sha256,
)


print(
    "Accepted Corpus records:",
    len(
        cell108_corpus
    ),
)


print(
    "Direct-Author Website records:",
    len(
        cell108_primary_website_records
    ),
)


print(
    "\nSearch scope:",
    "WEBSITE / DIRECT_AUTHOR ONLY",
)


for item in (
    cell108_source_mapping_report
):

    print(
        "\n"
        +
        "="
        *
        110
    )

    print(
        "PILOT:",
        item[
            "reference_pilot_number"
        ],
    )

    print(
        "Pilot ID:",
        item[
            "pilot_case_id"
        ],
    )

    print(
        "Type:",
        item[
            "pilot_type"
        ],
    )

    print(
        "Mapping purpose:",
        item[
            "mapping_purpose"
        ],
    )

    print(
        "\nQuestion:"
    )

    print(
        item[
            "derived_question"
        ]
    )


    if item[
        "expected_answer"
    ]:

        print(
            "\nCurrent Expected Answer:"
        )

        print(
            item[
                "expected_answer"
            ]
        )


    if item[
        "expected_behavior"
    ]:

        print(
            "\nCurrent Expected Behavior:"
        )

        print(
            item[
                "expected_behavior"
            ]
        )


    print(
        "\nSearch query:"
    )

    print(
        item[
            "search_query"
        ]
    )


    print(
        "\nRETRIEVAL SIGNAL:",
        item[
            "retrieval_signal"
        ],
    )


    print(
        "\n--- TOP REAL SOURCE CANDIDATES ---"
    )


    for candidate in (
        item[
            "top_candidates"
        ]
    ):

        print(
            "\n"
            +
            "-"
            *
            90
        )

        print(
            "Rank:",
            candidate[
                "rank"
            ],
        )

        print(
            "Document ID:",
            candidate[
                "document_id"
            ],
        )

        print(
            "Title:",
            candidate[
                "title"
            ],
        )

        print(
            "URL:",
            candidate[
                "url"
            ],
        )

        print(
            "Score:",
            candidate[
                "score"
            ],
        )

        print(
            "Query coverage:",
            candidate[
                "query_coverage"
            ],
        )

        print(
            "Matched terms:",
            candidate[
                "matched_terms"
            ],
        )

        print(
            "\nEvidence window:"
        )

        print(
            candidate[
                "evidence_window"
            ]
        )


    print(
        "\nHUMAN SOURCE DECISION:",
        "NOT_STARTED"
    )


# ============================================================
# 15) Safety / gate
# ============================================================

cell108_gate = (
    len(
        cell108_failures
    )
    ==
    0
)


print(
    "\n"
    +
    "="
    *
    110
)


print(
    "=== SAFETY ==="
)


print(
    "Registry write:",
    False,
)


print(
    "Author Review mutation:",
    False,
)


print(
    "Human ACCEPT/REJECT recorded:",
    False,
)


print(
    "Question changed:",
    False,
)


print(
    "Expected Answer changed:",
    False,
)


print(
    "Target Evidence changed:",
    False,
)


print(
    "Citation changed:",
    False,
)


print(
    "Approved 22 cases changed:",
    False,
)


print(
    "Markdown document changed:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell108_failures
)


print(
    "\nCODE CELL 108 GATE:",
    (
        "PASS"
        if cell108_gate
        else
        "FAIL"
    ),
)


assert cell108_gate, (
    "Pilot 23-28 Source Mapping diagnostic failed. "
    "Repair this SAME Code Cell 108 before Human Source Review."
)

=== CELL 108 — SOURCE MAPPING REVIEW: PILOTS 23-28 ===

=== FROZEN SOURCE ===
Corpus: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl
Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Accepted Corpus records: 19456
Direct-Author Website records: 76

Search scope: WEBSITE / DIRECT_AUTHOR ONLY

PILOT: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Type: AUTHOR_FEEDBACK_CONTENT
Mapping purpose: CONTENT_GROUNDING

Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟

Current Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.

Search query:
سواد مالی مدیریت پول میزان پول پولدار شدن مدیریت بهتر پول

RETRIEVAL SIGNAL: STRONG_SOURCE_CANDIDATE

--- TOP REAL SOURCE CANDIDATES ---

------------------------------------------------------------------------------------------
Rank: 1
Document ID: website:3A675BF6
Title: دختران خوب

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## تصمیم نهایی بازبینی منبع برای پایلوت‌های 23 تا 28

در Cell قبل، هر شش پایلوت مشتق‌شده از بازخورد فاز یک روی Accepted Corpus واقعی Phase 2 و فقط در محدوده Website / Direct Author بررسی شدند.

در این مرحله تصمیم انسانی Source Review ثبت می‌شود.

### تصمیم‌های تأییدشده

#### Pilot 23 — ACCEPT

سؤال و پاسخ فعلی حفظ می‌شوند.

Source اصلی Corpus:

`website:7C71884B — ابزارهای سواد مالی برای بهبود مدیریت هزینه`

این Source به‌صورت مستقیم بیان می‌کند که سواد مالی درباره میزان پول نیست، بلکه درباره درست مدیریت کردن پول است.

---

#### Pilot 24 — ACCEPT

سؤال و پاسخ فعلی حفظ می‌شوند.

Grounding ترکیبی:

- `website:662BDA7F — سواد مالی خانم‌ها؛ وضعیت چقدر بحرانی است؟`
- `website:E2311705 — آموزش سواد مالی در اروپا`

یکی بخش پرسشنامه و ارزیابی وضعیت را پشتیبانی می‌کند و دیگری ابعاد دانش، نگرش و مهارت را.

---

#### Pilot 25 — REVISE

این مورد دیگر به‌عنوان `MULTI_TURN_CONTEXT_CONTINUITY` فعال باقی نمی‌ماند.

علت:

در دموی فاز یک، Turnها Context مشترک قابل اتکایی نداشتند؛ بنابراین پاسخ بی‌ربط مدل نباید به‌عنوان یک Golden Regression Case محتوایی وارد مجموعه مرجع شود.

اما موضوع اصلی سؤال با Corpus واقعی پشتیبانی می‌شود.

Source اصلی:

`website:3E26E509 — سواد مالی خانم‌ها؛ پول‌ها کجا می‌روند؟`

سؤال جدید:

**وقتی نمی‌دانیم پولمان کجا می‌رود و در پایان ماه نمی‌دانیم آن را صرف چه چیزهایی کرده‌ایم، چه کاری باید انجام دهیم؟**

پاسخ جدید:

**باید هزینه‌ها را ثبت کنیم تا بدانیم پول را کجا و برای چه چیزهایی خرج کرده‌ایم. آگاهی از هزینه‌ها امکان مدیریت آن‌ها را فراهم می‌کند و ثبت هزینه‌ها می‌تواند به پیشگیری از ولخرجی و مدیریت بهتر مخارج کمک کند.**

نوع فعال جدید:

`AUTHOR_FEEDBACK_CONTENT`

تاریخچه حالت قبلی حذف نمی‌شود و به‌عنوان Superseded History حفظ خواهد شد.

---

#### Pilot 26 — REVISE / GROUNDING ONLY

سؤال و پاسخ فعلی تغییر نمی‌کنند.

بازخورد مستقیم دکتر همچنان Grounding اصلی هفت الگوی خرید است.

Corpus پشتیبانی تکمیلی ارائه می‌دهد:

- `website:9673DF51 — الگوهای نادرست خرید و راهکارهای پرهیز از آن`
- `website:BC0342B6 — سواد مالی خانم‌ها؛ تصمیم‌های خرید را خودمان می‌گیریم یا عادت‌های‌مان؟`

مقالات مفهوم الگوهای نادرست خرید، عوامل عاطفی و اثر تبلیغات/بازاریابی را پشتیبانی می‌کنند، اما تمام هفت مورد را مستقل و عیناً پوشش نمی‌دهند.

بنابراین:

`DIRECT AUTHOR FEEDBACK = PRIMARY GROUNDING`

`WEBSITE CORPUS = PARTIAL CORROBORATION`

---

#### Pilot 27 — REVISE / GROUNDING ONLY

سؤال و پاسخ فعلی تغییر نمی‌کنند.

چهار روش و ترتیب آن‌ها از بازخورد مستقیم دکتر آمده است.

Corpus فقط بخشی از مفاهیم مرتبط با افزایش درآمد، دارایی درآمدزا، سرمایه‌گذاری و سود را پشتیبانی می‌کند.

Sources تکمیلی:

- `website:3A675BF6 — دختران خوب ثروتمند می‌شوند اگر ...`
- `website:C042183F — «پول، پول می‌آورد» اما چگونه؟`

بنابراین:

`DIRECT AUTHOR FEEDBACK = PRIMARY GROUNDING`

`WEBSITE CORPUS = PARTIAL CORROBORATION`

---

#### Pilot 28 — ACCEPT

سؤال و پاسخ فعلی حفظ می‌شوند.

Source مهم Corpus:

`website:2F8F15FB — چگونه کمتر نگران پول باشیم؟`

این مقاله بخش اصلی مفهوم رابطه با پول، احساسات و تلاش برای جبران چیزهایی که فرد احساس می‌کند ندارد را پشتیبانی می‌کند.

بخش گسترده‌تر پاسخ درباره احساس فقر و نرسیدن به خواسته‌ها از بازخورد مستقیم دکتر حفظ می‌شود.

بنابراین Grounding ترکیبی است:

`DIRECT AUTHOR FEEDBACK + WEBSITE DIRECT AUTHOR`

---

### نتیجه مورد انتظار

پس از اجرای Cell بعد:

- 23 = ACCEPT
- 24 = ACCEPT
- 25 = REVISE → AUTHOR_FEEDBACK_CONTENT
- 26 = REVISE / GROUNDING_ONLY
- 27 = REVISE / GROUNDING_ONLY
- 28 = ACCEPT

هر دو Copy موجود در Registry باید Sync شوند:

1. `second_human_review.phase1_author_feedback_derived_pilots`
2. `author_review.cases_ready_for_review`

همچنان:

- Author Final Review = NOT_STARTED
- ۲۲ مورد قبلی تغییر نمی‌کنند.
- Raw Phase-1 Author Feedback تغییر نمی‌کند.
- History قبلی Pilot 25 حذف نمی‌شود.
- هیچ Commit یا Push انجام نمی‌شود.
- فایل Markdown ۲۸تایی در این Cell تغییر نمی‌کند.

</div>

In [ ]:
# Code Cell 109
# ============================================================
# APPLY HUMAN SOURCE REVIEW — PILOTS 23-28
#
# Decisions approved after Cell 108 Source Mapping:
#
# 23 -> ACCEPT
# 24 -> ACCEPT
# 25 -> REVISE:
#       MULTI_TURN_CONTEXT_CONTINUITY
#       -> AUTHOR_FEEDBACK_CONTENT
# 26 -> REVISE / GROUNDING_ONLY
# 27 -> REVISE / GROUNDING_ONLY
# 28 -> ACCEPT
#
# This cell:
# - verifies the frozen Accepted Corpus;
# - verifies all chosen source documents really exist;
# - applies the Source Review to BOTH Registry copies;
# - preserves raw Phase-1 conversation and Author Feedback;
# - preserves Pilot 25's superseded continuity history;
# - creates a new accepted Q/A for Pilot 25;
# - updates citation/source mappings;
# - keeps Author Final Review NOT_STARTED;
# - protects the original 22 cases;
# - creates a Registry backup;
# - reloads and validates persisted state;
# - is idempotent.
#
# It DOES NOT:
# - change the 28-case Markdown file;
# - record any decision on behalf of Dr. Roudi;
# - commit or push.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Constants
# ============================================================

CELL109_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40"
)

CELL109_EXPECTED_COMPLETED_CASES = 22
CELL109_EXPECTED_AUTHOR_QUEUE = 28
CELL109_EXPECTED_NEW_CASES = 6

CELL109_BATCH_KEY = (
    "phase1_author_feedback_derived_pilots"
)

CELL109_REVIEW_VERSION = (
    "phase1_author_feedback_source_review_v1"
)

CELL109_REFERENCE_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]


# ============================================================
# 2) Approved Human Review decisions
# ============================================================

CELL109_DECISIONS = {

    # --------------------------------------------------------
    # Pilot 23
    # --------------------------------------------------------

    23: {
        "decision":
            "ACCEPT",

        "revision_scope":
            "NONE",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PLUS_CORPUS_DIRECT_SUPPORT",

        "sources": [
            {
                "document_id":
                    "website:7C71884B",

                "role":
                    "PRIMARY_CORPUS_GROUNDING",

                "coverage":
                    "DIRECT_EXPLICIT",

                "review_note":
                    (
                        "The article explicitly states that financial "
                        "literacy is not about the amount of money, "
                        "but about managing money correctly."
                    ),
            },
        ],
    },


    # --------------------------------------------------------
    # Pilot 24
    # --------------------------------------------------------

    24: {
        "decision":
            "ACCEPT",

        "revision_scope":
            "NONE",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PLUS_COMBINED_CORPUS_SUPPORT",

        "sources": [
            {
                "document_id":
                    "website:662BDA7F",

                "role":
                    "ASSESSMENT_AND_QUESTIONNAIRE_GROUNDING",

                "coverage":
                    "DIRECT_SUPPORT",

                "review_note":
                    (
                        "Supports questionnaire-based assessment "
                        "and interpretation of financial condition."
                    ),
            },

            {
                "document_id":
                    "website:E2311705",

                "role":
                    "KNOWLEDGE_ATTITUDE_SKILL_GROUNDING",

                "coverage":
                    "DIRECT_SUPPORT",

                "review_note":
                    (
                        "Supports financial literacy as a combination "
                        "of knowledge, attitudes and abilities/skills."
                    ),
            },
        ],
    },


    # --------------------------------------------------------
    # Pilot 25
    # --------------------------------------------------------

    25: {
        "decision":
            "REVISE",

        "revision_scope":
            "QUESTION_ANSWER_AND_PILOT_TYPE",

        "grounding_mode":
            "CORPUS_DIRECT_SUPPORT",

        "new_pilot_type":
            "AUTHOR_FEEDBACK_CONTENT",

        "new_question":
            (
                "وقتی نمی‌دانیم پولمان کجا می‌رود و در پایان ماه "
                "نمی‌دانیم آن را صرف چه چیزهایی کرده‌ایم، "
                "چه کاری باید انجام دهیم؟"
            ),

        "new_expected_answer":
            (
                "باید هزینه‌ها را ثبت کنیم تا بدانیم پول را کجا و "
                "برای چه چیزهایی خرج کرده‌ایم. آگاهی از هزینه‌ها "
                "امکان مدیریت آن‌ها را فراهم می‌کند و ثبت هزینه‌ها "
                "می‌تواند به پیشگیری از ولخرجی و مدیریت بهتر مخارج "
                "کمک کند."
            ),

        "sources": [
            {
                "document_id":
                    "website:3E26E509",

                "role":
                    "PRIMARY_CORPUS_GROUNDING",

                "coverage":
                    "DIRECT_EXPLICIT",

                "review_note":
                    (
                        "Directly supports the problem of not knowing "
                        "where money went and the value of recording "
                        "expenses for awareness and better management."
                    ),
            },

            {
                "document_id":
                    "website:7C71884B",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "SUPPORTING",

                "review_note":
                    (
                        "Supports expense recording and using that "
                        "experience for better expense management "
                        "and budgeting."
                    ),
            },
        ],
    },


    # --------------------------------------------------------
    # Pilot 26
    # --------------------------------------------------------

    26: {
        "decision":
            "REVISE",

        "revision_scope":
            "GROUNDING_ONLY",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PRIMARY_WITH_PARTIAL_CORPUS_CORROBORATION",

        "sources": [
            {
                "document_id":
                    "website:9673DF51",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports incorrect purchasing patterns, "
                        "emotional drivers, and advertising/marketing "
                        "influence, but does not independently enumerate "
                        "all seven Author-feedback items."
                    ),
            },

            {
                "document_id":
                    "website:BC0342B6",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports emotional/psychological purchasing "
                        "patterns and broader incorrect buying behavior."
                    ),
            },
        ],

        "grounding_constraint":
            (
                "The seven-item expected answer remains primarily "
                "grounded in Dr. Roudi's direct historical feedback. "
                "Website evidence is corroborating, not an independent "
                "full source for all seven items."
            ),
    },


    # --------------------------------------------------------
    # Pilot 27
    # --------------------------------------------------------

    27: {
        "decision":
            "REVISE",

        "revision_scope":
            "GROUNDING_ONLY",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PRIMARY_WITH_PARTIAL_CORPUS_CORROBORATION",

        "sources": [
            {
                "document_id":
                    "website:3A675BF6",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports increasing income, work/income "
                        "capacity and income-producing assets, but "
                        "does not independently reproduce the complete "
                        "four-step Author sequence."
                    ),
            },

            {
                "document_id":
                    "website:C042183F",

                "role":
                    "CORROBORATING_CORPUS_SOURCE",

                "coverage":
                    "PARTIAL",

                "review_note":
                    (
                        "Supports the concept of investing money in "
                        "productive assets and generating profit."
                    ),
            },
        ],

        "grounding_constraint":
            (
                "The four methods and their ordering remain primarily "
                "grounded in Dr. Roudi's direct historical feedback. "
                "Website sources provide only partial corroboration."
            ),
    },


    # --------------------------------------------------------
    # Pilot 28
    # --------------------------------------------------------

    28: {
        "decision":
            "ACCEPT",

        "revision_scope":
            "NONE",

        "grounding_mode":
            "DIRECT_AUTHOR_FEEDBACK_PLUS_CORPUS_DIRECT_SUPPORT",

        "sources": [
            {
                "document_id":
                    "website:2F8F15FB",

                "role":
                    "PRIMARY_CORPUS_CORROBORATION",

                "coverage":
                    "STRONG_PARTIAL",

                "review_note":
                    (
                        "Strongly supports the relationship-with-money "
                        "concept and the idea that money can represent "
                        "things a person feels are missing, while the "
                        "broader poverty interpretation remains grounded "
                        "in direct Author feedback."
                    ),
            },
        ],
    },
}


# ============================================================
# 3) Utilities
# ============================================================

cell109_failures = []


def cell109_check(
    name,
    condition,
    details=None,
):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell109_failures.append(
            failure
        )

    return bool(condition)


def cell109_stable_id(
    prefix,
    payload,
):

    if not isinstance(
        payload,
        str,
    ):

        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell109_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell109_sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def cell109_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


def cell109_find_repo_root():

    candidates = []

    for variable_name in [
        "cell108_registry_path",
        "cell107_registry_path",
        "cell105_registry_path",
    ]:

        if variable_name in globals():

            try:

                p = Path(
                    globals()[
                        variable_name
                    ]
                ).resolve()

                candidates.extend(
                    [
                        p.parent,
                        *p.parents,
                    ]
                )

            except Exception:
                pass

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = (
            candidate.resolve()
        )

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_candidate = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        corpus_candidate = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "processed"
            /
            "documents_dedup_quality_accepted_v1.jsonl"
        )

        if (
            registry_candidate.exists()
            and
            corpus_candidate.exists()
        ):
            return candidate

    return None


def cell109_json_blob(value):

    try:
        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
        ).lower()

    except Exception:
        return str(
            value
        ).lower()


def cell109_is_direct_author_website(
    record,
):

    if (
        str(
            record.get(
                "source_type",
                ""
            )
        ).lower()
        !=
        "website"
    ):
        return False

    authority_blob = (
        cell109_json_blob({
            "attribution":
                record.get(
                    "attribution"
                ),

            "content_origin":
                record.get(
                    "content_origin"
                ),

            "source_metadata":
                record.get(
                    "source_metadata"
                ),
        })
    )

    return (
        "primary_author"
        in authority_blob
        or
        "direct_author"
        in authority_blob
    )


def cell109_get_title(record):

    if record.get(
        "title"
    ):
        return str(
            record[
                "title"
            ]
        )

    metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(
        metadata,
        dict,
    ):

        for key in [
            "title",
            "article_title",
            "page_title",
        ]:

            if metadata.get(
                key
            ):
                return str(
                    metadata[
                        key
                    ]
                )

    return "NO_TITLE"


def cell109_get_url(record):

    candidates = []

    locator = (
        record.get(
            "locator"
        )
        or
        {}
    )

    metadata = (
        record.get(
            "source_metadata"
        )
        or
        {}
    )

    if isinstance(
        locator,
        dict,
    ):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if locator.get(
                key
            ):
                candidates.append(
                    locator[
                        key
                    ]
                )

    if isinstance(
        metadata,
        dict,
    ):

        for key in [
            "canonical_url",
            "url",
            "source_url",
        ]:

            if metadata.get(
                key
            ):
                candidates.append(
                    metadata[
                        key
                    ]
                )

    return (
        candidates[0]
        if candidates
        else None
    )


# ============================================================
# 4) Resolve Registry + frozen Corpus
# ============================================================

cell109_repo_root = (
    cell109_find_repo_root()
)


cell109_check(
    "REPO_ROOT_NOT_FOUND",
    cell109_repo_root
    is not None,
)


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


cell109_registry_path = (
    cell109_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell109_corpus_path = (
    cell109_repo_root
    /
    "data"
    /
    "phase2"
    /
    "processed"
    /
    "documents_dedup_quality_accepted_v1.jsonl"
)


cell109_check(
    "REGISTRY_NOT_FOUND",
    cell109_registry_path.exists(),
)


cell109_check(
    "CORPUS_NOT_FOUND",
    cell109_corpus_path.exists(),
)


cell109_corpus_sha256 = (
    cell109_sha256_file(
        cell109_corpus_path
    )
)


cell109_check(
    "FROZEN_CORPUS_SHA256_MISMATCH",
    cell109_corpus_sha256
    ==
    CELL109_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL109_EXPECTED_CORPUS_SHA256,

        "actual":
            cell109_corpus_sha256,
    },
)


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


# ============================================================
# 5) Load only the exact required Corpus documents
# ============================================================

cell109_required_document_ids = sorted({
    source[
        "document_id"
    ]

    for decision
    in CELL109_DECISIONS.values()

    for source
    in decision[
        "sources"
    ]
})


cell109_source_documents = {}


with cell109_corpus_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(
            line
        )

        document_id = (
            record.get(
                "document_id"
            )
        )

        if (
            document_id
            in
            cell109_required_document_ids
        ):

            cell109_source_documents[
                document_id
            ] = record


cell109_check(
    "REQUIRED_SOURCE_DOCUMENTS_MISSING",
    sorted(
        cell109_source_documents.keys()
    )
    ==
    cell109_required_document_ids,
    {
        "required":
            cell109_required_document_ids,

        "found":
            sorted(
                cell109_source_documents.keys()
            ),
    },
)


for document_id, record in (
    cell109_source_documents.items()
):

    cell109_check(
        (
            "SOURCE_NOT_DIRECT_AUTHOR_WEBSITE:"
            +
            document_id
        ),
        cell109_is_direct_author_website(
            record
        ),
    )


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


# ============================================================
# 6) Load Registry fresh
# ============================================================

cell109_registry_before_text = (
    cell109_registry_path.read_text(
        encoding="utf-8"
    )
)


cell109_registry = json.loads(
    cell109_registry_before_text
)


cell109_registry_before_sha256 = (
    cell109_sha256_text(
        cell109_registry_before_text
    )
)


cell109_completed_cases = (
    cell109_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell109_author_review = (
    cell109_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell109_author_cases = (
    cell109_author_review.get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell109_second_review = (
    cell109_registry.get(
        "second_human_review",
        {}
    )
    or
    {}
)


cell109_feedback_batch = (
    cell109_second_review.get(
        CELL109_BATCH_KEY,
        {}
    )
    or
    {}
)


cell109_feedback_cases = (
    cell109_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


# ============================================================
# 7) Preconditions
# ============================================================

cell109_check(
    "COMPLETED_CASE_COUNT_NOT_22",
    len(
        cell109_completed_cases
    )
    ==
    CELL109_EXPECTED_COMPLETED_CASES,
)


cell109_check(
    "AUTHOR_QUEUE_COUNT_NOT_28",
    len(
        cell109_author_cases
    )
    ==
    CELL109_EXPECTED_AUTHOR_QUEUE,
)


cell109_check(
    "FEEDBACK_BATCH_COUNT_NOT_6",
    len(
        cell109_feedback_cases
    )
    ==
    CELL109_EXPECTED_NEW_CASES,
)


cell109_batch_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_feedback_cases
}


cell109_author_new_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_author_cases[22:]
}


cell109_check(
    "BATCH_REFERENCE_NUMBERS_MISMATCH",
    sorted(
        cell109_batch_by_number.keys()
    )
    ==
    CELL109_REFERENCE_NUMBERS,
)


cell109_check(
    "AUTHOR_REFERENCE_NUMBERS_MISMATCH",
    sorted(
        cell109_author_new_by_number.keys()
    )
    ==
    CELL109_REFERENCE_NUMBERS,
)


for number in (
    CELL109_REFERENCE_NUMBERS
):

    batch_id = (
        cell109_batch_by_number[
            number
        ].get(
            "pilot_case_id"
        )
    )

    author_id = (
        cell109_author_new_by_number[
            number
        ].get(
            "pilot_case_id"
        )
    )

    cell109_check(
        (
            "PILOT_ID_COPY_MISMATCH:"
            +
            str(number)
        ),
        batch_id
        ==
        author_id,
    )


    for case_copy in [
        cell109_batch_by_number[
            number
        ],
        cell109_author_new_by_number[
            number
        ],
    ]:

        cell109_check(
            (
                "AUTHOR_REVIEW_ALREADY_STARTED:"
                +
                str(number)
            ),
            (
                case_copy.get(
                    "author_final_review",
                    {}
                )
                or
                {}
            ).get(
                "status"
            )
            ==
            "NOT_STARTED",
        )


if cell109_failures:
    raise AssertionError(
        cell109_failures
    )


# ============================================================
# 8) Protect original 22 cases + raw historical data
# ============================================================

cell109_completed_snapshot = (
    deepcopy(
        cell109_completed_cases
    )
)


cell109_original_22_author_snapshot = (
    deepcopy(
        cell109_author_cases[:22]
    )
)


cell109_raw_history_before = {}


for number in (
    CELL109_REFERENCE_NUMBERS
):

    cell109_raw_history_before[
        number
    ] = {
        "batch":
            deepcopy(
                cell109_batch_by_number[
                    number
                ].get(
                    "historical_phase1_demo"
                )
            ),

        "author":
            deepcopy(
                cell109_author_new_by_number[
                    number
                ].get(
                    "historical_phase1_demo"
                )
            ),
    }


# ============================================================
# 9) Build verified source objects
# ============================================================

def cell109_verified_source(
    source_config,
):

    document_id = (
        source_config[
            "document_id"
        ]
    )

    record = (
        cell109_source_documents[
            document_id
        ]
    )

    text = (
        record.get(
            "text"
        )
        or
        ""
    )

    return {
        "document_id":
            document_id,

        "source_id":
            record.get(
                "source_id"
            ),

        "source_type":
            "website",

        "authority_class":
            "PRIMARY_DIRECT_AUTHOR",

        "title":
            cell109_get_title(
                record
            ),

        "url":
            cell109_get_url(
                record
            ),

        "role":
            source_config[
                "role"
            ],

        "coverage":
            source_config[
                "coverage"
            ],

        "review_note":
            source_config[
                "review_note"
            ],

        "document_text_sha256":
            cell109_sha256_text(
                text
            ),
    }


# ============================================================
# 10) Apply review to one case copy
# ============================================================

def cell109_apply_review(
    case,
    number,
):

    decision_config = (
        CELL109_DECISIONS[
            number
        ]
    )

    pilot_id = case[
        "pilot_case_id"
    ]


    # --------------------------------------------------------
    # Stable review event
    # --------------------------------------------------------

    event_id = (
        cell109_stable_id(
            "second-source-review",
            {
                "pilot_case_id":
                    pilot_id,

                "review_version":
                    CELL109_REVIEW_VERSION,

                "decision":
                    decision_config[
                        "decision"
                    ],

                "revision_scope":
                    decision_config[
                        "revision_scope"
                    ],

                "sources":
                    [
                        source[
                            "document_id"
                        ]

                        for source
                        in decision_config[
                            "sources"
                        ]
                    ],
            },
        )
    )


    history = case.setdefault(
        "source_review_history",
        [],
    )


    existing_event = None


    for event in history:

        if (
            isinstance(
                event,
                dict,
            )
            and
            event.get(
                "review_event_id"
            )
            ==
            event_id
        ):

            existing_event = event
            break


    if existing_event is not None:

        review_timestamp = (
            existing_event[
                "review_timestamp_utc"
            ]
        )

    else:

        review_timestamp = (
            cell109_now()
        )


    # --------------------------------------------------------
    # Preserve superseded Pilot 25 state BEFORE revision
    # --------------------------------------------------------

    superseded_snapshot = None


    if number == 25:

        previous_question = (
            (
                case.get(
                    "derived_question",
                    {}
                )
                or
                {}
            ).get(
                "question_text"
            )
        )

        previous_behavior = (
            (
                case.get(
                    "expected_behavior_review",
                    {}
                )
                or
                {}
            ).get(
                "expected_behavior"
            )
        )

        superseded_snapshot = {
            "previous_pilot_type":
                case.get(
                    "pilot_type"
                ),

            "previous_golden_content_case":
                case.get(
                    "golden_content_case"
                ),

            "previous_question":
                previous_question,

            "previous_expected_behavior":
                previous_behavior,

            "previous_regression_role":
                case.get(
                    "regression_role"
                ),

            "superseded_reason":
                (
                    "The historical demo turns did not provide "
                    "reliable connected-chat context. The old "
                    "continuity interpretation is preserved only "
                    "as history, while the active Pilot is revised "
                    "to a source-grounded content case."
                ),
        }


        new_question = (
            decision_config[
                "new_question"
            ]
        )

        new_answer = (
            decision_config[
                "new_expected_answer"
            ]
        )


        new_question_id = (
            cell109_stable_id(
                "question-draft",
                {
                    "pilot_case_id":
                        pilot_id,

                    "question":
                        new_question,

                    "review_version":
                        CELL109_REVIEW_VERSION,
                },
            )
        )


        new_answer_id = (
            cell109_stable_id(
                "expected-answer-draft",
                {
                    "pilot_case_id":
                        pilot_id,

                    "question_draft_id":
                        new_question_id,

                    "answer":
                        new_answer,

                    "review_version":
                        CELL109_REVIEW_VERSION,
                },
            )
        )


        case[
            "pilot_type"
        ] = (
            decision_config[
                "new_pilot_type"
            ]
        )


        case[
            "golden_content_case"
        ] = True


        case[
            "derived_question"
        ] = {
            "question_draft_id":
                new_question_id,

            "question_text":
                new_question,

            "review_origin":
                "SECOND_HUMAN_SOURCE_REVIEW",

            "human_decision":
                "REVISE",

            "external_knowledge_used":
                False,
        }


        case[
            "expected_answer_review"
        ] = {
            "expected_answer_draft_id":
                new_answer_id,

            "expected_answer_text":
                new_answer,

            "human_decision":
                "REVISE",

            "grounding_source":
                "PHASE2_ACCEPTED_DIRECT_AUTHOR_CORPUS",

            "external_knowledge_used":
                False,
        }


        # Do not delete old behavior evidence.
        old_behavior = (
            case.get(
                "expected_behavior_review"
            )
        )

        if isinstance(
            old_behavior,
            dict,
        ):

            old_behavior[
                "status"
            ] = (
                "SUPERSEDED_BY_SOURCE_GROUNDED_CONTENT_REVISION"
            )


        if (
            "regression_role"
            in case
        ):

            case[
                "regression_role_status"
            ] = (
                "SUPERSEDED"
            )


    # --------------------------------------------------------
    # Verified Corpus sources
    # --------------------------------------------------------

    verified_sources = [
        cell109_verified_source(
            source
        )

        for source
        in decision_config[
            "sources"
        ]
    ]


    source_mapping_id = (
        cell109_stable_id(
            "source-mapping",
            {
                "pilot_case_id":
                    pilot_id,

                "review_version":
                    CELL109_REVIEW_VERSION,

                "document_ids":
                    [
                        source[
                            "document_id"
                        ]

                        for source
                        in verified_sources
                    ],

                "grounding_mode":
                    decision_config[
                        "grounding_mode"
                    ],
            },
        )
    )


    review_event = {
        "review_event_id":
            event_id,

        "review_version":
            CELL109_REVIEW_VERSION,

        "review_timestamp_utc":
            review_timestamp,

        "decision":
            decision_config[
                "decision"
            ],

        "revision_scope":
            decision_config[
                "revision_scope"
            ],

        "grounding_mode":
            decision_config[
                "grounding_mode"
            ],

        "source_mapping_id":
            source_mapping_id,

        "verified_corpus_sources":
            deepcopy(
                verified_sources
            ),

        "direct_author_feedback_preserved":
            True,

        "external_knowledge_used":
            False,

        "author_final_review_recorded":
            False,
    }


    if (
        "grounding_constraint"
        in decision_config
    ):

        review_event[
            "grounding_constraint"
        ] = (
            decision_config[
                "grounding_constraint"
            ]
        )


    if superseded_snapshot is not None:

        review_event[
            "superseded_state"
        ] = (
            superseded_snapshot
        )


    # --------------------------------------------------------
    # Idempotent history append / exact-match validation
    # --------------------------------------------------------

    if existing_event is None:

        history.append(
            deepcopy(
                review_event
            )
        )

    else:

        cell109_check(
            (
                "EXISTING_REVIEW_EVENT_MISMATCH:"
                +
                pilot_id
            ),
            existing_event
            ==
            review_event,
        )


    # --------------------------------------------------------
    # Current Source Review state
    # --------------------------------------------------------

    case[
        "second_source_review"
    ] = {
        "status":
            "COMPLETE",

        "review_event_id":
            event_id,

        "review_version":
            CELL109_REVIEW_VERSION,

        "decision":
            decision_config[
                "decision"
            ],

        "revision_scope":
            decision_config[
                "revision_scope"
            ],

        "grounding_mode":
            decision_config[
                "grounding_mode"
            ],

        "source_mapping_id":
            source_mapping_id,

        "verified_corpus_sources":
            deepcopy(
                verified_sources
            ),

        "human_source_decision":
            decision_config[
                "decision"
            ],

        "review_timestamp_utc":
            review_timestamp,

        "direct_author_feedback_preserved":
            True,

        "author_final_review_status":
            "NOT_STARTED",
    }


    if (
        "grounding_constraint"
        in decision_config
    ):

        case[
            "second_source_review"
        ][
            "grounding_constraint"
        ] = (
            decision_config[
                "grounding_constraint"
            ]
        )


    # --------------------------------------------------------
    # Rebuild citation review WITHOUT deleting Author Feedback
    # --------------------------------------------------------

    old_citation_entries = (
        (
            case.get(
                "citation_review",
                {}
            )
            or
            {}
        ).get(
            "citation_entries",
            []
        )
        or
        []
    )


    direct_feedback_entries = [
        deepcopy(
            entry
        )

        for entry
        in old_citation_entries

        if (
            isinstance(
                entry,
                dict,
            )
            and
            entry.get(
                "source_type"
            )
            ==
            "DIRECT_AUTHOR_FEEDBACK"
        )
    ]


    if not direct_feedback_entries:

        historical = (
            case.get(
                "historical_phase1_demo",
                {}
            )
            or
            {}
        )

        raw_feedback = (
            historical.get(
                "raw_author_feedback"
            )
            or
            ""
        )

        feedback_source = (
            case.get(
                "feedback_source",
                {}
            )
            or
            {}
        )

        direct_feedback_entries = [
            {
                "source_type":
                    "DIRECT_AUTHOR_FEEDBACK",

                "feedback_source_id":
                    feedback_source.get(
                        "feedback_source_id"
                    ),

                "raw_author_feedback_sha256":
                    cell109_sha256_text(
                        raw_feedback
                    ),

                "authority_class":
                    "PRIMARY_DIRECT_AUTHOR_FEEDBACK",

                "role":
                    "PRIMARY_AUTHOR_EVIDENCE",
            },
        ]


    website_entries = []


    for source in verified_sources:

        website_entries.append({
            "source_type":
                "WEBSITE_DIRECT_AUTHOR",

            "document_id":
                source[
                    "document_id"
                ],

            "source_id":
                source[
                    "source_id"
                ],

            "title":
                source[
                    "title"
                ],

            "url":
                source[
                    "url"
                ],

            "authority_class":
                source[
                    "authority_class"
                ],

            "role":
                source[
                    "role"
                ],

            "coverage":
                source[
                    "coverage"
                ],

            "document_text_sha256":
                source[
                    "document_text_sha256"
                ],
        })


    citation_payload = {
        "pilot_case_id":
            pilot_id,

        "review_version":
            CELL109_REVIEW_VERSION,

        "direct_feedback":
            direct_feedback_entries,

        "website_sources":
            website_entries,
    }


    citation_mapping_id = (
        cell109_stable_id(
            "citation-mapping",
            citation_payload,
        )
    )


    citation_entries = []


    for entry in (
        direct_feedback_entries
        +
        website_entries
    ):

        entry = deepcopy(
            entry
        )

        entry[
            "citation_index"
        ] = len(
            citation_entries
        )

        citation_entries.append(
            entry
        )


    case[
        "citation_review"
    ] = {
        "citation_mapping_id":
            citation_mapping_id,

        "human_decision":
            "ACCEPT",

        "citation_entries":
            citation_entries,

        "grounding_mode":
            decision_config[
                "grounding_mode"
            ],

        "source_review_event_id":
            event_id,
    }


    # --------------------------------------------------------
    # Keep Author Review untouched
    # --------------------------------------------------------

    author_final_review = (
        case.setdefault(
            "author_final_review",
            {},
        )
    )


    author_final_review.update({
        "status":
            "NOT_STARTED",

        "reviewer":
            "AUTHOR",

        "decision":
            None,

        "notes":
            None,
    })


    # Promotion remains valid; source review is now completed.
    promotion = (
        case.setdefault(
            "promotion",
            {},
        )
    )

    promotion[
        "source_review_status"
    ] = (
        "COMPLETE"
    )


    return case


# ============================================================
# 11) Apply to BOTH copies
# ============================================================

for number in (
    CELL109_REFERENCE_NUMBERS
):

    cell109_apply_review(
        cell109_batch_by_number[
            number
        ],
        number,
    )


    cell109_apply_review(
        cell109_author_new_by_number[
            number
        ],
        number,
    )


if cell109_failures:

    print(
        cell109_failures
    )

    raise AssertionError(
        "Cell 109 source-review application failed."
    )


# ============================================================
# 12) Sync Batch-level status
# ============================================================

cell109_feedback_batch.update({
    "source_review_status":
        "COMPLETE",

    "source_review_version":
        CELL109_REVIEW_VERSION,

    "source_review_case_count":
        6,

    "source_review_decisions": {
        "ACCEPT":
            3,

        "REVISE":
            3,

        "REJECT":
            0,

        "RETARGET":
            0,
    },

    "content_pilot_count_after_source_review":
        6,

    "continuity_regression_pilot_count_after_source_review":
        0,

    "author_final_review_status":
        "NOT_STARTED",
})


# ============================================================
# 13) Strong in-memory validation
# ============================================================

cell109_check(
    "ORIGINAL_COMPLETED_22_MUTATED",
    cell109_registry.get(
        "completed_human_review_cases",
        []
    )
    ==
    cell109_completed_snapshot,
)


cell109_check(
    "ORIGINAL_AUTHOR_22_MUTATED",
    cell109_author_cases[:22]
    ==
    cell109_original_22_author_snapshot,
)


for number in (
    CELL109_REFERENCE_NUMBERS
):

    batch_case = (
        cell109_batch_by_number[
            number
        ]
    )

    author_case = (
        cell109_author_new_by_number[
            number
        ]
    )


    # Raw historical Phase-1 material must be untouched.
    cell109_check(
        (
            "RAW_BATCH_HISTORY_MUTATED:"
            +
            str(number)
        ),
        batch_case.get(
            "historical_phase1_demo"
        )
        ==
        cell109_raw_history_before[
            number
        ][
            "batch"
        ],
    )


    cell109_check(
        (
            "RAW_AUTHOR_HISTORY_MUTATED:"
            +
            str(number)
        ),
        author_case.get(
            "historical_phase1_demo"
        )
        ==
        cell109_raw_history_before[
            number
        ][
            "author"
        ],
    )


    # Both copies must agree on active content fields.
    for field_name in [
        "pilot_type",
        "golden_content_case",
        "derived_question",
        "expected_answer_review",
        "second_source_review",
        "citation_review",
    ]:

        cell109_check(
            (
                "COPY_SYNC_MISMATCH:"
                +
                str(number)
                +
                ":"
                +
                field_name
            ),
            batch_case.get(
                field_name
            )
            ==
            author_case.get(
                field_name
            ),
        )


    cell109_check(
        (
            "SOURCE_REVIEW_NOT_COMPLETE:"
            +
            str(number)
        ),
        (
            batch_case.get(
                "second_source_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "COMPLETE",
    )


    expected_decision = (
        CELL109_DECISIONS[
            number
        ][
            "decision"
        ]
    )


    cell109_check(
        (
            "SOURCE_DECISION_MISMATCH:"
            +
            str(number)
        ),
        (
            batch_case.get(
                "second_source_review",
                {}
            )
            or
            {}
        ).get(
            "decision"
        )
        ==
        expected_decision,
    )


    cell109_check(
        (
            "AUTHOR_REVIEW_STARTED:"
            +
            str(number)
        ),
        (
            author_case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )


# Pilot 25 special validation.
cell109_case25 = (
    cell109_batch_by_number[
        25
    ]
)


cell109_check(
    "PILOT25_NOT_CONVERTED_TO_CONTENT",
    cell109_case25.get(
        "pilot_type"
    )
    ==
    "AUTHOR_FEEDBACK_CONTENT",
)


cell109_check(
    "PILOT25_GOLDEN_CONTENT_FALSE",
    cell109_case25.get(
        "golden_content_case"
    )
    is True,
)


cell109_check(
    "PILOT25_NEW_QUESTION_MISMATCH",
    (
        cell109_case25.get(
            "derived_question",
            {}
        )
        or
        {}
    ).get(
        "question_text"
    )
    ==
    CELL109_DECISIONS[
        25
    ][
        "new_question"
    ],
)


cell109_check(
    "PILOT25_NEW_ANSWER_MISMATCH",
    (
        cell109_case25.get(
            "expected_answer_review",
            {}
        )
        or
        {}
    ).get(
        "expected_answer_text"
    )
    ==
    CELL109_DECISIONS[
        25
    ][
        "new_expected_answer"
    ],
)


cell109_check(
    "PILOT25_OLD_BEHAVIOR_HISTORY_NOT_PRESERVED",
    bool(
        (
            cell109_case25.get(
                "expected_behavior_review",
                {}
            )
            or
            {}
        ).get(
            "expected_behavior"
        )
    ),
)


if cell109_failures:

    print(
        "\n=== IN-MEMORY VALIDATION FAILURES ==="
    )

    print(
        cell109_failures
    )

    raise AssertionError(
        "Cell 109 failed before Registry write."
    )


# ============================================================
# 14) Serialize + backup + write
# ============================================================

cell109_registry_after_text = (
    json.dumps(
        cell109_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell109_change_required = (
    cell109_registry_after_text
    !=
    cell109_registry_before_text
)


cell109_backup_path = None


if cell109_change_required:

    timestamp = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )


    cell109_backup_path = (
        cell109_registry_path.parent
        /
        (
            cell109_registry_path.stem
            +
            ".before_pilots23_28_source_review_"
            +
            timestamp
            +
            cell109_registry_path.suffix
        )
    )


    cell109_backup_path.write_text(
        cell109_registry_before_text,
        encoding="utf-8",
    )


    temp_path = (
        cell109_registry_path.parent
        /
        (
            cell109_registry_path.name
            +
            ".cell109.tmp"
        )
    )


    temp_path.write_text(
        cell109_registry_after_text,
        encoding="utf-8",
    )


    # Parse temp before replacement.
    json.loads(
        temp_path.read_text(
            encoding="utf-8"
        )
    )


    temp_path.replace(
        cell109_registry_path
    )


# ============================================================
# 15) Reload exact persisted Registry
# ============================================================

cell109_persisted_text = (
    cell109_registry_path.read_text(
        encoding="utf-8"
    )
)


cell109_persisted_registry = (
    json.loads(
        cell109_persisted_text
    )
)


cell109_registry_after_sha256 = (
    cell109_sha256_text(
        cell109_persisted_text
    )
)


cell109_persisted_completed = (
    cell109_persisted_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell109_persisted_author_cases = (
    (
        cell109_persisted_registry.get(
            "author_review",
            {}
        )
        or
        {}
    ).get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell109_persisted_batch_cases = (
    (
        (
            cell109_persisted_registry.get(
                "second_human_review",
                {}
            )
            or
            {}
        ).get(
            CELL109_BATCH_KEY,
            {}
        )
        or
        {}
    ).get(
        "cases",
        []
    )
    or
    []
)


# ============================================================
# 16) Persisted final validation
# ============================================================

cell109_check(
    "PERSISTED_COMPLETED_COUNT_CHANGED",
    len(
        cell109_persisted_completed
    )
    ==
    22,
)


cell109_check(
    "PERSISTED_AUTHOR_QUEUE_NOT_28",
    len(
        cell109_persisted_author_cases
    )
    ==
    28,
)


cell109_check(
    "PERSISTED_BATCH_NOT_6",
    len(
        cell109_persisted_batch_cases
    )
    ==
    6,
)


cell109_persisted_batch_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_persisted_batch_cases
}


cell109_persisted_author_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell109_persisted_author_cases[22:]
}


for number in (
    CELL109_REFERENCE_NUMBERS
):

    batch_case = (
        cell109_persisted_batch_by_number[
            number
        ]
    )

    author_case = (
        cell109_persisted_author_by_number[
            number
        ]
    )


    cell109_check(
        (
            "PERSISTED_COPY_SOURCE_REVIEW_MISMATCH:"
            +
            str(number)
        ),
        batch_case.get(
            "second_source_review"
        )
        ==
        author_case.get(
            "second_source_review"
        ),
    )


    cell109_check(
        (
            "PERSISTED_COPY_QUESTION_MISMATCH:"
            +
            str(number)
        ),
        batch_case.get(
            "derived_question"
        )
        ==
        author_case.get(
            "derived_question"
        ),
    )


    cell109_check(
        (
            "PERSISTED_AUTHOR_NOT_NOT_STARTED:"
            +
            str(number)
        ),
        (
            author_case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )


# Final Pilot 25 persisted state.
cell109_persisted_case25 = (
    cell109_persisted_batch_by_number[
        25
    ]
)


cell109_check(
    "PERSISTED_PILOT25_TYPE_WRONG",
    cell109_persisted_case25.get(
        "pilot_type"
    )
    ==
    "AUTHOR_FEEDBACK_CONTENT",
)


cell109_check(
    "PERSISTED_PILOT25_ANSWER_MISSING",
    bool(
        (
            cell109_persisted_case25.get(
                "expected_answer_review",
                {}
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
    ),
)


# ============================================================
# 17) Final Gate / Human-readable summary
# ============================================================

cell109_gate = (
    len(
        cell109_failures
    )
    ==
    0
)


print(
    "=== CELL 109 — APPLY SOURCE REVIEW: PILOTS 23-28 ==="
)


print(
    "\nFrozen Corpus SHA256:",
    cell109_corpus_sha256,
)


print(
    "Verified Corpus Source Documents:",
    len(
        cell109_source_documents
    ),
)


print(
    "\n=== HUMAN SOURCE REVIEW DECISIONS ==="
)


for number in (
    CELL109_REFERENCE_NUMBERS
):

    case = (
        cell109_persisted_batch_by_number[
            number
        ]
    )

    review = (
        case.get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Pilot:",
        number,
    )

    print(
        "Pilot ID:",
        case.get(
            "pilot_case_id"
        ),
    )

    print(
        "Decision:",
        review.get(
            "decision"
        ),
    )

    print(
        "Revision scope:",
        review.get(
            "revision_scope"
        ),
    )

    print(
        "Pilot type:",
        case.get(
            "pilot_type"
        ),
    )

    print(
        "Golden content case:",
        case.get(
            "golden_content_case"
        ),
    )

    print(
        "Question:"
    )

    print(
        (
            case.get(
                "derived_question",
                {}
            )
            or
            {}
        ).get(
            "question_text"
        )
    )

    if case.get(
        "golden_content_case"
    ):

        print(
            "Expected Answer:"
        )

        print(
            (
                case.get(
                    "expected_answer_review",
                    {}
                )
                or
                {}
            ).get(
                "expected_answer_text"
            )
        )


    print(
        "Grounding mode:",
        review.get(
            "grounding_mode"
        ),
    )


    print(
        "Verified Corpus Sources:"
    )

    for source in (
        review.get(
            "verified_corpus_sources",
            []
        )
    ):

        print(
            " -",
            source.get(
                "document_id"
            ),
            "|",
            source.get(
                "title"
            ),
            "|",
            source.get(
                "coverage"
            ),
        )


    print(
        "Author Final Review:",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        ),
    )


print(
    "\n=== EXPECTED DECISION COUNTS ==="
)


print(
    "ACCEPT:",
    sum(
        1

        for number
        in CELL109_REFERENCE_NUMBERS

        if (
            CELL109_DECISIONS[
                number
            ][
                "decision"
            ]
            ==
            "ACCEPT"
        )
    ),
)


print(
    "REVISE:",
    sum(
        1

        for number
        in CELL109_REFERENCE_NUMBERS

        if (
            CELL109_DECISIONS[
                number
            ][
                "decision"
            ]
            ==
            "REVISE"
        )
    ),
)


print(
    "REJECT:",
    0,
)


print(
    "RETARGET:",
    0,
)


print(
    "\n=== POST-REVIEW ACTIVE PILOT TYPES ==="
)


print(
    "AUTHOR_FEEDBACK_CONTENT:",
    sum(
        1

        for case
        in cell109_persisted_batch_cases

        if (
            case.get(
                "pilot_type"
            )
            ==
            "AUTHOR_FEEDBACK_CONTENT"
        )
    ),
)


print(
    "MULTI_TURN_CONTEXT_CONTINUITY:",
    sum(
        1

        for case
        in cell109_persisted_batch_cases

        if (
            case.get(
                "pilot_type"
            )
            ==
            "MULTI_TURN_CONTEXT_CONTINUITY"
        )
    ),
)


print(
    "\n=== PROTECTION / SAFETY ==="
)


print(
    "Original completed 22 cases changed:",
    False,
)


print(
    "Original first 22 Author Review cases changed:",
    False,
)


print(
    "Raw Phase-1 conversations changed:",
    False,
)


print(
    "Raw Dr. Roudi feedback changed:",
    False,
)


print(
    "Pilot 25 old continuity history preserved:",
    True,
)


print(
    "Author decisions recorded:",
    False,
)


print(
    "Author Review queue count:",
    len(
        cell109_persisted_author_cases
    ),
)


print(
    "Markdown changed:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== WRITE ==="
)


print(
    "Registry write required:",
    cell109_change_required,
)


print(
    "Backup:",
    (
        str(
            cell109_backup_path
        )
        if cell109_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_APPLIED"
    ),
)


print(
    "Registry SHA256 before:",
    cell109_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell109_registry_after_sha256,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell109_failures
)


print(
    "\nCODE CELL 109 GATE:",
    (
        "PASS"
        if cell109_gate
        else
        "FAIL"
    ),
)


assert cell109_gate, (
    "Pilots 23-28 Source Review application failed. "
    "Repair this SAME Code Cell 109 before updating the "
    "28-case Author Review Markdown."
)

=== CELL 109 — APPLY SOURCE REVIEW: PILOTS 23-28 ===

Frozen Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Verified Corpus Source Documents: 9

=== HUMAN SOURCE REVIEW DECISIONS ===

Pilot: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Decision: ACCEPT
Revision scope: NONE
Pilot type: AUTHOR_FEEDBACK_CONTENT
Golden content case: True
Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟
Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.
Grounding mode: DIRECT_AUTHOR_FEEDBACK_PLUS_CORPUS_DIRECT_SUPPORT
Verified Corpus Sources:
 - website:7C71884B | ابزارهای سواد مالی برای بهبود مدیریت هزینه | DIRECT_EXPLICIT
Author Final Review: NOT_STARTED

Pilot: 24
Pilot ID: question-pilot:70628caeefebbfba48300176fafddc96
Decision: ACCEPT
Revision scope: NONE
Pilot type: AUTHOR_FEEDBACK_CONTENT
Golden content case: True
Question:
برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید 

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## ارتقای پایلوت‌های 23 تا 28 به صف بازبینی مؤلف

در مرحله قبل، ۶ پایلوت جدید بر اساس بازخورد واقعی دکتر کمیل رودی در دموی فاز یک ساخته و در یک Batch مستقل ثبت شدند.

وضعیت قبل از این مرحله:

- پایلوت‌های 1 تا 22: آماده بازبینی مؤلف
- پایلوت‌های 23 تا 28: ساخته‌شده و آماده Promotion
- تعداد کل فعلی در صف Author Review: 22
- Author Final Review: هنوز شروع نشده است

در این مرحله، ۶ پایلوت جدید بدون تغییر ۲۲ مورد قبلی به انتهای صف بازبینی مؤلف اضافه می‌شوند.

ترتیب جدید:

- موارد 1 تا 22: پایلوت‌های قبلی فاز دو
- مورد 23: تعریف سواد مالی
- مورد 24: ارزیابی دانش، نگرش و مهارت مالی
- مورد 25: تداوم Context در سؤال پیگیری
- مورد 26: الگوهای خرید و هدررفت پول
- مورد 27: روش‌های افزایش درآمد
- مورد 28: احساس فقر با وجود درآمد

پس از اجرای موفق Cell بعدی:

**Author Review Queue = 28 Cases**

توجه:

- ۲۲ مورد قبلی نباید تغییر کنند.
- `completed_human_review_cases` همچنان ۲۲ مورد باقی می‌ماند.
- ۶ مورد جدید از Batch بازخورد فاز یک وارد صف بازبینی دکتر می‌شوند.
- هیچ تصمیمی از طرف دکتر ثبت نمی‌شود.
- `Author Final Review = NOT_STARTED`
- Raw Feedback دکتر تغییر نمی‌کند.
- هیچ Commit یا Push انجام نمی‌شود.

</div>

In [ ]:
# Code Cell 110
# ============================================================
# PROMOTE PILOTS 23-28 TO THE AUTHOR REVIEW QUEUE
#
# Self-contained:
# - reads Registry directly from disk;
# - does NOT depend on Cell 107 variables in memory;
# - preserves the existing 22 Author Review cases byte-for-byte
#   at the object level;
# - appends the six Phase-1 Author Feedback derived Pilots;
# - updates Author Review queue count from 22 to 28;
# - updates promotion metadata for the six new Pilots;
# - preserves completed_human_review_cases at 22;
# - keeps all Author decisions NOT_STARTED;
# - creates backup before write;
# - reloads and verifies persisted state;
# - is idempotent.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Constants
# ============================================================

CELL110_EXPECTED_OLD_AUTHOR_COUNT = 22
CELL110_EXPECTED_NEW_AUTHOR_COUNT = 28
CELL110_EXPECTED_NEW_PILOTS = 6

CELL110_BATCH_KEY = (
    "phase1_author_feedback_derived_pilots"
)

CELL110_EXPECTED_REFERENCE_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]


# ============================================================
# 2) Utilities
# ============================================================

cell110_failures = []


def cell110_check(
    name,
    condition,
    details=None,
):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell110_failures.append(
            failure
        )

    return bool(condition)


def cell110_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


def cell110_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell110_stable_id(
    prefix,
    payload,
):

    if not isinstance(
        payload,
        str,
    ):

        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(",", ":"),
        )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell110_find_repo_root():

    candidates = []

    # Reuse known path only as a convenience if available.
    if "cell105_registry_path" in globals():

        known_registry = Path(
            cell105_registry_path
        ).resolve()

        candidates.extend(
            [
                known_registry.parent,
                *known_registry.parents,
            ]
        )

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_candidate = (
            candidate
            /
            "data"
            /
            "phase2"
            /
            "human_review"
            /
            "human_review_registry.json"
        )

        if registry_candidate.exists():
            return candidate

    return None


# ============================================================
# 3) Resolve Registry independently
# ============================================================

cell110_repo_root = (
    cell110_find_repo_root()
)


cell110_check(
    "REPO_ROOT_NOT_FOUND",
    cell110_repo_root is not None,
)


if cell110_failures:
    raise AssertionError(
        cell110_failures
    )


cell110_registry_path = (
    cell110_repo_root
    /
    "data"
    /
    "phase2"
    /
    "human_review"
    /
    "human_review_registry.json"
)


cell110_check(
    "REGISTRY_NOT_FOUND",
    cell110_registry_path.exists(),
    str(cell110_registry_path),
)


if cell110_failures:
    raise AssertionError(
        cell110_failures
    )


# ============================================================
# 4) Load Registry fresh from disk
# ============================================================

cell110_registry_before_text = (
    cell110_registry_path.read_text(
        encoding="utf-8"
    )
)


cell110_registry = json.loads(
    cell110_registry_before_text
)


cell110_registry_before_sha256 = (
    cell110_sha256_text(
        cell110_registry_before_text
    )
)


# ============================================================
# 5) Resolve protected existing collections
# ============================================================

cell110_completed_cases = (
    cell110_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell110_author_review = (
    cell110_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell110_author_cases = (
    cell110_author_review.get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell110_second_review = (
    cell110_registry.get(
        "second_human_review",
        {}
    )
    or
    {}
)


cell110_feedback_batch = (
    cell110_second_review.get(
        CELL110_BATCH_KEY,
        {}
    )
    or
    {}
)


cell110_feedback_cases = (
    cell110_feedback_batch.get(
        "cases",
        []
    )
    or
    []
)


# ============================================================
# 6) Validate source Batch
# ============================================================

cell110_check(
    "COMPLETED_HUMAN_REVIEW_COUNT_NOT_22",
    len(cell110_completed_cases) == 22,
    len(cell110_completed_cases),
)


cell110_check(
    "PHASE1_FEEDBACK_BATCH_NOT_FOUND",
    bool(cell110_feedback_batch),
)


cell110_check(
    "PHASE1_FEEDBACK_BATCH_CASE_COUNT_NOT_6",
    len(cell110_feedback_cases)
    ==
    CELL110_EXPECTED_NEW_PILOTS,
    len(cell110_feedback_cases),
)


cell110_reference_numbers = [
    case.get(
        "reference_pilot_number"
    )

    for case
    in cell110_feedback_cases
]


cell110_check(
    "PHASE1_FEEDBACK_REFERENCE_NUMBERS_MISMATCH",
    cell110_reference_numbers
    ==
    CELL110_EXPECTED_REFERENCE_NUMBERS,
    cell110_reference_numbers,
)


cell110_feedback_ids = [
    case.get(
        "pilot_case_id"
    )

    for case
    in cell110_feedback_cases
]


cell110_check(
    "PHASE1_FEEDBACK_IDS_NOT_UNIQUE",
    len(
        set(cell110_feedback_ids)
    )
    ==
    6,
)


for case in cell110_feedback_cases:

    pilot_id = case.get(
        "pilot_case_id",
        "UNKNOWN",
    )

    cell110_check(
        f"AUTHOR_REVIEW_ALREADY_STARTED:{pilot_id}",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    promotion_status = (
        case.get(
            "promotion",
            {}
        )
        or
        {}
    ).get(
        "status"
    )

    cell110_check(
        f"UNEXPECTED_PROMOTION_STATUS:{pilot_id}",
        promotion_status
        in
        {
            "READY_FOR_AUTHOR_REVIEW",
            "PROMOTED_TO_AUTHOR_REVIEW",
        },
        promotion_status,
    )


if cell110_failures:

    print(
        "\n=== SOURCE BATCH VALIDATION FAILURES ==="
    )

    print(
        cell110_failures
    )

    raise AssertionError(
        "Cell 110 blocked: source Batch is not valid."
    )


# ============================================================
# 7) Determine whether promotion was already completed
# ============================================================

cell110_existing_author_ids = [
    case.get(
        "pilot_case_id"
    )

    for case
    in cell110_author_cases
]


cell110_new_ids_already_present = [
    pilot_id

    for pilot_id
    in cell110_feedback_ids

    if pilot_id
    in
    cell110_existing_author_ids
]


cell110_already_promoted = (
    len(cell110_author_cases)
    ==
    CELL110_EXPECTED_NEW_AUTHOR_COUNT
    and
    len(cell110_new_ids_already_present)
    ==
    6
)


cell110_fresh_promotion = (
    len(cell110_author_cases)
    ==
    CELL110_EXPECTED_OLD_AUTHOR_COUNT
    and
    len(cell110_new_ids_already_present)
    ==
    0
)


cell110_check(
    "AUTHOR_QUEUE_IS_IN_PARTIAL_OR_UNKNOWN_STATE",
    (
        cell110_fresh_promotion
        or
        cell110_already_promoted
    ),
    {
        "current_author_queue_count":
            len(cell110_author_cases),

        "new_ids_already_present":
            len(
                cell110_new_ids_already_present
            ),
    },
)


if cell110_failures:

    print(
        cell110_failures
    )

    raise AssertionError(
        "Author Review queue is neither clean 22-state nor valid 28-state."
    )


# ============================================================
# 8) Freeze exact original 22 cases
# ============================================================

cell110_original_22_author_cases = deepcopy(
    cell110_author_cases[:22]
)


cell110_completed_cases_snapshot = deepcopy(
    cell110_completed_cases
)


# ============================================================
# 9) Promotion
# ============================================================

cell110_promotion_timestamp = (
    cell110_feedback_batch.get(
        "promotion_timestamp_utc"
    )
    or
    cell110_now()
)


if cell110_fresh_promotion:

    cell110_promoted_cases = []

    for expected_position, source_case in enumerate(
        cell110_feedback_cases,
        start=23,
    ):

        promoted_case = deepcopy(
            source_case
        )

        reference_number = (
            promoted_case.get(
                "reference_pilot_number"
            )
        )

        cell110_check(
            (
                "REFERENCE_NUMBER_POSITION_MISMATCH:"
                +
                str(reference_number)
            ),
            reference_number
            ==
            expected_position,
        )


        promotion = (
            promoted_case.setdefault(
                "promotion",
                {},
            )
        )

        promotion.update({
            "status":
                "PROMOTED_TO_AUTHOR_REVIEW",

            "promoted_to_author_review":
                True,

            "author_review_queue_position":
                expected_position,

            "promotion_timestamp_utc":
                cell110_promotion_timestamp,

            "future_target_total":
                28,
        })


        promoted_case[
            "author_review_queue_position"
        ] = (
            expected_position
        )


        author_final_review = (
            promoted_case.setdefault(
                "author_final_review",
                {},
            )
        )

        author_final_review.update({
            "status":
                "NOT_STARTED",

            "reviewer":
                "AUTHOR",

            "decision":
                None,

            "notes":
                None,
        })


        cell110_promoted_cases.append(
            promoted_case
        )


    if cell110_failures:
        raise AssertionError(
            cell110_failures
        )


    cell110_author_cases.extend(
        cell110_promoted_cases
    )


# ============================================================
# 10) Update source Batch promotion metadata
# ============================================================

for source_case in cell110_feedback_cases:

    reference_number = (
        source_case.get(
            "reference_pilot_number"
        )
    )

    promotion = source_case.setdefault(
        "promotion",
        {},
    )

    promotion.update({
        "status":
            "PROMOTED_TO_AUTHOR_REVIEW",

        "promoted_to_author_review":
            True,

        "author_review_queue_position":
            reference_number,

        "promotion_timestamp_utc":
            cell110_promotion_timestamp,

        "future_target_total":
            28,
    })


cell110_feedback_batch.update({
    "status":
        "COMPLETE_PROMOTED_TO_AUTHOR_REVIEW",

    "currently_promoted_to_author_review":
        True,

    "promotion_timestamp_utc":
        cell110_promotion_timestamp,

    "author_review_queue_count_before_promotion":
        22,

    "author_review_queue_count_after_promotion":
        28,
})


# ============================================================
# 11) Update Author Review queue metadata
# ============================================================

cell110_author_review[
    "cases_ready_for_review"
] = (
    cell110_author_cases
)


cell110_author_review[
    "case_count"
] = (
    len(
        cell110_author_cases
    )
)


# Keep review state untouched / not started.
if (
    "status"
    in
    cell110_author_review
):

    cell110_check(
        "AUTHOR_REVIEW_STATUS_ALREADY_CHANGED",
        cell110_author_review[
            "status"
        ]
        in
        {
            "NOT_STARTED",
            "READY",
            "READY_FOR_REVIEW",
        },
        cell110_author_review[
            "status"
        ],
    )


# ============================================================
# 12) Promotion event record
# ============================================================

cell110_promotion_event_id = (
    cell110_stable_id(
        "author-review-promotion",
        {
            "source_batch":
                CELL110_BATCH_KEY,

            "pilot_ids":
                cell110_feedback_ids,

            "from_count":
                22,

            "to_count":
                28,
        },
    )
)


cell110_promotion_history = (
    cell110_author_review.setdefault(
        "promotion_history",
        [],
    )
)


cell110_existing_event_ids = {
    event.get(
        "promotion_event_id"
    )

    for event
    in cell110_promotion_history

    if isinstance(
        event,
        dict,
    )
}


if (
    cell110_promotion_event_id
    not in
    cell110_existing_event_ids
):

    cell110_promotion_history.append({
        "promotion_event_id":
            cell110_promotion_event_id,

        "promotion_timestamp_utc":
            cell110_promotion_timestamp,

        "source":
            "PHASE1_DEMO_AUTHOR_FEEDBACK",

        "source_batch":
            CELL110_BATCH_KEY,

        "previous_author_review_case_count":
            22,

        "added_case_count":
            6,

        "new_author_review_case_count":
            28,

        "added_reference_pilot_numbers":
            deepcopy(
                CELL110_EXPECTED_REFERENCE_NUMBERS
            ),

        "added_pilot_case_ids":
            deepcopy(
                cell110_feedback_ids
            ),

        "author_decisions_recorded":
            False,
    })


# ============================================================
# 13) Strong in-memory validation
# ============================================================

cell110_check(
    "AUTHOR_QUEUE_COUNT_NOT_28",
    len(
        cell110_author_cases
    )
    ==
    28,
    len(
        cell110_author_cases
    ),
)


cell110_check(
    "AUTHOR_REVIEW_CASE_COUNT_FIELD_NOT_28",
    cell110_author_review.get(
        "case_count"
    )
    ==
    28,
    cell110_author_review.get(
        "case_count"
    ),
)


# First 22 must be exactly unchanged.
cell110_check(
    "ORIGINAL_22_AUTHOR_CASES_MUTATED",
    cell110_author_cases[:22]
    ==
    cell110_original_22_author_cases,
)


# completed_human_review_cases must remain exactly unchanged.
cell110_check(
    "COMPLETED_HUMAN_REVIEW_CASES_MUTATED",
    cell110_registry.get(
        "completed_human_review_cases",
        []
    )
    ==
    cell110_completed_cases_snapshot,
)


cell110_new_author_cases = (
    cell110_author_cases[22:]
)


cell110_check(
    "NEW_AUTHOR_CASE_COUNT_NOT_6",
    len(
        cell110_new_author_cases
    )
    ==
    6,
)


cell110_check(
    "NEW_AUTHOR_REFERENCE_ORDER_MISMATCH",
    [
        case.get(
            "reference_pilot_number"
        )

        for case
        in cell110_new_author_cases
    ]
    ==
    CELL110_EXPECTED_REFERENCE_NUMBERS,
)


cell110_check(
    "NEW_AUTHOR_PILOT_IDS_MISMATCH",
    [
        case.get(
            "pilot_case_id"
        )

        for case
        in cell110_new_author_cases
    ]
    ==
    cell110_feedback_ids,
)


for case in cell110_new_author_cases:

    pilot_id = case.get(
        "pilot_case_id"
    )

    cell110_check(
        f"NEW_CASE_AUTHOR_REVIEW_STARTED:{pilot_id}",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    cell110_check(
        f"NEW_CASE_NOT_MARKED_PROMOTED:{pilot_id}",
        (
            case.get(
                "promotion",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "PROMOTED_TO_AUTHOR_REVIEW",
    )


cell110_check(
    "SOURCE_BATCH_NOT_MARKED_PROMOTED",
    cell110_feedback_batch.get(
        "currently_promoted_to_author_review"
    )
    is True,
)


if cell110_failures:

    print(
        "\n=== IN-MEMORY FAILURES ==="
    )

    print(
        cell110_failures
    )

    raise AssertionError(
        "Cell 110 promotion failed before Registry write."
    )


# ============================================================
# 14) Serialize
# ============================================================

cell110_registry_after_text = (
    json.dumps(
        cell110_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell110_change_required = (
    cell110_registry_after_text
    !=
    cell110_registry_before_text
)


cell110_backup_path = None


# ============================================================
# 15) Backup + atomic-ish write
# ============================================================

if cell110_change_required:

    filename_timestamp = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )


    cell110_backup_path = (
        cell110_registry_path.parent
        /
        (
            cell110_registry_path.stem
            +
            ".before_author_review_28_promotion_"
            +
            filename_timestamp
            +
            cell110_registry_path.suffix
        )
    )


    cell110_backup_path.write_text(
        cell110_registry_before_text,
        encoding="utf-8",
    )


    cell110_temp_path = (
        cell110_registry_path.parent
        /
        (
            cell110_registry_path.name
            +
            ".cell110.tmp"
        )
    )


    cell110_temp_path.write_text(
        cell110_registry_after_text,
        encoding="utf-8",
    )


    # Ensure temp file parses before replacement.
    json.loads(
        cell110_temp_path.read_text(
            encoding="utf-8"
        )
    )


    cell110_temp_path.replace(
        cell110_registry_path
    )


# ============================================================
# 16) Reload persisted Registry
# ============================================================

cell110_persisted_text = (
    cell110_registry_path.read_text(
        encoding="utf-8"
    )
)


cell110_persisted_registry = (
    json.loads(
        cell110_persisted_text
    )
)


cell110_registry_after_sha256 = (
    cell110_sha256_text(
        cell110_persisted_text
    )
)


# ============================================================
# 17) Persisted validation
# ============================================================

cell110_persisted_completed = (
    cell110_persisted_registry.get(
        "completed_human_review_cases",
        []
    )
    or
    []
)


cell110_persisted_author_review = (
    cell110_persisted_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell110_persisted_author_cases = (
    cell110_persisted_author_review.get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell110_persisted_batch = (
    (
        cell110_persisted_registry.get(
            "second_human_review",
            {}
        )
        or
        {}
    ).get(
        CELL110_BATCH_KEY,
        {}
    )
    or
    {}
)


cell110_check(
    "PERSISTED_COMPLETED_CASES_CHANGED",
    cell110_persisted_completed
    ==
    cell110_completed_cases_snapshot,
)


cell110_check(
    "PERSISTED_AUTHOR_QUEUE_COUNT_NOT_28",
    len(
        cell110_persisted_author_cases
    )
    ==
    28,
    len(
        cell110_persisted_author_cases
    ),
)


cell110_check(
    "PERSISTED_AUTHOR_CASE_COUNT_FIELD_NOT_28",
    cell110_persisted_author_review.get(
        "case_count"
    )
    ==
    28,
)


cell110_check(
    "PERSISTED_FIRST_22_CHANGED",
    cell110_persisted_author_cases[:22]
    ==
    cell110_original_22_author_cases,
)


cell110_check(
    "PERSISTED_NEW_SIX_IDS_MISMATCH",
    [
        case.get(
            "pilot_case_id"
        )

        for case
        in
        cell110_persisted_author_cases[22:]
    ]
    ==
    cell110_feedback_ids,
)


cell110_check(
    "PERSISTED_BATCH_NOT_PROMOTED",
    cell110_persisted_batch.get(
        "currently_promoted_to_author_review"
    )
    is True,
)


cell110_check(
    "PERSISTED_BATCH_STATUS_MISMATCH",
    cell110_persisted_batch.get(
        "status"
    )
    ==
    "COMPLETE_PROMOTED_TO_AUTHOR_REVIEW",
)


for case in (
    cell110_persisted_author_cases[22:]
):

    pilot_id = case.get(
        "pilot_case_id"
    )

    cell110_check(
        f"PERSISTED_AUTHOR_REVIEW_STARTED:{pilot_id}",
        (
            case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )


# ============================================================
# 18) Final Gate
# ============================================================

cell110_gate = (
    len(
        cell110_failures
    )
    ==
    0
)


print(
    "=== AUTHOR REVIEW QUEUE PROMOTION: 22 -> 28 ==="
)


print(
    "\nExisting completed Human Review cases:",
    len(
        cell110_persisted_completed
    ),
)


print(
    "Author Review queue:",
    len(
        cell110_persisted_author_cases
    ),
)


print(
    "Newly promoted Pilots:",
    6,
)


print(
    "Reference Pilot range added:",
    "23-28",
)


print(
    "\n=== NEW AUTHOR REVIEW CASES ==="
)


for case in (
    cell110_persisted_author_cases[22:]
):

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Reference Pilot:",
        case.get(
            "reference_pilot_number"
        ),
    )

    print(
        "Pilot ID:",
        case.get(
            "pilot_case_id"
        ),
    )

    print(
        "Type:",
        case.get(
            "pilot_type"
        ),
    )

    print(
        "Question:"
    )

    print(
        (
            case.get(
                "derived_question"
            )
            or
            {}
        ).get(
            "question_text"
        )
    )

    if case.get(
        "golden_content_case"
    ):

        print(
            "Expected Answer:"
        )

        print(
            (
                case.get(
                    "expected_answer_review"
                )
                or
                {}
            ).get(
                "expected_answer_text"
            )
        )

    else:

        print(
            "Expected Behavior:"
        )

        print(
            (
                case.get(
                    "expected_behavior_review"
                )
                or
                {}
            ).get(
                "expected_behavior"
            )
        )

    print(
        "Author Review:",
        (
            case.get(
                "author_final_review"
            )
            or
            {}
        ).get(
            "status"
        ),
    )


print(
    "\n=== PROTECTION ==="
)


print(
    "Original 22 Author Review cases changed:",
    False,
)


print(
    "Completed Human Review cases changed:",
    False,
)


print(
    "Completed Human Review case count:",
    len(
        cell110_persisted_completed
    ),
)


print(
    "Author Review case count:",
    len(
        cell110_persisted_author_cases
    ),
)


print(
    "Author decisions recorded:",
    False,
)


print(
    "\n=== WRITE / SAFETY ==="
)


print(
    "Registry write required:",
    cell110_change_required,
)


print(
    "Backup path:",
    (
        str(
            cell110_backup_path
        )
        if cell110_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_PROMOTED"
    ),
)


print(
    "Registry SHA256 before:",
    cell110_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell110_registry_after_sha256,
)


print(
    "Raw Author Feedback mutated:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell110_failures
)


print(
    "\nCODE CELL 110 GATE:",
    (
        "PASS"
        if cell110_gate
        else
        "FAIL"
    ),
)


assert cell110_gate, (
    "Author Review Queue promotion failed. "
    "Repair this SAME Code Cell 110 before generating the "
    "28-case Author Review document."
)

=== AUTHOR REVIEW QUEUE PROMOTION: 22 -> 28 ===

Existing completed Human Review cases: 22
Author Review queue: 28
Newly promoted Pilots: 6
Reference Pilot range added: 23-28

=== NEW AUTHOR REVIEW CASES ===

Reference Pilot: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497
Type: AUTHOR_FEEDBACK_CONTENT
Question:
سواد مالی درباره میزان پولی است که داریم یا درباره مدیریت پول؟
Expected Answer:
سواد مالی درباره مدیریت پول است، نه درباره میزان پول.
Author Review: NOT_STARTED

Reference Pilot: 24
Pilot ID: question-pilot:70628caeefebbfba48300176fafddc96
Type: AUTHOR_FEEDBACK_CONTENT
Question:
برای اینکه بفهمیم از نظر دانش، نگرش و مهارت‌های مالی وضعیت قابل قبولی داریم، چه باید کرد؟
Expected Answer:
مدیریت مالی شخصی به دانش، نگرش و مهارت‌های خاص نیاز دارد. برای اینکه بدانیم وضعیت مالی ما از این جنبه‌ها به اندازه کافی مناسب است، باید پرسشنامه مربوط را تکمیل کنیم.
Author Review: NOT_STARTED

Reference Pilot: 25
Pilot ID: question-pilot:77d8b06bc6bccdbbfca4faab99051fe0
Type: AUTHOR_F

<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## تولید بسته ۲۸ موردی بازبینی نهایی مؤلف

در این مرحله، نسخه بازبینی مؤلف از ۲۲ مورد به ۲۸ مورد ارتقا داده می‌شود.

### منبع ساخت

- همان فایل Markdown تأییدشده ۲۲ موردی برای موارد 1 تا 22؛
- Registry فعلی برای پایلوت‌های 23 تا 28؛
- تاریخچه واقعی دموی فاز یک و بازخورد مستقیم دکتر برای موارد 23 تا 28؛
- نتایج Source Mapping و Second Human Source Review انجام‌شده برای پایلوت‌های 23 تا 28.

### وضعیت پایلوت‌های جدید پس از Source Review

هر ۶ پایلوت جدید اکنون از نوع:

`AUTHOR_FEEDBACK_CONTENT`

هستند.

تصمیم‌های Human Source Review:

- Pilot 23 → `ACCEPT`
- Pilot 24 → `ACCEPT`
- Pilot 25 → `REVISE`
- Pilot 26 → `REVISE / GROUNDING_ONLY`
- Pilot 27 → `REVISE / GROUNDING_ONLY`
- Pilot 28 → `ACCEPT`

Pilot 25 دیگر یک پایلوت فعال `MULTI_TURN_CONTEXT_CONTINUITY` نیست.

نسخه قبلی آن به‌عنوان History حفظ شده، اما سؤال و پاسخ فعال جدید آن بر اساس Source واقعی Corpus ساخته شده است.

### قواعد این مرحله

- محتوای تأییدشده موارد 1 تا 22 دوباره از صفر تولید نمی‌شود.
- همان Markdown تأییدشده قبلی مبنا قرار می‌گیرد.
- فقط شمارنده کل از 22 به 28 تغییر می‌کند.
- موارد 23 تا 28 به انتهای همان سند اضافه می‌شوند.
- سؤال‌ها، جواب‌های مدل فاز یک و بازخورد قبلی دکتر باید در History قابل مشاهده باشند.
- Feedback خام دکتر بازنویسی نمی‌شود.
- Sourceهای واقعی تأییدشده در Source Review باید برای موارد 23 تا 28 نمایش داده شوند.
- تصمیم Human Source Review شامل `ACCEPT` یا `REVISE` باید در روند بازبینی هر مورد نمایش داده شود.
- Pilot 25 باید با سؤال و پاسخ جدید فعال نمایش داده شود و History نسخه قبلی آن نیز حذف نشود.
- برای Pilots 26 و 27 باید روشن باشد که Direct Author Feedback منبع اصلی Grounding است و Website Corpus فقط Partial Corroboration ارائه می‌دهد.
- هیچ تصمیمی به نام دکتر ثبت نمی‌شود.
- بخش نظر مؤلف برای تمام موارد همچنان خالی باقی می‌ماند.
- اگر فایل ۲۲ موردی تأییدشده پیدا نشود، Build باید Fail-Closed متوقف شود.
- هیچ Commit یا Push انجام نمی‌شود.

### History مورد 27

جواب کامل مدل فاز یک برای مورد 27 قبلاً بازیابی و در هر دو Copy Registry همگام شده است.

در این مرحله فقط باید صحت باقی‌ماندن آن Verify شود و نباید دوباره به‌عنوان Repair جدید ثبت شود.

### خروجی مورد انتظار

`notebooks/phase2/author_review_28_pilots.md`

در پایان:

- دقیقاً 28 مورد وجود داشته باشد.
- موارد 1 تا 22 همان محتوای تأییدشده قبلی باشند.
- موارد 23 تا 28 دارای History کامل باشند.
- هر ۶ مورد جدید دارای Source Review واقعی باشند.
- هر ۶ مورد جدید `AUTHOR_FEEDBACK_CONTENT` باشند.
- Pilot 25 سؤال و پاسخ اصلاح‌شده فعلی را نمایش دهد.
- Sourceهای Corpus تأییدشده در موارد 23 تا 28 قابل مشاهده باشند.
- تصمیم‌های Human Source Review قابل مشاهده باشند.
- Author Review برای همه موارد جدید `NOT_STARTED` باقی بماند.
- Content Coverage برای هر ۲۸ مورد PASS باشد.

</div>

In [4]:
# Code Cell 111
# ============================================================
# REBUILD FINAL 28-CASE AUTHOR REVIEW MARKDOWN
# WITH ONE UNIFORM HTML / RTL TEMPLATE
#
# Purpose:
# - Cases 1-22 remain the exact approved frozen content.
# - Cases 23-28 are rebuilt from current Registry/Corpus.
# - Cases 23-28 MUST use the SAME HTML/RTL presentation style
#   as the approved 1-22 cases.
#
# IMPORTANT:
# - Registry is READ-ONLY.
# - No Author decision is recorded.
# - No Commit / Push.
# - Existing 28-case output is backed up before replacement.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import html
import json
import re
import unicodedata


# ============================================================
# 1) FROZEN EXPECTATIONS
# ============================================================

CELL111_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1"
    "da34869f671e6e32762864e0ad70ea40"
)

CELL111_EXPECTED_REGISTRY_SHA256 = (
    "10933065b1d4598dba1534964b8a6837"
    "f723cf79082d1af2bc8a4964e2a6766e"
)

CELL111_EXPECTED_SOURCE22_SHA256 = (
    "2351a6011387dbc1ea56a8f9ba5fe83c"
    "1f1f28d8a9ebb7bd8c262c1f1f27fcfc"
)

CELL111_BATCH_KEY = (
    "phase1_author_feedback_derived_pilots"
)

CELL111_NEW_CASES = [
    23,
    24,
    25,
    26,
    27,
    28,
]

CELL111_EXPECTED_DECISIONS = {
    23: "ACCEPT",
    24: "ACCEPT",
    25: "REVISE",
    26: "REVISE",
    27: "REVISE",
    28: "ACCEPT",
}

CELL111_EXPECTED_REVISION_SCOPES = {
    23: "NONE",
    24: "NONE",
    25: "QUESTION_ANSWER_AND_PILOT_TYPE",
    26: "GROUNDING_ONLY",
    27: "GROUNDING_ONLY",
    28: "NONE",
}

CELL111_EXPECTED_ACTIVE_TYPE = (
    "AUTHOR_FEEDBACK_CONTENT"
)

CELL111_CORPUS_RELATIVE_PATH = Path(
    "data/phase2/processed/"
    "documents_dedup_quality_accepted_v1.jsonl"
)

CELL111_REGISTRY_RELATIVE_PATH = Path(
    "data/phase2/human_review/"
    "human_review_registry.json"
)

CELL111_OUTPUT_RELATIVE_PATH = Path(
    "notebooks/phase2/"
    "author_review_28_pilots.md"
)

CELL111_SOURCE22_CANDIDATES = [
    Path(
        "notebooks/phase2/"
        "author_review_22_pilots_REVISED.md"
    ),
    Path(
        "notebooks/phase2/"
        "author_review_22_pilots.md"
    ),
    Path(
        "author_review_22_pilots_REVISED.md"
    ),
    Path(
        "author_review_22_pilots.md"
    ),
]


CELL111_EXCERPT_TERMS = {

    23: (
        "سواد مالی میزان پول مدیریت پول "
        "درست مدیریت کردن پول"
    ),

    24: (
        "پرسشنامه ارزیابی وضعیت "
        "دانش نگرش مهارت سواد مالی"
    ),

    25: (
        "پول کجا می رود ثبت هزینه "
        "ولخرجی مدیریت هزینه"
    ),

    26: (
        "الگوهای نادرست خرید احساس "
        "عاطفی تبلیغات بازاریابی"
    ),

    27: (
        "افزایش درآمد دارایی درآمدزا "
        "سرمایه گذاری سود"
    ),

    28: (
        "رابطه با پول چیزهایی که نداریم "
        "نگرانی پول"
    ),
}


CELL111_CASE27_FULL_PHASE1_MODEL_ANSWER = (
    "روش‌های افزایش درآمد عبارتند از:\n\n"
    "1. **قیمت‌گذاری مناسب محصولات**: افزایش سود با توجه به "
    "هزینه‌های تولید و ارزش ارائه شده به مشتری.\n"
    "2. **سرمایه‌گذاری درست**: انجام سرمایه‌گذاری‌هایی که حداقل "
    "به اندازۀ تورم سودآوری داشته باشند.\n"
    "3. **خرید دارایی‌های درآمدزا**: افزایش درآمد از طریق "
    "سرمایه‌گذاری در دارایی‌هایی که درآمد داشته باشند، نه فقط "
    "برای خرج کردن درآمد.\n"
    "4. **تغییر نقشه کسب‌وکار**: اگر درآمد مطلوبی حاصل نشود، "
    "باید روش‌های کسب‌وکار را بهبود بخشید."
)


# ============================================================
# 2) UTILITIES
# ============================================================

cell111_failures = []


def cell111_check(
    reason,
    condition,
    details=None,
):

    if not bool(condition):

        item = {
            "reason": reason,
        }

        if details is not None:
            item["details"] = details

        cell111_failures.append(
            item
        )

    return bool(condition)


def cell111_sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(chunk)

    return h.hexdigest()


def cell111_sha256_text(text):

    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def cell111_find_repo_root():

    candidates = []

    for variable_name in [
        "cell110_registry_path",
        "cell109_registry_path",
        "cell108_registry_path",
        "cell105_registry_path",
    ]:

        if variable_name in globals():

            try:

                p = Path(
                    globals()[
                        variable_name
                    ]
                ).resolve()

                candidates.extend(
                    [
                        p.parent,
                        *p.parents,
                    ]
                )

            except Exception:
                pass

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:

        candidate = (
            candidate.resolve()
        )

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        registry_candidate = (
            candidate
            /
            CELL111_REGISTRY_RELATIVE_PATH
        )

        corpus_candidate = (
            candidate
            /
            CELL111_CORPUS_RELATIVE_PATH
        )

        if (
            registry_candidate.exists()
            and
            corpus_candidate.exists()
        ):

            return candidate

    return None


def cell111_html_text(
    value,
):

    if value is None:
        return "—"

    value = str(
        value
    )

    escaped = html.escape(
        value,
        quote=False,
    )

    # Preserve simple historical Markdown emphasis visually.
    escaped = re.sub(
        r"\*\*(.+?)\*\*",
        r"<strong>\1</strong>",
        escaped,
        flags=re.DOTALL,
    )

    escaped = (
        escaped
        .replace(
            "\r\n",
            "\n",
        )
        .replace(
            "\r",
            "\n",
        )
    )

    escaped = escaped.replace(
        "\n",
        "<br>"
    )

    return escaped


def cell111_html_code(
    value,
):

    return (
        '<code dir="ltr">'
        +
        html.escape(
            str(
                value
            ),
            quote=False,
        )
        +
        "</code>"
    )


def cell111_p(
    value,
):

    return (
        '<p dir="rtl" align="right">'
        +
        cell111_html_text(
            value
        )
        +
        "</p>"
    )


def cell111_h4(
    value,
):

    return (
        '<h4 dir="rtl" align="right">'
        +
        html.escape(
            str(
                value
            ),
            quote=False,
        )
        +
        "</h4>"
    )


def cell111_table_cell(
    value,
):

    return (
        '<td align="right">'
        +
        cell111_html_text(
            value
        )
        +
        "</td>"
    )


def cell111_table_header(
    value,
):

    return (
        '<th align="right">'
        +
        html.escape(
            str(
                value
            ),
            quote=False,
        )
        +
        "</th>"
    )


def cell111_build_table(
    headers,
    rows,
):

    parts = [
        '<table dir="rtl">',
        "<thead>",
        "<tr>",
    ]

    for header in headers:

        parts.append(
            cell111_table_header(
                header
            )
        )

    parts.extend([
        "</tr>",
        "</thead>",
        "<tbody>",
    ])

    for row in rows:

        parts.append(
            "<tr>"
        )

        for value in row:

            parts.append(
                cell111_table_cell(
                    value
                )
            )

        parts.append(
            "</tr>"
        )

    parts.extend([
        "</tbody>",
        "</table>",
    ])

    return "\n".join(
        parts
    )


def cell111_normalize(
    text,
):

    text = (
        ""
        if text is None
        else str(
            text
        )
    )

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    replacements = {
        "ي": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
        "\u200c": " ",
        "\u200f": " ",
        "\u200e": " ",
    }

    for old, new in replacements.items():

        text = text.replace(
            old,
            new,
        )

    text = re.sub(
        r"[^\w\u0600-\u06FF]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


CELL111_STOPWORDS = {
    "از", "به", "در", "با",
    "برای", "را", "که", "و",
    "یا", "این", "آن", "یک",
    "است", "هست", "بود",
    "شد", "شود", "می",
    "چه", "چرا", "چگونه",
    "اگر", "اما", "هم",
    "روی", "خود", "ما",
    "من", "شما", "فرد",
    "افراد", "مورد", "درباره",
}


def cell111_tokens(
    text,
):

    return [
        token

        for token
        in cell111_normalize(
            text
        ).split()

        if (
            len(
                token
            )
            >=
            2
            and
            token
            not in
            CELL111_STOPWORDS
        )
    ]


# ============================================================
# 3) SOURCE EXCERPT SELECTOR
# ============================================================

def cell111_best_source_excerpt(
    text,
    query_text,
    max_chars=900,
):

    text = str(
        text
        or
        ""
    ).strip()

    if not text:
        return ""

    query_tokens = set(
        cell111_tokens(
            query_text
        )
    )

    paragraphs = [
        part.strip()

        for part
        in re.split(
            r"\n\s*\n+",
            text,
        )

        if part.strip()
    ]

    sentences = [
        part.strip()

        for part
        in re.split(
            r"(?<=[\.\!\؟])\s+|\n+",
            text,
        )

        if part.strip()
    ]

    windows = []

    for index in range(
        len(
            sentences
        )
    ):

        window = " ".join(
            sentences[
                index:index + 3
            ]
        ).strip()

        if window:
            windows.append(
                window
            )

    candidates = (
        paragraphs
        +
        windows
    )

    if not candidates:
        candidates = [text]

    best = None
    best_score = -1

    for candidate in candidates:

        candidate_tokens = set(
            cell111_tokens(
                candidate
            )
        )

        score = len(
            query_tokens
            &
            candidate_tokens
        )

        if score > best_score:

            best_score = score
            best = candidate

        elif (
            score == best_score
            and
            best is not None
            and
            len(
                candidate
            )
            <
            len(
                best
            )
        ):

            best = candidate

    if best is None:
        best = text

    if len(
        best
    ) <= max_chars:

        return best

    position = None

    for token in cell111_tokens(
        query_text
    ):

        found = best.find(
            token
        )

        if found >= 0:

            position = found
            break

    if position is None:
        position = 0

    start = max(
        0,
        position - 220,
    )

    end = min(
        len(
            best
        ),
        start + max_chars,
    )

    excerpt = (
        best[
            start:end
        ].strip()
    )

    if start > 0:
        excerpt = (
            "... "
            +
            excerpt
        )

    if end < len(
        best
    ):
        excerpt += " ..."

    return excerpt


# ============================================================
# 4) RESOLVE REPOSITORY
# ============================================================

cell111_repo_root = (
    cell111_find_repo_root()
)


cell111_check(
    "REPO_ROOT_NOT_FOUND",
    cell111_repo_root
    is not None,
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


cell111_registry_path = (
    cell111_repo_root
    /
    CELL111_REGISTRY_RELATIVE_PATH
)


cell111_corpus_path = (
    cell111_repo_root
    /
    CELL111_CORPUS_RELATIVE_PATH
)


cell111_output_path = (
    cell111_repo_root
    /
    CELL111_OUTPUT_RELATIVE_PATH
)


cell111_output_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5) VERIFY FROZEN CORPUS + REGISTRY
# ============================================================

cell111_corpus_sha256 = (
    cell111_sha256_file(
        cell111_corpus_path
    )
)


cell111_registry_sha256_before = (
    cell111_sha256_file(
        cell111_registry_path
    )
)


cell111_check(
    "CORPUS_SHA_MISMATCH",
    cell111_corpus_sha256
    ==
    CELL111_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL111_EXPECTED_CORPUS_SHA256,

        "actual":
            cell111_corpus_sha256,
    },
)


cell111_check(
    "REGISTRY_SHA_MISMATCH",
    cell111_registry_sha256_before
    ==
    CELL111_EXPECTED_REGISTRY_SHA256,
    {
        "expected":
            CELL111_EXPECTED_REGISTRY_SHA256,

        "actual":
            cell111_registry_sha256_before,
    },
)


if cell111_failures:

    print(
        "=== FROZEN INPUT FAILURE ==="
    )

    print(
        cell111_failures
    )

    raise AssertionError(
        "Cell 111 stopped Fail-Closed."
    )


# ============================================================
# 6) LOAD REGISTRY — READ ONLY
# ============================================================

cell111_registry = json.loads(
    cell111_registry_path.read_text(
        encoding="utf-8"
    )
)


cell111_author_cases = (
    (
        cell111_registry.get(
            "author_review",
            {}
        )
        or
        {}
    ).get(
        "cases_ready_for_review",
        []
    )
    or
    []
)


cell111_feedback_cases = (
    (
        (
            cell111_registry.get(
                "second_human_review",
                {}
            )
            or
            {}
        ).get(
            CELL111_BATCH_KEY,
            {}
        )
        or
        {}
    ).get(
        "cases",
        []
    )
    or
    []
)


cell111_check(
    "AUTHOR_QUEUE_NOT_28",
    len(
        cell111_author_cases
    )
    ==
    28,
    len(
        cell111_author_cases
    ),
)


cell111_check(
    "FEEDBACK_BATCH_NOT_6",
    len(
        cell111_feedback_cases
    )
    ==
    6,
    len(
        cell111_feedback_cases
    ),
)


cell111_new_author_cases = (
    cell111_author_cases[
        22:
    ]
)


cell111_author_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell111_new_author_cases
}


cell111_batch_by_number = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case
    in cell111_feedback_cases
}


cell111_check(
    "NEW_CASE_SEQUENCE_WRONG",
    [
        case.get(
            "reference_pilot_number"
        )

        for case
        in cell111_new_author_cases
    ]
    ==
    CELL111_NEW_CASES,
)


# ============================================================
# 7) VALIDATE NEW CASES
# ============================================================

for number in CELL111_NEW_CASES:

    author_case = (
        cell111_author_by_number.get(
            number
        )
    )

    batch_case = (
        cell111_batch_by_number.get(
            number
        )
    )

    cell111_check(
        f"AUTHOR_CASE_MISSING:{number}",
        author_case
        is not None,
    )

    cell111_check(
        f"BATCH_CASE_MISSING:{number}",
        batch_case
        is not None,
    )

    if (
        author_case is None
        or
        batch_case is None
    ):

        continue

    cell111_check(
        f"PILOT_ID_MISMATCH:{number}",
        author_case.get(
            "pilot_case_id"
        )
        ==
        batch_case.get(
            "pilot_case_id"
        ),
    )

    cell111_check(
        f"ACTIVE_TYPE_WRONG:{number}",
        author_case.get(
            "pilot_type"
        )
        ==
        CELL111_EXPECTED_ACTIVE_TYPE,
        author_case.get(
            "pilot_type"
        ),
    )

    cell111_check(
        f"GOLDEN_CONTENT_FALSE:{number}",
        author_case.get(
            "golden_content_case"
        )
        is True,
    )

    review = (
        author_case.get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    cell111_check(
        f"SOURCE_REVIEW_NOT_COMPLETE:{number}",
        review.get(
            "status"
        )
        ==
        "COMPLETE",
    )

    cell111_check(
        f"DECISION_WRONG:{number}",
        review.get(
            "decision"
        )
        ==
        CELL111_EXPECTED_DECISIONS[
            number
        ],
        review.get(
            "decision"
        ),
    )

    cell111_check(
        f"REVISION_SCOPE_WRONG:{number}",
        review.get(
            "revision_scope"
        )
        ==
        CELL111_EXPECTED_REVISION_SCOPES[
            number
        ],
        review.get(
            "revision_scope"
        ),
    )

    cell111_check(
        f"NO_VERIFIED_CORPUS_SOURCE:{number}",
        len(
            review.get(
                "verified_corpus_sources",
                []
            )
            or
            []
        )
        >=
        1,
    )

    cell111_check(
        f"AUTHOR_REVIEW_ALREADY_STARTED:{number}",
        (
            author_case.get(
                "author_final_review",
                {}
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )


if cell111_failures:

    print(
        "=== REGISTRY VALIDATION FAILURE ==="
    )

    print(
        cell111_failures
    )

    raise AssertionError(
        "Cell 111 stopped before build."
    )


# ============================================================
# 8) PILOT 25 SUPERSEDED HISTORY
# ============================================================

cell111_case25 = (
    cell111_author_by_number[
        25
    ]
)


cell111_case25_review = (
    cell111_case25.get(
        "second_source_review",
        {}
    )
    or
    {}
)


cell111_case25_event_id = (
    cell111_case25_review.get(
        "review_event_id"
    )
)


cell111_case25_event = None


for event in (
    cell111_case25.get(
        "source_review_history",
        []
    )
    or
    []
):

    if (
        isinstance(
            event,
            dict,
        )
        and
        event.get(
            "review_event_id"
        )
        ==
        cell111_case25_event_id
    ):

        cell111_case25_event = event
        break


cell111_case25_old_state = (
    (
        cell111_case25_event
        or
        {}
    ).get(
        "superseded_state"
    )
    or
    {}
)


cell111_check(
    "PILOT25_SUPERSEDED_HISTORY_MISSING",
    bool(
        cell111_case25_old_state
    ),
)


cell111_check(
    "PILOT25_PREVIOUS_TYPE_WRONG",
    cell111_case25_old_state.get(
        "previous_pilot_type"
    )
    ==
    "MULTI_TURN_CONTEXT_CONTINUITY",
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 9) PILOT 27 FULL HISTORY
# ============================================================

cell111_case27 = (
    cell111_author_by_number[
        27
    ]
)


cell111_case27_model_answers = [
    entry.get(
        "text"
    )

    for entry
    in (
        (
            cell111_case27.get(
                "historical_phase1_demo",
                {}
            )
            or
            {}
        ).get(
            "raw_conversation",
            []
        )
        or
        []
    )

    if (
        isinstance(
            entry,
            dict,
        )
        and
        entry.get(
            "speaker"
        )
        ==
        "PHASE1_MODEL"
    )
]


cell111_check(
    "CASE27_FULL_MODEL_ANSWER_MISSING",
    CELL111_CASE27_FULL_PHASE1_MODEL_ANSWER
    in
    cell111_case27_model_answers,
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 10) RESOLVE EXACT APPROVED 22-CASE MARKDOWN
# ============================================================

cell111_source22_path = None


for relative_path in (
    CELL111_SOURCE22_CANDIDATES
):

    candidate = (
        cell111_repo_root
        /
        relative_path
    )

    if not candidate.exists():
        continue

    candidate_sha = (
        cell111_sha256_file(
            candidate
        )
    )

    if (
        candidate_sha
        ==
        CELL111_EXPECTED_SOURCE22_SHA256
    ):

        cell111_source22_path = candidate
        break


cell111_check(
    "APPROVED_22_SOURCE_NOT_FOUND",
    cell111_source22_path
    is not None,
)


if cell111_failures:

    raise AssertionError(
        "Exact approved 22-case Markdown was not found."
    )


cell111_source22_text = (
    cell111_source22_path.read_text(
        encoding="utf-8"
    )
)


cell111_source22_sha256 = (
    cell111_sha256_text(
        cell111_source22_text
    )
)


# ============================================================
# 11) VERIFY APPROVED 22 IS HTML / RTL
# ============================================================

cell111_check(
    "SOURCE22_NO_RTL_DIV",
    cell111_source22_text.count(
        '<div dir="rtl" align="right">'
    )
    >=
    22,
)


cell111_check(
    "SOURCE22_NO_RTL_TABLES",
    '<table dir="rtl">'
    in
    cell111_source22_text,
)


cell111_check(
    "SOURCE22_NO_RTL_H2",
    '<h2 dir="rtl" align="right">'
    in
    cell111_source22_text,
)


if cell111_failures:

    raise AssertionError(
        "Approved 22-case source is not the expected RTL HTML artifact."
    )


# ============================================================
# 12) TRANSFORM ONLY DOCUMENT TOTAL 22 -> 28
# ============================================================

cell111_base28_text = (
    cell111_source22_text
)


# Structural case labels only.
cell111_base28_text = re.sub(
    r"(مورد\s+[0-9۰-۹]+\s+از\s+)۲۲",
    r"\g<1>۲۸",
    cell111_base28_text,
)


cell111_base28_text = re.sub(
    r"(مورد\s+[0-9۰-۹]+\s+از\s+)22",
    r"\g<1>28",
    cell111_base28_text,
)


# Change document-level totals only before first case.
first_case_match = re.search(
    r"<h2[^>]*>\s*مورد\s+1\s+از\s+۲۸\s*</h2>",
    cell111_base28_text,
)


cell111_check(
    "FIRST_CASE_NOT_FOUND_AFTER_22_TO_28",
    first_case_match
    is not None,
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


prefix = (
    cell111_base28_text[
        :first_case_match.start()
    ]
)


suffix = (
    cell111_base28_text[
        first_case_match.start():
    ]
)


prefix = (
    prefix
    .replace(
        "۲۲ مورد واقعی Golden Dataset",
        "۲۸ مورد واقعی Golden Dataset",
    )
    .replace(
        "۲۲ پایلوت",
        "۲۸ پایلوت",
    )
    .replace(
        "۲۲ مورد",
        "۲۸ مورد",
    )
    .replace(
        "22 pilots",
        "28 pilots",
    )
    .replace(
        "22 cases",
        "28 cases",
    )
)


cell111_base28_text = (
    prefix
    +
    suffix
)


# ============================================================
# 13) LOAD VERIFIED CORPUS DOCUMENTS
# ============================================================

cell111_required_document_ids = set()


for number in CELL111_NEW_CASES:

    review = (
        cell111_author_by_number[
            number
        ].get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    for source in (
        review.get(
            "verified_corpus_sources",
            []
        )
        or
        []
    ):

        document_id = source.get(
            "document_id"
        )

        if document_id:

            cell111_required_document_ids.add(
                document_id
            )


cell111_source_documents = {}


with cell111_corpus_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(
            line
        )

        document_id = (
            record.get(
                "document_id"
            )
        )

        if (
            document_id
            in
            cell111_required_document_ids
        ):

            cell111_source_documents[
                document_id
            ] = record


cell111_check(
    "VERIFIED_CORPUS_DOCUMENTS_MISSING",
    set(
        cell111_source_documents.keys()
    )
    ==
    cell111_required_document_ids,
    {
        "required":
            sorted(
                cell111_required_document_ids
            ),

        "found":
            sorted(
                cell111_source_documents.keys()
            ),
    },
)


if cell111_failures:

    raise AssertionError(
        cell111_failures
    )


# ============================================================
# 14) GROUNDING EXPLANATIONS
# ============================================================

def cell111_grounding_explanation(
    number,
):

    if number == 23:

        return (
            "بازخورد مستقیم مؤلف با یک منبع Direct Author "
            "از Corpus پشتیبانی مستقیم و صریح دارد."
        )

    if number == 24:

        return (
            "Grounding ترکیبی است: یک منبع بخش پرسشنامه و "
            "ارزیابی وضعیت و منبع دیگر ابعاد دانش، نگرش و "
            "مهارت مالی را پشتیبانی می‌کند."
        )

    if number == 25:

        return (
            "برداشت قبلی تداوم Context کنار گذاشته شد. "
            "مورد فعال اکنون یک سؤال محتوایی است و سؤال و "
            "پاسخ فعلی با منابع Corpus درباره ثبت و مدیریت "
            "هزینه‌ها Ground شده‌اند."
        )

    if number == 26:

        return (
            "فهرست هفت‌گانه عمدتاً بر بازخورد مستقیم دکتر "
            "استوار است. منابع Website فقط بخشی از مفهوم "
            "الگوهای خرید نادرست، عوامل عاطفی و اثر تبلیغات "
            "و بازاریابی را تأیید می‌کنند."
        )

    if number == 27:

        return (
            "چهار روش و ترتیب آن‌ها از بازخورد مستقیم دکتر "
            "می‌آید. منابع Website فقط بخشی از مفاهیم افزایش "
            "درآمد، دارایی درآمدزا، سرمایه‌گذاری و سود را "
            "تأیید می‌کنند."
        )

    if number == 28:

        return (
            "منبع Direct Author بخش اصلی رابطه فرد با پول و "
            "تلاش برای جبران چیزهایی که فرد احساس می‌کند "
            "ندارد را پشتیبانی می‌کند. تفسیر گسترده‌تر احساس "
            "فقر از بازخورد مستقیم دکتر حفظ شده است."
        )

    return "—"


# ============================================================
# 15) SPEAKER LABEL
# ============================================================

def cell111_speaker_label(
    entry,
):

    speaker = entry.get(
        "speaker"
    )

    turn = entry.get(
        "turn"
    )

    if speaker == "HUMAN":
        label = "سؤال انسان"

    elif speaker == "PHASE1_MODEL":
        label = "پاسخ مدل فاز یک"

    else:
        label = str(
            speaker
            or
            "گوینده"
        )

    if turn is not None:

        label += (
            " — نوبت "
            +
            str(
                turn
            )
        )

    return label


# ============================================================
# 16) BUILD ONE NEW CASE
#     EXACT HTML / RTL STYLE
# ============================================================

def cell111_build_new_case_html(
    case,
):

    number = (
        case[
            "reference_pilot_number"
        ]
    )

    history = (
        case.get(
            "historical_phase1_demo",
            {}
        )
        or
        {}
    )

    conversation = (
        history.get(
            "raw_conversation",
            []
        )
        or
        []
    )

    raw_feedback = (
        history.get(
            "raw_author_feedback"
        )
        or
        ""
    )

    question = (
        (
            case.get(
                "derived_question",
                {}
            )
            or
            {}
        ).get(
            "question_text"
        )
        or
        ""
    )

    answer = (
        (
            case.get(
                "expected_answer_review",
                {}
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        or
        ""
    )

    review = (
        case.get(
            "second_source_review",
            {}
        )
        or
        {}
    )

    decision = (
        review.get(
            "decision"
        )
        or
        ""
    )

    revision_scope = (
        review.get(
            "revision_scope"
        )
        or
        ""
    )

    verified_sources = (
        review.get(
            "verified_corpus_sources",
            []
        )
        or
        []
    )

    parts = []

    # --------------------------------------------------------
    # Wrapper + case title
    # --------------------------------------------------------

    parts.extend([
        "",
        "",
        "---",
        "",
        '<div dir="rtl" align="right">',
        "",
        (
            '<h2 dir="rtl" align="right">'
            f"مورد {number} از ۲۸"
            "</h2>"
        ),
        "",
        (
            '<h3 dir="rtl" align="right">'
            "۱. متن منبع (بخشی از متن اصلی)"
            "</h3>"
        ),
        "",
    ])

    # --------------------------------------------------------
    # Historical Phase-1 demo
    # --------------------------------------------------------

    parts.append(
        cell111_h4(
            "سابقه واقعی دموی فاز یک"
        )
    )

    parts.append(
        ""
    )

    for entry in conversation:

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>"
                +
                html.escape(
                    cell111_speaker_label(
                        entry
                    ),
                    quote=False,
                )
                +
                ":</strong>"
                "</p>"
            )
        )

        parts.append(
            cell111_p(
                entry.get(
                    "text"
                )
                or
                ""
            )
        )

        parts.append(
            ""
        )

    parts.append(
        (
            '<p dir="rtl" align="right">'
            "<strong>"
            "بازخورد مستقیم دکتر در فاز یک:"
            "</strong>"
            "</p>"
        )
    )

    parts.append(
        cell111_p(
            raw_feedback
        )
    )

    parts.append(
        ""
    )

    # --------------------------------------------------------
    # Pilot 25 superseded interpretation
    # --------------------------------------------------------

    if number == 25:

        parts.append(
            cell111_h4(
                "وضعیت قبلی که پس از Source Review کنار گذاشته شد"
            )
        )

        parts.append(
            ""
        )

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>نوع قبلی:</strong> "
                +
                cell111_html_code(
                    cell111_case25_old_state.get(
                        "previous_pilot_type"
                    )
                    or
                    "—"
                )
                +
                "</p>"
            )
        )

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>سؤال قبلی:</strong><br>"
                +
                cell111_html_text(
                    cell111_case25_old_state.get(
                        "previous_question"
                    )
                    or
                    "—"
                )
                +
                "</p>"
            )
        )

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>رفتار مورد انتظار قبلی:</strong><br>"
                +
                cell111_html_text(
                    cell111_case25_old_state.get(
                        "previous_expected_behavior"
                    )
                    or
                    "—"
                )
                +
                "</p>"
            )
        )

        parts.append(
            cell111_p(
                "دلیل کنار گذاشتن: Turnهای دموی فاز یک "
                "Context مشترک قابل اتکایی نداشتند. بنابراین "
                "برداشت قبلی به‌عنوان تست فعال تداوم Context "
                "کنار گذاشته شد و فقط به‌عنوان History حفظ شده است."
            )
        )

        parts.append(
            ""
        )

    # --------------------------------------------------------
    # Verified Corpus evidence
    # --------------------------------------------------------

    parts.append(
        cell111_h4(
            "شواهد Corpus تأییدشده در Source Review"
        )
    )

    parts.append(
        ""
    )

    for source_index, source in enumerate(
        verified_sources,
        start=1,
    ):

        document_id = source.get(
            "document_id"
        )

        record = (
            cell111_source_documents[
                document_id
            ]
        )

        title = (
            source.get(
                "title"
            )
            or
            record.get(
                "title"
            )
            or
            "بدون عنوان"
        )

        role = (
            source.get(
                "role"
            )
            or
            "—"
        )

        coverage = (
            source.get(
                "coverage"
            )
            or
            "—"
        )

        excerpt = (
            cell111_best_source_excerpt(
                record.get(
                    "text",
                    "",
                ),
                CELL111_EXCERPT_TERMS[
                    number
                ],
            )
        )

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>"
                f"منبع Corpus {source_index}: "
                +
                html.escape(
                    title,
                    quote=False,
                )
                +
                "</strong>"
                "</p>"
            )
        )

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>نقش در Grounding:</strong> "
                +
                cell111_html_code(
                    role
                )
                +
                "</p>"
            )
        )

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>میزان پوشش:</strong> "
                +
                cell111_html_code(
                    coverage
                )
                +
                "</p>"
            )
        )

        parts.append(
            (
                '<p dir="rtl" align="right">'
                "<strong>بخشی از متن منبع:</strong>"
                "</p>"
            )
        )

        parts.append(
            cell111_p(
                excerpt
            )
        )

        parts.append(
            ""
        )

    parts.append(
        (
            '<p dir="rtl" align="right">'
            "<strong>جمع‌بندی Grounding:</strong><br>"
            +
            cell111_html_text(
                cell111_grounding_explanation(
                    number
                )
            )
            +
            "</p>"
        )
    )

    parts.append(
        ""
    )

    # --------------------------------------------------------
    # Section 2: Revision history table
    # --------------------------------------------------------

    parts.append(
        (
            '<h3 dir="rtl" align="right">'
            "۲. سؤال‌ها و پاسخ‌ها — روند اصلاح و بازبینی"
            "</h3>"
        )
    )

    parts.append(
        ""
    )

    history_rows = []

    human_entries = [
        entry

        for entry
        in conversation

        if (
            isinstance(
                entry,
                dict,
            )
            and
            entry.get(
                "speaker"
            )
            ==
            "HUMAN"
        )
    ]

    model_entries = [
        entry

        for entry
        in conversation

        if (
            isinstance(
                entry,
                dict,
            )
            and
            entry.get(
                "speaker"
            )
            ==
            "PHASE1_MODEL"
        )
    ]

    pair_count = max(
        len(
            human_entries
        ),
        len(
            model_entries
        ),
    )

    for index in range(
        pair_count
    ):

        stage = (
            "دموی فاز یک"
            if index == 0
            else
            (
                "دموی فاز یک — نوبت "
                +
                str(
                    index + 1
                )
            )
        )

        human_text = (
            human_entries[
                index
            ].get(
                "text"
            )
            if (
                index
                <
                len(
                    human_entries
                )
            )
            else
            "—"
        )

        model_text = (
            model_entries[
                index
            ].get(
                "text"
            )
            if (
                index
                <
                len(
                    model_entries
                )
            )
            else
            "—"
        )

        history_rows.append(
            [
                stage,
                human_text,
                model_text,
                (
                    "پاسخ تاریخی مدل فاز یک است و "
                    "Ground Truth محسوب نمی‌شود."
                ),
            ]
        )

    if number == 25:

        history_rows.append(
            [
                "برداشت اولیه پیش از Source Review",

                (
                    cell111_case25_old_state.get(
                        "previous_question"
                    )
                    or
                    "—"
                ),

                (
                    cell111_case25_old_state.get(
                        "previous_expected_behavior"
                    )
                    or
                    "—"
                ),

                (
                    "این برداشت قبلی، مورد را به‌عنوان "
                    "MULTI_TURN_CONTEXT_CONTINUITY در نظر "
                    "می‌گرفت و پس از Source Review کنار "
                    "گذاشته شد."
                ),
            ]
        )

    if number == 25:

        final_note = (
            "سؤال و پاسخ بازطراحی شد و Pilot فعال به "
            "AUTHOR_FEEDBACK_CONTENT تبدیل شد."
        )

    elif revision_scope == "GROUNDING_ONLY":

        final_note = (
            "متن سؤال و پاسخ حفظ شد؛ اصلاح فقط در Grounding "
            "و تفکیک بازخورد مستقیم مؤلف از پشتیبانی جزئی "
            "Corpus انجام شد."
        )

    else:

        final_note = (
            "سؤال و پاسخ پس از تطبیق با Source واقعی Corpus "
            "پذیرفته شد."
        )

    history_rows.append(
        [
            "Second Human Source Review",
            question,
            answer,
            final_note,
        ]
    )

    parts.append(
        cell111_build_table(
            [
                "مرحله",
                "متن سؤال",
                "متن پاسخ",
                "توضیحات / دلیل تغییر",
            ],
            history_rows,
        )
    )

    parts.append(
        ""
    )

    # --------------------------------------------------------
    # Section 3: Decisions
    # --------------------------------------------------------

    parts.append(
        (
            '<h3 dir="rtl" align="right">'
            "۳. تصمیم من در روند بازبینی"
            "</h3>"
        )
    )

    parts.append(
        ""
    )

    decision_rows = [
        [
            "بازخورد فاز یک",
            "بازخورد مستقیم مؤلف",
            (
                "متن بازخورد دکتر بدون بازنویسی "
                "در History حفظ شده است."
            ),
        ],

        [
            "Source Mapping",
            "VERIFIED",
            (
                str(
                    len(
                        verified_sources
                    )
                )
                +
                " Source Direct Author از Accepted Corpus "
                "برای این مورد بررسی شد."
            ),
        ],

        [
            "Second Human Source Review",
            decision,
            (
                "Revision Scope: "
                +
                revision_scope
                +
                " — "
                +
                cell111_grounding_explanation(
                    number
                )
            ),
        ],

        [
            "وضعیت فعلی",
            "آماده برای بازبینی مؤلف",
            (
                "هیچ تصمیم نهایی به نام دکتر ثبت نشده "
                "و Author Final Review همچنان "
                "NOT_STARTED است."
            ),
        ],
    ]

    parts.append(
        cell111_build_table(
            [
                "مرحله",
                "تصمیم",
                "توضیحات",
            ],
            decision_rows,
        )
    )

    parts.append(
        ""
    )

    # --------------------------------------------------------
    # Section 4: Final Q/A
    # --------------------------------------------------------

    parts.append(
        (
            '<h3 dir="rtl" align="right">'
            "۴. سؤال و پاسخ نهایی (مورد بازبینی توسط دکتر)"
            "</h3>"
        )
    )

    parts.append(
        ""
    )

    final_rows = [
        [
            "منبع",
            (
                "همان سابقه، بازخورد مستقیم مؤلف و "
                "شواهد تأییدشده Corpus در بخش ۱ بالا."
            ),
        ],

        [
            "متن سؤال نهایی",
            question,
        ],

        [
            "متن پاسخ نهایی",
            answer,
        ],
    ]

    parts.append(
        cell111_build_table(
            [
                "عنوان",
                "محتوا",
            ],
            final_rows,
        )
    )

    parts.append(
        ""
    )

    # --------------------------------------------------------
    # Section 5: Blank Author Review
    # EXACT SAME STYLE AS 1-22
    # --------------------------------------------------------

    parts.append(
        (
            '<h3 dir="rtl" align="right">'
            "۵. نظر مؤلف (دکتر کمیل رودی)"
            "</h3>"
        )
    )

    parts.append(
        ""
    )

    parts.extend([
        '<table dir="rtl">',
        "<thead>",
        "<tr>",
        '<th align="right">تصمیم دکتر</th>',
        '<th align="right">توضیحات دکتر</th>',
        "</tr>",
        "</thead>",
        "<tbody>",
        "<tr>",
        '<td align="right">',
        "☐ تأیید<br>",
        "☐ نیاز به اصلاح<br>",
        "☐ رد<br>",
        "☐ باید دوباره سؤال ساخته شود",
        "</td>",
        '<td align="right">',
        "<br><br><br><br><br><br><br><br>",
        "</td>",
        "</tr>",
        "</tbody>",
        "</table>",
        "",
        "</div>",
    ])

    return "\n".join(
        parts
    )


# ============================================================
# 17) BUILD CASES 23-28
# ============================================================

cell111_new_blocks = []


for number in CELL111_NEW_CASES:

    block = (
        cell111_build_new_case_html(
            cell111_author_by_number[
                number
            ]
        )
    )

    cell111_new_blocks.append(
        block
    )


cell111_new_cases_text = "\n".join(
    cell111_new_blocks
)


# ============================================================
# 18) BUILD COMPLETE 28-CASE DOCUMENT
# ============================================================

cell111_full_markdown = (
    cell111_base28_text.rstrip()
    +
    "\n"
    +
    cell111_new_cases_text
    +
    "\n"
)


# ============================================================
# 19) VALIDATE ALL 28 CASE HEADERS
# ============================================================

cell111_case_header_pattern = re.compile(
    r'<h2\s+dir="rtl"\s+align="right">\s*'
    r'مورد\s+([0-9۰-۹]+)\s+از\s+۲۸\s*'
    r'</h2>'
)


PERSIAN_TO_ASCII = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹",
    "0123456789",
)


def cell111_int(
    value,
):

    return int(
        str(
            value
        ).translate(
            PERSIAN_TO_ASCII
        )
    )


cell111_case_matches = list(
    cell111_case_header_pattern.finditer(
        cell111_full_markdown
    )
)


cell111_case_numbers = [
    cell111_int(
        match.group(
            1
        )
    )

    for match
    in cell111_case_matches
]


cell111_check(
    "CASE_HEADER_COUNT_NOT_28",
    len(
        cell111_case_numbers
    )
    ==
    28,
    cell111_case_numbers,
)


cell111_check(
    "CASE_HEADER_SEQUENCE_NOT_1_TO_28",
    cell111_case_numbers
    ==
    list(
        range(
            1,
            29,
        )
    ),
    cell111_case_numbers,
)


# ============================================================
# 20) VALIDATE CASES 23-28 ARE PURE HTML/RTL
# ============================================================

for number, block in zip(
    CELL111_NEW_CASES,
    cell111_new_blocks,
):

    required = [

        '<div dir="rtl" align="right">',

        (
            '<h2 dir="rtl" align="right">'
            f"مورد {number} از ۲۸"
            "</h2>"
        ),

        (
            '<h3 dir="rtl" align="right">'
            "۱. متن منبع (بخشی از متن اصلی)"
            "</h3>"
        ),

        (
            '<h3 dir="rtl" align="right">'
            "۲. سؤال‌ها و پاسخ‌ها — روند اصلاح و بازبینی"
            "</h3>"
        ),

        (
            '<h3 dir="rtl" align="right">'
            "۳. تصمیم من در روند بازبینی"
            "</h3>"
        ),

        (
            '<h3 dir="rtl" align="right">'
            "۴. سؤال و پاسخ نهایی (مورد بازبینی توسط دکتر)"
            "</h3>"
        ),

        (
            '<h3 dir="rtl" align="right">'
            "۵. نظر مؤلف (دکتر کمیل رودی)"
            "</h3>"
        ),

        '<table dir="rtl">',

        '<th align="right">',

        '<td align="right">',

        "☐ تأیید",

        "☐ نیاز به اصلاح",

        "☐ رد",

        "☐ باید دوباره سؤال ساخته شود",

        "توضیحات دکتر",

        "</div>",
    ]

    missing = [
        marker

        for marker
        in required

        if marker
        not in
        block
    ]

    cell111_check(
        f"HTML_RTL_STRUCTURE_MISSING:{number}",
        len(
            missing
        )
        ==
        0,
        missing,
    )

    # --------------------------------------------------------
    # HARD FAIL if old raw Markdown structure survives
    # --------------------------------------------------------

    forbidden_patterns = [

        r"(?m)^##\s+مورد",

        r"(?m)^###\s+",

        r"(?m)^\|\s*مرحله\s*\|",

        r"(?m)^\|---",

        r"(?m)^-\s+☐\s+تأیید",

        r"(?m)^\*\*توضیحات دکتر",
    ]

    for pattern in forbidden_patterns:

        cell111_check(
            (
                f"RAW_MARKDOWN_SURVIVED:"
                f"{number}:"
                f"{pattern}"
            ),
            re.search(
                pattern,
                block,
            )
            is None,
        )


# ============================================================
# 21) VERIFY EVERY NEW TABLE IS RTL
# ============================================================

for number, block in zip(
    CELL111_NEW_CASES,
    cell111_new_blocks,
):

    table_open_count = len(
        re.findall(
            r"<table\b",
            block,
        )
    )

    rtl_table_count = (
        block.count(
            '<table dir="rtl">'
        )
    )

    cell111_check(
        f"TABLE_RTL_COUNT_MISMATCH:{number}",
        table_open_count
        ==
        rtl_table_count,
        {
            "all_tables":
                table_open_count,

            "rtl_tables":
                rtl_table_count,
        },
    )


# ============================================================
# 22) VERIFY NEW CONTENT
# ============================================================

for number, block in zip(
    CELL111_NEW_CASES,
    cell111_new_blocks,
):

    case = (
        cell111_author_by_number[
            number
        ]
    )

    question = (
        (
            case.get(
                "derived_question",
                {}
            )
            or
            {}
        ).get(
            "question_text"
        )
        or
        ""
    )

    answer = (
        (
            case.get(
                "expected_answer_review",
                {}
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        or
        ""
    )

    raw_feedback = (
        (
            case.get(
                "historical_phase1_demo",
                {}
            )
            or
            {}
        ).get(
            "raw_author_feedback"
        )
        or
        ""
    )

    cell111_check(
        f"QUESTION_MISSING:{number}",
        cell111_html_text(
            question
        )
        in
        block,
    )

    cell111_check(
        f"ANSWER_MISSING:{number}",
        cell111_html_text(
            answer
        )
        in
        block,
    )

    cell111_check(
        f"RAW_FEEDBACK_MISSING:{number}",
        cell111_html_text(
            raw_feedback
        )
        in
        block,
    )


# ============================================================
# 23) PILOT 25 + 27 SPECIAL VALIDATION
# ============================================================

cell111_case25_block = (
    cell111_new_blocks[
        2
    ]
)


cell111_check(
    "PILOT25_OLD_CONTINUITY_HISTORY_NOT_VISIBLE",
    "MULTI_TURN_CONTEXT_CONTINUITY"
    in
    cell111_case25_block,
)


cell111_check(
    "PILOT25_ACTIVE_CONTENT_NOT_VISIBLE",
    "AUTHOR_FEEDBACK_CONTENT"
    in
    cell111_case25_block,
)


cell111_case27_block = (
    cell111_new_blocks[
        4
    ]
)


cell111_check(
    "PILOT27_FULL_HISTORY_NOT_VISIBLE",
    cell111_html_text(
        CELL111_CASE27_FULL_PHASE1_MODEL_ANSWER
    )
    in
    cell111_case27_block,
)


# ============================================================
# 24) ACTIVE TYPE COUNTS
# ============================================================

cell111_active_content_count = sum(
    1

    for case
    in cell111_new_author_cases

    if (
        case.get(
            "pilot_type"
        )
        ==
        "AUTHOR_FEEDBACK_CONTENT"
        and
        case.get(
            "golden_content_case"
        )
        is True
    )
)


cell111_active_continuity_count = sum(
    1

    for case
    in cell111_new_author_cases

    if (
        case.get(
            "pilot_type"
        )
        ==
        "MULTI_TURN_CONTEXT_CONTINUITY"
    )
)


cell111_check(
    "ACTIVE_CONTENT_COUNT_NOT_6",
    cell111_active_content_count
    ==
    6,
)


cell111_check(
    "ACTIVE_CONTINUITY_COUNT_NOT_0",
    cell111_active_continuity_count
    ==
    0,
)


# ============================================================
# 25) REGISTRY MUST STILL BE UNCHANGED
# ============================================================

cell111_registry_sha256_prewrite = (
    cell111_sha256_file(
        cell111_registry_path
    )
)


cell111_check(
    "REGISTRY_CHANGED_DURING_BUILD",
    cell111_registry_sha256_prewrite
    ==
    cell111_registry_sha256_before,
    {
        "before":
            cell111_registry_sha256_before,

        "after":
            cell111_registry_sha256_prewrite,
    },
)


# ============================================================
# 26) PRE-WRITE GATE
# ============================================================

if cell111_failures:

    print(
        "\n=== PRE-WRITE VALIDATION FAILURES ==="
    )

    print(
        cell111_failures
    )

    raise AssertionError(
        "Cell 111 stopped before writing output."
    )


# ============================================================
# 27) BACKUP EXISTING 28-CASE OUTPUT
# ============================================================

cell111_existing_output = None


if cell111_output_path.exists():

    cell111_existing_output = (
        cell111_output_path.read_text(
            encoding="utf-8"
        )
    )


cell111_output_change_required = (
    cell111_existing_output
    !=
    cell111_full_markdown
)


cell111_output_backup_path = None


if cell111_output_change_required:

    if cell111_output_path.exists():

        timestamp = (
            datetime.now(
                timezone.utc
            ).strftime(
                "%Y%m%dT%H%M%SZ"
            )
        )

        cell111_output_backup_path = (
            cell111_output_path.parent
            /
            (
                cell111_output_path.stem
                +
                ".before_cell111_rtl_repair_"
                +
                timestamp
                +
                cell111_output_path.suffix
            )
        )

        cell111_output_backup_path.write_text(
            cell111_existing_output,
            encoding="utf-8",
        )

    temp_path = (
        cell111_output_path.parent
        /
        (
            cell111_output_path.name
            +
            ".cell111.rtl.tmp"
        )
    )

    temp_path.write_text(
        cell111_full_markdown,
        encoding="utf-8",
    )

    temp_readback = (
        temp_path.read_text(
            encoding="utf-8"
        )
    )

    cell111_check(
        "TEMP_READBACK_MISMATCH",
        temp_readback
        ==
        cell111_full_markdown,
    )

    if cell111_failures:

        raise AssertionError(
            cell111_failures
        )

    temp_path.replace(
        cell111_output_path
    )


# ============================================================
# 28) FINAL READBACK
# ============================================================

cell111_final_text = (
    cell111_output_path.read_text(
        encoding="utf-8"
    )
)


cell111_final_sha256 = (
    cell111_sha256_text(
        cell111_final_text
    )
)


cell111_check(
    "FINAL_TEXT_MISMATCH",
    cell111_final_text
    ==
    cell111_full_markdown,
)


# ============================================================
# 29) FINAL ALL-28 HTML HEADER VALIDATION
# ============================================================

cell111_final_case_matches = list(
    cell111_case_header_pattern.finditer(
        cell111_final_text
    )
)


cell111_final_case_numbers = [
    cell111_int(
        match.group(
            1
        )
    )

    for match
    in cell111_final_case_matches
]


cell111_check(
    "FINAL_CASE_SEQUENCE_NOT_1_TO_28",
    cell111_final_case_numbers
    ==
    list(
        range(
            1,
            29,
        )
    ),
    cell111_final_case_numbers,
)


# ============================================================
# 30) FINAL 23-28 SEGMENT CHECK
# ============================================================

case23_marker = (
    '<h2 dir="rtl" align="right">'
    "مورد 23 از ۲۸"
    "</h2>"
)


case23_position = (
    cell111_final_text.find(
        case23_marker
    )
)


cell111_check(
    "FINAL_CASE23_NOT_FOUND",
    case23_position
    >=
    0,
)


cell111_case23_div_position = (
    cell111_final_text.rfind(
        '<div dir="rtl" align="right">',
        0,
        case23_position,
    )
)


cell111_check(
    "FINAL_CASE23_RTL_DIV_NOT_FOUND",
    cell111_case23_div_position
    >=
    0,
)


cell111_new_segment = (
    cell111_final_text[
        cell111_case23_div_position:
    ]
    if (
        case23_position
        >=
        0
        and
        cell111_case23_div_position
        >=
        0
    )
    else
    ""
)


# No old Markdown headings/tables in 23-28.
cell111_check(
    "FINAL_NEW_SEGMENT_HAS_RAW_MARKDOWN_CASE_HEADER",
    re.search(
        r"(?m)^##\s+مورد",
        cell111_new_segment,
    )
    is None,
)


cell111_check(
    "FINAL_NEW_SEGMENT_HAS_RAW_MARKDOWN_H3",
    re.search(
        r"(?m)^###\s+",
        cell111_new_segment,
    )
    is None,
)


cell111_check(
    "FINAL_NEW_SEGMENT_HAS_PIPE_TABLE",
    re.search(
        r"(?m)^\|\s*مرحله\s*\|",
        cell111_new_segment,
    )
    is None,
)


cell111_check(
    "FINAL_NEW_SEGMENT_HAS_MARKDOWN_CHECKBOX_LIST",
    re.search(
        r"(?m)^-\s+☐",
        cell111_new_segment,
    )
    is None,
)


cell111_check(
    "FINAL_NEW_SEGMENT_RTL_DIV_COUNT_NOT_6",
    cell111_new_segment.count(
        '<div dir="rtl" align="right">'
    )
    ==
    6,
)


cell111_check(
    "FINAL_NEW_SEGMENT_AUTHOR_TABLE_COUNT_NOT_6",
    cell111_new_segment.count(
        "۵. نظر مؤلف (دکتر کمیل رودی)"
    )
    ==
    6,
)


# ============================================================
# 31) FINAL REGISTRY SAFETY
# ============================================================

cell111_registry_sha256_final = (
    cell111_sha256_file(
        cell111_registry_path
    )
)


cell111_check(
    "FINAL_REGISTRY_SHA_CHANGED",
    cell111_registry_sha256_final
    ==
    cell111_registry_sha256_before,
    {
        "before":
            cell111_registry_sha256_before,

        "after":
            cell111_registry_sha256_final,
    },
)


# ============================================================
# 32) FINAL GATE
# ============================================================

cell111_gate = (
    len(
        cell111_failures
    )
    ==
    0
)


print(
    "=== CELL 111 — FINAL 28-CASE RTL/HTML REPAIR ==="
)


print(
    "\n=== FROZEN INPUTS ==="
)


print(
    "Approved 22 source:",
    str(
        cell111_source22_path
    ),
)


print(
    "Approved 22 SHA256:",
    cell111_source22_sha256,
)


print(
    "Accepted Corpus SHA256:",
    cell111_corpus_sha256,
)


print(
    "Registry SHA256:",
    cell111_registry_sha256_before,
)


print(
    "\n=== AUTHOR REVIEW STATE ==="
)


print(
    "Author Review queue:",
    len(
        cell111_author_cases
    ),
)


print(
    "Frozen old cases:",
    22,
)


print(
    "New cases:",
    6,
)


print(
    "AUTHOR_FEEDBACK_CONTENT:",
    cell111_active_content_count,
)


print(
    "MULTI_TURN_CONTEXT_CONTINUITY:",
    cell111_active_continuity_count,
)


print(
    "\n=== FORMAT REPAIR ==="
)


print(
    "Cases 1-22 regenerated:",
    False,
)


print(
    "Cases 1-22 protected by exact SHA:",
    True,
)


print(
    "Cases 23-28 renderer:",
    "HTML_RTL_MATCHING_APPROVED_22_TEMPLATE",
)


print(
    "New RTL wrappers:",
    cell111_new_segment.count(
        '<div dir="rtl" align="right">'
    ),
)


print(
    "New Author Review sections:",
    cell111_new_segment.count(
        "۵. نظر مؤلف (دکتر کمیل رودی)"
    ),
)


print(
    "Raw Markdown case headings in 23-28:",
    bool(
        re.search(
            r"(?m)^##\s+مورد",
            cell111_new_segment,
        )
    ),
)


print(
    "Raw Markdown H3 headings in 23-28:",
    bool(
        re.search(
            r"(?m)^###\s+",
            cell111_new_segment,
        )
    ),
)


print(
    "Markdown pipe tables in 23-28:",
    bool(
        re.search(
            r"(?m)^\|\s*مرحله\s*\|",
            cell111_new_segment,
        )
    ),
)


print(
    "\n=== OUTPUT ==="
)


print(
    "Output:",
    str(
        cell111_output_path
    ),
)


print(
    "Case sequence:",
    (
        f"{cell111_final_case_numbers[0]}"
        "-"
        f"{cell111_final_case_numbers[-1]}"
    ),
)


print(
    "Output write required:",
    cell111_output_change_required,
)


print(
    "Backup:",
    (
        str(
            cell111_output_backup_path
        )
        if
        cell111_output_backup_path
        is not None
        else
        "NOT_NEEDED"
    ),
)


print(
    "Output SHA256:",
    cell111_final_sha256,
)


print(
    "\n=== SAFETY ==="
)


print(
    "Registry mutated:",
    False,
)


print(
    "Registry SHA256 after:",
    cell111_registry_sha256_final,
)


print(
    "Author decisions recorded:",
    False,
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell111_failures
)


print(
    "\nCODE CELL 111 GATE:",
    (
        "PASS"
        if cell111_gate
        else
        "FAIL"
    ),
)


assert cell111_gate, (
    "Cell 111 RTL/HTML repair failed. "
    "Repair this SAME Cell 111 before continuing."
)

=== CELL 111 — FINAL 28-CASE RTL/HTML REPAIR ===

=== FROZEN INPUTS ===
Approved 22 source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/author_review_22_pilots_REVISED.md
Approved 22 SHA256: 2351a6011387dbc1ea56a8f9ba5fe83c1f1f28d8a9ebb7bd8c262c1f1f27fcfc
Accepted Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Registry SHA256: 10933065b1d4598dba1534964b8a6837f723cf79082d1af2bc8a4964e2a6766e

=== AUTHOR REVIEW STATE ===
Author Review queue: 28
Frozen old cases: 22
New cases: 6
AUTHOR_FEEDBACK_CONTENT: 6
MULTI_TURN_CONTEXT_CONTINUITY: 0

=== FORMAT REPAIR ===
Cases 1-22 regenerated: False
Cases 1-22 protected by exact SHA: True
Cases 23-28 renderer: HTML_RTL_MATCHING_APPROVED_22_TEMPLATE
New RTL wrappers: 6
New Author Review sections: 6
Raw Markdown case headings in 23-28: False
Raw Markdown H3 headings in 23-28: False
Markdown pipe tables in 23-28: False

=== OUTPUT ===
Output: /Users/macbookpro/AMIR_DATA/00

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازیابی اولیه Provenance منبع برای پایلوت‌های ۲۳ تا ۲۸</h3>

<h4>هدف</h4>

<p>
هدف این مرحله بررسی این است که آیا برای شش مورد واقعی اضافه‌شده از دموی فاز یک،
می‌توان منبعی را که پاسخ تاریخی مدل بر اساس آن ساخته شده است با شواهد موجود در Repository
یا مجموعه منابع مستقیم دکتر کمیل رودی بازیابی کرد یا خیر.
</p>

<h4>چرا این مرحله لازم است؟</h4>

<p>
در مراحل قبلی برای پایلوت‌های ۲۳ تا ۲۸، سؤال تاریخی، پاسخ تاریخی مدل،
بازخورد مستقیم دکتر و Source Grounding محتوایی ثبت شده بود.
اما Content Grounding به‌تنهایی اثبات نمی‌کند که همان مقاله،
منبع واقعی مورد استفاده در اجرای دموی فاز یک بوده است.
</p>

<p>
بنابراین پیش از قرار دادن «متن منبع اصلی» در بسته بازبینی مؤلف،
باید Provenance واقعی تا حدی که Evidence موجود اجازه می‌دهد بررسی شود.
</p>

<h4>ورودی‌های این مرحله</h4>

<ul>
<li>Registry فعلی Human / Author Review و شش پایلوت ۲۳ تا ۲۸.</li>
<li>سؤال‌های واقعی ثبت‌شده در دموی فاز یک.</li>
<li>پاسخ‌های واقعی مدل فاز یک.</li>
<li>بازخورد مستقیم دکتر کمیل رودی.</li>
<li>Accepted Corpus فعلی پروژه.</li>
<li>Artifactهای موجود مرتبط با Phase 1 / Baseline / Demo در Repository.</li>
<li>۷۶ سند Website با Authority نوع <code>DIRECT_AUTHOR</code>.</li>
</ul>

<h4>عملیات</h4>

<p>
این مرحله ابتدا Artifactهای موجود فاز یک را به‌صورت Read-Only بررسی می‌کند
تا مشخص شود آیا Question، Answer، Document ID، URL، Citation یا Source Reference
واقعی از اجرای دمو باقی مانده است یا خیر.
</p>

<p>
اگر Provenance صریح پیدا نشود، متن سؤال تاریخی، پاسخ مدل و بازخورد دکتر
با ۷۶ مقاله Direct Author موجود در Corpus مقایسه می‌شود
و نزدیک‌ترین Candidateها فقط برای بررسی انسانی گزارش می‌شوند.
</p>

<h4>تفکیک مفهومی الزامی</h4>

<p>
در این مرحله سه وضعیت نباید با یکدیگر اشتباه شوند:
</p>

<ul>
<li>
<strong>Original Source Provenance:</strong>
مدرکی که Source واقعی استفاده‌شده در اجرای دموی فاز یک را اثبات کند.
</li>

<li>
<strong>Source Match:</strong>
منبعی که شباهت محتوایی بسیار بالایی با سؤال یا پاسخ تاریخی دارد،
اما استفاده واقعی آن توسط Demo اثبات نشده است.
</li>

<li>
<strong>Supporting Grounding:</strong>
منبعی که محتوای درست را پشتیبانی می‌کند،
ولی ممکن است اصلاً Source پاسخ تاریخی مدل نبوده باشد.
</li>
</ul>

<h4>خروجی مورد انتظار</h4>

<p>
برای هر یک از پایلوت‌های ۲۳ تا ۲۸ باید موارد زیر گزارش شود:
</p>

<ul>
<li>سؤال تاریخی.</li>
<li>پاسخ تاریخی مدل.</li>
<li>بازخورد مستقیم دکتر.</li>
<li>وجود یا عدم وجود Artifact واقعی فاز یک.</li>
<li>وجود یا عدم وجود Source Reference صریح.</li>
<li>Candidateهای برتر در مجموعه ۷۶ مقاله Direct Author.</li>
<li>بخشی از متن واقعی Candidateها.</li>
<li>وضعیت Provenance بدون ادعای فراتر از Evidence.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>هر شش پایلوت ۲۳ تا ۲۸ بررسی شوند.</li>
<li>دقیقاً Scope موردنظر این مرحله یعنی ۷۶ سند Direct Author بررسی شود.</li>
<li>Artifactهای قابل‌دسترسی فاز یک بدون تغییر بررسی شوند.</li>
<li>نبود Provenance صریح مخفی نشود.</li>
<li>Source Match با Original Source اشتباه گرفته نشود.</li>
<li>Registry، Corpus و فایل Author Review تغییر نکنند.</li>
<li>خروجی نهایی Cell برابر <code>CODE CELL 112 GATE: PASS</code> باشد.</li>
</ul>

<h4>معیار FAIL</h4>

<ul>
<li>یکی از شش پایلوت از بررسی جا بماند.</li>
<li>Scope ۷۶ سند Direct Author به‌درستی بازیابی نشود.</li>
<li>Hash یکی از Artifactهای Freeze‌شده تغییر کند.</li>
<li>یک Candidate صرفاً بر اساس شباهت به‌عنوان Original Source قطعی معرفی شود.</li>
<li>Registry، Corpus یا Author Review Markdown تغییر کند.</li>
</ul>

<h4>خارج از Scope</h4>

<p>
این مرحله فایل Author Review را اصلاح نمی‌کند،
تصمیم مؤلف ثبت نمی‌کند و هیچ Source Candidate را هنوز Accepted نمی‌کند.
همچنین جست‌وجوی تمام Websiteهای Corpus در Scope این مرحله نیست
و در Step بعدی به‌صورت مستقل انجام می‌شود.
</p>

<h4>ایمنی نوشتن</h4>

<ul>
<li>Registry Write: ممنوع.</li>
<li>Corpus Write: ممنوع.</li>
<li>Author Review Markdown Write: ممنوع.</li>
<li>Author Decision Write: ممنوع.</li>
<li>Commit / Push: ممنوع.</li>
</ul>

</div>

In [5]:
# Code Cell 112
# ============================================================
# PHASE-1 ORIGINAL SOURCE PROVENANCE RECOVERY
# PILOTS 23-28
#
# GOAL:
# 1) First search Phase-1 baseline/demo artifacts for an
#    explicit source/citation tied to the historical Q/A.
# 2) Independently search ONLY the 76 Direct-Author Website
#    documents as a fallback / cross-check.
# 3) DO NOT mutate Registry.
# 4) DO NOT mutate author_review_28_pilots.md.
# 5) DO NOT claim "original source" unless explicit provenance
#    evidence is found.
# ============================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import math
import re
import unicodedata


# ============================================================
# 1) Constants
# ============================================================

CELL112_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1"
    "da34869f671e6e32762864e0ad70ea40"
)

CELL112_EXPECTED_REGISTRY_SHA256 = (
    "10933065b1d4598dba1534964b8a6837"
    "f723cf79082d1af2bc8a4964e2a6766e"
)

CELL112_EXPECTED_DIRECT_AUTHOR_WEBSITE_COUNT = 76

CELL112_CASE_NUMBERS = [23, 24, 25, 26, 27, 28]

CELL112_BATCH_KEY = "phase1_author_feedback_derived_pilots"

CELL112_CORPUS_REL = Path(
    "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl"
)

CELL112_REGISTRY_REL = Path(
    "data/phase2/human_review/human_review_registry.json"
)

CELL112_AUTHOR_MD_REL = Path(
    "notebooks/phase2/author_review_28_pilots.md"
)

CELL112_KNOWN_PHASE1_ARTIFACTS = [
    Path("reports/baseline/phase1_results.json"),
    Path("docs/baseline/phase1_baseline.md"),
    Path("reports/phase2/baseline/phase1_baseline.json"),
    Path("reports/phase2/baseline/phase1_results.json"),
    Path("reports/phase2/baseline/known_limitations.md"),
    Path("artifacts/phase2/run_context.json"),
]

CELL112_SCAN_ROOTS = [
    "reports",
    "docs",
    "artifacts",
]

CELL112_SCAN_EXTENSIONS = {
    ".json",
    ".jsonl",
    ".md",
    ".txt",
    ".csv",
}

CELL112_MAX_SCAN_FILE_BYTES = 25 * 1024 * 1024

CELL112_TOP_K = 5

cell112_failures = []


# ============================================================
# 2) Basic utilities
# ============================================================

def cell112_check(reason, condition, details=None):

    if not bool(condition):

        item = {
            "reason": reason,
        }

        if details is not None:
            item["details"] = details

        cell112_failures.append(item)

    return bool(condition)


def cell112_sha256_file(path):

    h = hashlib.sha256()

    with path.open("rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


def cell112_find_repo_root():

    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
    ]

    for name in [
        "cell111_repo_root",
        "cell110_repo_root",
        "cell109_repo_root",
    ]:

        if name in globals():

            try:
                candidates.insert(
                    0,
                    Path(globals()[name]).resolve(),
                )
            except Exception:
                pass

    seen = set()

    for candidate in candidates:

        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(str(candidate))

        if (
            (candidate / CELL112_CORPUS_REL).exists()
            and
            (candidate / CELL112_REGISTRY_REL).exists()
        ):
            return candidate

    return None


def cell112_normalize(text):

    text = "" if text is None else str(text)

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    replacements = {
        "ي": "ی",
        "ك": "ک",
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "إ": "ا",
        "أ": "ا",
        "ٱ": "ا",
        "\u200c": " ",
        "\u200f": " ",
        "\u200e": " ",
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(
        r"[^\w\u0600-\u06FF:/\.\-]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.lower()


CELL112_STOPWORDS = {
    "از", "به", "در", "با", "برای", "را",
    "که", "و", "یا", "این", "آن", "یک",
    "است", "هست", "بود", "شد", "شود",
    "می", "چه", "چرا", "چگونه", "اگر",
    "اما", "هم", "روی", "خود", "ما",
    "من", "شما", "فرد", "افراد",
    "درباره", "همین", "دیگه",
}


def cell112_tokens(text):

    return [
        token
        for token in cell112_normalize(text).split()
        if (
            len(token) >= 2
            and
            token not in CELL112_STOPWORDS
        )
    ]


def cell112_recursive_values(obj, wanted_key):

    results = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            if str(key).lower() == wanted_key.lower():
                results.append(value)

            results.extend(
                cell112_recursive_values(
                    value,
                    wanted_key,
                )
            )

    elif isinstance(obj, list):

        for value in obj:

            results.extend(
                cell112_recursive_values(
                    value,
                    wanted_key,
                )
            )

    return results


def cell112_first_recursive_value(
    obj,
    keys,
):

    for key in keys:

        values = cell112_recursive_values(
            obj,
            key,
        )

        for value in values:

            if (
                value is not None
                and
                not isinstance(
                    value,
                    (dict, list),
                )
                and
                str(value).strip()
            ):
                return value

    return None


# ============================================================
# 3) Resolve repository
# ============================================================

cell112_repo_root = cell112_find_repo_root()

cell112_check(
    "REPO_ROOT_NOT_FOUND",
    cell112_repo_root is not None,
)

if cell112_failures:
    raise AssertionError(cell112_failures)


cell112_corpus_path = (
    cell112_repo_root
    /
    CELL112_CORPUS_REL
)

cell112_registry_path = (
    cell112_repo_root
    /
    CELL112_REGISTRY_REL
)

cell112_author_md_path = (
    cell112_repo_root
    /
    CELL112_AUTHOR_MD_REL
)


# ============================================================
# 4) Freeze bytes before any inspection
# ============================================================

cell112_corpus_sha_before = (
    cell112_sha256_file(
        cell112_corpus_path
    )
)

cell112_registry_sha_before = (
    cell112_sha256_file(
        cell112_registry_path
    )
)

cell112_author_md_sha_before = (
    cell112_sha256_file(
        cell112_author_md_path
    )
    if cell112_author_md_path.exists()
    else None
)


cell112_check(
    "CORPUS_SHA_MISMATCH",
    cell112_corpus_sha_before
    ==
    CELL112_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL112_EXPECTED_CORPUS_SHA256,
        "actual":
            cell112_corpus_sha_before,
    },
)

cell112_check(
    "REGISTRY_SHA_MISMATCH",
    cell112_registry_sha_before
    ==
    CELL112_EXPECTED_REGISTRY_SHA256,
    {
        "expected":
            CELL112_EXPECTED_REGISTRY_SHA256,
        "actual":
            cell112_registry_sha_before,
    },
)

if cell112_failures:

    print("=== FROZEN INPUT FAILURE ===")
    print(cell112_failures)

    raise AssertionError(
        "Cell 112 stopped Fail-Closed."
    )


# ============================================================
# 5) Load six historical Phase-1 cases
# ============================================================

cell112_registry = json.loads(
    cell112_registry_path.read_text(
        encoding="utf-8"
    )
)

cell112_author_cases = (
    (
        cell112_registry.get(
            "author_review",
            {},
        )
        or
        {}
    ).get(
        "cases_ready_for_review",
        [],
    )
    or
    []
)

cell112_cases = {
    case.get(
        "reference_pilot_number"
    ):
        case

    for case in cell112_author_cases

    if case.get(
        "reference_pilot_number"
    )
    in CELL112_CASE_NUMBERS
}

cell112_check(
    "PILOTS_23_28_NOT_ALL_FOUND",
    sorted(cell112_cases.keys())
    ==
    CELL112_CASE_NUMBERS,
    sorted(cell112_cases.keys()),
)

if cell112_failures:
    raise AssertionError(cell112_failures)


def cell112_history_payload(case):

    history = (
        case.get(
            "historical_phase1_demo",
            {},
        )
        or
        {}
    )

    conversation = (
        history.get(
            "raw_conversation",
            [],
        )
        or
        []
    )

    human_texts = [
        str(row.get("text") or "")
        for row in conversation
        if (
            isinstance(row, dict)
            and
            row.get("speaker") == "HUMAN"
        )
    ]

    model_texts = [
        str(row.get("text") or "")
        for row in conversation
        if (
            isinstance(row, dict)
            and
            row.get("speaker") == "PHASE1_MODEL"
        )
    ]

    raw_feedback = str(
        history.get(
            "raw_author_feedback"
        )
        or
        ""
    )

    return {
        "human_texts":
            human_texts,

        "model_texts":
            model_texts,

        "raw_feedback":
            raw_feedback,

        "human_joined":
            "\n".join(human_texts),

        "model_joined":
            "\n".join(model_texts),
    }


cell112_history_by_number = {
    number:
        cell112_history_payload(
            cell112_cases[number]
        )

    for number
    in CELL112_CASE_NUMBERS
}


# ============================================================
# 6) Load accepted Corpus and isolate exactly
#    the 76 Website / Direct-Author records
# ============================================================

cell112_corpus_records = []

with cell112_corpus_path.open(
    "r",
    encoding="utf-8",
) as f:

    for line_number, line in enumerate(
        f,
        start=1,
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        record["_cell112_line_number"] = (
            line_number
        )

        cell112_corpus_records.append(
            record
        )


def cell112_is_direct_author_website(record):

    document_id = str(
        record.get(
            "document_id"
        )
        or
        ""
    ).lower()

    source_type_values = [
        str(v).lower()
        for v in cell112_recursive_values(
            record,
            "source_type",
        )
        if not isinstance(
            v,
            (dict, list),
        )
    ]

    authority_values = [
        str(v).lower()
        for v in cell112_recursive_values(
            record,
            "content_authority_type",
        )
        if not isinstance(
            v,
            (dict, list),
        )
    ]

    website_ok = (
        document_id.startswith(
            "website:"
        )
        or
        "website"
        in
        source_type_values
    )

    direct_ok = (
        "direct_author"
        in
        authority_values
    )

    return (
        website_ok
        and
        direct_ok
    )


cell112_direct_author_records = [
    record
    for record in cell112_corpus_records
    if cell112_is_direct_author_website(
        record
    )
]


cell112_check(
    "DIRECT_AUTHOR_WEBSITE_COUNT_NOT_76",
    len(
        cell112_direct_author_records
    )
    ==
    CELL112_EXPECTED_DIRECT_AUTHOR_WEBSITE_COUNT,
    len(
        cell112_direct_author_records
    ),
)

if cell112_failures:

    print(
        "=== DIRECT AUTHOR FILTER FAILURE ==="
    )

    print(cell112_failures)

    raise AssertionError(
        "Do not continue until the exact 76-record scope is recovered."
    )


# ============================================================
# 7) Standardized document metadata
# ============================================================

def cell112_record_text(record):

    value = record.get("text")

    if isinstance(value, str) and value.strip():
        return value

    value = cell112_first_recursive_value(
        record,
        [
            "source_text",
            "normalized_text",
            "content_text",
            "article_text",
        ],
    )

    return (
        str(value)
        if value is not None
        else ""
    )


def cell112_record_title(record):

    value = record.get("title")

    if value:
        return str(value)

    value = cell112_first_recursive_value(
        record,
        [
            "title",
            "article_title",
        ],
    )

    return (
        str(value)
        if value
        else "بدون عنوان"
    )


def cell112_record_url(record):

    value = cell112_first_recursive_value(
        record,
        [
            "canonical_url",
            "url",
        ],
    )

    return (
        str(value)
        if value
        else ""
    )


cell112_direct_docs = []

for record in cell112_direct_author_records:

    cell112_direct_docs.append({
        "document_id":
            str(
                record.get(
                    "document_id"
                )
                or
                ""
            ),

        "title":
            cell112_record_title(
                record
            ),

        "url":
            cell112_record_url(
                record
            ),

        "text":
            cell112_record_text(
                record
            ),

        "record":
            record,
    })


cell112_doc_by_id = {
    row["document_id"]: row
    for row in cell112_direct_docs
}

cell112_doc_by_url = {
    cell112_normalize(row["url"]):
        row

    for row in cell112_direct_docs

    if row["url"]
}

cell112_doc_by_title = {
    cell112_normalize(row["title"]):
        row

    for row in cell112_direct_docs

    if row["title"]
}


# ============================================================
# 8) Discover Phase-1 / demo / baseline artifacts
# ============================================================

cell112_artifact_paths = []

for rel in CELL112_KNOWN_PHASE1_ARTIFACTS:

    path = (
        cell112_repo_root
        /
        rel
    )

    if (
        path.exists()
        and
        path.is_file()
    ):
        cell112_artifact_paths.append(
            path
        )


for root_name in CELL112_SCAN_ROOTS:

    root = (
        cell112_repo_root
        /
        root_name
    )

    if not root.exists():
        continue

    for path in root.rglob("*"):

        if not path.is_file():
            continue

        if (
            path.suffix.lower()
            not in
            CELL112_SCAN_EXTENSIONS
        ):
            continue

        lowered = str(path).lower()

        if not any(
            cue in lowered
            for cue in [
                "phase1",
                "phase_1",
                "baseline",
                "demo",
                "feedback",
            ]
        ):
            continue

        try:
            size = path.stat().st_size
        except Exception:
            continue

        if (
            size
            >
            CELL112_MAX_SCAN_FILE_BYTES
        ):
            continue

        cell112_artifact_paths.append(
            path
        )


# Deduplicate.
cell112_artifact_paths = sorted(
    set(
        p.resolve()
        for p
        in cell112_artifact_paths
    ),
    key=str,
)


# ============================================================
# 9) Read candidate Phase-1 artifacts
# ============================================================

cell112_artifact_texts = {}

for path in cell112_artifact_paths:

    try:

        cell112_artifact_texts[path] = (
            path.read_text(
                encoding="utf-8",
                errors="replace",
            )
        )

    except Exception:
        pass


# ============================================================
# 10) Find exact historical Q/A traces
# ============================================================

def cell112_find_text_hits(
    needle,
):

    needle_norm = cell112_normalize(
        needle
    )

    results = []

    if not needle_norm:
        return results

    for path, text in (
        cell112_artifact_texts.items()
    ):

        normalized = cell112_normalize(
            text
        )

        if needle_norm in normalized:

            raw_index = (
                text.find(
                    str(needle).strip()
                )
            )

            results.append({
                "path":
                    path,

                "raw_exact":
                    raw_index
                    >=
                    0,

                "normalized_match":
                    True,
            })

    return results


cell112_baseline_hits = {}

for number in CELL112_CASE_NUMBERS:

    history = (
        cell112_history_by_number[
            number
        ]
    )

    case_hits = {
        "human": [],
        "model": [],
    }

    for text in history["human_texts"]:

        case_hits[
            "human"
        ].extend(
            cell112_find_text_hits(
                text
            )
        )

    for text in history["model_texts"]:

        case_hits[
            "model"
        ].extend(
            cell112_find_text_hits(
                text
            )
        )

    cell112_baseline_hits[
        number
    ] = case_hits


# ============================================================
# 11) Extract local context around a Phase-1 hit
# ============================================================

def cell112_context_window(
    full_text,
    needle,
    radius=2500,
):

    if not full_text:
        return ""

    needle = str(
        needle
        or
        ""
    ).strip()

    if not needle:
        return ""

    pos = full_text.find(
        needle
    )

    if pos < 0:

        norm_full = cell112_normalize(
            full_text
        )

        norm_needle = cell112_normalize(
            needle
        )

        pos_norm = norm_full.find(
            norm_needle
        )

        if pos_norm < 0:
            return ""

        # Normalization destroys exact offsets.
        # In this fallback return beginning-limited context.
        return full_text[
            :radius * 2
        ]

    start = max(
        0,
        pos - radius,
    )

    end = min(
        len(full_text),
        pos + len(needle) + radius,
    )

    return full_text[
        start:end
    ]


# ============================================================
# 12) Detect explicit source references near matched Q/A
# ============================================================

def cell112_match_known_source_refs(
    text,
):

    text_norm = cell112_normalize(
        text
    )

    matches = {}

    for doc in cell112_direct_docs:

        doc_id = doc["document_id"]
        title = doc["title"]
        url = doc["url"]

        reasons = []

        if (
            doc_id
            and
            cell112_normalize(
                doc_id
            )
            in
            text_norm
        ):
            reasons.append(
                "DOCUMENT_ID"
            )

        if (
            url
            and
            cell112_normalize(
                url
            )
            in
            text_norm
        ):
            reasons.append(
                "URL"
            )

        title_norm = (
            cell112_normalize(
                title
            )
        )

        if (
            title_norm
            and
            len(title_norm) >= 8
            and
            title_norm
            in
            text_norm
        ):
            reasons.append(
                "TITLE"
            )

        if reasons:

            matches[
                doc_id
            ] = {
                "document_id":
                    doc_id,

                "title":
                    title,

                "url":
                    url,

                "match_reasons":
                    reasons,
            }

    return list(
        matches.values()
    )


cell112_explicit_phase1_refs = {
    number: []
    for number in CELL112_CASE_NUMBERS
}


for number in CELL112_CASE_NUMBERS:

    history = (
        cell112_history_by_number[
            number
        ]
    )

    needles = (
        history[
            "human_texts"
        ]
        +
        history[
            "model_texts"
        ]
    )

    seen = set()

    for path, full_text in (
        cell112_artifact_texts.items()
    ):

        for needle in needles:

            if (
                cell112_normalize(
                    needle
                )
                not in
                cell112_normalize(
                    full_text
                )
            ):
                continue

            context = (
                cell112_context_window(
                    full_text,
                    needle,
                    radius=3500,
                )
            )

            refs = (
                cell112_match_known_source_refs(
                    context
                )
            )

            for ref in refs:

                key = (
                    str(path),
                    ref[
                        "document_id"
                    ],
                )

                if key in seen:
                    continue

                seen.add(key)

                cell112_explicit_phase1_refs[
                    number
                ].append({
                    "artifact":
                        str(
                            path.relative_to(
                                cell112_repo_root
                            )
                        ),

                    **ref,
                })


# ============================================================
# 13) IDF over exactly the 76 articles
# ============================================================

cell112_doc_token_sets = {}

cell112_df = Counter()

for doc in cell112_direct_docs:

    tokens = set(
        cell112_tokens(
            doc["text"]
        )
    )

    cell112_doc_token_sets[
        doc["document_id"]
    ] = tokens

    for token in tokens:
        cell112_df[token] += 1


def cell112_idf(token):

    return (
        math.log(
            (
                1
                +
                CELL112_EXPECTED_DIRECT_AUTHOR_WEBSITE_COUNT
            )
            /
            (
                1
                +
                cell112_df.get(
                    token,
                    0,
                )
            )
        )
        +
        1.0
    )


# ============================================================
# 14) Phrase overlap
# ============================================================

def cell112_ngrams(
    tokens,
    n,
):

    return [
        " ".join(
            tokens[
                i:i+n
            ]
        )
        for i in range(
            max(
                0,
                len(tokens)
                -
                n
                +
                1,
            )
        )
    ]


def cell112_phrase_hits(
    source_text,
    target_text,
):

    source_norm = cell112_normalize(
        source_text
    )

    target_tokens = cell112_tokens(
        target_text
    )

    hits = []

    for n in [5, 4, 3]:

        for phrase in cell112_ngrams(
            target_tokens,
            n,
        ):

            if phrase in source_norm:

                hits.append({
                    "n":
                        n,

                    "phrase":
                        phrase,
                })

    # Stable unique phrases.
    unique = []
    seen = set()

    for row in hits:

        key = (
            row["n"],
            row["phrase"],
        )

        if key in seen:
            continue

        seen.add(key)
        unique.append(row)

    return unique


# ============================================================
# 15) Rank 76 articles for each historical demo case
# ============================================================

def cell112_weighted_coverage(
    query_text,
    doc_token_set,
):

    query_tokens = set(
        cell112_tokens(
            query_text
        )
    )

    if not query_tokens:
        return 0.0, []

    matched = (
        query_tokens
        &
        doc_token_set
    )

    denominator = sum(
        cell112_idf(token)
        for token
        in query_tokens
    )

    numerator = sum(
        cell112_idf(token)
        for token
        in matched
    )

    coverage = (
        numerator
        /
        denominator
        if denominator
        else
        0.0
    )

    matched_sorted = sorted(
        matched,
        key=lambda token:
            (
                -cell112_idf(
                    token
                ),
                token,
            )
    )

    return (
        coverage,
        matched_sorted,
    )


def cell112_rank_case(
    number,
):

    history = (
        cell112_history_by_number[
            number
        ]
    )

    human_text = (
        history[
            "human_joined"
        ]
    )

    model_text = (
        history[
            "model_joined"
        ]
    )

    feedback_text = (
        history[
            "raw_feedback"
        ]
    )

    rows = []

    explicit_ids = {
        row["document_id"]
        for row
        in
        cell112_explicit_phase1_refs[
            number
        ]
    }

    for doc in cell112_direct_docs:

        doc_id = doc[
            "document_id"
        ]

        token_set = (
            cell112_doc_token_sets[
                doc_id
            ]
        )

        human_cov, human_terms = (
            cell112_weighted_coverage(
                human_text,
                token_set,
            )
        )

        model_cov, model_terms = (
            cell112_weighted_coverage(
                model_text,
                token_set,
            )
        )

        feedback_cov, feedback_terms = (
            cell112_weighted_coverage(
                feedback_text,
                token_set,
            )
        )

        model_phrase_hits = (
            cell112_phrase_hits(
                doc["text"],
                model_text,
            )
        )

        feedback_phrase_hits = (
            cell112_phrase_hits(
                doc["text"],
                feedback_text,
            )
        )

        phrase_bonus = sum(
            hit["n"] ** 2
            for hit
            in
            model_phrase_hits[:12]
        )

        feedback_phrase_bonus = sum(
            hit["n"]
            for hit
            in
            feedback_phrase_hits[:8]
        )

        explicit_bonus = (
            1000.0
            if doc_id in explicit_ids
            else
            0.0
        )

        score = (
            explicit_bonus
            +
            (human_cov * 20.0)
            +
            (model_cov * 50.0)
            +
            (feedback_cov * 30.0)
            +
            phrase_bonus
            +
            feedback_phrase_bonus
        )

        rows.append({
            "document_id":
                doc_id,

            "title":
                doc["title"],

            "url":
                doc["url"],

            "score":
                round(
                    score,
                    4,
                ),

            "human_coverage":
                round(
                    human_cov,
                    4,
                ),

            "model_coverage":
                round(
                    model_cov,
                    4,
                ),

            "feedback_coverage":
                round(
                    feedback_cov,
                    4,
                ),

            "model_phrase_hits":
                model_phrase_hits[:8],

            "feedback_phrase_hits":
                feedback_phrase_hits[:5],

            "matched_model_terms":
                model_terms[:15],

            "matched_feedback_terms":
                feedback_terms[:15],

            "explicit_phase1_reference":
                doc_id
                in
                explicit_ids,

            "text":
                doc["text"],
        })

    rows.sort(
        key=lambda row: (
            -row["score"],
            row["document_id"],
        )
    )

    return rows


cell112_rankings = {
    number:
        cell112_rank_case(
            number
        )

    for number
    in CELL112_CASE_NUMBERS
}


# ============================================================
# 16) Evidence window from each article
# ============================================================

def cell112_best_evidence_window(
    text,
    query_parts,
    max_chars=1200,
):

    text = str(
        text
        or
        ""
    ).strip()

    if not text:
        return ""

    query_tokens = set()

    for part in query_parts:
        query_tokens.update(
            cell112_tokens(
                part
            )
        )

    sentences = [
        part.strip()
        for part in re.split(
            r"(?<=[\.\!\؟])\s+|\n+",
            text,
        )
        if part.strip()
    ]

    if not sentences:
        return text[
            :max_chars
        ]

    best_window = None
    best_score = -1.0

    for i in range(
        len(sentences)
    ):

        window = " ".join(
            sentences[
                i:i+4
            ]
        ).strip()

        tokens = set(
            cell112_tokens(
                window
            )
        )

        matched = (
            query_tokens
            &
            tokens
        )

        score = sum(
            cell112_idf(
                token
            )
            for token
            in matched
        )

        if score > best_score:

            best_score = score
            best_window = window

    best_window = (
        best_window
        or
        text
    )

    if len(
        best_window
    ) > max_chars:

        best_window = (
            best_window[
                :max_chars
            ].rstrip()
            +
            " ..."
        )

    return best_window


# ============================================================
# 17) Conservative provenance classification
# ============================================================

def cell112_classify(
    number,
):

    explicit_refs = (
        cell112_explicit_phase1_refs[
            number
        ]
    )

    if explicit_refs:

        return (
            "EXACT_ORIGINAL_SOURCE_CONFIRMED"
        )

    top = (
        cell112_rankings[
            number
        ][0]
    )

    strong_phrase = any(
        hit["n"] >= 4
        for hit
        in top[
            "model_phrase_hits"
        ]
    )

    if (
        top["model_coverage"] >= 0.55
        and
        (
            strong_phrase
            or
            top["feedback_coverage"]
            >=
            0.55
        )
    ):

        return (
            "HIGH_CONFIDENCE_SOURCE_MATCH"
        )

    return (
        "SUPPORTING_SOURCE_NOT_PROVEN_ORIGINAL"
    )


cell112_status_by_number = {
    number:
        cell112_classify(
            number
        )

    for number
    in CELL112_CASE_NUMBERS
}


# ============================================================
# 18) Print full Human Review packet
# ============================================================

print(
    "=== CELL 112 — PHASE-1 ORIGINAL SOURCE PROVENANCE RECOVERY ==="
)

print(
    "\n=== FROZEN INPUTS ==="
)

print(
    "Repo:",
    cell112_repo_root,
)

print(
    "Corpus:",
    cell112_corpus_path,
)

print(
    "Corpus SHA256:",
    cell112_corpus_sha_before,
)

print(
    "Registry SHA256:",
    cell112_registry_sha_before,
)

print(
    "Author MD SHA256:",
    cell112_author_md_sha_before,
)

print(
    "Accepted Corpus records:",
    len(
        cell112_corpus_records
    ),
)

print(
    "Direct-Author Website records:",
    len(
        cell112_direct_author_records
    ),
)

print(
    "\nPhase-1 / baseline artifacts scanned:",
    len(
        cell112_artifact_texts
    ),
)

for path in cell112_artifact_texts:

    print(
        "-",
        path.relative_to(
            cell112_repo_root
        ),
    )


for number in CELL112_CASE_NUMBERS:

    case = (
        cell112_cases[
            number
        ]
    )

    history = (
        cell112_history_by_number[
            number
        ]
    )

    ranking = (
        cell112_rankings[
            number
        ]
    )

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "PILOT:",
        number,
    )

    print(
        "Pilot ID:",
        case.get(
            "pilot_case_id"
        ),
    )

    print(
        "\n--- HISTORICAL PHASE-1 QUESTION(S) ---"
    )

    for text in history[
        "human_texts"
    ]:
        print(text)

    print(
        "\n--- HISTORICAL PHASE-1 MODEL ANSWER(S) ---"
    )

    for text in history[
        "model_texts"
    ]:
        print(text)

    print(
        "\n--- DR. ROUDI PHASE-1 FEEDBACK ---"
    )

    print(
        history[
            "raw_feedback"
        ]
    )

    print(
        "\n--- EXPLICIT PHASE-1 ARTIFACT HITS ---"
    )

    human_hits = (
        cell112_baseline_hits[
            number
        ][
            "human"
        ]
    )

    model_hits = (
        cell112_baseline_hits[
            number
        ][
            "model"
        ]
    )

    print(
        "Historical question artifact hits:",
        len(
            human_hits
        ),
    )

    for hit in human_hits:

        print(
            "-",
            hit["path"].relative_to(
                cell112_repo_root
            ),
            "| raw_exact =",
            hit[
                "raw_exact"
            ],
        )

    print(
        "Historical model-answer artifact hits:",
        len(
            model_hits
        ),
    )

    for hit in model_hits:

        print(
            "-",
            hit["path"].relative_to(
                cell112_repo_root
            ),
            "| raw_exact =",
            hit[
                "raw_exact"
            ],
        )

    print(
        "\n--- EXPLICIT SOURCE REFERENCES NEAR PHASE-1 Q/A ---"
    )

    explicit_refs = (
        cell112_explicit_phase1_refs[
            number
        ]
    )

    if explicit_refs:

        for ref in explicit_refs:

            print(
                "- Artifact:",
                ref[
                    "artifact"
                ],
            )

            print(
                "  Document ID:",
                ref[
                    "document_id"
                ],
            )

            print(
                "  Title:",
                ref[
                    "title"
                ],
            )

            print(
                "  URL:",
                ref[
                    "url"
                ],
            )

            print(
                "  Match reasons:",
                ref[
                    "match_reasons"
                ],
            )

    else:

        print(
            "NONE — no explicit Phase-1 source reference was proven."
        )

    print(
        "\n--- PROVENANCE CLASSIFICATION ---"
    )

    print(
        cell112_status_by_number[
            number
        ]
    )

    print(
        "\n--- TOP 76-ARTICLE SOURCE CANDIDATES ---"
    )

    for rank, row in enumerate(
        ranking[
            :CELL112_TOP_K
        ],
        start=1,
    ):

        print(
            "\n",
            "-" * 100,
            sep="",
        )

        print(
            "Rank:",
            rank,
        )

        print(
            "Document ID:",
            row[
                "document_id"
            ],
        )

        print(
            "Title:",
            row[
                "title"
            ],
        )

        print(
            "URL:",
            row[
                "url"
            ],
        )

        print(
            "Score:",
            row[
                "score"
            ],
        )

        print(
            "Explicit Phase-1 reference:",
            row[
                "explicit_phase1_reference"
            ],
        )

        print(
            "Historical question coverage:",
            row[
                "human_coverage"
            ],
        )

        print(
            "Historical model answer coverage:",
            row[
                "model_coverage"
            ],
        )

        print(
            "Dr feedback coverage:",
            row[
                "feedback_coverage"
            ],
        )

        print(
            "Model phrase hits:",
            row[
                "model_phrase_hits"
            ],
        )

        print(
            "Matched model terms:",
            row[
                "matched_model_terms"
            ],
        )

        print(
            "\nEvidence window:"
        )

        print(
            cell112_best_evidence_window(
                row[
                    "text"
                ],
                [
                    history[
                        "human_joined"
                    ],
                    history[
                        "model_joined"
                    ],
                    history[
                        "raw_feedback"
                    ],
                ],
            )
        )

    print(
        "\nHUMAN PROVENANCE DECISION: NOT_STARTED"
    )


# ============================================================
# 19) Verify absolute read-only behavior
# ============================================================

cell112_corpus_sha_after = (
    cell112_sha256_file(
        cell112_corpus_path
    )
)

cell112_registry_sha_after = (
    cell112_sha256_file(
        cell112_registry_path
    )
)

cell112_author_md_sha_after = (
    cell112_sha256_file(
        cell112_author_md_path
    )
    if cell112_author_md_path.exists()
    else None
)


cell112_check(
    "CORPUS_MUTATED",
    cell112_corpus_sha_after
    ==
    cell112_corpus_sha_before,
)

cell112_check(
    "REGISTRY_MUTATED",
    cell112_registry_sha_after
    ==
    cell112_registry_sha_before,
)

cell112_check(
    "AUTHOR_MD_MUTATED",
    cell112_author_md_sha_after
    ==
    cell112_author_md_sha_before,
)


# ============================================================
# 20) Final summary
# ============================================================

print(
    "\n"
    +
    "=" * 120
)

print(
    "=== PROVENANCE SUMMARY ==="
)

for number in CELL112_CASE_NUMBERS:

    top = (
        cell112_rankings[
            number
        ][0]
    )

    print(
        (
            f"Pilot {number}: "
            f"{cell112_status_by_number[number]}"
        )
    )

    print(
        "  Top candidate:",
        top[
            "document_id"
        ],
        "|",
        top[
            "title"
        ],
    )

    print(
        "  Explicit Phase-1 refs:",
        len(
            cell112_explicit_phase1_refs[
                number
            ]
        ),
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    False,
)

print(
    "Author MD write:",
    False,
)

print(
    "Corpus write:",
    False,
)

print(
    "Author decisions recorded:",
    False,
)

print(
    "Commit / Push:",
    False,
)

print(
    "\n=== FAILURES ==="
)

print(
    cell112_failures
)

cell112_gate = (
    len(
        cell112_failures
    )
    ==
    0
)

print(
    "\nCODE CELL 112 GATE:",
    (
        "PASS"
        if cell112_gate
        else
        "FAIL"
    ),
)

assert cell112_gate, (
    "Cell 112 provenance recovery failed. "
    "Do not modify the 28-case Author Review document yet."
)

=== CELL 112 — PHASE-1 ORIGINAL SOURCE PROVENANCE RECOVERY ===

=== FROZEN INPUTS ===
Repo: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Corpus: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl
Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Registry SHA256: 10933065b1d4598dba1534964b8a6837f723cf79082d1af2bc8a4964e2a6766e
Author MD SHA256: f9b2e04a50e61060b150c958d444f1308379c76815edb299c25b96257b031f09
Accepted Corpus records: 19456
Direct-Author Website records: 76

Phase-1 / baseline artifacts scanned: 1
- reports/phase2/environment_baseline_v1.json

PILOT: 23
Pilot ID: question-pilot:a0f61a7f0c946fe1d79095dbb96a9497

--- HISTORICAL PHASE-1 QUESTION(S) ---
سواد مالی درباره پول دار شدن دیگه؟

--- HISTORICAL PHASE-1 MODEL ANSWER(S) ---
سواد مالی به رابطۀ انسان با پول می‌پردازد، نه فقط به خود پول. پول در سواد مالی، یک پدیده‌ای بیرونی نیست

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>جست‌وجوی کامل Website Corpus برای منبع پایلوت‌های ۲۳ تا ۲۸</h3>

<h4>هدف</h4>

<p>
هدف این مرحله حذف محدودیت Direct Author و بررسی تمام محتوای Website موجود
در Accepted Corpus است تا مشخص شود آیا متن یا مقاله‌ای که سؤال،
پاسخ تاریخی مدل یا بازخورد دکتر به آن نزدیک است در بخش دیگری از Website Corpus وجود دارد یا خیر.
</p>

<h4>چرا این مرحله لازم است؟</h4>

<p>
مرحله قبل فقط ۷۶ مقاله Direct Author را بررسی کرد و برای هیچ‌یک از شش پایلوت
Source Reference صریح از اجرای دموی فاز یک پیدا نشد.
</p>

<p>
این نتیجه ثابت نمی‌کند که Source در Corpus وجود ندارد؛
فقط ثابت می‌کند که جست‌وجوی محدود قبلی Provenance قطعی تولید نکرده است.
ممکن است متن مرتبط در Website Record دیگری،
Article دیگری، نسخه دیگری از محتوا یا رکوردی با Attribution متفاوت قرار داشته باشد.
</p>

<p>
بنابراین در این مرحله هیچ محدودیتی بر اساس Author، Authority یا Primary Knowledge Owner اعمال نمی‌شود.
هر رکوردی که واقعاً از نوع Website باشد در Scope بررسی قرار می‌گیرد.
</p>

<h4>Source of Truth</h4>

<p>
Source of Truth این مرحله همان Accepted Corpus فعلی است:
</p>

<p>
<code>data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl</code>
</p>

<p>
تعداد Website Recordها از خود Artifact کشف می‌شود و هیچ Countی از قبل فرض نمی‌شود.
</p>

<h4>دامنه بررسی</h4>

<ul>
<li>تمام رکوردهای Accepted Corpus که واقعاً Website محسوب می‌شوند.</li>
<li>بدون محدودیت Author.</li>
<li>بدون محدودیت Authority.</li>
<li>بدون محدودیت Direct Author.</li>
<li>بدون فرض اینکه یک مقاله مشخص حتماً Source اصلی بوده است.</li>
</ul>

<h4>روش بررسی</h4>

<p>
برای جلوگیری از تکرار مشکل Ranking مرحله قبل،
سیگنال‌های مختلف با یک Score ترکیبی مخلوط نمی‌شوند.
</p>

<p>
برای هر پایلوت، هر نوع Evidence به‌صورت مستقل بررسی می‌شود:
</p>

<ul>
<li>هر Human Turn تاریخی به‌صورت مستقل.</li>
<li>هر Model Answer Turn تاریخی به‌صورت مستقل.</li>
<li>بازخورد مستقیم دکتر به‌صورت مستقل.</li>
</ul>

<p>
برای هر Signal، تمام Website Recordها اسکن می‌شوند و Candidateها
بر اساس پوشش واژگان همان Signal رتبه‌بندی می‌شوند.
</p>

<p>
برای Candidateهای برتر نیز عبارت‌های چندکلمه‌ای مشترک و بخش واقعی متن مقاله
نمایش داده می‌شود تا تصمیم نهایی انسانی بر اساس متن واقعی انجام شود.
</p>

<h4>چرا Turnها جدا بررسی می‌شوند؟</h4>

<p>
در برخی از موارد، خصوصاً پایلوت‌های ۲۵ و ۲۶،
پاسخ‌های بعدی مدل فاز یک بی‌ربط یا اشتباه بوده‌اند.
اگر سؤال انسان، پاسخ مدل و بازخورد دکتر در یک Score واحد ترکیب شوند،
ممکن است Source مربوط به پاسخ غلط مدل بالاتر از Source واقعی موضوع سؤال قرار گیرد.
</p>

<p>
بنابراین این مرحله هیچ Signalی را به Signal دیگر تبدیل نمی‌کند
و هیچ Weight دلخواهی برای مخلوط کردن آن‌ها نمی‌سازد.
</p>

<h4>خروجی مورد انتظار</h4>

<p>
برای هر پایلوت باید گزارش شود:
</p>

<ul>
<li>Ranking مستقل هر Human Turn.</li>
<li>Ranking مستقل هر Model Answer Turn.</li>
<li>Ranking مستقل بازخورد دکتر.</li>
<li>Document ID هر Candidate.</li>
<li>عنوان واقعی مقاله یا صفحه.</li>
<li>URL موجود در Corpus.</li>
<li>میزان پوشش واژگانی Signal.</li>
<li>طولانی‌ترین عبارت چندکلمه‌ای مشترک قابل‌مشاهده.</li>
<li>بخشی از متن واقعی Candidate.</li>
</ul>

<h4>تفسیر خروجی</h4>

<p>
این Cell فقط Candidate Discovery انجام می‌دهد.
حتی Rank 1 نیز به‌تنهایی Original Source محسوب نمی‌شود.
</p>

<p>
بعد از مشاهده Output، برای هر پایلوت Human Source Review جداگانه انجام خواهد شد
تا مشخص شود متن موردنظر واقعاً در Website Corpus وجود دارد یا خیر
و اگر وجود دارد، کدام Source یا Sourceها باید وارد بسته بازبینی مؤلف شوند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Website Recordهای موجود در Accepted Corpus شناسایی شوند.</li>
<li>هیچ فیلتر Author یا Authority روی Website Scope اعمال نشود.</li>
<li>هر شش پایلوت ۲۳ تا ۲۸ بررسی شوند.</li>
<li>تمام Human Turnها، Model Turnها و Author Feedbackهای موجود جداگانه اسکن شوند.</li>
<li>Ranking هر Signal بر اساس همان Signal انجام شود.</li>
<li>هیچ Source صرفاً به دلیل Rank بالا Original Source اعلام نشود.</li>
<li>Registry، Corpus و Author Review Markdown بدون تغییر باقی بمانند.</li>
<li>خروجی نهایی برابر <code>CODE CELL 113 GATE: PASS</code> باشد.</li>
</ul>

<h4>معیار FAIL</h4>

<ul>
<li>بخشی از Website Corpus به دلیل فیلتر Author/Authority حذف شود.</li>
<li>یکی از شش پایلوت یا یکی از Turnهای ثبت‌شده بررسی نشود.</li>
<li>Human Question، Model Answer و Author Feedback دوباره در یک Ranking مبهم مخلوط شوند.</li>
<li>Count رکوردهای Scan‌شده با Website Scope کشف‌شده سازگار نباشد.</li>
<li>Registry، Corpus یا فایل Author Review تغییر کند.</li>
<li>نتیجه Candidate Discovery به‌عنوان Provenance قطعی گزارش شود.</li>
</ul>

<h4>خارج از Scope</h4>

<p>
این مرحله هنوز فایل <code>author_review_28_pilots.md</code> را تغییر نمی‌دهد
و هنوز هیچ Source نهایی برای مؤلف ثبت نمی‌کند.
اصلاح بسته Author Review فقط پس از مشاهده و تأیید Human Source Review انجام خواهد شد.
</p>

<h4>ایمنی نوشتن</h4>

<ul>
<li>Registry Write: ممنوع.</li>
<li>Corpus Write: ممنوع.</li>
<li>Author Review Markdown Write: ممنوع.</li>
<li>Author Decision Write: ممنوع.</li>
<li>Commit / Push: ممنوع.</li>
</ul>

</div>

In [6]:
# Code Cell 113

cell113_failures = []


def cell113_check(reason, condition, details=None):
    if not bool(condition):
        item = {"reason": reason}
        if details is not None:
            item["details"] = details
        cell113_failures.append(item)
    return bool(condition)


required_cell112_names = [
    "cell112_gate",
    "cell112_repo_root",
    "cell112_corpus_path",
    "cell112_registry_path",
    "cell112_author_md_path",
    "cell112_corpus_records",
    "cell112_cases",
    "cell112_history_by_number",
    "cell112_sha256_file",
    "cell112_normalize",
    "cell112_tokens",
    "cell112_recursive_values",
    "cell112_record_text",
    "cell112_record_title",
    "cell112_record_url",
]

missing_cell112_names = [
    name
    for name in required_cell112_names
    if name not in globals()
]

cell113_check(
    "CELL112_REQUIRED_STATE_MISSING",
    len(missing_cell112_names) == 0,
    missing_cell112_names,
)

if cell113_failures:
    print("=== PREFLIGHT FAILURES ===")
    print(cell113_failures)
    raise AssertionError(
        "Code Cell 113 requires the accepted output of Code Cell 112."
    )


cell113_check(
    "CELL112_GATE_NOT_PASS",
    cell112_gate is True,
)

if cell113_failures:
    raise AssertionError(cell113_failures)


CELL113_CASE_NUMBERS = [23, 24, 25, 26, 27, 28]
CELL113_DISPLAY_TOP_K = 5
CELL113_PHRASE_MAX_N = 12
CELL113_PHRASE_MIN_N = 3


cell113_corpus_sha_before = cell112_sha256_file(
    cell112_corpus_path
)

cell113_registry_sha_before = cell112_sha256_file(
    cell112_registry_path
)

cell113_author_md_sha_before = (
    cell112_sha256_file(
        cell112_author_md_path
    )
    if cell112_author_md_path.exists()
    else None
)


def cell113_is_website(record):
    document_id = str(
        record.get("document_id") or ""
    ).strip().lower()

    if document_id.startswith("website:"):
        return True

    source_type_values = []

    for key in [
        "source_type",
        "source_kind",
        "content_type",
    ]:
        source_type_values.extend(
            cell112_recursive_values(
                record,
                key,
            )
        )

    normalized_values = {
        cell112_normalize(value)
        for value in source_type_values
        if not isinstance(
            value,
            (dict, list),
        )
    }

    return any(
        (
            value == "website"
            or
            "website" in value
        )
        for value in normalized_values
    )


cell113_website_records = [
    record
    for record in cell112_corpus_records
    if cell113_is_website(record)
]


cell113_check(
    "NO_WEBSITE_RECORDS_FOUND",
    len(cell113_website_records) > 0,
)


def cell113_record_document_id(record):
    return str(
        record.get("document_id") or ""
    )


cell113_website_docs = []

for record in cell113_website_records:
    cell113_website_docs.append({
        "document_id":
            cell113_record_document_id(
                record
            ),
        "title":
            cell112_record_title(
                record
            ),
        "url":
            cell112_record_url(
                record
            ),
        "text":
            cell112_record_text(
                record
            ),
        "record":
            record,
    })


cell113_check(
    "WEBSITE_DOC_COUNT_MISMATCH",
    len(cell113_website_docs)
    ==
    len(cell113_website_records),
)


cell113_empty_text_docs = [
    doc["document_id"]
    for doc in cell113_website_docs
    if not str(
        doc["text"] or ""
    ).strip()
]


cell113_nonempty_docs = [
    doc
    for doc in cell113_website_docs
    if str(
        doc["text"] or ""
    ).strip()
]


cell113_check(
    "NO_NONEMPTY_WEBSITE_TEXT",
    len(cell113_nonempty_docs) > 0,
)


cell113_doc_token_sets = {}
cell113_document_frequency = Counter()

for doc in cell113_nonempty_docs:
    token_set = set(
        cell112_tokens(
            doc["text"]
        )
    )

    cell113_doc_token_sets[
        doc["document_id"]
    ] = token_set

    for token in token_set:
        cell113_document_frequency[
            token
        ] += 1


cell113_document_count = len(
    cell113_nonempty_docs
)


def cell113_idf(token):
    return (
        math.log(
            (
                1
                +
                cell113_document_count
            )
            /
            (
                1
                +
                cell113_document_frequency.get(
                    token,
                    0,
                )
            )
        )
        +
        1.0
    )


def cell113_weighted_coverage(
    query_text,
    document_id,
):
    query_tokens = set(
        cell112_tokens(
            query_text
        )
    )

    if not query_tokens:
        return 0.0, []

    document_tokens = (
        cell113_doc_token_sets.get(
            document_id,
            set(),
        )
    )

    matched_tokens = (
        query_tokens
        &
        document_tokens
    )

    denominator = sum(
        cell113_idf(token)
        for token in query_tokens
    )

    numerator = sum(
        cell113_idf(token)
        for token in matched_tokens
    )

    coverage = (
        numerator
        /
        denominator
        if denominator
        else
        0.0
    )

    matched_sorted = sorted(
        matched_tokens,
        key=lambda token: (
            -cell113_idf(token),
            token,
        ),
    )

    return (
        coverage,
        matched_sorted,
    )


def cell113_longest_phrase_hits(
    source_text,
    query_text,
    max_results=5,
):
    source_normalized = (
        cell112_normalize(
            source_text
        )
    )

    query_tokens = (
        cell112_tokens(
            query_text
        )
    )

    if (
        not source_normalized
        or
        len(query_tokens)
        <
        CELL113_PHRASE_MIN_N
    ):
        return []

    maximum_n = min(
        CELL113_PHRASE_MAX_N,
        len(query_tokens),
    )

    results = []
    seen = set()

    for n in range(
        maximum_n,
        CELL113_PHRASE_MIN_N - 1,
        -1,
    ):
        for index in range(
            0,
            len(query_tokens) - n + 1,
        ):
            phrase = " ".join(
                query_tokens[
                    index:index + n
                ]
            )

            if phrase in seen:
                continue

            if phrase in source_normalized:
                seen.add(phrase)

                results.append({
                    "n": n,
                    "phrase": phrase,
                })

                if len(results) >= max_results:
                    return results

    return results


def cell113_best_evidence_window(
    source_text,
    query_text,
    max_chars=1200,
):
    source_text = str(
        source_text or ""
    ).strip()

    if not source_text:
        return ""

    query_tokens = set(
        cell112_tokens(
            query_text
        )
    )

    sentences = [
        part.strip()
        for part in re.split(
            r"(?<=[\.\!\؟])\s+|\n+",
            source_text,
        )
        if part.strip()
    ]

    if not sentences:
        return source_text[:max_chars]

    best_window = None
    best_score = -1.0

    for index in range(
        len(sentences)
    ):
        window = " ".join(
            sentences[
                index:index + 4
            ]
        ).strip()

        window_tokens = set(
            cell112_tokens(
                window
            )
        )

        matched = (
            query_tokens
            &
            window_tokens
        )

        score = sum(
            cell113_idf(token)
            for token in matched
        )

        if score > best_score:
            best_score = score
            best_window = window

    if best_window is None:
        best_window = source_text

    if len(best_window) > max_chars:
        best_window = (
            best_window[:max_chars].rstrip()
            +
            " ..."
        )

    return best_window


def cell113_rank_signal(
    signal_text,
):
    rows = []

    for doc in cell113_nonempty_docs:
        coverage, matched_terms = (
            cell113_weighted_coverage(
                signal_text,
                doc["document_id"],
            )
        )

        rows.append({
            "document_id":
                doc["document_id"],
            "title":
                doc["title"],
            "url":
                doc["url"],
            "coverage":
                coverage,
            "matched_terms":
                matched_terms,
            "text":
                doc["text"],
        })

    rows.sort(
        key=lambda row: (
            -row["coverage"],
            row["document_id"],
        )
    )

    top_rows = rows[
        :CELL113_DISPLAY_TOP_K
    ]

    for row in top_rows:
        row[
            "phrase_hits"
        ] = cell113_longest_phrase_hits(
            row["text"],
            signal_text,
        )

        row[
            "evidence_window"
        ] = cell113_best_evidence_window(
            row["text"],
            signal_text,
        )

    return {
        "scanned_documents":
            len(rows),
        "ranking":
            rows,
        "top":
            top_rows,
    }


cell113_results = {}


for number in CELL113_CASE_NUMBERS:
    history = (
        cell112_history_by_number[
            number
        ]
    )

    human_results = []

    for turn_index, text in enumerate(
        history["human_texts"],
        start=1,
    ):
        human_results.append({
            "turn":
                turn_index,
            "text":
                text,
            "result":
                cell113_rank_signal(
                    text
                ),
        })

    model_results = []

    for turn_index, text in enumerate(
        history["model_texts"],
        start=1,
    ):
        model_results.append({
            "turn":
                turn_index,
            "text":
                text,
            "result":
                cell113_rank_signal(
                    text
                ),
        })

    feedback_result = (
        cell113_rank_signal(
            history["raw_feedback"]
        )
        if history["raw_feedback"].strip()
        else None
    )

    cell113_results[
        number
    ] = {
        "human":
            human_results,
        "model":
            model_results,
        "feedback":
            feedback_result,
    }


cell113_check(
    "PILOT_RESULT_COUNT_NOT_6",
    sorted(
        cell113_results.keys()
    )
    ==
    CELL113_CASE_NUMBERS,
)


def cell113_print_signal(
    label,
    signal_text,
    result,
):
    print(
        "\n---",
        label,
        "---",
    )

    print(
        "Signal text:"
    )

    print(
        signal_text
    )

    print(
        "\nWebsite documents scanned:",
        result[
            "scanned_documents"
        ],
    )

    for rank, row in enumerate(
        result["top"],
        start=1,
    ):
        print(
            "\n",
            "-" * 100,
            sep="",
        )

        print(
            "Rank:",
            rank,
        )

        print(
            "Document ID:",
            row[
                "document_id"
            ],
        )

        print(
            "Title:",
            row[
                "title"
            ],
        )

        print(
            "URL:",
            row[
                "url"
            ],
        )

        print(
            "Weighted token coverage:",
            round(
                row[
                    "coverage"
                ],
                4,
            ),
        )

        print(
            "Longest phrase hits:",
            row[
                "phrase_hits"
            ],
        )

        print(
            "Matched terms:",
            row[
                "matched_terms"
            ][:20],
        )

        print(
            "\nEvidence window:"
        )

        print(
            row[
                "evidence_window"
            ]
        )


print(
    "=== CODE CELL 113 — FULL WEBSITE CORPUS SOURCE DISCOVERY ==="
)

print(
    "\n=== SCOPE ==="
)

print(
    "Accepted Corpus records:",
    len(
        cell112_corpus_records
    ),
)

print(
    "Website records discovered:",
    len(
        cell113_website_records
    ),
)

print(
    "Website records with non-empty text:",
    len(
        cell113_nonempty_docs
    ),
)

print(
    "Website records with empty text:",
    len(
        cell113_empty_text_docs
    ),
)

print(
    "Author filter applied:",
    False,
)

print(
    "Authority filter applied:",
    False,
)

print(
    "Direct-Author-only filter applied:",
    False,
)


for number in CELL113_CASE_NUMBERS:
    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "PILOT:",
        number,
    )

    pilot_result = (
        cell113_results[
            number
        ]
    )

    for item in pilot_result[
        "human"
    ]:
        cell113_print_signal(
            (
                "HISTORICAL HUMAN TURN "
                +
                str(
                    item["turn"]
                )
            ),
            item["text"],
            item["result"],
        )

    for item in pilot_result[
        "model"
    ]:
        cell113_print_signal(
            (
                "HISTORICAL MODEL TURN "
                +
                str(
                    item["turn"]
                )
            ),
            item["text"],
            item["result"],
        )

    if (
        pilot_result[
            "feedback"
        ]
        is not None
    ):
        cell113_print_signal(
            "DR. ROUDI FEEDBACK",
            cell112_history_by_number[
                number
            ][
                "raw_feedback"
            ],
            pilot_result[
                "feedback"
            ],
        )

    print(
        "\nHUMAN SOURCE DECISION: NOT_STARTED"
    )


print(
    "\n"
    +
    "=" * 120
)

print(
    "=== TOP-1 SIGNAL SUMMARY ==="
)


for number in CELL113_CASE_NUMBERS:
    print(
        "\nPilot",
        number,
    )

    result = (
        cell113_results[
            number
        ]
    )

    for item in result[
        "human"
    ]:
        top = (
            item[
                "result"
            ][
                "top"
            ][0]
        )

        print(
            (
                "  Human turn "
                +
                str(
                    item[
                        "turn"
                    ]
                )
                +
                ":"
            ),
            top[
                "document_id"
            ],
            "|",
            top[
                "title"
            ],
            "| coverage =",
            round(
                top[
                    "coverage"
                ],
                4,
            ),
        )

    for item in result[
        "model"
    ]:
        top = (
            item[
                "result"
            ][
                "top"
            ][0]
        )

        print(
            (
                "  Model turn "
                +
                str(
                    item[
                        "turn"
                    ]
                )
                +
                ":"
            ),
            top[
                "document_id"
            ],
            "|",
            top[
                "title"
            ],
            "| coverage =",
            round(
                top[
                    "coverage"
                ],
                4,
            ),
        )

    if result[
        "feedback"
    ] is not None:
        top = (
            result[
                "feedback"
            ][
                "top"
            ][0]
        )

        print(
            "  Dr feedback:",
            top[
                "document_id"
            ],
            "|",
            top[
                "title"
            ],
            "| coverage =",
            round(
                top[
                    "coverage"
                ],
                4,
            ),
        )


cell113_corpus_sha_after = (
    cell112_sha256_file(
        cell112_corpus_path
    )
)

cell113_registry_sha_after = (
    cell112_sha256_file(
        cell112_registry_path
    )
)

cell113_author_md_sha_after = (
    cell112_sha256_file(
        cell112_author_md_path
    )
    if cell112_author_md_path.exists()
    else None
)


cell113_check(
    "CORPUS_MUTATED",
    cell113_corpus_sha_after
    ==
    cell113_corpus_sha_before,
)

cell113_check(
    "REGISTRY_MUTATED",
    cell113_registry_sha_after
    ==
    cell113_registry_sha_before,
)

cell113_check(
    "AUTHOR_MD_MUTATED",
    cell113_author_md_sha_after
    ==
    cell113_author_md_sha_before,
)


expected_signal_count = 0
actual_signal_count = 0

for number in CELL113_CASE_NUMBERS:
    history = (
        cell112_history_by_number[
            number
        ]
    )

    expected_signal_count += (
        len(
            history[
                "human_texts"
            ]
        )
        +
        len(
            history[
                "model_texts"
            ]
        )
        +
        (
            1
            if history[
                "raw_feedback"
            ].strip()
            else
            0
        )
    )

    result = (
        cell113_results[
            number
        ]
    )

    actual_signal_count += (
        len(
            result[
                "human"
            ]
        )
        +
        len(
            result[
                "model"
            ]
        )
        +
        (
            1
            if result[
                "feedback"
            ]
            is not None
            else
            0
        )
    )


cell113_check(
    "SIGNAL_COUNT_MISMATCH",
    actual_signal_count
    ==
    expected_signal_count,
    {
        "expected":
            expected_signal_count,
        "actual":
            actual_signal_count,
    },
)


cell113_check(
    "WEBSITE_SCAN_COUNT_INCONSISTENT",
    all(
        signal[
            "result"
        ][
            "scanned_documents"
        ]
        ==
        len(
            cell113_nonempty_docs
        )
        for number
        in CELL113_CASE_NUMBERS
        for signal
        in (
            cell113_results[
                number
            ][
                "human"
            ]
            +
            cell113_results[
                number
            ][
                "model"
            ]
        )
    ),
)


for number in CELL113_CASE_NUMBERS:
    feedback_result = (
        cell113_results[
            number
        ][
            "feedback"
        ]
    )

    if feedback_result is not None:
        cell113_check(
            (
                "FEEDBACK_SCAN_COUNT_INCONSISTENT:"
                +
                str(
                    number
                )
            ),
            feedback_result[
                "scanned_documents"
            ]
            ==
            len(
                cell113_nonempty_docs
            ),
        )


print(
    "\n=== VALIDATION ==="
)

print(
    "Pilots expected:",
    len(
        CELL113_CASE_NUMBERS
    ),
)

print(
    "Pilots processed:",
    len(
        cell113_results
    ),
)

print(
    "Signals expected:",
    expected_signal_count,
)

print(
    "Signals processed:",
    actual_signal_count,
)

print(
    "Website records scanned per signal:",
    len(
        cell113_nonempty_docs
    ),
)

print(
    "Corpus mutated:",
    cell113_corpus_sha_after
    !=
    cell113_corpus_sha_before,
)

print(
    "Registry mutated:",
    cell113_registry_sha_after
    !=
    cell113_registry_sha_before,
)

print(
    "Author Review Markdown mutated:",
    cell113_author_md_sha_after
    !=
    cell113_author_md_sha_before,
)

print(
    "Author decisions recorded:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell113_failures
)


cell113_gate = (
    len(
        cell113_failures
    )
    ==
    0
)


print(
    "\nCODE CELL 113 GATE:",
    (
        "PASS"
        if cell113_gate
        else
        "FAIL"
    ),
)


assert cell113_gate, (
    "Full Website Corpus source discovery failed. "
    "Repair this SAME Code Cell 113 before continuing."
)

=== CODE CELL 113 — FULL WEBSITE CORPUS SOURCE DISCOVERY ===

=== SCOPE ===
Accepted Corpus records: 19456
Website records discovered: 529
Website records with non-empty text: 529
Website records with empty text: 0
Author filter applied: False
Authority filter applied: False
Direct-Author-only filter applied: False

PILOT: 23

--- HISTORICAL HUMAN TURN 1 ---
Signal text:
سواد مالی درباره پول دار شدن دیگه؟

Website documents scanned: 529

----------------------------------------------------------------------------------------------------
Rank: 1
Document ID: website:E4A15578
Title: معرفی کتاب «همه چیز دربارۀ پول»
URL: https://www.fintelligence.ir/Blog/Detail/E4A15578
Weighted token coverage: 0.6055
Longest phrase hits: []
Matched terms: ['دار', 'شدن', 'پول', 'سواد', 'مالی']

Evidence window:
انتشارات مهرسا با هدف آشنایی دختران نوجوان با سرفصل‌های سواد مالی، کتابی با عنوان «همه چیز دربارۀ پول؛ چطوری پول پارو کنم، پس‌انداز کنم و خرج کنم» را منتشر کرده که توسط تیم علمی آکادمی هوش مالی بررس

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی هدفمند متن کامل Website Corpus برای پایلوت‌های ۲۳ تا ۲۸</h3>

<h4>هدف</h4>

<p>
هدف این مرحله یافتن متن‌های واقعاً مرتبط با شش پایلوت ۲۳ تا ۲۸ در تمام
۵۲۹ رکورد Website موجود در Accepted Corpus است.
</p>

<p>
در مرحله قبل تمام Website Corpus بررسی شد، اما Ranking عمومی مبتنی بر
هم‌پوشانی واژگانی نشان داد که Rank 1 همیشه بهترین Source محتوایی نیست.
بنابراین در این مرحله به‌جای Ranking عمومی، عبارت‌ها و مفهوم‌های متمایز
هر مورد مستقیماً در متن کامل تمام Websiteها جست‌وجو می‌شوند.
</p>

<h4>دلیل این مرحله</h4>

<p>
سؤال‌های کوتاه یا پاسخ‌های اشتباه مدل ممکن است باعث شوند مقاله‌ای با چند واژه عمومی
امتیاز بالاتری از مقاله‌ای بگیرد که از نظر محتوایی دقیقاً همان مسئله را توضیح می‌دهد.
</p>

<p>
برای نمونه، در پایلوت ۲۵ مقاله «سواد مالی خانم‌ها؛ پول‌ها کجا می‌روند؟»
از نظر متن بسیار نزدیک به موضوع اصلی است، اما در Ranking پاسخ تاریخی مدل Rank 4 شده است.
این نشان می‌دهد انتخاب Source نهایی باید بر Evidence واقعی متن متکی باشد، نه صرفاً Rank.
</p>

<h4>Source of Truth</h4>

<p>
همان ۵۲۹ رکورد Website کشف‌شده در Code Cell 113 از:
</p>

<p>
<code>data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl</code>
</p>

<h4>روش بررسی</h4>

<p>
برای هر پایلوت مجموعه‌ای از عبارت‌ها و مفهوم‌های متمایز از سؤال واقعی،
پاسخ تاریخی مدل و بازخورد مستقیم دکتر تعریف می‌شود.
</p>

<p>
سپس متن کامل تمام ۵۲۹ Website برای این عبارت‌ها بررسی می‌شود.
برای هر Document گزارش خواهد شد:
</p>

<ul>
<li>چند عبارت هدف در متن پیدا شده است.</li>
<li>کدام عبارت‌ها پیدا شده‌اند.</li>
<li>Document ID.</li>
<li>عنوان.</li>
<li>URL.</li>
<li>بخش واقعی متن در اطراف قوی‌ترین Match.</li>
</ul>

<h4>قاعده مهم</h4>

<p>
وجود Match محتوایی هنوز به معنی اثبات Original Provenance اجرای دموی فاز یک نیست.
این مرحله فقط مشخص می‌کند که متن مرتبط واقعاً در Website Corpus وجود دارد یا خیر.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام ۵۲۹ Website بررسی شوند.</li>
<li>هر شش پایلوت ۲۳ تا ۲۸ بررسی شوند.</li>
<li>هیچ فیلتر Author یا Authority اعمال نشود.</li>
<li>جست‌وجو بر متن کامل مقاله انجام شود.</li>
<li>Matchهای واقعی همراه با بخش متن مقاله نمایش داده شوند.</li>
<li>Registry، Corpus و Author Review Markdown تغییر نکنند.</li>
<li>خروجی نهایی برابر <code>CODE CELL 114 GATE: PASS</code> باشد.</li>
</ul>

<h4>معیار FAIL</h4>

<ul>
<li>تعداد Websiteهای بررسی‌شده کمتر از Scope کشف‌شده Cell 113 باشد.</li>
<li>یکی از شش پایلوت بررسی نشود.</li>
<li>یک Match صرفاً بر اساس Rank یا نام مقاله پذیرفته شود.</li>
<li>Corpus، Registry یا فایل Author Review تغییر کند.</li>
</ul>

<h4>خارج از Scope</h4>

<p>
این مرحله هنوز هیچ Source نهایی را وارد فایل
<code>author_review_28_pilots.md</code>
نمی‌کند.
انتخاب Source نهایی پس از مشاهده Output و Human Source Review انجام می‌شود.
</p>

<h4>ایمنی نوشتن</h4>

<ul>
<li>Corpus Write: ممنوع.</li>
<li>Registry Write: ممنوع.</li>
<li>Author Review Markdown Write: ممنوع.</li>
<li>Author Decision Write: ممنوع.</li>
<li>Commit / Push: ممنوع.</li>
</ul>

</div>

In [7]:
# Code Cell 114

cell114_failures = []


def cell114_check(reason, condition, details=None):
    if not bool(condition):
        item = {"reason": reason}
        if details is not None:
            item["details"] = details
        cell114_failures.append(item)
    return bool(condition)


cell114_check(
    "CELL113_GATE_NOT_PASS",
    "cell113_gate" in globals()
    and cell113_gate is True,
)

cell114_check(
    "CELL113_WEBSITE_SCOPE_MISSING",
    "cell113_nonempty_docs" in globals()
    and len(cell113_nonempty_docs) > 0,
)

if cell114_failures:
    raise AssertionError(cell114_failures)


CELL114_CASE_NUMBERS = [23, 24, 25, 26, 27, 28]
CELL114_TOP_K = 10


CELL114_TARGETS = {
    23: [
        "سواد مالی درباره مدیریت پول",
        "سواد مالی درباره میزان پول",
        "مدیریت پول",
        "میزان پول",
        "درست مدیریت کردن پول",
        "پول دار شدن",
        "ثروتمند شدن",
    ],

    24: [
        "دانش نگرش مهارت",
        "دانش مهارت نگرش",
        "دانش و نگرش و مهارت",
        "پرسشنامه",
        "اندازه گیری سواد مالی",
        "سنجش سواد مالی",
        "وضعیت سواد مالی",
        "مدیریت مالی شخصی",
    ],

    25: [
        "پول ها کجا می روند",
        "پولم کجا می رود",
        "نمی دانم چگونه خرج کرده ام",
        "نمی دانم برای چه چیزی خرج کرده ام",
        "نوشتن هزینه ها",
        "ثبت هزینه ها",
        "مدیریت هزینه ها",
        "هدررفت پول",
    ],

    26: [
        "الگوهای نادرست خرید",
        "الگوی خرید نامناسب",
        "خرید به خاطر تنش",
        "خرید هنگام ناراحتی",
        "خرید هیجانی",
        "هیجان زود گذر",
        "گروه های مرجع",
        "تبلیغات و بازاریابی",
        "خرید به خاطر تبلیغات",
        "خرید به خاطر فداکاری",
        "خرید با حجم زیاد",
        "خرید با تنوع زیاد",
    ],

    27: [
        "افزایش درآمد",
        "قیمت گذاری مناسب",
        "هزینه های تولید",
        "ارزش پیشنهادی",
        "تغییر نقشه کسب و کار",
        "دارایی های درآمدزا",
        "خرید دارایی درآمدزا",
        "درآمد منفعل",
        "درآمد غیرفعال",
        "فروش وسایلی که نیاز نداریم",
        "کار بیشتر به کارفرما",
        "پروژه بیشتر",
        "سرمایه گذاری دوباره سود",
    ],

    28: [
        "رابطه با پول",
        "رابطه ما با پول",
        "چیزهایی که نداریم",
        "پول چیزهایی است که نداریم",
        "مقام امنیت آزادی خوشبختی",
        "احساس فقر",
        "اعتماد به نفس",
        "پول ابزاری است",
        "بدون پول زندگی ام چیزی کم دارد",
    ],
}


def cell114_normalized_contains(
    source_text,
    phrase,
):
    return (
        cell112_normalize(phrase)
        in
        cell112_normalize(source_text)
    )


def cell114_find_best_window(
    text,
    matched_phrases,
    radius=700,
):
    text = str(text or "")

    if not text:
        return ""

    normalized_text = cell112_normalize(
        text
    )

    best_phrase = None
    best_pos = None

    for phrase in matched_phrases:
        normalized_phrase = (
            cell112_normalize(
                phrase
            )
        )

        pos = normalized_text.find(
            normalized_phrase
        )

        if pos >= 0:
            best_phrase = phrase
            best_pos = pos
            break

    if best_phrase is None:
        return text[:1400]

    raw_phrase_tokens = (
        str(best_phrase)
        .split()
    )

    raw_anchor = (
        raw_phrase_tokens[0]
        if raw_phrase_tokens
        else ""
    )

    raw_pos = (
        text.find(raw_anchor)
        if raw_anchor
        else -1
    )

    if raw_pos < 0:
        raw_pos = 0

    start = max(
        0,
        raw_pos - radius,
    )

    end = min(
        len(text),
        raw_pos + radius,
    )

    excerpt = text[
        start:end
    ].strip()

    if start > 0:
        excerpt = "... " + excerpt

    if end < len(text):
        excerpt += " ..."

    return excerpt


def cell114_search_case(
    number,
):
    target_phrases = (
        CELL114_TARGETS[
            number
        ]
    )

    rows = []

    for doc in cell113_nonempty_docs:
        text = doc["text"]

        matched_phrases = [
            phrase
            for phrase in target_phrases
            if cell114_normalized_contains(
                text,
                phrase,
            )
        ]

        if not matched_phrases:
            continue

        matched_token_union = set()

        for phrase in matched_phrases:
            matched_token_union.update(
                cell112_tokens(
                    phrase
                )
            )

        rows.append({
            "document_id":
                doc["document_id"],

            "title":
                doc["title"],

            "url":
                doc["url"],

            "matched_phrase_count":
                len(
                    matched_phrases
                ),

            "matched_phrases":
                matched_phrases,

            "distinct_target_tokens":
                len(
                    matched_token_union
                ),

            "evidence_window":
                cell114_find_best_window(
                    text,
                    matched_phrases,
                ),
        })

    rows.sort(
        key=lambda row: (
            -row[
                "matched_phrase_count"
            ],
            -row[
                "distinct_target_tokens"
            ],
            row[
                "document_id"
            ],
        )
    )

    return rows


cell114_results = {
    number:
        cell114_search_case(
            number
        )

    for number
    in CELL114_CASE_NUMBERS
}


cell114_check(
    "CASE_RESULT_COUNT_NOT_6",
    sorted(
        cell114_results.keys()
    )
    ==
    CELL114_CASE_NUMBERS,
)


print(
    "=== CODE CELL 114 — TARGETED FULL-WEBSITE TEXT AUDIT ==="
)

print(
    "\n=== SCOPE ==="
)

print(
    "Website documents available:",
    len(
        cell113_nonempty_docs
    ),
)

print(
    "Website documents scanned per pilot:",
    len(
        cell113_nonempty_docs
    ),
)

print(
    "Author filter applied:",
    False,
)

print(
    "Authority filter applied:",
    False,
)


for number in CELL114_CASE_NUMBERS:
    rows = (
        cell114_results[
            number
        ]
    )

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "PILOT:",
        number,
    )

    print(
        "Target phrases:",
        len(
            CELL114_TARGETS[
                number
            ]
        ),
    )

    print(
        "Website documents with at least one target match:",
        len(
            rows
        ),
    )

    if not rows:
        print(
            "\nNO TARGETED MATCH FOUND"
        )
        continue

    for rank, row in enumerate(
        rows[:CELL114_TOP_K],
        start=1,
    ):
        print(
            "\n"
            +
            "-" * 100
        )

        print(
            "Rank:",
            rank,
        )

        print(
            "Document ID:",
            row[
                "document_id"
            ],
        )

        print(
            "Title:",
            row[
                "title"
            ],
        )

        print(
            "URL:",
            row[
                "url"
            ],
        )

        print(
            "Matched phrase count:",
            row[
                "matched_phrase_count"
            ],
        )

        print(
            "Matched phrases:",
        )

        for phrase in row[
            "matched_phrases"
        ]:
            print(
                " -",
                phrase,
            )

        print(
            "\nEvidence window:"
        )

        print(
            row[
                "evidence_window"
            ]
        )


cell114_corpus_sha_after = (
    cell112_sha256_file(
        cell112_corpus_path
    )
)

cell114_registry_sha_after = (
    cell112_sha256_file(
        cell112_registry_path
    )
)

cell114_author_md_sha_after = (
    cell112_sha256_file(
        cell112_author_md_path
    )
    if cell112_author_md_path.exists()
    else None
)


cell114_check(
    "CORPUS_MUTATED",
    cell114_corpus_sha_after
    ==
    cell113_corpus_sha_before,
)

cell114_check(
    "REGISTRY_MUTATED",
    cell114_registry_sha_after
    ==
    cell113_registry_sha_before,
)

cell114_check(
    "AUTHOR_MD_MUTATED",
    cell114_author_md_sha_after
    ==
    cell113_author_md_sha_before,
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "=== TARGETED AUDIT SUMMARY ==="
)

for number in CELL114_CASE_NUMBERS:
    rows = (
        cell114_results[
            number
        ]
    )

    print(
        "\nPilot",
        number,
    )

    print(
        "  Matching Website documents:",
        len(
            rows
        ),
    )

    if rows:
        best = rows[0]

        print(
            "  Best targeted candidate:",
            best[
                "document_id"
            ],
            "|",
            best[
                "title"
            ],
        )

        print(
            "  Matched phrases:",
            best[
                "matched_phrase_count"
            ],
        )

    else:
        print(
            "  Best targeted candidate: NONE"
        )


print(
    "\n=== SAFETY ==="
)

print(
    "Corpus mutated:",
    cell114_corpus_sha_after
    !=
    cell113_corpus_sha_before,
)

print(
    "Registry mutated:",
    cell114_registry_sha_after
    !=
    cell113_registry_sha_before,
)

print(
    "Author Review Markdown mutated:",
    cell114_author_md_sha_after
    !=
    cell113_author_md_sha_before,
)

print(
    "Author decisions recorded:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell114_failures
)


cell114_gate = (
    len(
        cell114_failures
    )
    ==
    0
)


print(
    "\nCODE CELL 114 GATE:",
    (
        "PASS"
        if cell114_gate
        else
        "FAIL"
    ),
)


assert cell114_gate, (
    "Targeted full-Website text audit failed. "
    "Repair this SAME Code Cell 114 before continuing."
)

=== CODE CELL 114 — TARGETED FULL-WEBSITE TEXT AUDIT ===

=== SCOPE ===
Website documents available: 529
Website documents scanned per pilot: 529
Author filter applied: False
Authority filter applied: False

PILOT: 23
Target phrases: 7
Website documents with at least one target match: 33

----------------------------------------------------------------------------------------------------
Rank: 1
Document ID: website:7C71884B
Title: ابزارهای سواد مالی برای بهبود مدیریت هزینه
URL: https://www.fintelligence.ir/Blog/Detail/7C71884B
Matched phrase count: 3
Matched phrases:
 - سواد مالی درباره میزان پول
 - میزان پول
 - درست مدیریت کردن پول

Evidence window:
ابزار دوم سواد مالی: بودجه‌بندی
هر جا پولی وارد یا خارج شود، بودجه معنا پیدا می‌کند. حتی در سطح خانواده و افراد.
بودجه‌بندی
یعنی قبل از اینکه هزینه‌ای انجام دهیم، به آن فکر کنیم. اگر می‌خواهیم توانگر باشیم باید از قبل بدانیم در هر حوزه‌ای چقدر و چطور می‌خواهیم هزینه کنیم. توصیه می‌شود بعد از سه ماه ثبت هزینه‌ها و استفاده از تجربه به دست‌آ

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی پوشش جزءبه‌جزء منبع برای پایلوت‌های ۲۳ تا ۲۸</h3>

<h4>هدف</h4>

<p>
هدف این مرحله مشخص کردن این است که هر جزء محتوایی از پاسخ نهایی
پایلوت‌های ۲۳ تا ۲۸ دقیقاً در کدام متن یا متن‌های Website Corpus وجود دارد.
</p>

<p>
مرحله قبل Candidateهای اصلی را از میان تمام ۵۲۹ Website پیدا کرد.
اما برای مواردی مانند پایلوت ۲۶ و ۲۷، پاسخ یا بازخورد دکتر شامل چند ادعای مستقل است
و ممکن است تمام اجزا در یک مقاله واحد قرار نداشته باشند.
</p>

<h4>چرا این مرحله لازم است؟</h4>

<p>
هدف بسته Author Review فقط نمایش یک عنوان مقاله مشابه نیست.
در بخش «متن منبع» باید متن واقعی‌ای قرار گیرد که محتوای سؤال و پاسخ نهایی را پشتیبانی کند.
</p>

<p>
بنابراین لازم است پیش از اصلاح فایل Author Review بدانیم:
</p>

<ul>
<li>کدام جزء پاسخ در Corpus وجود دارد.</li>
<li>در کدام Document وجود دارد.</li>
<li>عبارت واقعی مقاله چیست.</li>
<li>آیا یک Source کافی است یا چند Source لازم است.</li>
<li>کدام بخش پاسخ فقط از بازخورد مستقیم دکتر آمده و در Website Corpus پیدا نشده است.</li>
</ul>

<h4>Source of Truth</h4>

<p>
دامنه بررسی همان ۵۲۹ Website Record تأییدشده در Code Cell 113 است.
</p>

<p>
هیچ محدودیتی بر اساس Author، Authority یا Direct Author اعمال نمی‌شود.
</p>

<h4>روش بررسی</h4>

<p>
برای هر پایلوت، پاسخ موردنظر به چند Claim مستقل شکسته می‌شود.
برای هر Claim نیز چند عبارت یا شکل نوشتاری نزدیک تعریف می‌شود.
</p>

<p>
سپس متن کامل تمام ۵۲۹ Website برای هر Claim به‌صورت مستقل اسکن می‌شود.
</p>

<p>
این مرحله از Score ترکیبی استفاده نمی‌کند.
نتیجه هر Claim مستقل گزارش می‌شود.
</p>

<h4>خروجی مورد انتظار</h4>

<p>
برای هر Claim باید گزارش شود:
</p>

<ul>
<li>نام و توضیح Claim.</li>
<li>تعداد Websiteهای Match‌شده.</li>
<li>Document ID.</li>
<li>عنوان.</li>
<li>URL.</li>
<li>عبارت‌های واقعی Match‌شده.</li>
<li>بخش واقعی متن مقاله در اطراف Match.</li>
</ul>

<p>
در پایان هر Pilot باید مشخص شود چه تعداد Claim از کل Claimهای تعریف‌شده
دارای حداقل یک Match واقعی هستند.
</p>

<h4>تفسیر نتیجه</h4>

<p>
وجود Match در این مرحله به معنی اثبات Provenance تاریخی اجرای Demo نیست.
این مرحله «Content Source Recovery» انجام می‌دهد.
</p>

<p>
اگر تمام یا بیشتر اجزای پاسخ در یک مقاله واحد یافت شوند،
آن مقاله Candidate اصلی برای بخش «متن منبع» خواهد بود.
اگر اجزا بین چند مقاله تقسیم شده باشند، Source باید چندبخشی گزارش شود.
</p>

<p>
اگر Claimی در کل ۵۲۹ Website پیدا نشود، نباید برای آن Source ساختگی ایجاد شود.
در این حالت باید صریحاً ثبت شود که آن بخش فقط از بازخورد مستقیم دکتر یا Evidence دیگری آمده است.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر شش پایلوت ۲۳ تا ۲۸ بررسی شوند.</li>
<li>تمام Claimهای تعریف‌شده برای هر Pilot بررسی شوند.</li>
<li>هر Claim روی تمام ۵۲۹ Website اجرا شود.</li>
<li>Match واقعی همراه با متن واقعی Source نمایش داده شود.</li>
<li>Claim بدون Source مخفی نشود.</li>
<li>هیچ Candidate به‌صورت خودکار Accepted نشود.</li>
<li>Corpus، Registry و Author Review Markdown تغییر نکنند.</li>
<li>خروجی نهایی برابر <code>CODE CELL 115 GATE: PASS</code> باشد.</li>
</ul>

<h4>معیار FAIL</h4>

<ul>
<li>یکی از Pilotها یا Claimها بررسی نشود.</li>
<li>دامنه اسکن کمتر از ۵۲۹ Website باشد.</li>
<li>Source براساس شباهت نام یا Rank بدون متن واقعی پذیرفته شود.</li>
<li>Claim بدون Evidence به‌عنوان Covered گزارش شود.</li>
<li>Corpus، Registry یا Author Review Markdown تغییر کند.</li>
</ul>

<h4>خارج از Scope</h4>

<p>
این مرحله هنوز فایل <code>author_review_28_pilots.md</code> را تغییر نمی‌دهد.
انتخاب و درج Source نهایی فقط بعد از Human Source Review خروجی این Cell انجام می‌شود.
</p>

<h4>ایمنی نوشتن</h4>

<ul>
<li>Corpus Write: ممنوع.</li>
<li>Registry Write: ممنوع.</li>
<li>Author Review Markdown Write: ممنوع.</li>
<li>Author Decision Write: ممنوع.</li>
<li>Commit / Push: ممنوع.</li>
</ul>

</div>

In [10]:
# Code Cell 115

cell115_failures = []


def cell115_check(reason, condition, details=None):
    if not bool(condition):
        item = {
            "reason": reason,
        }

        if details is not None:
            item["details"] = details

        cell115_failures.append(item)

    return bool(condition)


cell115_check(
    "CELL114_GATE_NOT_PASS",
    (
        "cell114_gate" in globals()
        and
        cell114_gate is True
    ),
)

cell115_check(
    "WEBSITE_SCOPE_NOT_529",
    (
        "cell113_nonempty_docs" in globals()
        and
        len(cell113_nonempty_docs) == 529
    ),
    (
        len(cell113_nonempty_docs)
        if "cell113_nonempty_docs" in globals()
        else None
    ),
)

if cell115_failures:
    raise AssertionError(cell115_failures)


CELL115_CASE_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]

CELL115_TOP_K_PER_CLAIM = 5

CELL115_SENTENCE_WINDOW = 4


# ============================================================
# Claim contract
#
# Each support_pattern is a list of phrases.
# ALL phrases inside one pattern must appear inside the SAME
# local sentence window.
#
# A claim is STRICT_FOUND only if at least one complete pattern
# is satisfied.
# ============================================================

CELL115_CLAIMS = {

    23: [
        {
            "claim_id": "23-A",
            "description":
                "سواد مالی درباره مدیریت پول است، نه میزان پول.",

            "support_patterns": [
                [
                    "سواد مالی",
                    "میزان پول نیست",
                    "مدیریت کردن پول",
                ],
                [
                    "سواد مالی",
                    "میزان پول نیست",
                    "مدیریت پول",
                ],
            ],
        },
    ],

    24: [
        {
            "claim_id": "24-A",
            "description":
                "سواد مالی دارای ابعاد دانش، مهارت و نگرش است.",

            "support_patterns": [
                [
                    "سواد مالی",
                    "دانش",
                    "مهارت",
                    "نگرش",
                ],
                [
                    "قابلیت های مالی",
                    "دانش",
                    "مهارت",
                    "نگرش",
                ],
            ],
        },

        {
            "claim_id": "24-B",
            "description":
                "برای ارزیابی وضعیت سواد مالی از پرسشنامه یا خودارزیابی استفاده می‌شود.",

            "support_patterns": [
                [
                    "سواد مالی",
                    "پرسشنامه",
                    "ارزیابی",
                ],
                [
                    "سواد مالی",
                    "پرسشنامه",
                    "اندازه گیری",
                ],
                [
                    "سواد مالی",
                    "خودارزیابی",
                ],
                [
                    "وضعیت مالی",
                    "پرسشنامه",
                ],
            ],
        },
    ],

    25: [
        {
            "claim_id": "25-A",
            "description":
                "فرد در پایان دوره نمی‌داند پول را چگونه و برای چه چیزی خرج کرده است.",

            "support_patterns": [
                [
                    "موجودی حساب",
                    "یادمان نمی آید",
                    "خرج کرده ایم",
                ],
                [
                    "چگونه و برای چه چیزی خرج کرده ایم",
                ],
                [
                    "پول ها کجا می روند",
                ],
            ],
        },

        {
            "claim_id": "25-B",
            "description":
                "گام نخست مدیریت مالی، نوشتن یا ثبت هزینه‌هاست.",

            "support_patterns": [
                [
                    "گام نخست",
                    "مدیریت مالی شخصی",
                    "نوشتن هزینه",
                ],
                [
                    "ثبت و ردیابی هزینه",
                ],
                [
                    "ثبت هزینه",
                    "مدیریت",
                ],
            ],
        },

        {
            "claim_id": "25-C",
            "description":
                "ثبت هزینه‌ها به پیشگیری از ولخرجی، کاهش هدررفت یا مدیریت بهتر هزینه‌ها کمک می‌کند.",

            "support_patterns": [
                [
                    "نوشتن هزینه",
                    "پیشگیری از ولخرجی",
                ],
                [
                    "نوشتن هزینه",
                    "مدیریت بهتر هزینه",
                ],
                [
                    "ثبت هزینه",
                    "رفتارهای مالی",
                ],
                [
                    "هدررفت پول",
                    "نوشتن هزینه",
                ],
            ],
        },
    ],

    26: [
        {
            "claim_id": "26-A",
            "description":
                "خرید با تنوع یا حجم زیاد می‌تواند از الگوهای نامناسب خرید باشد.",

            "support_patterns": [
                [
                    "خرید با تنوع زیاد",
                ],
                [
                    "خرید با حجم زیاد",
                ],
                [
                    "تنوع زیاد خرید",
                ],
                [
                    "حجم زیاد خرید",
                ],
                [
                    "خریدهای زیاد و متنوع",
                ],
            ],
        },

        {
            "claim_id": "26-B",
            "description":
                "ناراحتی، استرس یا تنش عاطفی می‌تواند محرک خرید باشد.",

            "support_patterns": [
                [
                    "خرید",
                    "ناراحتی",
                ],
                [
                    "خرید",
                    "استرس",
                ],
                [
                    "خرید",
                    "تنش عاطفی",
                ],
                [
                    "هنگام ناراحتی",
                    "خرید",
                ],
                [
                    "مقابله با ناراحتی",
                    "خرید",
                ],
            ],
        },

        {
            "claim_id": "26-C",
            "description":
                "هیجان یا شتاب‌زدگی می‌تواند محرک خرید باشد.",

            "support_patterns": [
                [
                    "خرید هیجانی",
                ],
                [
                    "خرید",
                    "هیجان زودگذر",
                ],
                [
                    "خرید",
                    "هیجانی",
                ],
                [
                    "خرید",
                    "شتاب زده",
                ],
            ],
        },

        {
            "claim_id": "26-D",
            "description":
                "خرید برای پذیرفته‌شدن یا تأیید شدن در گروه مرجع.",

            "support_patterns": [
                [
                    "خرید",
                    "گروه مرجع",
                ],
                [
                    "خرید",
                    "گروه های مرجع",
                ],
                [
                    "خرید",
                    "پذیرفته شدن",
                    "گروه",
                ],
                [
                    "خرید",
                    "تایید دیگران",
                ],
                [
                    "خرید",
                    "پذیرش اجتماعی",
                ],
            ],
        },

        {
            "claim_id": "26-E",
            "description":
                "تبلیغات یا بازاریابی می‌تواند محرک خرید نامناسب باشد.",

            "support_patterns": [
                [
                    "خرید",
                    "تبلیغات",
                ],
                [
                    "فهرست خرید",
                    "تبلیغات",
                ],
                [
                    "خرید",
                    "بازاریابی",
                ],
                [
                    "الگوهای نادرست خرید",
                    "بازاریابانه",
                ],
            ],
        },

        {
            "claim_id": "26-F",
            "description":
                "خرید پس از یک دوره سخت برای پرداختن یا پاداش دادن به خود.",

            "support_patterns": [
                [
                    "خرید",
                    "دوره سخت",
                ],
                [
                    "خرید",
                    "پرداختن به خود",
                ],
                [
                    "خرید",
                    "برای خودم",
                    "جبران",
                ],
                [
                    "خرید",
                    "پاداش به خود",
                ],
            ],
        },

        {
            "claim_id": "26-G",
            "description":
                "خرید یا خرج‌کردن ناشی از فداکاری برای دیگران.",

            "support_patterns": [
                [
                    "به خاطر فداکاری",
                    "خرید",
                ],
                [
                    "به دلیل فداکاری",
                    "خرید",
                ],
                [
                    "به خاطر فداکاری",
                    "خرج",
                ],
                [
                    "به دلیل فداکاری",
                    "خرج",
                ],
                [
                    "فداکاری",
                    "برای دیگران خرید",
                ],
                [
                    "فداکاری",
                    "برای دیگران خرج",
                ],
            ],
        },
    ],

    27: [
        {
            "claim_id": "27-A",
            "description":
                "فروش وسایلی که دیگر به آن‌ها نیاز نداریم می‌تواند راه افزایش درآمد باشد.",

            "support_patterns": [
                [
                    "فروش وسایلی",
                    "نیاز ندارند",
                ],
                [
                    "فروش",
                    "وسایل",
                    "نیاز نداریم",
                ],
            ],
        },

        {
            "claim_id": "27-B",
            "description":
                "پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.",

            "support_patterns": [
                [
                    "پیشنهاد",
                    "کارفرما",
                    "پروژه",
                ],
                [
                    "پیشنهاد",
                    "کارفرما",
                    "کار بیشتر",
                ],
                [
                    "کارفرمای فعلی",
                    "پیشنهاد",
                    "پروژه",
                ],
                [
                    "کارفرمای فعلی",
                    "پیشنهاد",
                    "کار",
                ],
                [
                    "از کارفرما",
                    "کار بیشتر",
                ],
                [
                    "از کارفرما",
                    "پروژه بیشتر",
                ],
            ],
        },

        {
            "claim_id": "27-C",
            "description":
                "خرید دارایی درآمدزا.",

            "support_patterns": [
                [
                    "خرید دارایی های درآمدزا",
                ],
                [
                    "خرید دارایی درآمدزا",
                ],
                [
                    "درآمد",
                    "دارایی های درآمدزا",
                ],
            ],
        },

        {
            "claim_id": "27-D",
            "description":
                "سرمایه‌گذاری دوباره یا مجدد سود حاصل.",

            "support_patterns": [
                [
                    "سرمایه گذاری دوباره سود",
                ],
                [
                    "سرمایه گذاری مجدد سود",
                ],
                [
                    "سود",
                    "دوباره سرمایه گذاری",
                ],
                [
                    "سود",
                    "مجدد سرمایه گذاری",
                ],
            ],
        },

        {
            "claim_id": "27-E",
            "description":
                "ایجاد درآمد منفعل یا غیرفعال که به فعالیت مستمر شغلی وابسته نیست.",

            "support_patterns": [
                [
                    "درآمد غیرفعال",
                    "نیازی به فعالیت",
                ],
                [
                    "درآمد غیرفعال",
                    "بدون نیاز به فعالیت",
                ],
                [
                    "درآمد منفعل",
                    "فعالیت",
                ],
                [
                    "درآمد غیرفعال",
                    "پول",
                    "کار می کند",
                ],
            ],
        },
    ],

    28: [
        {
            "claim_id": "28-A",
            "description":
                "مسئله می‌تواند به رابطه فرد با پول مربوط باشد.",

            "support_patterns": [
                [
                    "رابطه ما با پول",
                ],
                [
                    "رابطه با پول",
                    "نگرانی",
                ],
                [
                    "ارتباط ما با پول",
                    "نگرانی",
                ],
            ],
        },

        {
            "claim_id": "28-B",
            "description":
                "پول می‌تواند نماینده چیزهایی باشد که فرد احساس می‌کند ندارد.",

            "support_patterns": [
                [
                    "پول",
                    "چیزهایی ست که نداریم",
                ],
                [
                    "پول",
                    "چیزهایی است که نداریم",
                ],
                [
                    "پول",
                    "مقام",
                    "امنیت",
                    "آزادی",
                    "خوشبختی",
                ],
            ],
        },

        {
            "claim_id": "28-C",
            "description":
                "پول می‌تواند برای اثبات خود یا جبران کمبودهای روانی استفاده شود.",

            "support_patterns": [
                [
                    "پول",
                    "اثبات خود",
                ],
                [
                    "پول ابزاری است",
                    "ثابت کنم",
                ],
                [
                    "خرج",
                    "اعتماد به نفس",
                    "اثبات خود",
                ],
                [
                    "پول",
                    "خلاء های عاطفی",
                ],
            ],
        },

        {
            "claim_id": "28-D",
            "description":
                "احساس فقر با نرسیدن به خواسته‌ها یا تمرکز بر نداشته‌ها ارتباط دارد.",

            "support_patterns": [
                [
                    "فقر",
                    "نرسیدن",
                    "خواسته",
                ],
                [
                    "احساس فقر",
                    "خواسته",
                    "نداشته",
                ],
                [
                    "احساس فقر",
                    "چیزهایی",
                    "ندار",
                ],
            ],
        },
    ],
}


def cell115_split_sentences(text):
    return [
        part.strip()
        for part in re.split(
            r"(?<=[\.\!\؟])\s+|\n+",
            str(text or ""),
        )
        if part.strip()
    ]


def cell115_pattern_matches_window(
    normalized_window,
    pattern,
):
    normalized_parts = [
        cell112_normalize(part)
        for part in pattern
        if cell112_normalize(part)
    ]

    if not normalized_parts:
        return False

    return all(
        part in normalized_window
        for part in normalized_parts
    )


def cell115_find_support_windows(
    source_text,
    support_patterns,
):
    sentences = cell115_split_sentences(
        source_text
    )

    if not sentences:
        return []

    results = []

    for start in range(
        len(sentences)
    ):
        end = min(
            len(sentences),
            start
            +
            CELL115_SENTENCE_WINDOW,
        )

        window_sentences = (
            sentences[
                start:end
            ]
        )

        window_text = " ".join(
            window_sentences
        )

        normalized_window = (
            cell112_normalize(
                window_text
            )
        )

        matched_patterns = []

        for pattern in support_patterns:
            if cell115_pattern_matches_window(
                normalized_window,
                pattern,
            ):
                matched_patterns.append(
                    pattern
                )

        if not matched_patterns:
            continue

        results.append({
            "start_sentence":
                start,

            "end_sentence":
                end,

            "window":
                window_text,

            "matched_patterns":
                matched_patterns,
        })

    return results


def cell115_search_claim(
    claim,
):
    matches = []

    for doc in cell113_nonempty_docs:

        support_windows = (
            cell115_find_support_windows(
                doc["text"],
                claim[
                    "support_patterns"
                ],
            )
        )

        if not support_windows:
            continue

        best_window = sorted(
            support_windows,
            key=lambda row: (
                -len(
                    row[
                        "matched_patterns"
                    ]
                ),
                len(
                    row[
                        "window"
                    ]
                ),
                row[
                    "start_sentence"
                ],
            ),
        )[0]

        matches.append({
            "document_id":
                doc[
                    "document_id"
                ],

            "title":
                doc[
                    "title"
                ],

            "url":
                doc[
                    "url"
                ],

            "matched_pattern_count":
                len(
                    best_window[
                        "matched_patterns"
                    ]
                ),

            "matched_patterns":
                best_window[
                    "matched_patterns"
                ],

            "evidence_window":
                best_window[
                    "window"
                ],
        })

    matches.sort(
        key=lambda row: (
            -row[
                "matched_pattern_count"
            ],
            row[
                "document_id"
            ],
        )
    )

    return matches


cell115_results = {}


for number in CELL115_CASE_NUMBERS:

    pilot_claims = []

    for claim in CELL115_CLAIMS[
        number
    ]:

        matches = (
            cell115_search_claim(
                claim
            )
        )

        pilot_claims.append({
            "claim_id":
                claim[
                    "claim_id"
                ],

            "description":
                claim[
                    "description"
                ],

            "support_patterns":
                claim[
                    "support_patterns"
                ],

            "matches":
                matches,

            "strict_found":
                len(matches) > 0,
        })

    cell115_results[
        number
    ] = pilot_claims


expected_claim_count = sum(
    len(
        CELL115_CLAIMS[
            number
        ]
    )
    for number
    in CELL115_CASE_NUMBERS
)

actual_claim_count = sum(
    len(
        cell115_results[
            number
        ]
    )
    for number
    in CELL115_CASE_NUMBERS
)


cell115_check(
    "PILOT_RESULT_COUNT_NOT_6",
    sorted(
        cell115_results.keys()
    )
    ==
    CELL115_CASE_NUMBERS,
)

cell115_check(
    "CLAIM_COUNT_MISMATCH",
    actual_claim_count
    ==
    expected_claim_count,
    {
        "expected":
            expected_claim_count,

        "actual":
            actual_claim_count,
    },
)


print(
    "=== CODE CELL 115 — STRICT CLAIM SUPPORT AUDIT ==="
)

print(
    "\n=== REPAIR PURPOSE ==="
)

print(
    "Previous broad phrase validator accepted semantic false positives."
)

print(
    "Current rule: all phrases in at least one support pattern must co-occur inside the same local sentence window."
)

print(
    "Sentence window:",
    CELL115_SENTENCE_WINDOW,
)

print(
    "\n=== SCOPE ==="
)

print(
    "Website documents scanned per claim:",
    len(
        cell113_nonempty_docs
    ),
)

print(
    "Pilots:",
    CELL115_CASE_NUMBERS,
)

print(
    "Total atomic claims:",
    expected_claim_count,
)

print(
    "Author filter applied:",
    False,
)

print(
    "Authority filter applied:",
    False,
)


for number in CELL115_CASE_NUMBERS:

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "PILOT:",
        number,
    )

    for claim_result in (
        cell115_results[
            number
        ]
    ):

        print(
            "\n"
            +
            "-" * 100
        )

        print(
            "Claim:",
            claim_result[
                "claim_id"
            ],
        )

        print(
            "Description:",
            claim_result[
                "description"
            ],
        )

        print(
            "Website documents scanned:",
            len(
                cell113_nonempty_docs
            ),
        )

        print(
            "Matching documents:",
            len(
                claim_result[
                    "matches"
                ]
            ),
        )

        print(
            "Strict support:",
            (
                "FOUND"
                if claim_result[
                    "strict_found"
                ]
                else
                "NOT_FOUND"
            ),
        )

        if not claim_result[
            "matches"
        ]:
            continue

        for rank, row in enumerate(
            claim_result[
                "matches"
            ][
                :CELL115_TOP_K_PER_CLAIM
            ],
            start=1,
        ):

            print(
                "\nRank:",
                rank,
            )

            print(
                "Document ID:",
                row[
                    "document_id"
                ],
            )

            print(
                "Title:",
                row[
                    "title"
                ],
            )

            print(
                "URL:",
                row[
                    "url"
                ],
            )

            print(
                "Matched support patterns:",
                row[
                    "matched_patterns"
                ],
            )

            print(
                "Evidence window:"
            )

            print(
                row[
                    "evidence_window"
                ]
            )


print(
    "\n"
    +
    "=" * 120
)

print(
    "=== STRICT CLAIM COVERAGE SUMMARY ==="
)


for number in CELL115_CASE_NUMBERS:

    results = (
        cell115_results[
            number
        ]
    )

    found_count = sum(
        1
        for row in results
        if row[
            "strict_found"
        ]
    )

    total_count = len(
        results
    )

    print(
        (
            f"Pilot {number}: "
            f"{found_count}/{total_count} "
            f"claims STRICT_FOUND"
        )
    )

    for row in results:

        print(
            " ",
            row[
                "claim_id"
            ],
            "=",
            (
                "STRICT_FOUND"
                if row[
                    "strict_found"
                ]
                else
                "NOT_FOUND"
            ),
        )


cell115_corpus_sha_after = (
    cell112_sha256_file(
        cell112_corpus_path
    )
)

cell115_registry_sha_after = (
    cell112_sha256_file(
        cell112_registry_path
    )
)

cell115_author_md_sha_after = (
    cell112_sha256_file(
        cell112_author_md_path
    )
    if cell112_author_md_path.exists()
    else None
)


cell115_check(
    "CORPUS_MUTATED",
    cell115_corpus_sha_after
    ==
    cell113_corpus_sha_before,
)

cell115_check(
    "REGISTRY_MUTATED",
    cell115_registry_sha_after
    ==
    cell113_registry_sha_before,
)

cell115_check(
    "AUTHOR_MD_MUTATED",
    cell115_author_md_sha_after
    ==
    cell113_author_md_sha_before,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Corpus mutated:",
    cell115_corpus_sha_after
    !=
    cell113_corpus_sha_before,
)

print(
    "Registry mutated:",
    cell115_registry_sha_after
    !=
    cell113_registry_sha_before,
)

print(
    "Author Review Markdown mutated:",
    cell115_author_md_sha_after
    !=
    cell113_author_md_sha_before,
)

print(
    "Author decisions recorded:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell115_failures
)


cell115_gate = (
    len(
        cell115_failures
    )
    ==
    0
)


print(
    "\nCODE CELL 115 GATE:",
    (
        "PASS"
        if cell115_gate
        else
        "FAIL"
    ),
)


assert cell115_gate, (
    "Strict claim support audit failed. "
    "Repair this SAME Code Cell 115 before continuing."
)

=== CODE CELL 115 — STRICT CLAIM SUPPORT AUDIT ===

=== REPAIR PURPOSE ===
Previous broad phrase validator accepted semantic false positives.
Current rule: all phrases in at least one support pattern must co-occur inside the same local sentence window.
Sentence window: 4

=== SCOPE ===
Website documents scanned per claim: 529
Pilots: [23, 24, 25, 26, 27, 28]
Total atomic claims: 22
Author filter applied: False
Authority filter applied: False

PILOT: 23

----------------------------------------------------------------------------------------------------
Claim: 23-A
Description: سواد مالی درباره مدیریت پول است، نه میزان پول.
Website documents scanned: 529
Matching documents: 1
Strict support: FOUND

Rank: 1
Document ID: website:7C71884B
Title: ابزارهای سواد مالی برای بهبود مدیریت هزینه
URL: https://www.fintelligence.ir/Blog/Detail/7C71884B
Matched support patterns: [['سواد مالی', 'میزان پول نیست', 'مدیریت کردن پول']]
Evidence window:
فرهاد جم : وقتی درآمدمان کم است، سواد مالی چطور به ما 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ممیزی نهایی Claimهای حل‌نشده در پایلوت‌های ۲۶ و ۲۷</h3>

<h4>هدف</h4>

<p>
پس از ممیزی کامل ۵۲۹ رکورد Website و اجرای Strict Claim Audit،
بیشتر Claimهای پایلوت‌های ۲۳ تا ۲۸ دارای Source محتوایی قابل اتکا شدند.
با این حال چند Claim هنوز یا Source قطعی محتوایی ندارند
یا Validator برای آن‌ها False Positive تولید کرده است.
</p>

<p>
هدف این مرحله فقط بررسی همین Claimهای حل‌نشده است؛
نه اجرای دوباره جست‌وجوی عمومی کل پروژه.
</p>

<h4>Claimهای باقی‌مانده</h4>

<ul>
<li>26-A — خرید با تنوع یا حجم زیاد.</li>
<li>26-D — خرید برای پذیرفته‌شدن در گروه مرجع یا گروه خاص.</li>
<li>26-F — خرید پس از یک دوره سخت برای پرداختن یا پاداش دادن به خود.</li>
<li>26-G — خرید یا خرج‌کردن ناشی از فداکاری برای دیگران.</li>
<li>27-B — پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.</li>
<li>27-D — سرمایه‌گذاری دوباره یا مجدد سود حاصل.</li>
</ul>

<h4>دلیل این مرحله</h4>

<p>
در Cell 115 مشخص شد که صرف هم‌زمانی چند واژه در یک پنجرۀ متنی
برای اثبات رابطۀ معنایی کافی نیست.
برای نمونه، وجود واژه‌های «پیشنهاد»، «کارفرما» و «پروژه»
در یک پاراگراف الزاماً به معنای پیشنهاد پروژه بیشتر به کارفرمای فعلی نیست.
</p>

<p>
در مقابل، برای 26-D یک Evidence مهم در مقاله
«ضرورت یادگیری مهارت‌های مالی در دوران دانشجویی»
مشاهده شد که به «مورد قبول واقع شدن در گروه خاص» اشاره می‌کند.
بنابراین لازم است متن کامل اطراف این بخش و Candidateهای مشابه بررسی شود.
</p>

<h4>Source of Truth</h4>

<p>
همان ۵۲۹ Website Record موجود در Accepted Corpus.
</p>

<p>
هیچ فیلتر Author یا Authority اعمال نمی‌شود.
</p>

<h4>روش بررسی</h4>

<p>
برای هر Claim حل‌نشده، مجموعه‌ای گسترده‌تر از واژه‌ها و عبارت‌های معنایی تعریف می‌شود.
سپس در تمام ۵۲۹ Website، تمام پنجره‌هایی که حداقل چند Signal مرتبط را هم‌زمان دارند
استخراج و نمایش داده می‌شوند.
</p>

<p>
این Cell به‌صورت خودکار هیچ Claim را FOUND یا NOT_FOUND نهایی اعلام نمی‌کند.
هدف آن نمایش Evidence واقعی برای Human Source Review است.
</p>

<h4>خروجی مورد انتظار</h4>

<ul>
<li>Claim مورد بررسی.</li>
<li>تمام Candidateهای مرتبط تا سقف تعیین‌شده.</li>
<li>Document ID.</li>
<li>عنوان.</li>
<li>URL.</li>
<li>Signalهای Match‌شده.</li>
<li>متن واقعی و نسبتاً بلند اطراف Match.</li>
</ul>

<h4>معیار PASS</h4>

<ul>
<li>هر شش Claim باقی‌مانده بررسی شوند.</li>
<li>هر Claim روی تمام ۵۲۹ Website اجرا شود.</li>
<li>هیچ Candidate به‌طور خودکار Source نهایی اعلام نشود.</li>
<li>متن واقعی Candidateها برای تصمیم انسانی نمایش داده شود.</li>
<li>Corpus، Registry و Author Review Markdown تغییر نکنند.</li>
<li>خروجی نهایی برابر <code>CODE CELL 116 GATE: PASS</code> باشد.</li>
</ul>

<h4>معیار FAIL</h4>

<ul>
<li>یکی از شش Claim بررسی نشود.</li>
<li>Scope کمتر از ۵۲۹ Website باشد.</li>
<li>Source صرفاً بر اساس هم‌زمانی واژگان Accepted شود.</li>
<li>Corpus، Registry یا Author Review Markdown تغییر کند.</li>
</ul>

<h4>خارج از Scope</h4>

<p>
این مرحله هنوز فایل <code>author_review_28_pilots.md</code> را تغییر نمی‌دهد.
تصمیم نهایی Source پس از بررسی Human Review همین Output انجام می‌شود.
</p>

<h4>ایمنی نوشتن</h4>

<ul>
<li>Corpus Write: ممنوع.</li>
<li>Registry Write: ممنوع.</li>
<li>Author Review Markdown Write: ممنوع.</li>
<li>Author Decision Write: ممنوع.</li>
<li>Commit / Push: ممنوع.</li>
</ul>

</div>

In [11]:
# Code Cell 116

cell116_failures = []


def cell116_check(reason, condition, details=None):
    if not bool(condition):
        item = {"reason": reason}

        if details is not None:
            item["details"] = details

        cell116_failures.append(item)

    return bool(condition)


cell116_check(
    "CELL115_GATE_NOT_PASS",
    (
        "cell115_gate" in globals()
        and
        cell115_gate is True
    ),
)

cell116_check(
    "WEBSITE_SCOPE_NOT_529",
    (
        "cell113_nonempty_docs" in globals()
        and
        len(cell113_nonempty_docs) == 529
    ),
    (
        len(cell113_nonempty_docs)
        if "cell113_nonempty_docs" in globals()
        else None
    ),
)

if cell116_failures:
    raise AssertionError(cell116_failures)


CELL116_TOP_K = 15
CELL116_WINDOW_SENTENCES = 8


CELL116_UNRESOLVED = {

    "26-A": {
        "description":
            "خرید با تنوع یا حجم زیاد",

        "signal_groups": [
            [
                "خرید",
                "تنوع",
            ],
            [
                "خرید",
                "متنوع",
            ],
            [
                "خرید",
                "حجم",
            ],
            [
                "خریدهای",
                "زیاد",
            ],
            [
                "کالا",
                "زیاد",
                "خرید",
            ],
            [
                "چیز",
                "زیاد",
                "خرید",
            ],
        ],
    },

    "26-D": {
        "description":
            "خرید برای پذیرفته‌شدن یا تأیید شدن در گروه مرجع",

        "signal_groups": [
            [
                "خرید",
                "گروه",
                "پذیرفته",
            ],
            [
                "خرید",
                "گروه خاص",
            ],
            [
                "هزینه",
                "گروه خاص",
            ],
            [
                "هزینه کرد",
                "مورد قبول",
                "گروه",
            ],
            [
                "خرج",
                "تایید دیگران",
            ],
            [
                "خرج",
                "پذیرش اجتماعی",
            ],
            [
                "خرید",
                "همسالان",
            ],
            [
                "خرید",
                "دوستان",
                "قبول",
            ],
        ],
    },

    "26-F": {
        "description":
            "خرید پس از یک دوره سخت برای پرداختن یا پاداش دادن به خود",

        "signal_groups": [
            [
                "خرید",
                "دوره سخت",
            ],
            [
                "خرید",
                "روزهای سخت",
            ],
            [
                "خرید",
                "سختی",
                "خود",
            ],
            [
                "خرید",
                "پاداش",
                "خود",
            ],
            [
                "خرید",
                "حال خود",
            ],
            [
                "خرج",
                "پاداش",
                "خود",
            ],
            [
                "خرج",
                "سختی",
                "خود",
            ],
            [
                "خرید",
                "جبران",
                "خود",
            ],
        ],
    },

    "26-G": {
        "description":
            "خرید یا خرج‌کردن ناشی از فداکاری برای دیگران",

        "signal_groups": [
            [
                "خرید",
                "فداکاری",
            ],
            [
                "خرج",
                "فداکاری",
            ],
            [
                "هزینه",
                "فداکاری",
            ],
            [
                "خرید",
                "ایثار",
            ],
            [
                "خرج",
                "ایثار",
            ],
            [
                "خرید",
                "از خودگذشتگی",
            ],
            [
                "خرج",
                "از خودگذشتگی",
            ],
            [
                "برای دیگران",
                "خرج",
                "خود",
            ],
            [
                "برای دیگران",
                "خرید",
                "خود",
            ],
        ],
    },

    "27-B": {
        "description":
            "پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی",

        "signal_groups": [
            [
                "کارفرمای فعلی",
                "پروژه",
            ],
            [
                "کارفرمای فعلی",
                "کار بیشتر",
            ],
            [
                "کارفرما",
                "پیشنهاد پروژه",
            ],
            [
                "کارفرما",
                "پروژه بیشتر",
            ],
            [
                "کارفرما",
                "کار بیشتر",
            ],
            [
                "کارفرما",
                "مسئولیت بیشتر",
            ],
            [
                "کارفرما",
                "اضافه کاری",
            ],
            [
                "افزایش درآمد",
                "کارفرما",
                "پروژه",
            ],
        ],
    },

    "27-D": {
        "description":
            "سرمایه‌گذاری دوباره یا مجدد سود حاصل",

        "signal_groups": [
            [
                "سود",
                "سرمایه گذاری مجدد",
            ],
            [
                "سود",
                "سرمایه گذاری دوباره",
            ],
            [
                "سود را",
                "دوباره سرمایه گذاری",
            ],
            [
                "سود را",
                "مجدد سرمایه گذاری",
            ],
            [
                "سود",
                "بازسرمایه گذاری",
            ],
            [
                "درآمد",
                "دوباره سرمایه گذاری",
            ],
            [
                "سود حاصل",
                "سرمایه گذاری",
                "دوباره",
            ],
            [
                "بازده",
                "سرمایه گذاری مجدد",
            ],
        ],
    },
}


def cell116_split_sentences(text):
    return [
        part.strip()
        for part in re.split(
            r"(?<=[\.\!\؟])\s+|\n+",
            str(text or ""),
        )
        if part.strip()
    ]


def cell116_group_matches(
    normalized_text,
    group,
):
    normalized_signals = [
        cell112_normalize(signal)
        for signal in group
        if cell112_normalize(signal)
    ]

    if not normalized_signals:
        return False

    return all(
        signal in normalized_text
        for signal in normalized_signals
    )


def cell116_find_candidate_windows(
    source_text,
    signal_groups,
):
    sentences = cell116_split_sentences(
        source_text
    )

    rows = []

    if not sentences:
        return rows

    for start in range(
        len(sentences)
    ):
        end = min(
            len(sentences),
            start + CELL116_WINDOW_SENTENCES,
        )

        window = " ".join(
            sentences[start:end]
        )

        normalized_window = (
            cell112_normalize(
                window
            )
        )

        matched_groups = []

        for group in signal_groups:
            if cell116_group_matches(
                normalized_window,
                group,
            ):
                matched_groups.append(
                    group
                )

        if not matched_groups:
            continue

        rows.append({
            "start_sentence":
                start,

            "end_sentence":
                end,

            "window":
                window,

            "matched_groups":
                matched_groups,
        })

    return rows


def cell116_search_claim(
    config,
):
    candidates = []

    for doc in cell113_nonempty_docs:

        windows = (
            cell116_find_candidate_windows(
                doc["text"],
                config[
                    "signal_groups"
                ],
            )
        )

        if not windows:
            continue

        best_window = sorted(
            windows,
            key=lambda row: (
                -len(
                    row[
                        "matched_groups"
                    ]
                ),
                row[
                    "start_sentence"
                ],
            ),
        )[0]

        candidates.append({
            "document_id":
                doc[
                    "document_id"
                ],

            "title":
                doc[
                    "title"
                ],

            "url":
                doc[
                    "url"
                ],

            "matched_group_count":
                len(
                    best_window[
                        "matched_groups"
                    ]
                ),

            "matched_groups":
                best_window[
                    "matched_groups"
                ],

            "evidence_window":
                best_window[
                    "window"
                ],
        })

    candidates.sort(
        key=lambda row: (
            -row[
                "matched_group_count"
            ],
            row[
                "document_id"
            ],
        )
    )

    return candidates


cell116_results = {}


for claim_id, config in (
    CELL116_UNRESOLVED.items()
):

    cell116_results[
        claim_id
    ] = cell116_search_claim(
        config
    )


cell116_check(
    "UNRESOLVED_CLAIM_COUNT_MISMATCH",
    len(
        cell116_results
    )
    ==
    len(
        CELL116_UNRESOLVED
    ),
)


print(
    "=== CODE CELL 116 — FINAL UNRESOLVED CLAIM EVIDENCE AUDIT ==="
)

print(
    "\n=== SCOPE ==="
)

print(
    "Website documents scanned per claim:",
    len(
        cell113_nonempty_docs
    ),
)

print(
    "Unresolved claims:",
    len(
        CELL116_UNRESOLVED
    ),
)

print(
    "Author filter applied:",
    False,
)

print(
    "Authority filter applied:",
    False,
)

print(
    "Automatic Human Source Decision:",
    False,
)


for claim_id, config in (
    CELL116_UNRESOLVED.items()
):

    candidates = (
        cell116_results[
            claim_id
        ]
    )

    print(
        "\n"
        +
        "=" * 120
    )

    print(
        "CLAIM:",
        claim_id,
    )

    print(
        "Description:",
        config[
            "description"
        ],
    )

    print(
        "Website documents scanned:",
        len(
            cell113_nonempty_docs
        ),
    )

    print(
        "Candidate documents:",
        len(
            candidates
        ),
    )

    if not candidates:

        print(
            "\nNO CANDIDATE WINDOW FOUND"
        )

        print(
            "HUMAN SOURCE DECISION: NOT_STARTED"
        )

        continue

    for rank, row in enumerate(
        candidates[
            :CELL116_TOP_K
        ],
        start=1,
    ):

        print(
            "\n"
            +
            "-" * 100
        )

        print(
            "Rank:",
            rank,
        )

        print(
            "Document ID:",
            row[
                "document_id"
            ],
        )

        print(
            "Title:",
            row[
                "title"
            ],
        )

        print(
            "URL:",
            row[
                "url"
            ],
        )

        print(
            "Matched signal groups:",
            row[
                "matched_groups"
            ],
        )

        print(
            "\nEvidence window:"
        )

        print(
            row[
                "evidence_window"
            ]
        )

    print(
        "\nHUMAN SOURCE DECISION: NOT_STARTED"
    )


cell116_corpus_sha_after = (
    cell112_sha256_file(
        cell112_corpus_path
    )
)

cell116_registry_sha_after = (
    cell112_sha256_file(
        cell112_registry_path
    )
)

cell116_author_md_sha_after = (
    cell112_sha256_file(
        cell112_author_md_path
    )
    if cell112_author_md_path.exists()
    else None
)


cell116_check(
    "CORPUS_MUTATED",
    cell116_corpus_sha_after
    ==
    cell113_corpus_sha_before,
)

cell116_check(
    "REGISTRY_MUTATED",
    cell116_registry_sha_after
    ==
    cell113_registry_sha_before,
)

cell116_check(
    "AUTHOR_MD_MUTATED",
    cell116_author_md_sha_after
    ==
    cell113_author_md_sha_before,
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "=== UNRESOLVED CLAIM SUMMARY ==="
)


for claim_id in (
    CELL116_UNRESOLVED
):

    print(
        claim_id,
        "| candidate documents =",
        len(
            cell116_results[
                claim_id
            ]
        ),
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Corpus mutated:",
    cell116_corpus_sha_after
    !=
    cell113_corpus_sha_before,
)

print(
    "Registry mutated:",
    cell116_registry_sha_after
    !=
    cell113_registry_sha_before,
)

print(
    "Author Review Markdown mutated:",
    cell116_author_md_sha_after
    !=
    cell113_author_md_sha_before,
)

print(
    "Author decisions recorded:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell116_failures
)


cell116_gate = (
    len(
        cell116_failures
    )
    ==
    0
)


print(
    "\nCODE CELL 116 GATE:",
    (
        "PASS"
        if cell116_gate
        else
        "FAIL"
    ),
)


assert cell116_gate, (
    "Final unresolved-claim evidence audit failed. "
    "Repair this SAME Code Cell 116 before continuing."
)

=== CODE CELL 116 — FINAL UNRESOLVED CLAIM EVIDENCE AUDIT ===

=== SCOPE ===
Website documents scanned per claim: 529
Unresolved claims: 6
Author filter applied: False
Authority filter applied: False
Automatic Human Source Decision: False

CLAIM: 26-A
Description: خرید با تنوع یا حجم زیاد
Website documents scanned: 529
Candidate documents: 39

----------------------------------------------------------------------------------------------------
Rank: 1
Document ID: website:F6928B2E
Title: پلتفرم‌های مبتنی بر اجارۀ کالا چه تأثیری بر میزان پس‌انداز دارند؟
URL: https://www.fintelligence.ir/Blog/Detail/F6928B2E
Matched signal groups: [['خرید', 'تنوع'], ['خرید', 'متنوع'], ['خریدهای', 'زیاد']]

Evidence window:
دو توصیه‌‌ برای سیاستگذاران این پژوهش نشان داد با وجود مزایای بسیار زیاد خدمات مبتنی بر دسترسی، یک نکتۀ منفی دربارۀ آن‌ها وجود دارد: این‌گونه خدمات، تمرکز بر زمان حال را افزایش داده و برنامه‌ریزی برای آینده را کم‌اهمیت جلوه می‌دهند. با توجه به عرضۀ گستردۀ این خدمات در بازار، به نظر می‌ر

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>ثبت Human Source Review و ساخت Source Packet نهایی برای پایلوت‌های ۲۳ تا ۲۸</h3>

<h4>هدف</h4>

<p>
پس از اجرای Source Recovery در کل Website Corpus و انجام چند مرحلۀ
Targeted Audit، Strict Claim Audit و ممیزی Claimهای حل‌نشده،
در این مرحله تصمیم‌های Human Source Review برای پایلوت‌های ۲۳ تا ۲۸
به‌صورت رسمی ثبت می‌شوند.
</p>

<p>
همچنین یک Source Packet ساختاریافته ایجاد می‌شود تا در مرحلۀ بعد،
فایل Author Review بر اساس Evidence واقعی اصلاح شود.
</p>

<h4>اصل مهم این مرحله</h4>

<p>
این مرحله بین دو نوع Authority تفاوت صریح قائل می‌شود:
</p>

<ul>
<li>
<strong>Website Source Backed:</strong>
Claimی که متن واقعی مرتبط با آن در Website Corpus بازیابی و پس از Human Review تأیید شده است.
</li>

<li>
<strong>Primary Direct Author Feedback Only:</strong>
Claimی که متن کافی و قابل اتکایی برای آن در کل ۵۲۹ Website بازیابی نشده،
اما خود دکتر کمیل رودی آن را مستقیماً در بازخورد فاز یک بیان کرده است.
</li>
</ul>

<p>
در حالت دوم، Claim حذف نمی‌شود و Source ساختگی نیز برای آن ایجاد نمی‌شود.
Authority آن Claim همان بازخورد مستقیم مؤلف باقی می‌ماند.
</p>

<h4>وضعیت Provenance تاریخی Demo</h4>

<p>
با وجود بازیابی Sourceهای محتوایی قوی، هیچ Artifact صریحی از اجرای دموی فاز یک
که Document ID یا Citation واقعی استفاده‌شده در Retrieval آن اجرا را اثبات کند پیدا نشد.
</p>

<p>
بنابراین وضعیت زیر برای تمام پایلوت‌های ۲۳ تا ۲۸ حفظ می‌شود:
</p>

<p>
<code>ORIGINAL_PHASE1_DEMO_PROVENANCE = NOT_PROVEN_FROM_AVAILABLE_ARTIFACTS</code>
</p>

<p>
Sourceهای بازیابی‌شده در این مرحله به‌عنوان
<strong>Content Source Recovery</strong>
ثبت می‌شوند، نه Provenance قطعی اجرای تاریخی Demo.
</p>

<h4>تصمیم نهایی پایلوت ۲۳</h4>

<ul>
<li>23-A — Website Source Backed.</li>
<li>Source: <code>website:7C71884B</code></li>
<li>موضوع: سواد مالی درباره مدیریت پول است، نه میزان پول.</li>
</ul>

<h4>تصمیم نهایی پایلوت ۲۴</h4>

<ul>
<li>24-A — Website Source Backed.</li>
<li>Source: <code>website:D76E641C</code></li>
<li>موضوع: دانش، مهارت و نگرش.</li>

<li>24-B — Website Source Backed.</li>
<li>Source: <code>website:662BDA7F</code></li>
<li>موضوع: سنجش وضعیت مالی با پرسشنامه / خودارزیابی.</li>
</ul>

<h4>تصمیم نهایی پایلوت ۲۵</h4>

<ul>
<li>25-A — Website Source Backed.</li>
<li>25-B — Website Source Backed.</li>
<li>25-C — Website Source Backed.</li>
<li>Source اصلی هر سه Claim: <code>website:3E26E509</code></li>
</ul>

<h4>تصمیم نهایی پایلوت ۲۶</h4>

<ul>
<li>26-A — Primary Direct Author Feedback Only.</li>

<li>26-B — Website Source Backed.</li>
<li>Source: <code>website:BC0342B6</code></li>

<li>26-C — Website Source Backed.</li>
<li>Source: <code>website:AE081F78</code></li>

<li>26-D — Website Source Backed.</li>
<li>Source: <code>website:D4DBC461</code></li>

<li>26-E — Website Source Backed.</li>
<li>Source: <code>website:D4DBC461</code></li>

<li>26-F — Primary Direct Author Feedback Only.</li>

<li>26-G — Primary Direct Author Feedback Only.</li>
</ul>

<h4>تصمیم نهایی پایلوت ۲۷</h4>

<ul>
<li>27-A — Website Source Backed.</li>
<li>Source: <code>website:0B74EC18</code></li>

<li>27-B — Primary Direct Author Feedback Only.</li>

<li>27-C — Website Source Backed.</li>
<li>Source: <code>website:7AB68D23</code></li>

<li>27-D — Primary Direct Author Feedback Only.</li>

<li>27-E — Website Source Backed.</li>
<li>Source: <code>website:A8746170</code></li>
</ul>

<h4>تصمیم نهایی پایلوت ۲۸</h4>

<ul>
<li>28-A — Website Source Backed.</li>
<li>28-B — Website Source Backed.</li>
<li>28-C — Website Source Backed.</li>
<li>Source: <code>website:2F8F15FB</code></li>

<li>28-D — Website Source Backed.</li>
<li>Source: <code>website:E3C871BC</code></li>
</ul>

<h4>خروجی‌های این مرحله</h4>

<p>
این مرحله دو خروجی ساختاریافته ایجاد می‌کند:
</p>

<ul>
<li>
ثبت تصمیم‌های Human Source Review در Registry.
</li>

<li>
ایجاد فایل:
<code>data/phase2/human_review/source_packet_pilots23_28_v1.json</code>
</li>
</ul>

<p>
Source Packet شامل Claimها، نوع Authority، Document ID، عنوان،
URL داخلی و Evidence Window واقعی است.
URLها صرفاً برای Provenance داخلی پروژه نگهداری می‌شوند
و قرار نیست در نسخۀ Author-Facing نمایش داده شوند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>هر شش پایلوت ۲۳ تا ۲۸ ثبت شوند.</li>
<li>هر ۲۲ Claim اتمیک ثبت شود.</li>
<li>۱۷ Claim به‌عنوان Website Source Backed ثبت شوند.</li>
<li>۵ Claim به‌عنوان Primary Direct Author Feedback Only ثبت شوند.</li>
<li>هیچ Claim بدون Evidence به Website نسبت داده نشود.</li>
<li>Original Demo Provenance همچنان NOT_PROVEN باقی بماند.</li>
<li>Raw Phase-1 Conversation تغییر نکند.</li>
<li>Raw Author Feedback تغییر نکند.</li>
<li>Author Review Queue تغییر نکند.</li>
<li>Completed Human Review Cases تغییر نکنند.</li>
<li>فایل <code>author_review_28_pilots.md</code> تغییر نکند.</li>
<li>Corpus تغییر نکند.</li>
<li>هیچ Author Decision ثبت نشود.</li>
<li>خروجی نهایی برابر <code>CODE CELL 117 GATE: PASS</code> باشد.</li>
</ul>

<h4>معیار FAIL</h4>

<ul>
<li>تعداد Claimها یا Pilotها با تصمیم تأییدشده سازگار نباشد.</li>
<li>Claim فاقد Source به‌صورت Website Source Backed ثبت شود.</li>
<li>Claim دارای Source تأییدشده بدون Evidence Window ثبت شود.</li>
<li>Registry در بخشی غیر از Source Review جدید تغییر کند.</li>
<li>Author Review Markdown یا Corpus تغییر کند.</li>
<li>Author Decision ثبت شود.</li>
</ul>

<h4>خارج از Scope</h4>

<p>
این مرحله هنوز متن فایل
<code>author_review_28_pilots.md</code>
را تغییر نمی‌دهد.
</p>

<p>
در مرحلۀ بعد، فقط بر اساس Source Packet تأییدشده،
بخش «متن منبع» پایلوت‌های ۲۳ تا ۲۸ اصلاح خواهد شد.
</p>

<h4>ایمنی نوشتن</h4>

<ul>
<li>Registry Write: فقط Source Review جدید مجاز است.</li>
<li>Source Packet Write: مجاز است.</li>
<li>Corpus Write: ممنوع.</li>
<li>Author Review Markdown Write: ممنوع.</li>
<li>Author Decision Write: ممنوع.</li>
<li>Commit / Push: ممنوع.</li>
</ul>

</div>

In [12]:
# Code Cell 117
# ============================================================
# HUMAN SOURCE REVIEW REGISTRATION
# + FINAL SOURCE PACKET FOR PILOTS 23-28
#
# IMPORTANT:
# - Registry mutation is allowed ONLY for the dedicated
#   source-provenance review section.
# - author_review_28_pilots.md MUST remain unchanged.
# - Corpus MUST remain unchanged.
# - No Author Decision is recorded.
# - No Commit / Push.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import copy
import hashlib
import json
import os


# ============================================================
# 1) Constants
# ============================================================

CELL117_EXPECTED_BASELINE_REGISTRY_SHA256 = (
    "10933065b1d4598dba1534964b8a6837"
    "f723cf79082d1af2bc8a4964e2a6766e"
)

CELL117_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1"
    "da34869f671e6e32762864e0ad70ea40"
)

CELL117_EXPECTED_AUTHOR_MD_SHA256 = (
    "f9b2e04a50e61060b150c958d444f130"
    "8379c76815edb299c25b96257b031f09"
)

CELL117_CASE_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]

CELL117_EXPECTED_ATOMIC_CLAIMS = 22

CELL117_EXPECTED_WEBSITE_BACKED = 17

CELL117_EXPECTED_AUTHOR_FEEDBACK_ONLY = 5

CELL117_EXPECTED_WEBSITE_SCOPE = 529

CELL117_REGISTRY_REVIEW_KEY = (
    "source_provenance_review_pilots23_28_v1"
)

CELL117_SOURCE_PACKET_REL = Path(
    "data/phase2/human_review/"
    "source_packet_pilots23_28_v1.json"
)

CELL117_ORIGINAL_DEMO_PROVENANCE_STATUS = (
    "NOT_PROVEN_FROM_AVAILABLE_ARTIFACTS"
)

CELL117_WEBSITE_STATUS = (
    "WEBSITE_SOURCE_BACKED"
)

CELL117_AUTHOR_ONLY_STATUS = (
    "PRIMARY_DIRECT_AUTHOR_FEEDBACK_ONLY"
)

cell117_failures = []


# ============================================================
# 2) Validation helper
# ============================================================

def cell117_check(
    reason,
    condition,
    details=None,
):
    if not bool(condition):

        item = {
            "reason": reason,
        }

        if details is not None:
            item["details"] = details

        cell117_failures.append(
            item
        )

    return bool(condition)


# ============================================================
# 3) Required previous state
# ============================================================

required_names = [
    "cell112_repo_root",
    "cell112_corpus_path",
    "cell112_registry_path",
    "cell112_author_md_path",
    "cell112_sha256_file",
    "cell112_cases",
    "cell113_nonempty_docs",
    "cell115_gate",
    "cell115_results",
    "CELL115_CLAIMS",
    "cell116_gate",
    "cell116_results",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]


cell117_check(
    "REQUIRED_PREVIOUS_STATE_MISSING",
    len(
        missing_names
    )
    ==
    0,
    missing_names,
)


if cell117_failures:

    print(
        "=== CELL 117 PREFLIGHT FAILURES ==="
    )

    print(
        cell117_failures
    )

    raise AssertionError(
        "Cell 117 requires accepted Cells 112-116."
    )


cell117_check(
    "CELL115_GATE_NOT_PASS",
    cell115_gate is True,
)

cell117_check(
    "CELL116_GATE_NOT_PASS",
    cell116_gate is True,
)

cell117_check(
    "WEBSITE_SCOPE_NOT_529",
    len(
        cell113_nonempty_docs
    )
    ==
    CELL117_EXPECTED_WEBSITE_SCOPE,
    len(
        cell113_nonempty_docs
    ),
)


if cell117_failures:
    raise AssertionError(
        cell117_failures
    )


# ============================================================
# 4) Resolve paths
# ============================================================

cell117_repo_root = Path(
    cell112_repo_root
).resolve()

cell117_corpus_path = Path(
    cell112_corpus_path
).resolve()

cell117_registry_path = Path(
    cell112_registry_path
).resolve()

cell117_author_md_path = Path(
    cell112_author_md_path
).resolve()

cell117_source_packet_path = (
    cell117_repo_root
    /
    CELL117_SOURCE_PACKET_REL
)


# ============================================================
# 5) Freeze immutable artifacts
# ============================================================

cell117_corpus_sha_before = (
    cell112_sha256_file(
        cell117_corpus_path
    )
)

cell117_registry_sha_before = (
    cell112_sha256_file(
        cell117_registry_path
    )
)

cell117_author_md_sha_before = (
    cell112_sha256_file(
        cell117_author_md_path
    )
)


cell117_check(
    "CORPUS_SHA_MISMATCH",
    cell117_corpus_sha_before
    ==
    CELL117_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL117_EXPECTED_CORPUS_SHA256,

        "actual":
            cell117_corpus_sha_before,
    },
)


cell117_check(
    "AUTHOR_MD_SHA_MISMATCH",
    cell117_author_md_sha_before
    ==
    CELL117_EXPECTED_AUTHOR_MD_SHA256,
    {
        "expected":
            CELL117_EXPECTED_AUTHOR_MD_SHA256,

        "actual":
            cell117_author_md_sha_before,
    },
)


if cell117_failures:

    print(
        "=== CELL 117 FROZEN INPUT FAILURE ==="
    )

    print(
        cell117_failures
    )

    raise AssertionError(
        "Cell 117 stopped Fail-Closed."
    )


# ============================================================
# 6) Load current Registry
# ============================================================

cell117_registry_before = json.loads(
    cell117_registry_path.read_text(
        encoding="utf-8"
    )
)

cell117_second_review_before = (
    cell117_registry_before.get(
        "second_human_review",
        {},
    )
    or
    {}
)

cell117_existing_review = (
    cell117_second_review_before.get(
        CELL117_REGISTRY_REVIEW_KEY
    )
)


# ============================================================
# 7) Claim description lookup
# ============================================================

cell117_claim_description = {}

for pilot_number in (
    CELL117_CASE_NUMBERS
):

    for claim in (
        CELL115_CLAIMS[
            pilot_number
        ]
    ):

        cell117_claim_description[
            claim[
                "claim_id"
            ]
        ] = claim[
            "description"
        ]


cell117_check(
    "CLAIM_DESCRIPTION_COUNT_NOT_22",
    len(
        cell117_claim_description
    )
    ==
    CELL117_EXPECTED_ATOMIC_CLAIMS,
    len(
        cell117_claim_description
    ),
)


# ============================================================
# 8) Approved Human Source Review decisions
# ============================================================

CELL117_DECISIONS = {

    23: [
        {
            "claim_id":
                "23-A",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:7C71884B",
        },
    ],

    24: [
        {
            "claim_id":
                "24-A",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:D76E641C",
        },

        {
            "claim_id":
                "24-B",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:662BDA7F",
        },
    ],

    25: [
        {
            "claim_id":
                "25-A",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:3E26E509",
        },

        {
            "claim_id":
                "25-B",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:3E26E509",
        },

        {
            "claim_id":
                "25-C",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:3E26E509",
        },
    ],

    26: [
        {
            "claim_id":
                "26-A",

            "status":
                CELL117_AUTHOR_ONLY_STATUS,

            "reason":
                (
                    "No semantically sufficient Website source "
                    "was recovered after full 529-record review."
                ),
        },

        {
            "claim_id":
                "26-B",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:BC0342B6",
        },

        {
            "claim_id":
                "26-C",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:AE081F78",
        },

        {
            "claim_id":
                "26-D",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL116",

            "document_id":
                "website:D4DBC461",
        },

        {
            "claim_id":
                "26-E",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:D4DBC461",
        },

        {
            "claim_id":
                "26-F",

            "status":
                CELL117_AUTHOR_ONLY_STATUS,

            "reason":
                (
                    "Candidate windows existed, but Human Source "
                    "Review rejected them as semantically insufficient."
                ),
        },

        {
            "claim_id":
                "26-G",

            "status":
                CELL117_AUTHOR_ONLY_STATUS,

            "reason":
                (
                    "Candidate windows discussed sacrifice, but did "
                    "not establish buying/spending because of sacrifice."
                ),
        },
    ],

    27: [
        {
            "claim_id":
                "27-A",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:0B74EC18",
        },

        {
            "claim_id":
                "27-B",

            "status":
                CELL117_AUTHOR_ONLY_STATUS,

            "reason":
                (
                    "Cell 115 produced a semantic false positive and "
                    "Cell 116 found no valid candidate window."
                ),
        },

        {
            "claim_id":
                "27-C",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:7AB68D23",
        },

        {
            "claim_id":
                "27-D",

            "status":
                CELL117_AUTHOR_ONLY_STATUS,

            "reason":
                (
                    "No Website candidate for reinvesting the resulting "
                    "profit was recovered from the full 529-record scope."
                ),
        },

        {
            "claim_id":
                "27-E",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:A8746170",
        },
    ],

    28: [
        {
            "claim_id":
                "28-A",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:2F8F15FB",
        },

        {
            "claim_id":
                "28-B",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:2F8F15FB",
        },

        {
            "claim_id":
                "28-C",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:2F8F15FB",
        },

        {
            "claim_id":
                "28-D",

            "status":
                CELL117_WEBSITE_STATUS,

            "evidence_source":
                "CELL115",

            "document_id":
                "website:E3C871BC",
        },
    ],
}


# ============================================================
# 9) Decision-shape validation
# ============================================================

cell117_all_decisions = [
    decision

    for pilot_number
    in CELL117_CASE_NUMBERS

    for decision
    in CELL117_DECISIONS[
        pilot_number
    ]
]


cell117_check(
    "ATOMIC_DECISION_COUNT_NOT_22",
    len(
        cell117_all_decisions
    )
    ==
    CELL117_EXPECTED_ATOMIC_CLAIMS,
    len(
        cell117_all_decisions
    ),
)


cell117_decision_claim_ids = [
    decision[
        "claim_id"
    ]
    for decision
    in cell117_all_decisions
]


cell117_check(
    "DUPLICATE_CLAIM_DECISION",
    len(
        cell117_decision_claim_ids
    )
    ==
    len(
        set(
            cell117_decision_claim_ids
        )
    ),
)


cell117_check(
    "CLAIM_SET_MISMATCH",
    set(
        cell117_decision_claim_ids
    )
    ==
    set(
        cell117_claim_description.keys()
    ),
    {
        "decision_claims":
            sorted(
                cell117_decision_claim_ids
            ),

        "known_claims":
            sorted(
                cell117_claim_description.keys()
            ),
    },
)


cell117_website_backed_count = sum(
    1
    for decision
    in cell117_all_decisions
    if decision[
        "status"
    ]
    ==
    CELL117_WEBSITE_STATUS
)

cell117_author_only_count = sum(
    1
    for decision
    in cell117_all_decisions
    if decision[
        "status"
    ]
    ==
    CELL117_AUTHOR_ONLY_STATUS
)


cell117_check(
    "WEBSITE_BACKED_COUNT_NOT_17",
    cell117_website_backed_count
    ==
    CELL117_EXPECTED_WEBSITE_BACKED,
    cell117_website_backed_count,
)


cell117_check(
    "AUTHOR_FEEDBACK_ONLY_COUNT_NOT_5",
    cell117_author_only_count
    ==
    CELL117_EXPECTED_AUTHOR_FEEDBACK_ONLY,
    cell117_author_only_count,
)


# ============================================================
# 10) Helpers to resolve approved evidence
# ============================================================

def cell117_find_cell115_claim(
    pilot_number,
    claim_id,
):

    for row in (
        cell115_results[
            pilot_number
        ]
    ):

        if row[
            "claim_id"
        ] == claim_id:

            return row

    return None


def cell117_find_cell115_evidence(
    pilot_number,
    claim_id,
    document_id,
):

    claim_row = (
        cell117_find_cell115_claim(
            pilot_number,
            claim_id,
        )
    )

    if claim_row is None:
        return None

    for match in (
        claim_row.get(
            "matches",
            [],
        )
        or
        []
    ):

        if match.get(
            "document_id"
        ) == document_id:

            return copy.deepcopy(
                match
            )

    return None


def cell117_find_cell116_evidence(
    claim_id,
    document_id,
):

    for match in (
        cell116_results.get(
            claim_id,
            [],
        )
        or
        []
    ):

        if match.get(
            "document_id"
        ) == document_id:

            return copy.deepcopy(
                match
            )

    return None


# ============================================================
# 11) Website document lookup
# ============================================================

cell117_doc_lookup = {
    doc[
        "document_id"
    ]:
        doc

    for doc
    in cell113_nonempty_docs
}


# ============================================================
# 12) Build final Source Packet
# ============================================================

cell117_packet_pilots = []


for pilot_number in (
    CELL117_CASE_NUMBERS
):

    case = (
        cell112_cases[
            pilot_number
        ]
    )

    pilot_claims = []

    accepted_document_ids = []

    author_feedback_only_claims = []


    for decision in (
        CELL117_DECISIONS[
            pilot_number
        ]
    ):

        claim_id = (
            decision[
                "claim_id"
            ]
        )

        claim_record = {
            "claim_id":
                claim_id,

            "description":
                cell117_claim_description[
                    claim_id
                ],

            "human_source_review_status":
                decision[
                    "status"
                ],
        }


        if (
            decision[
                "status"
            ]
            ==
            CELL117_WEBSITE_STATUS
        ):

            document_id = (
                decision[
                    "document_id"
                ]
            )

            evidence_source = (
                decision[
                    "evidence_source"
                ]
            )


            if evidence_source == "CELL115":

                evidence = (
                    cell117_find_cell115_evidence(
                        pilot_number,
                        claim_id,
                        document_id,
                    )
                )

            elif evidence_source == "CELL116":

                evidence = (
                    cell117_find_cell116_evidence(
                        claim_id,
                        document_id,
                    )
                )

            else:

                evidence = None


            cell117_check(
                (
                    "APPROVED_EVIDENCE_NOT_FOUND:"
                    +
                    claim_id
                ),
                evidence
                is not None,
                {
                    "pilot":
                        pilot_number,

                    "document_id":
                        document_id,

                    "evidence_source":
                        evidence_source,
                },
            )


            doc = (
                cell117_doc_lookup.get(
                    document_id
                )
            )


            cell117_check(
                (
                    "APPROVED_DOCUMENT_NOT_IN_WEBSITE_SCOPE:"
                    +
                    claim_id
                ),
                doc
                is not None,
                document_id,
            )


            if (
                evidence is not None
                and
                doc is not None
            ):

                claim_record[
                    "content_source"
                ] = {
                    "document_id":
                        document_id,

                    "title":
                        doc.get(
                            "title"
                        ),

                    "url":
                        doc.get(
                            "url"
                        ),

                    "evidence_source_cell":
                        evidence_source,

                    "evidence_window":
                        evidence.get(
                            "evidence_window"
                        ),

                    "matched_patterns":
                        (
                            evidence.get(
                                "matched_patterns"
                            )
                            or
                            evidence.get(
                                "matched_groups"
                            )
                            or
                            []
                        ),
                }


                cell117_check(
                    (
                        "EMPTY_APPROVED_EVIDENCE_WINDOW:"
                        +
                        claim_id
                    ),
                    bool(
                        str(
                            claim_record[
                                "content_source"
                            ][
                                "evidence_window"
                            ]
                            or
                            ""
                        ).strip()
                    ),
                )


                accepted_document_ids.append(
                    document_id
                )


        elif (
            decision[
                "status"
            ]
            ==
            CELL117_AUTHOR_ONLY_STATUS
        ):

            claim_record[
                "authority"
            ] = {
                "type":
                    "PRIMARY_DIRECT_AUTHOR_FEEDBACK",

                "primary_author":
                    "Dr. Komeil Roudi",

                "secondary_to_primary_promotion":
                    "FORBIDDEN",

                "website_source_status":
                    "NOT_ACCEPTED_AFTER_FULL_529_WEBSITE_HUMAN_REVIEW",

                "reason":
                    decision[
                        "reason"
                    ],
            }


            author_feedback_only_claims.append(
                claim_id
            )


        else:

            cell117_check(
                (
                    "UNKNOWN_DECISION_STATUS:"
                    +
                    claim_id
                ),
                False,
                decision[
                    "status"
                ],
            )


        pilot_claims.append(
            claim_record
        )


    unique_accepted_document_ids = sorted(
        set(
            accepted_document_ids
        )
    )


    cell117_packet_pilots.append({
        "reference_pilot_number":
            pilot_number,

        "pilot_case_id":
            case.get(
                "pilot_case_id"
            ),

        "original_phase1_demo_provenance":
            CELL117_ORIGINAL_DEMO_PROVENANCE_STATUS,

        "content_source_recovery":
            "HUMAN_REVIEWED",

        "website_scope_reviewed":
            CELL117_EXPECTED_WEBSITE_SCOPE,

        "accepted_website_document_ids":
            unique_accepted_document_ids,

        "primary_direct_author_feedback_only_claim_ids":
            sorted(
                author_feedback_only_claims
            ),

        "claims":
            pilot_claims,
    })


cell117_source_packet = {
    "schema_version":
        "source_packet_pilots23_28_v1",

    "purpose":
        "HUMAN_REVIEWED_CONTENT_SOURCE_RECOVERY_FOR_AUTHOR_REVIEW",

    "reference_pilot_numbers":
        CELL117_CASE_NUMBERS,

    "source_of_truth":
        str(
            cell117_corpus_path.relative_to(
                cell117_repo_root
            )
        ),

    "accepted_corpus_sha256":
        CELL117_EXPECTED_CORPUS_SHA256,

    "website_scope_reviewed":
        CELL117_EXPECTED_WEBSITE_SCOPE,

    "author_filter_applied":
        False,

    "authority_filter_applied":
        False,

    "original_phase1_demo_provenance":
        CELL117_ORIGINAL_DEMO_PROVENANCE_STATUS,

    "total_atomic_claims":
        CELL117_EXPECTED_ATOMIC_CLAIMS,

    "website_source_backed_claims":
        CELL117_EXPECTED_WEBSITE_BACKED,

    "primary_direct_author_feedback_only_claims":
        CELL117_EXPECTED_AUTHOR_FEEDBACK_ONLY,

    "author_facing_source_urls":
        False,

    "pilots":
        cell117_packet_pilots,
}


# ============================================================
# 13) Validate final packet counts
# ============================================================

cell117_packet_claims = [
    claim

    for pilot
    in cell117_source_packet[
        "pilots"
    ]

    for claim
    in pilot[
        "claims"
    ]
]


cell117_packet_website_count = sum(
    1

    for claim
    in cell117_packet_claims

    if claim[
        "human_source_review_status"
    ]
    ==
    CELL117_WEBSITE_STATUS
)


cell117_packet_author_only_count = sum(
    1

    for claim
    in cell117_packet_claims

    if claim[
        "human_source_review_status"
    ]
    ==
    CELL117_AUTHOR_ONLY_STATUS
)


cell117_check(
    "PACKET_PILOT_COUNT_NOT_6",
    len(
        cell117_source_packet[
            "pilots"
        ]
    )
    ==
    6,
)


cell117_check(
    "PACKET_CLAIM_COUNT_NOT_22",
    len(
        cell117_packet_claims
    )
    ==
    CELL117_EXPECTED_ATOMIC_CLAIMS,
    len(
        cell117_packet_claims
    ),
)


cell117_check(
    "PACKET_WEBSITE_COUNT_NOT_17",
    cell117_packet_website_count
    ==
    CELL117_EXPECTED_WEBSITE_BACKED,
    cell117_packet_website_count,
)


cell117_check(
    "PACKET_AUTHOR_ONLY_COUNT_NOT_5",
    cell117_packet_author_only_count
    ==
    CELL117_EXPECTED_AUTHOR_FEEDBACK_ONLY,
    cell117_packet_author_only_count,
)


# ============================================================
# 14) Build compact Registry review payload
# ============================================================

cell117_registry_pilot_reviews = []


for pilot in (
    cell117_source_packet[
        "pilots"
    ]
):

    registry_claims = []


    for claim in pilot[
        "claims"
    ]:

        registry_claim = {
            "claim_id":
                claim[
                    "claim_id"
                ],

            "status":
                claim[
                    "human_source_review_status"
                ],
        }


        if (
            claim[
                "human_source_review_status"
            ]
            ==
            CELL117_WEBSITE_STATUS
        ):

            registry_claim[
                "document_id"
            ] = (
                claim[
                    "content_source"
                ][
                    "document_id"
                ]
            )

        else:

            registry_claim[
                "authority"
            ] = (
                "PRIMARY_DIRECT_AUTHOR_FEEDBACK"
            )


        registry_claims.append(
            registry_claim
        )


    cell117_registry_pilot_reviews.append({
        "reference_pilot_number":
            pilot[
                "reference_pilot_number"
            ],

        "pilot_case_id":
            pilot[
                "pilot_case_id"
            ],

        "original_phase1_demo_provenance":
            CELL117_ORIGINAL_DEMO_PROVENANCE_STATUS,

        "claims":
            registry_claims,

        "accepted_website_document_ids":
            pilot[
                "accepted_website_document_ids"
            ],

        "primary_direct_author_feedback_only_claim_ids":
            pilot[
                "primary_direct_author_feedback_only_claim_ids"
            ],
    })


cell117_registry_review_payload = {
    "review_status":
        "HUMAN_SOURCE_REVIEW_APPROVED",

    "reference_pilot_numbers":
        CELL117_CASE_NUMBERS,

    "website_scope_reviewed":
        CELL117_EXPECTED_WEBSITE_SCOPE,

    "original_phase1_demo_provenance":
        CELL117_ORIGINAL_DEMO_PROVENANCE_STATUS,

    "content_source_recovery_status":
        "COMPLETE_FOR_AUTHOR_REVIEW",

    "total_atomic_claims":
        CELL117_EXPECTED_ATOMIC_CLAIMS,

    "website_source_backed_claims":
        CELL117_EXPECTED_WEBSITE_BACKED,

    "primary_direct_author_feedback_only_claims":
        CELL117_EXPECTED_AUTHOR_FEEDBACK_ONLY,

    "secondary_to_primary_promotion":
        "FORBIDDEN",

    "source_packet":
        str(
            CELL117_SOURCE_PACKET_REL
        ),

    "pilots":
        cell117_registry_pilot_reviews,
}


# ============================================================
# 15) Existing Registry state / idempotence
# ============================================================

if (
    cell117_registry_sha_before
    !=
    CELL117_EXPECTED_BASELINE_REGISTRY_SHA256
):

    cell117_check(
        "NON_BASELINE_REGISTRY_WITHOUT_MATCHING_CELL117_REVIEW",
        cell117_existing_review
        ==
        cell117_registry_review_payload,
        {
            "registry_sha":
                cell117_registry_sha_before,

            "existing_review_present":
                cell117_existing_review
                is not None,
        },
    )


if (
    cell117_existing_review
    is not None
):

    cell117_check(
        "EXISTING_CELL117_REVIEW_DIFFERS",
        cell117_existing_review
        ==
        cell117_registry_review_payload,
    )


if cell117_failures:

    print(
        "=== CELL 117 PRE-WRITE VALIDATION FAILURES ==="
    )

    print(
        cell117_failures
    )

    raise AssertionError(
        "Cell 117 stopped before write."
    )


# ============================================================
# 16) JSON serialization
# ============================================================

def cell117_json_text(
    obj
):

    return (
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        +
        "\n"
    )


cell117_source_packet_text = (
    cell117_json_text(
        cell117_source_packet
    )
)


# ============================================================
# 17) Atomic write helper
# ============================================================

def cell117_atomic_write_text(
    path,
    text,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp_path = path.with_name(
        path.name
        +
        ".tmp_cell117"
    )

    tmp_path.write_text(
        text,
        encoding="utf-8",
    )

    os.replace(
        tmp_path,
        path,
    )


# ============================================================
# 18) Write Source Packet if needed
# ============================================================

cell117_source_packet_write_required = True

cell117_source_packet_backup = None


if cell117_source_packet_path.exists():

    existing_text = (
        cell117_source_packet_path.read_text(
            encoding="utf-8"
        )
    )

    cell117_source_packet_write_required = (
        existing_text
        !=
        cell117_source_packet_text
    )


if (
    cell117_source_packet_write_required
    and
    cell117_source_packet_path.exists()
):

    timestamp = (
        datetime.now(
            timezone.utc
        )
        .strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )

    cell117_source_packet_backup = (
        cell117_source_packet_path.with_name(
            (
                cell117_source_packet_path.stem
                +
                ".before_cell117_"
                +
                timestamp
                +
                cell117_source_packet_path.suffix
            )
        )
    )

    cell117_source_packet_backup.write_bytes(
        cell117_source_packet_path.read_bytes()
    )


if cell117_source_packet_write_required:

    cell117_atomic_write_text(
        cell117_source_packet_path,
        cell117_source_packet_text,
    )


# ============================================================
# 19) Prepare Registry mutation
# ============================================================

cell117_registry_write_required = (
    cell117_existing_review
    !=
    cell117_registry_review_payload
)

cell117_registry_backup = None


if cell117_registry_write_required:

    cell117_registry_after_obj = copy.deepcopy(
        cell117_registry_before
    )

    cell117_registry_after_obj.setdefault(
        "second_human_review",
        {},
    )

    cell117_registry_after_obj[
        "second_human_review"
    ][
        CELL117_REGISTRY_REVIEW_KEY
    ] = copy.deepcopy(
        cell117_registry_review_payload
    )


    # --------------------------------------------------------
    # Allowed-diff validation:
    # The ENTIRE Registry must be identical except for this
    # single dedicated review key.
    # --------------------------------------------------------

    before_without_target = copy.deepcopy(
        cell117_registry_before
    )

    after_without_target = copy.deepcopy(
        cell117_registry_after_obj
    )


    before_second = (
        before_without_target.setdefault(
            "second_human_review",
            {},
        )
    )

    after_second = (
        after_without_target.setdefault(
            "second_human_review",
            {},
        )
    )


    before_second.pop(
        CELL117_REGISTRY_REVIEW_KEY,
        None,
    )

    after_second.pop(
        CELL117_REGISTRY_REVIEW_KEY,
        None,
    )


    cell117_check(
        "REGISTRY_MUTATION_ESCAPED_ALLOWED_SOURCE_REVIEW_KEY",
        before_without_target
        ==
        after_without_target,
    )


    cell117_check(
        "AUTHOR_REVIEW_QUEUE_MUTATED_IN_MEMORY",
        (
            cell117_registry_before.get(
                "author_review"
            )
            ==
            cell117_registry_after_obj.get(
                "author_review"
            )
        ),
    )


    if cell117_failures:

        print(
            "=== CELL 117 IN-MEMORY MUTATION FAILURE ==="
        )

        print(
            cell117_failures
        )

        raise AssertionError(
            "Cell 117 failed before Registry write."
        )


    timestamp = (
        datetime.now(
            timezone.utc
        )
        .strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )

    cell117_registry_backup = (
        cell117_registry_path.with_name(
            (
                cell117_registry_path.stem
                +
                ".before_cell117_source_review_"
                +
                timestamp
                +
                cell117_registry_path.suffix
            )
        )
    )


    cell117_registry_backup.write_bytes(
        cell117_registry_path.read_bytes()
    )


    cell117_atomic_write_text(
        cell117_registry_path,
        cell117_json_text(
            cell117_registry_after_obj
        ),
    )


# ============================================================
# 20) Reload written artifacts
# ============================================================

cell117_registry_reloaded = json.loads(
    cell117_registry_path.read_text(
        encoding="utf-8"
    )
)

cell117_source_packet_reloaded = json.loads(
    cell117_source_packet_path.read_text(
        encoding="utf-8"
    )
)


cell117_reloaded_review = (
    (
        cell117_registry_reloaded.get(
            "second_human_review",
            {},
        )
        or
        {}
    ).get(
        CELL117_REGISTRY_REVIEW_KEY
    )
)


# ============================================================
# 21) Post-write semantic checks
# ============================================================

cell117_check(
    "RELOADED_REGISTRY_REVIEW_MISMATCH",
    cell117_reloaded_review
    ==
    cell117_registry_review_payload,
)


cell117_check(
    "RELOADED_SOURCE_PACKET_MISMATCH",
    cell117_source_packet_reloaded
    ==
    cell117_source_packet,
)


cell117_reloaded_packet_claims = [
    claim

    for pilot
    in cell117_source_packet_reloaded[
        "pilots"
    ]

    for claim
    in pilot[
        "claims"
    ]
]


cell117_check(
    "RELOADED_PACKET_CLAIM_COUNT_NOT_22",
    len(
        cell117_reloaded_packet_claims
    )
    ==
    CELL117_EXPECTED_ATOMIC_CLAIMS,
)


cell117_check(
    "RELOADED_PACKET_HAS_WRONG_PROVENANCE_STATUS",
    all(
        pilot[
            "original_phase1_demo_provenance"
        ]
        ==
        CELL117_ORIGINAL_DEMO_PROVENANCE_STATUS

        for pilot
        in cell117_source_packet_reloaded[
            "pilots"
        ]
    ),
)


# ============================================================
# 22) Immutable artifact checks
# ============================================================

cell117_corpus_sha_after = (
    cell112_sha256_file(
        cell117_corpus_path
    )
)

cell117_registry_sha_after = (
    cell112_sha256_file(
        cell117_registry_path
    )
)

cell117_author_md_sha_after = (
    cell112_sha256_file(
        cell117_author_md_path
    )
)

cell117_source_packet_sha = (
    cell112_sha256_file(
        cell117_source_packet_path
    )
)


cell117_check(
    "CORPUS_MUTATED",
    cell117_corpus_sha_after
    ==
    cell117_corpus_sha_before,
)


cell117_check(
    "AUTHOR_REVIEW_MD_MUTATED",
    cell117_author_md_sha_after
    ==
    cell117_author_md_sha_before,
)


# ============================================================
# 23) Explicit protection of Author Review / decisions
# ============================================================

cell117_check(
    "AUTHOR_REVIEW_QUEUE_CHANGED",
    (
        cell117_registry_before.get(
            "author_review"
        )
        ==
        cell117_registry_reloaded.get(
            "author_review"
        )
    ),
)


cell117_author_decision_values = []

for key in [
    "author_decision",
    "author_decisions",
    "author_review_decision",
]:

    def _collect_values(obj):

        values = []

        if isinstance(obj, dict):

            for k, v in obj.items():

                if str(k).lower() == key:
                    values.append(v)

                values.extend(
                    _collect_values(
                        v
                    )
                )

        elif isinstance(obj, list):

            for item in obj:

                values.extend(
                    _collect_values(
                        item
                    )
                )

        return values


    cell117_author_decision_values.extend(
        _collect_values(
            cell117_registry_reloaded
        )
    )


# We do not require the Registry to have zero historical
# author-decision fields globally.
# We require the NEW Cell117 payload itself to contain none.

cell117_review_payload_text = (
    json.dumps(
        cell117_registry_review_payload,
        ensure_ascii=False,
    )
    .lower()
)


cell117_check(
    "CELL117_REVIEW_CONTAINS_AUTHOR_DECISION_FIELD",
    "author_decision"
    not in
    cell117_review_payload_text,
)


# ============================================================
# 24) Final counts
# ============================================================

cell117_unique_website_documents = sorted(
    {
        claim[
            "content_source"
        ][
            "document_id"
        ]

        for claim
        in cell117_packet_claims

        if claim[
            "human_source_review_status"
        ]
        ==
        CELL117_WEBSITE_STATUS
    }
)


cell117_author_only_claim_ids = sorted(
    [
        claim[
            "claim_id"
        ]

        for claim
        in cell117_packet_claims

        if claim[
            "human_source_review_status"
        ]
        ==
        CELL117_AUTHOR_ONLY_STATUS
    ]
)


# ============================================================
# 25) Report
# ============================================================

print(
    "=== CODE CELL 117 — HUMAN SOURCE REVIEW REGISTRATION ==="
)

print(
    "\n=== INPUT STATE ==="
)

print(
    "Corpus SHA256:",
    cell117_corpus_sha_before,
)

print(
    "Registry SHA256 before:",
    cell117_registry_sha_before,
)

print(
    "Author Review MD SHA256:",
    cell117_author_md_sha_before,
)

print(
    "Website scope reviewed:",
    CELL117_EXPECTED_WEBSITE_SCOPE,
)


print(
    "\n=== HUMAN SOURCE REVIEW ==="
)

print(
    "Pilots:",
    CELL117_CASE_NUMBERS,
)

print(
    "Atomic claims:",
    len(
        cell117_packet_claims
    ),
)

print(
    "Website Source Backed:",
    cell117_packet_website_count,
)

print(
    "Primary Direct Author Feedback Only:",
    cell117_packet_author_only_count,
)

print(
    "Original Phase-1 Demo Provenance:",
    CELL117_ORIGINAL_DEMO_PROVENANCE_STATUS,
)


print(
    "\n=== AUTHOR-FEEDBACK-ONLY CLAIMS ==="
)

for claim_id in (
    cell117_author_only_claim_ids
):

    print(
        "-",
        claim_id,
        "|",
        cell117_claim_description[
            claim_id
        ],
    )


print(
    "\n=== ACCEPTED WEBSITE DOCUMENTS ==="
)

for document_id in (
    cell117_unique_website_documents
):

    doc = (
        cell117_doc_lookup[
            document_id
        ]
    )

    print(
        "-",
        document_id,
        "|",
        doc[
            "title"
        ],
    )


print(
    "\n=== SOURCE PACKET ==="
)

print(
    "Path:",
    cell117_source_packet_path,
)

print(
    "Write required:",
    cell117_source_packet_write_required,
)

print(
    "Backup:",
    (
        cell117_source_packet_backup
        if cell117_source_packet_backup
        is not None
        else
        "NOT_NEEDED"
    ),
)

print(
    "SHA256:",
    cell117_source_packet_sha,
)


print(
    "\n=== REGISTRY ==="
)

print(
    "Review key:",
    CELL117_REGISTRY_REVIEW_KEY,
)

print(
    "Write required:",
    cell117_registry_write_required,
)

print(
    "Backup:",
    (
        cell117_registry_backup
        if cell117_registry_backup
        is not None
        else
        "NOT_NEEDED"
    ),
)

print(
    "Registry SHA256 after:",
    cell117_registry_sha_after,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Corpus mutated:",
    cell117_corpus_sha_after
    !=
    cell117_corpus_sha_before,
)

print(
    "Author Review Markdown mutated:",
    cell117_author_md_sha_after
    !=
    cell117_author_md_sha_before,
)

print(
    "Author Review queue mutated:",
    (
        cell117_registry_before.get(
            "author_review"
        )
        !=
        cell117_registry_reloaded.get(
            "author_review"
        )
    ),
)

print(
    "Raw source Corpus write:",
    False,
)

print(
    "Author decisions recorded by Cell117:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell117_failures
)


cell117_gate = (
    len(
        cell117_failures
    )
    ==
    0
)


print(
    "\nCODE CELL 117 GATE:",
    (
        "PASS"
        if cell117_gate
        else
        "FAIL"
    ),
)


assert cell117_gate, (
    "Human Source Review registration failed. "
    "Repair this SAME Code Cell 117 before modifying "
    "author_review_28_pilots.md."
)

=== CODE CELL 117 — HUMAN SOURCE REVIEW REGISTRATION ===

=== INPUT STATE ===
Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Registry SHA256 before: 10933065b1d4598dba1534964b8a6837f723cf79082d1af2bc8a4964e2a6766e
Author Review MD SHA256: f9b2e04a50e61060b150c958d444f1308379c76815edb299c25b96257b031f09
Website scope reviewed: 529

=== HUMAN SOURCE REVIEW ===
Pilots: [23, 24, 25, 26, 27, 28]
Atomic claims: 22
Website Source Backed: 17
Primary Direct Author Feedback Only: 5
Original Phase-1 Demo Provenance: NOT_PROVEN_FROM_AVAILABLE_ARTIFACTS

=== AUTHOR-FEEDBACK-ONLY CLAIMS ===
- 26-A | خرید با تنوع یا حجم زیاد می‌تواند از الگوهای نامناسب خرید باشد.
- 26-F | خرید پس از یک دوره سخت برای پرداختن یا پاداش دادن به خود.
- 26-G | خرید یا خرج‌کردن ناشی از فداکاری برای دیگران.
- 27-B | پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.
- 27-D | سرمایه‌گذاری دوباره یا مجدد سود حاصل.

=== ACCEPTED WEBSITE DOCUMENTS ===
- website:0B74EC18 | فروش و هنر مذاکره؛ تجربه‌ای ار

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اصلاح نهایی متن منبع در بسته Author Review برای پایلوت‌های ۲۳ تا ۲۸</h3>

<h4>وضعیت مرحله قبل</h4>

<p>
Human Source Review برای پایلوت‌های ۲۳ تا ۲۸ در Code Cell 117 تکمیل و ثبت شد.
</p>

<p>
نتیجۀ پذیرفته‌شده شامل ۲۲ Claim اتمیک است:
</p>

<ul>
<li>۱۷ Claim دارای Website Source بازیابی‌شده و تأییدشده هستند.</li>
<li>۵ Claim در کل ۵۲۹ Website شاهد مستقل کافی ندارند و Authority آن‌ها بازخورد مستقیم دکتر کمیل رودی است.</li>
</ul>

<p>
همچنین Source Packet نهایی در فایل زیر ایجاد شده است:
</p>

<p>
<code>data/phase2/human_review/source_packet_pilots23_28_v1.json</code>
</p>

<h4>هدف</h4>

<p>
هدف این مرحله اصلاح فایل Author Review است تا در موارد ۲۳ تا ۲۸،
بخش «متن منبع» واقعاً شامل بخش‌هایی از متن اصلی Sourceهای بازیابی‌شده باشد.
</p>

<p>
در نسخۀ قبلی، تاریخچۀ دموی فاز یک و Source Grounding اولیه وجود داشت،
اما نمایش Source برای بازبینی مؤلف هنوز در بعضی بخش‌ها به عباراتی مانند
«همان سابقه»، «همان منبع» یا ارجاع کلی به History متکی بود.
</p>

<p>
پس از تکمیل Source Recovery، دیگر نباید چنین Placeholderهایی جای متن واقعی منبع را بگیرند.
</p>

<h4>ورودی‌های Freeze‌شده</h4>

<ul>
<li>
Accepted Corpus:
<code>9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40</code>
</li>

<li>
Registry پس از Cell 117:
<code>a851f1a20641f8627f12121bef6497e0fcfb72e6c0bc4e47096f4021e6604b04</code>
</li>

<li>
Source Packet:
<code>a233d25207124b808eb3747054860bc3278670b4e12444160dd809693c8ea3a4</code>
</li>

<li>
نسخۀ فعلی Author Review پیش از اصلاح:
<code>f9b2e04a50e61060b150c958d444f1308379c76815edb299c25b96257b031f09</code>
</li>
</ul>

<h4>عملیات</h4>

<p>
فقط Cases 23 تا 28 اصلاح می‌شوند.
</p>

<p>
برای هر Case، Source Packet خوانده می‌شود و Sourceهای تأییدشده
بر اساس Human Source Review استخراج می‌شوند.
</p>

<p>
برای هر Website Source تأییدشده، موارد زیر در بخش «متن منبع» نمایش داده می‌شود:
</p>

<ul>
<li>عنوان Source.</li>
<li>بخش واقعی و تأییدشدۀ متن منبع.</li>
</ul>

<p>
URL و Document ID در نسخۀ Author-Facing نمایش داده نمی‌شوند.
این اطلاعات فقط در Source Packet داخلی پروژه باقی می‌مانند.
</p>

<h4>Claimهای بدون Website Source مستقل</h4>

<p>
برای پنج Claim زیر Source ساختگی ایجاد نمی‌شود:
</p>

<ul>
<li>26-A — خرید با تنوع یا حجم زیاد.</li>
<li>26-F — خرید پس از یک دوره سخت برای پرداختن یا پاداش دادن به خود.</li>
<li>26-G — خرید یا خرج‌کردن ناشی از فداکاری برای دیگران.</li>
<li>27-B — پیشنهاد پروژه یا کار بیشتر به کارفرمای فعلی.</li>
<li>27-D — سرمایه‌گذاری دوباره یا مجدد سود حاصل.</li>
</ul>

<p>
در Author Review صریحاً نوشته می‌شود که برای این اجزا،
در کل Website Corpus شاهد مستقل پذیرفته‌شده بازیابی نشده
و این بخش‌ها بر بازخورد مستقیم دکتر کمیل رودی متکی‌اند.
</p>

<h4>حفظ تاریخچۀ واقعی</h4>

<p>
سؤال تاریخی Demo، پاسخ تاریخی مدل، بازخورد مستقیم دکتر،
تصمیم‌های Source Mapping و روند اصلاح سؤال و پاسخ حذف یا بازنویسی نمی‌شوند.
</p>

<p>
فقط نمایش Source Evidence اصلاح می‌شود.
</p>

<h4>حفاظت از موارد ۱ تا ۲۲</h4>

<p>
Cases 1 تا 22 کاملاً Freeze هستند و این Cell حق تغییر حتی یک کاراکتر
در آن بخش‌ها را ندارد.
</p>

<h4>اصلاح ردیف «منبع» در سؤال و پاسخ نهایی</h4>

<p>
در Cases 23 تا 28، ردیف «منبع» در بخش سؤال و پاسخ نهایی نیز اصلاح می‌شود.
</p>

<p>
عبارت‌های مبهمی مانند «همان سابقه» یا «همان شواهد» با توضیح روشن جایگزین می‌شوند:
</p>

<ul>
<li>
برای Caseهای کاملاً Source-Backed:
«متن‌های منبع بازیابی‌شده و تأییدشده در بخش ۱ بالا.»
</li>

<li>
برای Caseهای ۲۶ و ۲۷:
علاوه بر Sourceهای Website، صریحاً ذکر می‌شود که اجزای بدون شاهد مستقل Website
بر بازخورد مستقیم دکتر کمیل رودی متکی‌اند.
</li>
</ul>

<h4>قاعدۀ Provenance</h4>

<p>
Sourceهای درج‌شده Content Sourceهای بازیابی‌شده و Human-Reviewed هستند.
</p>

<p>
این مرحله ادعا نمی‌کند که Artifactهای موجود ثابت کرده‌اند
دقیقاً همین Documentها در Retrieval اجرای تاریخی Demo استفاده شده‌اند.
وضعیت Historical Demo Provenance همچنان:
</p>

<p>
<code>NOT_PROVEN_FROM_AVAILABLE_ARTIFACTS</code>
</p>

<h4>خروجی مورد انتظار</h4>

<p>
فایل زیر در محل فعلی اصلاح می‌شود:
</p>

<p>
<code>notebooks/phase2/author_review_28_pilots.md</code>
</p>

<p>
پیش از هر Write واقعی، Backup از نسخۀ فعلی ساخته می‌شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Cases 1 تا 22 دقیقاً بدون تغییر باقی بمانند.</li>
<li>Cases 23 تا 28 هر شش مورد دارای Source Block جدید باشند.</li>
<li>تمام Website Sourceهای پذیرفته‌شده در Source Packet در Case مربوط نمایش داده شوند.</li>
<li>Evidence Window واقعی هر Source در فایل Author Review درج شود.</li>
<li>هیچ URL در Source Block Author-Facing نمایش داده نشود.</li>
<li>هیچ Document ID داخلی در Source Block نمایش داده نشود.</li>
<li>هر پنج Claim فاقد Website Source صریحاً مشخص شوند.</li>
<li>History واقعی Demo و بازخورد دکتر حفظ شود.</li>
<li>بخش سؤال و پاسخ، تصمیم‌های Review و نظر مؤلف تغییر نکنند.</li>
<li>فقط ردیف «منبع» در بخش سؤال و پاسخ نهایی مجاز به اصلاح است.</li>
<li>Registry تغییر نکند.</li>
<li>Source Packet تغییر نکند.</li>
<li>Corpus تغییر نکند.</li>
<li>هیچ Author Decision ثبت نشود.</li>
<li>Commit / Push انجام نشود.</li>
<li>خروجی نهایی برابر <code>CODE CELL 118 GATE: PASS</code> باشد.</li>
</ul>

<h4>معیار FAIL</h4>

<ul>
<li>حتی یک Case از 1 تا 22 تغییر کند.</li>
<li>یکی از Cases 23 تا 28 Source Block نداشته باشد.</li>
<li>Source تأییدنشده وارد Author Review شود.</li>
<li>Evidence Window ساختگی یا خالی باشد.</li>
<li>URL یا Document ID داخلی در Author-Facing Source Block نمایش داده شود.</li>
<li>یکی از Claimهای Author-Feedback-Only به Website نسبت داده شود.</li>
<li>History واقعی Demo حذف یا بازنویسی شود.</li>
<li>Registry، Source Packet یا Corpus تغییر کند.</li>
</ul>

<h4>خارج از Scope</h4>

<ul>
<li>تغییر سؤال نهایی ممنوع است.</li>
<li>تغییر پاسخ نهایی ممنوع است.</li>
<li>تغییر تصمیم‌های Human Review ممنوع است.</li>
<li>ثبت نظر دکتر ممنوع است.</li>
<li>تغییر Cases 1 تا 22 ممنوع است.</li>
<li>Commit و Push ممنوع است.</li>
</ul>

<h4>تعریف Done</h4>

<p>
این مرحله فقط زمانی Done محسوب می‌شود که Cases 23 تا 28
در همان قالب RTL/HTML فعلی، متن واقعی Sourceهای بازیابی‌شده را نمایش دهند،
Claimهای بدون Source مستقل صریحاً مشخص باشند،
و تمام بخش‌های دیگر فایل بدون تغییر باقی مانده باشند.
</p>

</div>

In [13]:
# Code Cell 118
# ============================================================
# AUTHOR REVIEW 28 — RECOVERED SOURCE TEXT INSERTION
# CASES 23-28 ONLY
#
# ALLOWED:
# - Modify source section body of Cases 23-28.
# - Modify ONLY the "منبع" value inside the final Q/A table
#   of Cases 23-28.
#
# FORBIDDEN:
# - Any change to Cases 1-22.
# - Any change to final Question / Answer.
# - Any Registry mutation.
# - Any Source Packet mutation.
# - Any Corpus mutation.
# - Any Author Decision.
# - Commit / Push.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import OrderedDict
import hashlib
import html
import json
import os
import re


# ============================================================
# 1) Constants
# ============================================================

CELL118_EXPECTED_CORPUS_SHA256 = (
    "9d72c709f81b1218c5aac064052c7ed1"
    "da34869f671e6e32762864e0ad70ea40"
)

CELL118_EXPECTED_REGISTRY_SHA256 = (
    "a851f1a20641f8627f12121bef6497e0"
    "fcfb72e6c0bc4e47096f4021e6604b04"
)

CELL118_EXPECTED_SOURCE_PACKET_SHA256 = (
    "a233d25207124b808eb3747054860bc3"
    "278670b4e12444160dd809693c8ea3a4"
)

CELL118_EXPECTED_BASE_AUTHOR_MD_SHA256 = (
    "f9b2e04a50e61060b150c958d444f130"
    "8379c76815edb299c25b96257b031f09"
)

CELL118_CASE_NUMBERS = [
    23,
    24,
    25,
    26,
    27,
    28,
]

CELL118_EXPECTED_WEBSITE_BACKED = 17

CELL118_EXPECTED_AUTHOR_ONLY = 5

CELL118_EXPECTED_UNIQUE_RENDERED_SOURCE_BLOCKS = 12

CELL118_SOURCE_PACKET_REL = Path(
    "data/phase2/human_review/"
    "source_packet_pilots23_28_v1.json"
)

CELL118_AUTHOR_MD_REL = Path(
    "notebooks/phase2/"
    "author_review_28_pilots.md"
)

CELL118_START_MARKER = (
    "<!-- CELL118_RECOVERED_SOURCE_START -->"
)

CELL118_END_MARKER = (
    "<!-- CELL118_RECOVERED_SOURCE_END -->"
)

cell118_failures = []


# ============================================================
# 2) Validation helper
# ============================================================

def cell118_check(
    reason,
    condition,
    details=None,
):

    if not bool(condition):

        item = {
            "reason": reason,
        }

        if details is not None:
            item["details"] = details

        cell118_failures.append(
            item
        )

    return bool(condition)


# ============================================================
# 3) Required previous state
# ============================================================

required_names = [
    "cell117_gate",
    "cell117_repo_root",
    "cell117_corpus_path",
    "cell117_registry_path",
    "cell117_source_packet_path",
    "cell112_sha256_file",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]


cell118_check(
    "REQUIRED_CELL117_STATE_MISSING",
    len(
        missing_names
    )
    ==
    0,
    missing_names,
)


if cell118_failures:

    print(
        "=== CELL 118 PREFLIGHT FAILURE ==="
    )

    print(
        cell118_failures
    )

    raise AssertionError(
        "Code Cell 118 requires accepted Cell 117 state."
    )


cell118_check(
    "CELL117_GATE_NOT_PASS",
    cell117_gate is True,
)


if cell118_failures:
    raise AssertionError(
        cell118_failures
    )


# ============================================================
# 4) Resolve paths
# ============================================================

cell118_repo_root = Path(
    cell117_repo_root
).resolve()

cell118_corpus_path = Path(
    cell117_corpus_path
).resolve()

cell118_registry_path = Path(
    cell117_registry_path
).resolve()

cell118_source_packet_path = Path(
    cell117_source_packet_path
).resolve()

cell118_author_md_path = (
    cell118_repo_root
    /
    CELL118_AUTHOR_MD_REL
)


cell118_check(
    "AUTHOR_REVIEW_MD_NOT_FOUND",
    cell118_author_md_path.exists(),
    str(
        cell118_author_md_path
    ),
)


if cell118_failures:
    raise AssertionError(
        cell118_failures
    )


# ============================================================
# 5) Freeze input hashes
# ============================================================

cell118_corpus_sha_before = (
    cell112_sha256_file(
        cell118_corpus_path
    )
)

cell118_registry_sha_before = (
    cell112_sha256_file(
        cell118_registry_path
    )
)

cell118_source_packet_sha_before = (
    cell112_sha256_file(
        cell118_source_packet_path
    )
)

cell118_author_md_sha_before = (
    cell112_sha256_file(
        cell118_author_md_path
    )
)


cell118_check(
    "CORPUS_SHA_MISMATCH",
    cell118_corpus_sha_before
    ==
    CELL118_EXPECTED_CORPUS_SHA256,
    {
        "expected":
            CELL118_EXPECTED_CORPUS_SHA256,

        "actual":
            cell118_corpus_sha_before,
    },
)


cell118_check(
    "REGISTRY_SHA_MISMATCH",
    cell118_registry_sha_before
    ==
    CELL118_EXPECTED_REGISTRY_SHA256,
    {
        "expected":
            CELL118_EXPECTED_REGISTRY_SHA256,

        "actual":
            cell118_registry_sha_before,
    },
)


cell118_check(
    "SOURCE_PACKET_SHA_MISMATCH",
    cell118_source_packet_sha_before
    ==
    CELL118_EXPECTED_SOURCE_PACKET_SHA256,
    {
        "expected":
            CELL118_EXPECTED_SOURCE_PACKET_SHA256,

        "actual":
            cell118_source_packet_sha_before,
    },
)


if cell118_failures:

    print(
        "=== CELL 118 FROZEN INPUT FAILURE ==="
    )

    print(
        cell118_failures
    )

    raise AssertionError(
        "Cell 118 stopped Fail-Closed."
    )


# ============================================================
# 6) Load Source Packet + Author Review
# ============================================================

cell118_source_packet = json.loads(
    cell118_source_packet_path.read_text(
        encoding="utf-8"
    )
)

cell118_original_text = (
    cell118_author_md_path.read_text(
        encoding="utf-8"
    )
)


# ============================================================
# 7) Source Packet validation
# ============================================================

cell118_packet_pilots = (
    cell118_source_packet.get(
        "pilots",
        [],
    )
    or
    []
)

cell118_packet_by_number = {
    pilot[
        "reference_pilot_number"
    ]:
        pilot

    for pilot
    in cell118_packet_pilots
}


cell118_check(
    "SOURCE_PACKET_PILOT_SET_MISMATCH",
    sorted(
        cell118_packet_by_number.keys()
    )
    ==
    CELL118_CASE_NUMBERS,
    sorted(
        cell118_packet_by_number.keys()
    ),
)


cell118_packet_claims = [
    claim

    for pilot
    in cell118_packet_pilots

    for claim
    in (
        pilot.get(
            "claims",
            [],
        )
        or
        []
    )
]


cell118_website_claims = [
    claim

    for claim
    in cell118_packet_claims

    if claim.get(
        "human_source_review_status"
    )
    ==
    "WEBSITE_SOURCE_BACKED"
]


cell118_author_only_claims = [
    claim

    for claim
    in cell118_packet_claims

    if claim.get(
        "human_source_review_status"
    )
    ==
    "PRIMARY_DIRECT_AUTHOR_FEEDBACK_ONLY"
]


cell118_check(
    "WEBSITE_BACKED_COUNT_NOT_17",
    len(
        cell118_website_claims
    )
    ==
    CELL118_EXPECTED_WEBSITE_BACKED,
    len(
        cell118_website_claims
    ),
)


cell118_check(
    "AUTHOR_ONLY_COUNT_NOT_5",
    len(
        cell118_author_only_claims
    )
    ==
    CELL118_EXPECTED_AUTHOR_ONLY,
    len(
        cell118_author_only_claims
    ),
)


cell118_check(
    "WRONG_DEMO_PROVENANCE_STATUS",
    cell118_source_packet.get(
        "original_phase1_demo_provenance"
    )
    ==
    "NOT_PROVEN_FROM_AVAILABLE_ARTIFACTS",
)


if cell118_failures:
    raise AssertionError(
        cell118_failures
    )


# ============================================================
# 8) HTML / Persian helpers
# ============================================================

CELL118_PERSIAN_DIGITS = str.maketrans(
    "0123456789",
    "۰۱۲۳۴۵۶۷۸۹",
)


def cell118_fa_number(value):

    return str(
        value
    ).translate(
        CELL118_PERSIAN_DIGITS
    )


def cell118_html_text(value):

    value = str(
        value
        or
        ""
    )

    value = html.escape(
        value,
        quote=False,
    )

    value = value.replace(
        "\n",
        "<br>",
    )

    return value


# ============================================================
# 9) Case marker discovery
# ============================================================

CELL118_CASE_H2_RE = re.compile(
    r"""
    <h2
    [^>]*>
    \s*
    مورد
    \s+
    ([0-9۰-۹]+)
    \s+
    از
    \s+
    ۲۸
    \s*
    </h2>
    """,
    re.VERBOSE,
)


def cell118_number_to_int(value):

    value = str(
        value
    )

    translation = str.maketrans(
        "۰۱۲۳۴۵۶۷۸۹",
        "0123456789",
    )

    return int(
        value.translate(
            translation
        )
    )


def cell118_case_matches(text):

    matches = []

    for match in (
        CELL118_CASE_H2_RE.finditer(
            text
        )
    ):

        number = (
            cell118_number_to_int(
                match.group(
                    1
                )
            )
        )

        matches.append({
            "number":
                number,

            "start":
                match.start(),

            "end":
                match.end(),
        })

    return matches


cell118_original_case_matches = (
    cell118_case_matches(
        cell118_original_text
    )
)


cell118_check(
    "AUTHOR_MD_CASE_SEQUENCE_NOT_1_28",
    [
        row[
            "number"
        ]
        for row
        in cell118_original_case_matches
    ]
    ==
    list(
        range(
            1,
            29,
        )
    ),
    [
        row[
            "number"
        ]
        for row
        in cell118_original_case_matches
    ],
)


if cell118_failures:
    raise AssertionError(
        cell118_failures
    )


# ============================================================
# 10) Case block extraction
# ============================================================

def cell118_extract_case_blocks(text):

    matches = (
        cell118_case_matches(
            text
        )
    )

    result = {}

    for index, row in enumerate(
        matches
    ):

        start = row[
            "start"
        ]

        end = (
            matches[
                index + 1
            ][
                "start"
            ]
            if index + 1
            <
            len(
                matches
            )
            else
            len(
                text
            )
        )

        result[
            row[
                "number"
            ]
        ] = text[
            start:end
        ]

    return result


cell118_original_blocks = (
    cell118_extract_case_blocks(
        cell118_original_text
    )
)


# ============================================================
# 11) Section regexes
# ============================================================

CELL118_SOURCE_HEADING_RE = re.compile(
    r"""
    <h3
    [^>]*>
    \s*
    (?:
        ۱
        \.
        \s*
    )?
    متن
    \s+
    منبع
    \s*
    \(بخشی
    \s+
    از
    \s+
    متن
    \s+
    اصلی\)
    \s*
    </h3>
    """,
    re.VERBOSE,
)


CELL118_QA_HISTORY_HEADING_RE = re.compile(
    r"""
    <h3
    [^>]*>
    \s*
    (?:
        ۲
        \.
        \s*
    )?
    سؤال‌ها
    \s+
    و
    \s+
    پاسخ‌ها
    \s*
    [—\-]
    \s*
    روند
    \s+
    اصلاح
    \s+
    و
    \s+
    بازبینی
    \s*
    </h3>
    """,
    re.VERBOSE,
)


CELL118_FINAL_QA_HEADING_RE = re.compile(
    r"""
    <h3
    [^>]*>
    \s*
    (?:
        ۴
        \.
        \s*
    )?
    سؤال
    \s+
    و
    \s+
    پاسخ
    \s+
    نهایی
    \s*
    \(مورد
    \s+
    بازبینی
    \s+
    توسط
    \s+
    دکتر\)
    \s*
    </h3>
    """,
    re.VERBOSE,
)


CELL118_AUTHOR_HEADING_RE = re.compile(
    r"""
    <h3
    [^>]*>
    \s*
    (?:
        ۵
        \.
        \s*
    )?
    نظر
    \s+
    مؤلف
    \s*
    \(دکتر
    \s+
    کمیل
    \s+
    رودی\)
    \s*
    </h3>
    """,
    re.VERBOSE,
)


CELL118_OLD_CORPUS_EVIDENCE_RE = re.compile(
    r"""
    <h4
    [^>]*>
    \s*
    شواهد
    \s+
    Corpus
    .*?
    </h4>
    """,
    re.VERBOSE
    |
    re.DOTALL
    |
    re.IGNORECASE,
)


CELL118_OLD_CORPUS_EVIDENCE_MD_RE = re.compile(
    r"""
    ^\s*
    \#{4}
    \s+
    شواهد
    \s+
    Corpus
    .*$
    """,
    re.VERBOSE
    |
    re.MULTILINE
    |
    re.IGNORECASE,
)


CELL118_INSERTED_BLOCK_RE = re.compile(
    re.escape(
        CELL118_START_MARKER
    )
    +
    r".*?"
    +
    re.escape(
        CELL118_END_MARKER
    ),
    re.DOTALL,
)


# ============================================================
# 12) Build recovered-source block from Source Packet
# ============================================================

def cell118_build_source_block(
    pilot_packet,
):

    grouped_sources = OrderedDict()

    author_only = []


    for claim in (
        pilot_packet.get(
            "claims",
            [],
        )
        or
        []
    ):

        status = claim.get(
            "human_source_review_status"
        )


        if (
            status
            ==
            "WEBSITE_SOURCE_BACKED"
        ):

            source = (
                claim.get(
                    "content_source",
                    {},
                )
                or
                {}
            )

            document_id = source.get(
                "document_id"
            )

            title = source.get(
                "title"
            )

            evidence_window = source.get(
                "evidence_window"
            )


            cell118_check(
                (
                    "EMPTY_SOURCE_DOCUMENT_ID:"
                    +
                    str(
                        claim.get(
                            "claim_id"
                        )
                    )
                ),
                bool(
                    str(
                        document_id
                        or
                        ""
                    ).strip()
                ),
            )


            cell118_check(
                (
                    "EMPTY_SOURCE_TITLE:"
                    +
                    str(
                        claim.get(
                            "claim_id"
                        )
                    )
                ),
                bool(
                    str(
                        title
                        or
                        ""
                    ).strip()
                ),
            )


            cell118_check(
                (
                    "EMPTY_EVIDENCE_WINDOW:"
                    +
                    str(
                        claim.get(
                            "claim_id"
                        )
                    )
                ),
                bool(
                    str(
                        evidence_window
                        or
                        ""
                    ).strip()
                ),
            )


            if document_id not in grouped_sources:

                grouped_sources[
                    document_id
                ] = {
                    "title":
                        title,

                    "evidence_windows":
                        [],
                }


            if (
                evidence_window
                not in
                grouped_sources[
                    document_id
                ][
                    "evidence_windows"
                ]
            ):

                grouped_sources[
                    document_id
                ][
                    "evidence_windows"
                ].append(
                    evidence_window
                )


        elif (
            status
            ==
            "PRIMARY_DIRECT_AUTHOR_FEEDBACK_ONLY"
        ):

            author_only.append({
                "claim_id":
                    claim.get(
                        "claim_id"
                    ),

                "description":
                    claim.get(
                        "description"
                    ),
            })


        else:

            cell118_check(
                (
                    "UNKNOWN_SOURCE_REVIEW_STATUS:"
                    +
                    str(
                        claim.get(
                            "claim_id"
                        )
                    )
                ),
                False,
                status,
            )


    lines = [
        CELL118_START_MARKER,
        "",
        (
            '<h4 dir="rtl" align="right">'
            'متن‌های منبع بازیابی‌شده و تأییدشده'
            '</h4>'
        ),
        "",
        (
            '<p dir="rtl" align="right">'
            'متن‌های زیر از منابع موجود پروژه بازیابی شده‌اند '
            'و در Human Source Review تأیید شده‌اند. '
            'از Artifactهای باقی‌مانده فاز یک نمی‌توان با قطعیت '
            'ثابت کرد که دقیقاً کدام Document در Retrieval اجرای '
            'تاریخی Demo استفاده شده است؛ بنابراین این بخش، '
            'Content Source بازیابی‌شده را نمایش می‌دهد.'
            '</p>'
        ),
        "",
    ]


    for source_index, (
        document_id,
        source_data,
    ) in enumerate(
        grouped_sources.items(),
        start=1,
    ):

        title = (
            cell118_html_text(
                source_data[
                    "title"
                ]
            )
        )


        lines.extend([
            (
                '<h4 dir="rtl" align="right">'
                f'منبع بازیابی‌شده {cell118_fa_number(source_index)}: '
                f'{title}'
                '</h4>'
            ),
            "",
        ])


        for evidence_window in (
            source_data[
                "evidence_windows"
            ]
        ):

            evidence_html = (
                cell118_html_text(
                    evidence_window
                )
            )

            lines.extend([
                (
                    '<p dir="rtl" align="right">'
                    f'{evidence_html}'
                    '</p>'
                ),
                "",
            ])


    if author_only:

        lines.extend([
            (
                '<h4 dir="rtl" align="right">'
                'بخش‌های متکی به بازخورد مستقیم مؤلف'
                '</h4>'
            ),
            "",
            (
                '<p dir="rtl" align="right">'
                'برای اجزای زیر، پس از بررسی کامل ۵۲۹ Website، '
                'شاهد مستقل Website که در Human Source Review '
                'قابل پذیرش باشد بازیابی نشد. این اجزا حذف نشده‌اند، '
                'زیرا مستقیماً در بازخورد دکتر کمیل رودی بیان شده‌اند.'
                '</p>'
            ),
            "",
            '<ul dir="rtl">',
        ])


        for item in author_only:

            description = (
                cell118_html_text(
                    item[
                        "description"
                    ]
                )
            )

            lines.append(
                (
                    '<li>'
                    f'{description}'
                    '</li>'
                )
            )


        lines.extend([
            '</ul>',
            "",
            (
                '<p dir="rtl" align="right">'
                'Authority این اجزا: بازخورد مستقیم مؤلف '
                '(دکتر کمیل رودی).'
                '</p>'
            ),
            "",
        ])


    lines.extend([
        CELL118_END_MARKER,
    ])


    return (
        "\n".join(
            lines
        ),
        grouped_sources,
        author_only,
    )


# ============================================================
# 13) Remove previous Cell118 block / obsolete Corpus block
#     while preserving exact historical Demo content
# ============================================================

def cell118_clean_existing_source_body(
    source_body,
):

    cleaned = (
        CELL118_INSERTED_BLOCK_RE.sub(
            "",
            source_body,
        )
    )


    candidate_positions = []


    old_html = (
        CELL118_OLD_CORPUS_EVIDENCE_RE.search(
            cleaned
        )
    )

    if old_html is not None:

        candidate_positions.append(
            old_html.start()
        )


    old_md = (
        CELL118_OLD_CORPUS_EVIDENCE_MD_RE.search(
            cleaned
        )
    )

    if old_md is not None:

        candidate_positions.append(
            old_md.start()
        )


    if candidate_positions:

        cleaned = cleaned[
            :
            min(
                candidate_positions
            )
        ]


    return cleaned.strip()


# ============================================================
# 14) Replace final "منبع" row only
# ============================================================

CELL118_FINAL_SOURCE_CELL_RE = re.compile(
    r"""
    (
        <td
        \s+
        align="right"
        >
        \s*
        منبع
        \s*
        </td>
        \s*
        <td
        \s+
        align="right"
        >
    )
    .*?
    (
        </td>
    )
    """,
    re.VERBOSE
    |
    re.DOTALL,
)


def cell118_replace_final_source_cell(
    block,
    author_only_exists,
):

    final_heading = (
        CELL118_FINAL_QA_HEADING_RE.search(
            block
        )
    )

    author_heading = (
        CELL118_AUTHOR_HEADING_RE.search(
            block
        )
    )


    cell118_check(
        "FINAL_QA_HEADING_NOT_FOUND",
        final_heading
        is not None,
    )

    cell118_check(
        "AUTHOR_HEADING_NOT_FOUND",
        author_heading
        is not None,
    )


    if (
        final_heading
        is None
        or
        author_heading
        is None
    ):
        return block


    final_segment = block[
        final_heading.end()
        :
        author_heading.start()
    ]


    if author_only_exists:

        replacement_text = (
            "متن‌های منبع بازیابی‌شده و تأییدشده در بخش ۱ بالا؛ "
            "بخش‌هایی که در Website Corpus شاهد مستقل پذیرفته‌شده "
            "برای آن‌ها بازیابی نشد، بر بازخورد مستقیم "
            "دکتر کمیل رودی متکی‌اند."
        )

    else:

        replacement_text = (
            "متن‌های منبع بازیابی‌شده و تأییدشده در بخش ۱ بالا."
        )


    replacement_html = (
        cell118_html_text(
            replacement_text
        )
    )


    new_final_segment, count = (
        CELL118_FINAL_SOURCE_CELL_RE.subn(
            (
                r"\1"
                +
                replacement_html
                +
                r"\2"
            ),
            final_segment,
            count=1,
        )
    )


    cell118_check(
        "FINAL_SOURCE_CELL_REPLACEMENT_COUNT_NOT_1",
        count == 1,
        count,
    )


    return (
        block[
            :
            final_heading.end()
        ]
        +
        new_final_segment
        +
        block[
            author_heading.start()
            :
        ]
    )


# ============================================================
# 15) Transform one Case 23-28
# ============================================================

cell118_case_audit = {}


def cell118_transform_case(
    number,
    block,
):

    pilot_packet = (
        cell118_packet_by_number[
            number
        ]
    )


    source_heading = (
        CELL118_SOURCE_HEADING_RE.search(
            block
        )
    )


    cell118_check(
        (
            "SOURCE_HEADING_NOT_FOUND:"
            +
            str(
                number
            )
        ),
        source_heading
        is not None,
    )


    if source_heading is None:
        return block


    qa_heading = (
        CELL118_QA_HISTORY_HEADING_RE.search(
            block,
            source_heading.end(),
        )
    )


    cell118_check(
        (
            "QA_HISTORY_HEADING_NOT_FOUND:"
            +
            str(
                number
            )
        ),
        qa_heading
        is not None,
    )


    if qa_heading is None:
        return block


    original_source_body = block[
        source_heading.end()
        :
        qa_heading.start()
    ]


    historical_body = (
        cell118_clean_existing_source_body(
            original_source_body
        )
    )


    recovered_block, grouped_sources, author_only = (
        cell118_build_source_block(
            pilot_packet
        )
    )


    new_source_body_parts = [
        "",
        recovered_block,
        "",
    ]


    if historical_body:

        new_source_body_parts.extend([
            historical_body,
            "",
        ])


    new_source_body = (
        "\n".join(
            new_source_body_parts
        )
    )


    transformed = (
        block[
            :
            source_heading.end()
        ]
        +
        new_source_body
        +
        block[
            qa_heading.start()
            :
        ]
    )


    transformed = (
        cell118_replace_final_source_cell(
            transformed,
            author_only_exists=bool(
                author_only
            ),
        )
    )


    # --------------------------------------------------------
    # Preserve historical content exactly.
    # --------------------------------------------------------

    if historical_body:

        cell118_check(
            (
                "HISTORICAL_SOURCE_BODY_NOT_PRESERVED:"
                +
                str(
                    number
                )
            ),
            historical_body
            in
            transformed,
        )


    # --------------------------------------------------------
    # Every approved Source title + Evidence must render.
    # --------------------------------------------------------

    for source_data in (
        grouped_sources.values()
    ):

        title_html = (
            cell118_html_text(
                source_data[
                    "title"
                ]
            )
        )


        cell118_check(
            (
                "APPROVED_SOURCE_TITLE_NOT_RENDERED:"
                +
                str(
                    number
                )
                +
                ":"
                +
                source_data[
                    "title"
                ]
            ),
            title_html
            in
            transformed,
        )


        for evidence_window in (
            source_data[
                "evidence_windows"
            ]
        ):

            evidence_html = (
                cell118_html_text(
                    evidence_window
                )
            )


            cell118_check(
                (
                    "APPROVED_EVIDENCE_NOT_RENDERED:"
                    +
                    str(
                        number
                    )
                ),
                evidence_html
                in
                transformed,
            )


    # --------------------------------------------------------
    # Every Author-Feedback-only Claim must be explicit.
    # --------------------------------------------------------

    for item in author_only:

        description_html = (
            cell118_html_text(
                item[
                    "description"
                ]
            )
        )


        cell118_check(
            (
                "AUTHOR_ONLY_CLAIM_NOT_RENDERED:"
                +
                str(
                    number
                )
                +
                ":"
                +
                str(
                    item[
                        "claim_id"
                    ]
                )
            ),
            description_html
            in
            transformed,
        )


    rendered_source_fragment_match = re.search(
        re.escape(
            CELL118_START_MARKER
        )
        +
        r"(.*?)"
        +
        re.escape(
            CELL118_END_MARKER
        ),
        transformed,
        re.DOTALL,
    )


    cell118_check(
        (
            "RECOVERED_SOURCE_FRAGMENT_NOT_FOUND:"
            +
            str(
                number
            )
        ),
        rendered_source_fragment_match
        is not None,
    )


    if rendered_source_fragment_match is not None:

        fragment = (
            rendered_source_fragment_match.group(
                1
            )
        )


        cell118_check(
            (
                "AUTHOR_FACING_SOURCE_CONTAINS_URL:"
                +
                str(
                    number
                )
            ),
            "https://"
            not in
            fragment
            and
            "http://"
            not in
            fragment,
        )


        cell118_check(
            (
                "AUTHOR_FACING_SOURCE_CONTAINS_DOCUMENT_ID:"
                +
                str(
                    number
                )
            ),
            "website:"
            not in
            fragment,
        )


    cell118_case_audit[
        number
    ] = {
        "website_source_documents":
            len(
                grouped_sources
            ),

        "author_feedback_only_claims":
            len(
                author_only
            ),

        "historical_body_preserved":
            (
                historical_body
                in
                transformed
                if historical_body
                else
                True
            ),
    }


    return transformed


# ============================================================
# 16) Build candidate final Author Review text
# ============================================================

cell118_candidate_text = (
    cell118_original_text
)


# Replace from 28 down to 23 so string offsets remain irrelevant.
for number in reversed(
    CELL118_CASE_NUMBERS
):

    current_blocks = (
        cell118_extract_case_blocks(
            cell118_candidate_text
        )
    )

    old_block = (
        current_blocks[
            number
        ]
    )

    new_block = (
        cell118_transform_case(
            number,
            old_block,
        )
    )


    cell118_check(
        (
            "CASE_BLOCK_NOT_UNIQUELY_FOUND:"
            +
            str(
                number
            )
        ),
        cell118_candidate_text.count(
            old_block
        )
        ==
        1,
    )


    cell118_candidate_text = (
        cell118_candidate_text.replace(
            old_block,
            new_block,
            1,
        )
    )


# ============================================================
# 17) Idempotence / allowed starting state
# ============================================================

cell118_marker_count_before = (
    cell118_original_text.count(
        CELL118_START_MARKER
    )
)

cell118_is_baseline_input = (
    cell118_author_md_sha_before
    ==
    CELL118_EXPECTED_BASE_AUTHOR_MD_SHA256
)

cell118_is_already_applied = (
    (
        not cell118_is_baseline_input
    )
    and
    cell118_marker_count_before
    ==
    6
    and
    cell118_candidate_text
    ==
    cell118_original_text
)


cell118_check(
    "AUTHOR_MD_UNEXPECTED_STARTING_STATE",
    (
        cell118_is_baseline_input
        or
        cell118_is_already_applied
    ),
    {
        "sha256":
            cell118_author_md_sha_before,

        "cell118_marker_count":
            cell118_marker_count_before,

        "candidate_equals_original":
            cell118_candidate_text
            ==
            cell118_original_text,
    },
)


if cell118_failures:

    print(
        "=== CELL 118 STARTING STATE FAILURE ==="
    )

    print(
        cell118_failures
    )

    raise AssertionError(
        "Unexpected Author Review state. "
        "Cell 118 stopped before write."
    )


# ============================================================
# 18) Structural protection
# ============================================================

cell118_final_blocks_in_memory = (
    cell118_extract_case_blocks(
        cell118_candidate_text
    )
)


# ------------------------------------------------------------
# Cases 1-22 MUST be byte-identical.
# ------------------------------------------------------------

for number in range(
    1,
    23,
):

    cell118_check(
        (
            "FROZEN_CASE_MUTATED:"
            +
            str(
                number
            )
        ),
        cell118_original_blocks[
            number
        ]
        ==
        cell118_final_blocks_in_memory[
            number
        ],
    )


# ============================================================
# 19) Mask only the two allowed mutable regions in 23-28
# ============================================================

def cell118_mask_allowed_regions(
    block,
):

    source_heading = (
        CELL118_SOURCE_HEADING_RE.search(
            block
        )
    )

    if source_heading is None:
        return None


    qa_heading = (
        CELL118_QA_HISTORY_HEADING_RE.search(
            block,
            source_heading.end(),
        )
    )

    if qa_heading is None:
        return None


    masked = (
        block[
            :
            source_heading.end()
        ]
        +
        "\n[ALLOWED_SOURCE_SECTION_BODY]\n"
        +
        block[
            qa_heading.start()
            :
        ]
    )


    final_heading = (
        CELL118_FINAL_QA_HEADING_RE.search(
            masked
        )
    )

    author_heading = (
        CELL118_AUTHOR_HEADING_RE.search(
            masked
        )
    )


    if (
        final_heading is None
        or
        author_heading is None
    ):
        return None


    final_segment = masked[
        final_heading.end()
        :
        author_heading.start()
    ]


    masked_final_segment, count = (
        CELL118_FINAL_SOURCE_CELL_RE.subn(
            r"\1[ALLOWED_FINAL_SOURCE_VALUE]\2",
            final_segment,
            count=1,
        )
    )


    if count != 1:
        return None


    return (
        masked[
            :
            final_heading.end()
        ]
        +
        masked_final_segment
        +
        masked[
            author_heading.start()
            :
        ]
    )


for number in (
    CELL118_CASE_NUMBERS
):

    before_masked = (
        cell118_mask_allowed_regions(
            cell118_original_blocks[
                number
            ]
        )
    )

    after_masked = (
        cell118_mask_allowed_regions(
            cell118_final_blocks_in_memory[
                number
            ]
        )
    )


    cell118_check(
        (
            "ALLOWED_REGION_MASK_FAILED:"
            +
            str(
                number
            )
        ),
        before_masked
        is not None
        and
        after_masked
        is not None,
    )


    if (
        before_masked is not None
        and
        after_masked is not None
    ):

        cell118_check(
            (
                "CASE_CHANGED_OUTSIDE_ALLOWED_REGIONS:"
                +
                str(
                    number
                )
            ),
            before_masked
            ==
            after_masked,
        )


# ============================================================
# 20) Final format checks
# ============================================================

cell118_final_case_matches = (
    cell118_case_matches(
        cell118_candidate_text
    )
)


cell118_check(
    "FINAL_CASE_SEQUENCE_NOT_1_28",
    [
        row[
            "number"
        ]
        for row
        in cell118_final_case_matches
    ]
    ==
    list(
        range(
            1,
            29,
        )
    ),
)


cell118_check(
    "RECOVERED_SOURCE_BLOCK_COUNT_NOT_6",
    cell118_candidate_text.count(
        CELL118_START_MARKER
    )
    ==
    6,
    cell118_candidate_text.count(
        CELL118_START_MARKER
    ),
)


cell118_check(
    "RECOVERED_SOURCE_END_BLOCK_COUNT_NOT_6",
    cell118_candidate_text.count(
        CELL118_END_MARKER
    )
    ==
    6,
)


cell118_new_segment_start = (
    cell118_candidate_text.find(
        '<h2 dir="rtl" align="right">مورد 23 از ۲۸</h2>'
    )
)


cell118_check(
    "CASE23_MARKER_NOT_FOUND_AFTER_REPAIR",
    cell118_new_segment_start
    >=
    0,
)


cell118_new_segment = (
    cell118_candidate_text[
        cell118_new_segment_start:
    ]
    if cell118_new_segment_start
    >=
    0
    else
    ""
)


cell118_check(
    "RAW_MARKDOWN_H2_SURVIVED_IN_23_28",
    re.search(
        r"(?m)^##\s+مورد\s+",
        cell118_new_segment,
    )
    is None,
)


cell118_check(
    "RAW_MARKDOWN_H3_SURVIVED_IN_23_28",
    re.search(
        r"(?m)^###\s+",
        cell118_new_segment,
    )
    is None,
)


# ============================================================
# 21) Validate rendered source counts
# ============================================================

cell118_rendered_source_document_count = sum(
    row[
        "website_source_documents"
    ]

    for row
    in cell118_case_audit.values()
)


cell118_rendered_author_only_count = sum(
    row[
        "author_feedback_only_claims"
    ]

    for row
    in cell118_case_audit.values()
)


cell118_check(
    "RENDERED_SOURCE_DOCUMENT_COUNT_NOT_12",
    cell118_rendered_source_document_count
    ==
    CELL118_EXPECTED_UNIQUE_RENDERED_SOURCE_BLOCKS,
    cell118_rendered_source_document_count,
)


cell118_check(
    "RENDERED_AUTHOR_ONLY_COUNT_NOT_5",
    cell118_rendered_author_only_count
    ==
    CELL118_EXPECTED_AUTHOR_ONLY,
    cell118_rendered_author_only_count,
)


# ============================================================
# 22) No old ambiguous final-source wording in 23-28
# ============================================================

cell118_check(
    "AMBIGUOUS_SAME_SOURCE_TEXT_SURVIVED_IN_NEW_CASES",
    (
        "همان سابقه"
        not in
        cell118_new_segment
    ),
)


# ============================================================
# 23) Stop before write if any in-memory failure
# ============================================================

if cell118_failures:

    print(
        "=== CELL 118 IN-MEMORY VALIDATION FAILURES ==="
    )

    print(
        cell118_failures
    )

    raise AssertionError(
        "Cell 118 failed before Author Review write."
    )


# ============================================================
# 24) Write only if content changed
# ============================================================

cell118_write_required = (
    cell118_candidate_text
    !=
    cell118_original_text
)

cell118_backup_path = None


if cell118_write_required:

    timestamp = (
        datetime.now(
            timezone.utc
        )
        .strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )


    cell118_backup_path = (
        cell118_author_md_path.with_name(
            (
                cell118_author_md_path.stem
                +
                ".before_cell118_source_text_"
                +
                timestamp
                +
                cell118_author_md_path.suffix
            )
        )
    )


    cell118_backup_path.write_bytes(
        cell118_author_md_path.read_bytes()
    )


    cell118_tmp_path = (
        cell118_author_md_path.with_name(
            cell118_author_md_path.name
            +
            ".tmp_cell118"
        )
    )


    cell118_tmp_path.write_text(
        cell118_candidate_text,
        encoding="utf-8",
    )


    os.replace(
        cell118_tmp_path,
        cell118_author_md_path,
    )


# ============================================================
# 25) Reload final file
# ============================================================

cell118_final_text = (
    cell118_author_md_path.read_text(
        encoding="utf-8"
    )
)


cell118_check(
    "RELOADED_AUTHOR_MD_MISMATCH",
    cell118_final_text
    ==
    cell118_candidate_text,
)


cell118_author_md_sha_after = (
    cell112_sha256_file(
        cell118_author_md_path
    )
)


# ============================================================
# 26) Immutable artifacts after write
# ============================================================

cell118_corpus_sha_after = (
    cell112_sha256_file(
        cell118_corpus_path
    )
)

cell118_registry_sha_after = (
    cell112_sha256_file(
        cell118_registry_path
    )
)

cell118_source_packet_sha_after = (
    cell112_sha256_file(
        cell118_source_packet_path
    )
)


cell118_check(
    "CORPUS_MUTATED",
    cell118_corpus_sha_after
    ==
    cell118_corpus_sha_before,
)


cell118_check(
    "REGISTRY_MUTATED",
    cell118_registry_sha_after
    ==
    cell118_registry_sha_before,
)


cell118_check(
    "SOURCE_PACKET_MUTATED",
    cell118_source_packet_sha_after
    ==
    cell118_source_packet_sha_before,
)


# ============================================================
# 27) Reload + protect Cases 1-22 independently
# ============================================================

cell118_reloaded_blocks = (
    cell118_extract_case_blocks(
        cell118_final_text
    )
)


for number in range(
    1,
    23,
):

    cell118_check(
        (
            "RELOADED_FROZEN_CASE_MUTATED:"
            +
            str(
                number
            )
        ),
        cell118_original_blocks[
            number
        ]
        ==
        cell118_reloaded_blocks[
            number
        ],
    )


# ============================================================
# 28) Final Case 23-28 audit
# ============================================================

for number in (
    CELL118_CASE_NUMBERS
):

    block = (
        cell118_reloaded_blocks[
            number
        ]
    )


    cell118_check(
        (
            "FINAL_CASE_MISSING_CELL118_SOURCE_BLOCK:"
            +
            str(
                number
            )
        ),
        CELL118_START_MARKER
        in
        block
        and
        CELL118_END_MARKER
        in
        block,
    )


    pilot_packet = (
        cell118_packet_by_number[
            number
        ]
    )


    for claim in (
        pilot_packet.get(
            "claims",
            [],
        )
        or
        []
    ):

        status = claim.get(
            "human_source_review_status"
        )


        if (
            status
            ==
            "WEBSITE_SOURCE_BACKED"
        ):

            source = (
                claim[
                    "content_source"
                ]
            )


            cell118_check(
                (
                    "FINAL_APPROVED_EVIDENCE_MISSING:"
                    +
                    str(
                        number
                    )
                    +
                    ":"
                    +
                    str(
                        claim.get(
                            "claim_id"
                        )
                    )
                ),
                cell118_html_text(
                    source[
                        "evidence_window"
                    ]
                )
                in
                block,
            )


        elif (
            status
            ==
            "PRIMARY_DIRECT_AUTHOR_FEEDBACK_ONLY"
        ):

            cell118_check(
                (
                    "FINAL_AUTHOR_ONLY_DESCRIPTION_MISSING:"
                    +
                    str(
                        number
                    )
                    +
                    ":"
                    +
                    str(
                        claim.get(
                            "claim_id"
                        )
                    )
                ),
                cell118_html_text(
                    claim[
                        "description"
                    ]
                )
                in
                block,
            )


# ============================================================
# 29) Report
# ============================================================

print(
    "=== CODE CELL 118 — AUTHOR REVIEW SOURCE TEXT REPAIR ==="
)


print(
    "\n=== FROZEN INPUTS ==="
)

print(
    "Corpus SHA256:",
    cell118_corpus_sha_before,
)

print(
    "Registry SHA256:",
    cell118_registry_sha_before,
)

print(
    "Source Packet SHA256:",
    cell118_source_packet_sha_before,
)

print(
    "Author Review SHA256 before:",
    cell118_author_md_sha_before,
)


print(
    "\n=== SOURCE REVIEW INPUT ==="
)

print(
    "Pilots:",
    CELL118_CASE_NUMBERS,
)

print(
    "Website-backed claims:",
    len(
        cell118_website_claims
    ),
)

print(
    "Author-feedback-only claims:",
    len(
        cell118_author_only_claims
    ),
)

print(
    "Rendered Website source sections:",
    cell118_rendered_source_document_count,
)


print(
    "\n=== CASE AUDIT ==="
)

for number in (
    CELL118_CASE_NUMBERS
):

    audit = (
        cell118_case_audit[
            number
        ]
    )

    print(
        (
            f"Case {number}: "
            f"Website sources = "
            f"{audit['website_source_documents']} | "
            f"Author-feedback-only claims = "
            f"{audit['author_feedback_only_claims']} | "
            f"Historical body preserved = "
            f"{audit['historical_body_preserved']}"
        )
    )


print(
    "\n=== PROTECTION ==="
)

print(
    "Cases 1-22 regenerated:",
    False,
)

print(
    "Cases 1-22 byte-preserved:",
    all(
        cell118_original_blocks[
            number
        ]
        ==
        cell118_reloaded_blocks[
            number
        ]

        for number
        in range(
            1,
            23,
        )
    ),
)

print(
    "Final Question / Answer rewrite:",
    False,
)

print(
    "Only final Source value allowed outside Section 1:",
    True,
)


print(
    "\n=== AUTHOR-FACING SOURCE POLICY ==="
)

print(
    "Source URLs rendered:",
    False,
)

print(
    "Internal Document IDs rendered:",
    False,
)

print(
    "Historical Demo Provenance claimed as exact:",
    False,
)

print(
    "Historical Demo Provenance status:",
    "NOT_PROVEN_FROM_AVAILABLE_ARTIFACTS",
)


print(
    "\n=== OUTPUT ==="
)

print(
    "Output:",
    cell118_author_md_path,
)

print(
    "Write required:",
    cell118_write_required,
)

print(
    "Backup:",
    (
        cell118_backup_path
        if cell118_backup_path
        is not None
        else
        "NOT_NEEDED"
    ),
)

print(
    "Author Review SHA256 after:",
    cell118_author_md_sha_after,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Corpus mutated:",
    cell118_corpus_sha_after
    !=
    cell118_corpus_sha_before,
)

print(
    "Registry mutated:",
    cell118_registry_sha_after
    !=
    cell118_registry_sha_before,
)

print(
    "Source Packet mutated:",
    cell118_source_packet_sha_after
    !=
    cell118_source_packet_sha_before,
)

print(
    "Author decisions recorded:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell118_failures
)


cell118_gate = (
    len(
        cell118_failures
    )
    ==
    0
)


print(
    "\nCODE CELL 118 GATE:",
    (
        "PASS"
        if cell118_gate
        else
        "FAIL"
    ),
)


assert cell118_gate, (
    "Author Review source-text repair failed. "
    "Repair this SAME Code Cell 118 before continuing."
)

=== CODE CELL 118 — AUTHOR REVIEW SOURCE TEXT REPAIR ===

=== FROZEN INPUTS ===
Corpus SHA256: 9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40
Registry SHA256: a851f1a20641f8627f12121bef6497e0fcfb72e6c0bc4e47096f4021e6604b04
Source Packet SHA256: a233d25207124b808eb3747054860bc3278670b4e12444160dd809693c8ea3a4
Author Review SHA256 before: f9b2e04a50e61060b150c958d444f1308379c76815edb299c25b96257b031f09

=== SOURCE REVIEW INPUT ===
Pilots: [23, 24, 25, 26, 27, 28]
Website-backed claims: 17
Author-feedback-only claims: 5
Rendered Website source sections: 12

=== CASE AUDIT ===
Case 23: Website sources = 1 | Author-feedback-only claims = 0 | Historical body preserved = True
Case 24: Website sources = 2 | Author-feedback-only claims = 0 | Historical body preserved = True
Case 25: Website sources = 1 | Author-feedback-only claims = 0 | Historical body preserved = True
Case 26: Website sources = 3 | Author-feedback-only claims = 3 | Historical body preserved = True
Case 27: 